# 03 — Iatan 2, estados operacionais e mineração de processos

**Escopo:** Iatan, Facility 6065, Unit 2, ano 2025. Esta etapa transforma a carga horária em estados e eventos e prepara atributos para a análise de emissões. A modelagem preditiva de CO₂ fica para a Fase 4.

**Execução:** use a raiz comum dos notebooks, com as entradas produzidas pelo 02 em data/interim, data/processed e pipeline. No Colab, a primeira célula usa a pasta de execução do projeto no Drive. Defina IATAN_PROJECT_ROOT para outra localização. Execute as células em ordem. A Fase 2 cria um diretório próprio e imprime seu identificador para a Fase 4.

**Evidência histórica:** as saídas visíveis em células de código preservadas são da entrega do grupo. As saídas das células editadas foram limpas. A revisão altera configuração e verificações, mantendo os critérios e atributos do experimento. Produza novas saídas em uma execução completa e registre o identificador e as versões. A partição de teste de 2025 já foi examinada no estudo.

**Limite temporal:** o forward fornece estados a partir das observações até a hora t. A janela de ajuste e seleção dos parâmetros do HMM continua sendo um requisito distinto. Para avaliar preditores em outro corte, compare o limite de ajuste e seleção registrado no manifesto com o início da avaliação. As comparações exploratórias com cortes sobrepostos precisam de reajuste por janela para sustentar inferência fora da amostra.


In [ ]:
# Registros do experimento usam caminhos relativos à raiz configurada.
# Versões de bibliotecas e resultados são preservados sem identidade da sessão.
import builtins as _builtins
import json as _json
import os as _os
import re as _re
import warnings as _warnings
from pathlib import Path as _Path

def _project_root():
    for name in ("PROJECT_ROOT", "ROOT33", "ROOT", "PROJECT", "SOURCE", "root"):
        value = globals().get(name)
        if isinstance(value, _Path) and value.is_dir():
            return value.resolve()
    return _Path.cwd()

def portable_path(path, root=None):
    path = _Path(path).expanduser()
    root = _Path(root) if root is not None else _project_root()
    try:
        return path.resolve().relative_to(root.resolve()).as_posix()
    except (ValueError, OSError):
        return "external/" + path.name

def public_record(value):
    if isinstance(value, _Path):
        return portable_path(value)
    if isinstance(value, dict):
        return {k: public_record(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [public_record(v) for v in value]
    if isinstance(value, str):
        root = _project_root()
        for prefix in {str(root), root.as_posix()}:
            if value.startswith(prefix + "/") or value.startswith(prefix + "\\"):
                return value[len(prefix)+1:].replace("\\", "/")
            value = value.replace(prefix + "/", "").replace(prefix + "\\", "")
            value = value.replace(prefix, ".")
        if _re.match(r"^(?:[A-Za-z]:[\\/]|/(?:content|tmp|home|Users|mnt)/)", value):
            return "external/" + value.replace("\\", "/").rstrip("/").rsplit("/", 1)[-1]
        return value
    # Cópias de tabelas exportadas normalizam somente colunas textuais.
    if type(value).__module__.startswith("pandas.") and hasattr(value, "columns"):
        result = value.copy()
        for column in result.select_dtypes(include=["object", "string"]).columns:
            result[column] = result[column].map(public_record)
        return result
    return value

def _project_print(*values, **kwargs):
    root = _project_root()
    if root.is_dir():
        _os.chdir(root)
    return _builtins.print(*(public_record(v) for v in values), **kwargs)

def _project_dump(value, *args, **kwargs):
    return _json.dump(public_record(value), *args, **kwargs)

def _project_dumps(value, *args, **kwargs):
    return _json.dumps(public_record(value), *args, **kwargs)

def _project_csv(frame, *args, **kwargs):
    return public_record(frame).to_csv(*args, **kwargs)

def _project_write_text(path, text, *args, **kwargs):
    return path.write_text(public_record(text), *args, **kwargs)

def _project_display(*values, **kwargs):
    from IPython.display import display as _display
    return _display(*(public_record(v) for v in values), **kwargs)

# Avisos mantêm o conteúdo sem diretórios temporários ou IDs do kernel.
_warnings.formatwarning = lambda message, category, *args, **kwargs: (
    f"{category.__name__}: {public_record(str(message))}\n"
)
_os.environ["MLFLOW_DISABLE_AGENT_HINT"] = "1"
_context = _json.loads(_os.environ.get("MLFLOW_RUN_CONTEXT", "{}"))
_context.update({
    "mlflow.user": "equipe", "mlflow.source.name": "iatan",
    "mlflow.source.type": "LOCAL", "mlflow.source.git.commit": "",
    "mlflow.source.git.branch": "", "mlflow.source.git.repoURL": "",
    "mlflow.source.git.diff": "", "mlflow.source.git.dirty": "",
})
_os.environ["MLFLOW_RUN_CONTEXT"] = _json.dumps(_context)

# ============================================================
# CÉLULA 01/36 — CONFIGURAÇÃO GERAL DO EXPERIMENTO
# FASE 2: HMM + PROCESS MINING | IATAN, FACILITY 6065, UNIT 2
# ============================================================
# Objetivo: configurar ambiente, dependências, caminhos e
# parâmetros. Esta célula NÃO carrega dados nem treina modelos.

import os
import sys
import subprocess
import importlib
from pathlib import Path
from datetime import datetime, timezone
import json
import random
import platform
import importlib.metadata as metadata

# 1. Dependências mínimas: versões registradas para reprodução.
# A instalação ocorre somente quando a biblioteca não existe.
DEPENDENCIAS = {
    "numpy": "numpy",
    "pandas": "pandas",
    "sklearn": "scikit-learn",
    "hmmlearn": "hmmlearn",
    "pm4py": "pm4py",
    "matplotlib": "matplotlib",
    "seaborn": "seaborn",
    "pyarrow": "pyarrow",
    "graphviz": "graphviz",
    "scipy": "scipy",
    "statsmodels": "statsmodels",
}
for modulo, pacote in DEPENDENCIAS.items():
    try:
        importlib.import_module(modulo)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pacote])

import numpy as np
import pandas as pd

# 2. Configuração da raiz do projeto.
MOUNT_POINT = Path("/content/drive")
COLAB_ROOT = (
    MOUNT_POINT / "MyDrive"
    / "Ciência de Dados - Iatan - Process Mining e Experimentos"
    / "Iatan 2025 - entrega revisada" / "execucao"
)
try:
    from google.colab import drive
except ImportError:
    drive = None

# No Colab a raiz padrão fica no Drive. Em Jupyter local, use a raiz
# corrente ou informe IATAN_PROJECT_ROOT em qualquer ambiente.
configured_root = os.environ.get("IATAN_PROJECT_ROOT")
ROOT = Path(configured_root).expanduser() if configured_root else (
    COLAB_ROOT if drive is not None else Path.cwd()
)
if drive is not None and ROOT.is_relative_to(MOUNT_POINT):
    if not (MOUNT_POINT / "MyDrive").is_dir():
        drive.mount(str(MOUNT_POINT))
ROOT = ROOT.resolve()

# 3. Diretórios compatíveis com os notebooks anteriores.
if not ROOT.is_dir():
    raise FileNotFoundError(
        f"Pasta do projeto não encontrada: {ROOT}. "
        "Informe a raiz com data/ e pipeline/ em IATAN_PROJECT_ROOT."
    )
DATA = ROOT / "data"
DATA_RAW = DATA / "raw"
DATA_INTERIM = DATA / "interim"
DATA_PROCESSED = DATA / "processed"
PIPELINE_DIR = ROOT / "pipeline"
NOTEBOOKS = ROOT / "notebooks"
OUTPUT = ROOT / "outputs"

# 4. Identificação e parâmetros centrais do experimento.
EXPERIMENTO = {
    "fase": "hmm_process_mining",
    "facility_id": 6065,
    "unit_id": "2",
    "facility_name": "Iatan",
    "year": 2025,
    "seed": 42,
    "hmm_k_candidates": list(range(2, 9)),
    "hmm_random_seeds": [11, 23, 42, 71, 101],
    "inductive_miner_noise": [0.00, 0.05, 0.10, 0.20],
    "plan_code_cells": 36,
    "experiment_version": "v1",
    "notebook_revision": "entrega_revisada_2026_11",
}
SEED = EXPERIMENTO["seed"]
random.seed(SEED)
np.random.seed(SEED)

# 5. Pasta exclusiva de execução: não sobrescrever resultados.
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
RUN_DIR = OUTPUT / "fase2_hmm_process_mining" / f"iatan_6065_unit2_2025_{RUN_ID}"
if RUN_DIR.exists():
    raise FileExistsError(f"Identificador de execução já existe: {RUN_DIR}")
FIGURES = RUN_DIR / "figures"
TABLES = RUN_DIR / "tables"
REPORTS = RUN_DIR / "reports"
MODELS = RUN_DIR / "models"
EVENT_LOGS = RUN_DIR / "event_logs"
FEATURES = RUN_DIR / "features"
for pasta in [FIGURES, TABLES, REPORTS, MODELS, EVENT_LOGS, FEATURES]:
    pasta.mkdir(parents=True, exist_ok=False)

# 6. Arquivos candidatos da fase anterior (somente caminhos).
ENTRADAS_CANDIDATAS = {
    "operational_hourly": DATA_INTERIM / "step05_feature_engineering.parquet",
    "modeling_operating_hours": DATA_PROCESSED / "modeling_dataset_operating_hours.parquet",
    "selected_unit_metadata": PIPELINE_DIR / "selected_unit.json",
}

# 7. Manifesto inicial de reprodução.
VERSOES = {}
for pacote in DEPENDENCIAS.values():
    try:
        VERSOES[pacote] = metadata.version(pacote)
    except metadata.PackageNotFoundError:
        VERSOES[pacote] = "não detectada"

MANIFESTO = {
    "experiment": EXPERIMENTO,
    "run_id": RUN_ID,
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "python": platform.python_version(),
    "packages": VERSOES,
    "project_root": str(ROOT),
    "input_candidates": {k: str(v) for k, v in ENTRADAS_CANDIDATAS.items()},
    "note": "Célula 01: configuração; seleção e validação da entrada na Célula 02.",
}
MANIFEST_PATH = REPORTS / "manifest_initial.json"
_project_write_text(MANIFEST_PATH, 
    _project_dumps(MANIFESTO, ensure_ascii=False, indent=2), encoding="utf-8"
)

# 8. Resumo da execução (sem leitura ou análise dos dados).
_project_print("=" * 72)
_project_print("FASE 2 — CÉLULA 01/36 | CONFIGURAÇÃO DO EXPERIMENTO")
_project_print("=" * 72)
_project_print(f"Unidade: {EXPERIMENTO['facility_name']} | Facility {EXPERIMENTO['facility_id']} | Unit {EXPERIMENTO['unit_id']}")
_project_print(f"Ano: {EXPERIMENTO['year']} | Seed: {SEED}")
_project_print(f"Projeto: {ROOT}")
_project_print(f"Execução: {RUN_DIR}")
_project_print("\nEntradas candidatas (verificação definitiva na Célula 02):")
for nome, caminho in ENTRADAS_CANDIDATAS.items():
    _project_print(f"  {'OK' if caminho.is_file() else 'AUSENTE'} | {nome}: {caminho}")
_project_print(f"\nManifesto: {MANIFEST_PATH}")
_project_print("CÉLULA 01 CONFIGURADA — nenhum dado carregado; nenhum modelo treinado.")


## 01–05 — Fonte horária, integridade e ciclos ON

A unidade e as medidas vêm do EPA CEMS. O guia CAMD, p. 1, descreve CO₂ em short tons. A distinção ON/OFF, as regras para horas ausentes e o ciclo como caso são decisões do estudo. Diba et al. separam extração, correlação e abstração. A Aula 04, pp. 29–31, aplica essa distinção à definição dos casos e à granularidade dos eventos. [Guia CAMD](https://www.epa.gov/power-sector/power-sector-emissions-data) · [Diba et al. (2020)](https://doi.org/10.1002/widm.1346)


In [ ]:
# ============================================================
# CÉLULA 02/36 — CARREGAMENTO E IDENTIFICAÇÃO DOS DADOS
# Escopo: Iatan (Facility 6065), Unit 2, ano 2025.
# Não realiza limpeza, imputação, seleção de features ou treino.
# ============================================================
from pathlib import Path
import hashlib
import json
import pandas as pd
import numpy as np

assert "ENTRADAS_CANDIDATAS" in globals() and "EXPERIMENTO" in globals(), (
    "Execute primeiro a Célula 01 nesta sessão do Colab."
)

# 1. Metadados e inventário dos dois arquivos efetivamente encontrados.
meta_path = Path(ENTRADAS_CANDIDATAS["selected_unit_metadata"])
metadata_unidade = json.loads(meta_path.read_text(encoding="utf-8")) if meta_path.is_file() else {}

arquivos = {
    "operational_hourly": Path(ENTRADAS_CANDIDATAS["operational_hourly"]),
    "modeling_operating_hours": Path(ENTRADAS_CANDIDATAS["modeling_operating_hours"]),
}

CAMPOS_REFERENCIA = [
    "timestamp", "Operating Time", "Gross Load (MW)",
    "CO2 Mass (short tons)", "load_current_mw", "delta_load_1h",
    "facilityId_norm", "unitId_norm",
]

bases_fase1 = {}
resumos = []
for chave, caminho in arquivos.items():
    if not caminho.is_file():
        raise FileNotFoundError(f"Arquivo da Fase 1 não encontrado: {caminho}")
    tabela = pd.read_parquet(caminho)
    if "timestamp" not in tabela.columns:
        raise ValueError(f"{chave}: coluna 'timestamp' ausente. Colunas: {list(tabela.columns)}")
    # Converter somente em memória; não modificar os arquivos da Fase 1.
    tabela["timestamp"] = pd.to_datetime(tabela["timestamp"], errors="coerce")
    if tabela["timestamp"].isna().any():
        raise ValueError(f"{chave}: timestamps inválidos ({tabela['timestamp'].isna().sum()}).")
    bases_fase1[chave] = tabela
    resumos.append({
        "base": chave,
        "linhas": len(tabela),
        "colunas": len(tabela.columns),
        "inicio": str(tabela["timestamp"].min()),
        "fim": str(tabela["timestamp"].max()),
        "timestamps_unicos": int(tabela["timestamp"].nunique()),
        "duplicatas_timestamp": int(tabela["timestamp"].duplicated().sum()),
        "campos_referencia": ", ".join(c for c in CAMPOS_REFERENCIA if c in tabela.columns),
    })

inventario_fase1 = pd.DataFrame(resumos)
_project_print("=" * 72)
_project_print("FASE 2 — CÉLULA 02/36 | CARREGAMENTO E IDENTIFICAÇÃO")
_project_print("=" * 72)
_project_print("\nINVENTÁRIO DAS BASES DA FASE 1")
_project_print(inventario_fase1.to_string(index=False))
_project_print("\nMETADADOS DA UNIDADE (selected_unit.json)")
_project_print(_project_dumps(metadata_unidade, ensure_ascii=False, indent=2, default=str)[:4000])

# 2. Escolha explícita da fonte horária principal.
# A base de modelagem ON é complementar, pois pode excluir horas OFF.
principal = bases_fase1["operational_hourly"]
necessarios = ["timestamp", "Operating Time", "Gross Load (MW)", "CO2 Mass (short tons)"]
ausentes = [c for c in necessarios if c not in principal.columns]
if ausentes:
    raise ValueError(f"Base operacional não contém campos necessários: {ausentes}")

ano = int(EXPERIMENTO["year"])
mascara_ano = principal["timestamp"].dt.year.eq(ano)
df_fase2 = principal.loc[mascara_ano].copy().sort_values("timestamp").reset_index(drop=True)
if df_fase2.empty:
    raise ValueError(f"A base operacional não possui registros para {ano}.")

# 3. Validar identidade quando os campos existirem, sem pressupor nomes de metadados.
for coluna, esperado in [("facilityId_norm", EXPERIMENTO["facility_id"]),
                         ("unitId_norm", EXPERIMENTO["unit_id"])]:
    if coluna in df_fase2.columns:
        encontrados = set(df_fase2[coluna].dropna().astype(str).str.strip().unique())
        if encontrados and encontrados != {str(esperado)}:
            raise ValueError(f"Identidade incompatível em {coluna}: {encontrados}; esperado {esperado}")
        _project_print(f"Identidade conferida: {coluna} = {encontrados or 'sem valores'}")
    else:
        _project_print(f"AVISO: {coluna} não existe na base; identidade depende dos metadados da Fase 1.")

# 4. Inventário de variáveis e prévia, sem auditoria aprofundada (Célula 03).
dicionario_fase2 = pd.DataFrame({
    "coluna": df_fase2.columns,
    "dtype": [str(t) for t in df_fase2.dtypes],
    "nulos": [int(df_fase2[c].isna().sum()) for c in df_fase2.columns],
})
_project_print("\nBASE PRINCIPAL: data/interim/step05_feature_engineering.parquet")
_project_print(f"Registros {ano}: {len(df_fase2):,} | Colunas: {len(df_fase2.columns)}")
_project_print("\nDICIONÁRIO DE VARIÁVEIS")
_project_print(dicionario_fase2.to_string(index=False))
_project_print("\nAMOSTRA DA BASE (sem alteração dos dados)")
_project_print(df_fase2[necessarios].head(8).to_string(index=False))

# 5. Disponibilizar objetos para as células seguintes; salvar apenas relatório.
relatorio02 = {
    "cell": 2,
    "year": ano,
    "main_source": str(arquivos["operational_hourly"]),
    "secondary_source": str(arquivos["modeling_operating_hours"]),
    "input_sha256": {
        key: hashlib.sha256(path.read_bytes()).hexdigest()
        for key, path in {**arquivos, "selected_unit_metadata": meta_path}.items()
        if path.is_file()
    },
    "main_rows": int(len(df_fase2)),
    "main_columns": list(df_fase2.columns),
    "timestamp_min": str(df_fase2["timestamp"].min()),
    "timestamp_max": str(df_fase2["timestamp"].max()),
    "identity_check": "verified where identifier columns exist",
    "note": "Integridade temporal/física na Célula 03; ON/OFF na Célula 04.",
}
relatorio_path = REPORTS / "cell02_input_inventory.json"
_project_write_text(relatorio_path, _project_dumps(relatorio02, indent=2, ensure_ascii=False), encoding="utf-8")
_project_print(f"\nRelatório: {relatorio_path}")
_project_print("CÉLULA 02 CONCLUÍDA — dados carregados; nenhuma limpeza ou modelagem executada.")


In [ ]:

# ============================================================
# FASE 2 — CÉLULA 03/36
# AUDITORIA DE INTEGRIDADE TEMPORAL E FÍSICA
# ============================================================
#
# OBJETIVO
# Verificar a qualidade e a consistência da base horária
# da Iatan (Facility 6065, Unit 2), ano 2025.
#
# ENTRADA
# df_fase2 — produzido na Célula 02.
#
# SAÍDAS
# audit03_summary
# audit03_temporal
# audit03_missing
# audit03_operational
# audit03_treated
# audit03_issues
#
# RESTRIÇÕES
# - Não modificar df_fase2.
# - Não preencher valores ausentes.
# - Não remover observações.
# - Não definir ON/OFF nesta célula.
# - Não utilizar informações futuras.
# - Não treinar modelos.
# ============================================================

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import display

_project_print("=" * 72)
_project_print("FASE 2 — CÉLULA 03/36 | AUDITORIA TEMPORAL E FÍSICA")
_project_print("=" * 72)

# ------------------------------------------------------------
# 1. IDENTIFICAÇÃO DA BASE E VERIFICAÇÃO DE PRÉ-REQUISITOS
# ------------------------------------------------------------

if "df_fase2" not in globals():
    raise RuntimeError(
        "df_fase2 não encontrado. Execute primeiro a Célula 02."
    )

required_cols = [
    "timestamp",
    "Operating Time",
    "Gross Load (MW)",
    "CO2 Mass (short tons)",
    "gross_load_mw_treated",
    "co2_mass_short_tons_treated",
    "load_current_mw",
    "is_on",
    "is_off",
    "is_partial_operation",
]

missing_cols = [
    col for col in required_cols if col not in df_fase2.columns
]

if missing_cols:
    raise ValueError(
        f"Colunas obrigatórias ausentes: {missing_cols}"
    )

# Cópia apenas para análise; original preservado.
audit03_df = df_fase2.copy(deep=True)

audit03_df["timestamp"] = pd.to_datetime(
    audit03_df["timestamp"],
    errors="coerce"
)

audit03_df = audit03_df.sort_values(
    "timestamp",
    kind="stable",
    na_position="last"
).reset_index(drop=True)

N = len(audit03_df)

_project_print(f"\nRegistros analisados: {N:,}")
_project_print(f"Colunas: {audit03_df.shape[1]}")
_project_print("Base original preservada: df_fase2")


# ------------------------------------------------------------
# 2. INTEGRIDADE TEMPORAL
# ------------------------------------------------------------

ts = audit03_df["timestamp"]

valid_ts = ts.dropna()

expected_index = pd.date_range(
    start="2025-01-01 00:00:00",
    end="2025-12-31 23:00:00",
    freq="h"
)

observed_index = pd.DatetimeIndex(valid_ts.unique())

missing_hours = expected_index.difference(observed_index)
outside_hours = observed_index.difference(expected_index)

duplicate_count = int(valid_ts.duplicated().sum())
invalid_ts_count = int(ts.isna().sum())

time_diffs = valid_ts.diff().dropna()

non_hourly_steps = int(
    ((time_diffs != pd.Timedelta(hours=1)) &
     (time_diffs != pd.Timedelta(0))).sum()
)

audit03_temporal = pd.DataFrame([{
    "registros": N,
    "horas_esperadas_2025": len(expected_index),
    "timestamps_validos": len(valid_ts),
    "timestamps_unicos": len(observed_index),
    "timestamps_invalidos": invalid_ts_count,
    "duplicatas": duplicate_count,
    "horas_ausentes": len(missing_hours),
    "horas_fora_2025": len(outside_hours),
    "passos_nao_horarios": non_hourly_steps,
    "inicio": str(valid_ts.min()) if len(valid_ts) else None,
    "fim": str(valid_ts.max()) if len(valid_ts) else None,
}])

_project_print("\n1. INTEGRIDADE TEMPORAL")
_project_display(audit03_temporal)

if len(missing_hours):
    _project_print("\nPrimeiras horas ausentes:")
    _project_display(pd.DataFrame({
        "timestamp_ausente": missing_hours[:20]
    }))


# ------------------------------------------------------------
# 3. AUSÊNCIAS E DISTRIBUIÇÕES DAS VARIÁVEIS PRINCIPAIS
# ------------------------------------------------------------

audit_columns = [
    "Operating Time",
    "Gross Load (MW)",
    "CO2 Mass (short tons)",
    "gross_load_mw_treated",
    "co2_mass_short_tons_treated",
    "load_current_mw",
    "delta_load_1h",
    "abs_delta_load_1h",
]

audit_columns = [
    c for c in audit_columns if c in audit03_df.columns
]

missing_rows = []

for col in audit_columns:
    series = pd.to_numeric(
        audit03_df[col],
        errors="coerce"
    )

    finite = series.replace(
        [np.inf, -np.inf],
        np.nan
    )

    missing_rows.append({
        "variavel": col,
        "nulos": int(series.isna().sum()),
        "infinitos": int(np.isinf(series).sum()),
        "percentual_nulo": round(
            100 * series.isna().mean(), 3
        ),
        "minimo": finite.min(),
        "mediana": finite.median(),
        "maximo": finite.max(),
    })

audit03_missing = pd.DataFrame(missing_rows)

_project_print("\n2. AUSÊNCIAS E DISTRIBUIÇÕES")
_project_display(audit03_missing)


# ------------------------------------------------------------
# 4. CONSISTÊNCIA FÍSICA DAS VARIÁVEIS ORIGINAIS
# ------------------------------------------------------------

operating = pd.to_numeric(
    audit03_df["Operating Time"],
    errors="coerce"
)

load_original = pd.to_numeric(
    audit03_df["Gross Load (MW)"],
    errors="coerce"
)

co2_original = pd.to_numeric(
    audit03_df["CO2 Mass (short tons)"],
    errors="coerce"
)

# Tolerância numérica para comparações com zero.
EPS = 1e-9

masks = {
    "operating_time_negativo": operating < -EPS,
    "operating_time_maior_1": operating > 1 + EPS,
    "geracao_original_negativa": load_original < -EPS,
    "co2_original_negativo": co2_original < -EPS,
    "operando_sem_geracao_original": (
        (operating > EPS) & load_original.isna()
    ),
    "operando_sem_co2_original": (
        (operating > EPS) & co2_original.isna()
    ),
    "sem_operacao_com_geracao_positiva": (
        (operating <= EPS) & (load_original > EPS)
    ),
    "sem_operacao_com_co2_positivo": (
        (operating <= EPS) & (co2_original > EPS)
    ),
    "geracao_positiva_com_co2_zero": (
        (load_original > EPS) &
        co2_original.notna() &
        (co2_original.abs() <= EPS)
    ),
    "co2_positivo_com_geracao_zero": (
        (co2_original > EPS) &
        load_original.notna() &
        (load_original.abs() <= EPS)
    ),
    "geracao_ausente_co2_presente": (
        load_original.isna() & co2_original.notna()
    ),
    "geracao_presente_co2_ausente": (
        load_original.notna() & co2_original.isna()
    ),
}

audit03_operational = pd.DataFrame([
    {
        "verificacao": name,
        "ocorrencias": int(mask.fillna(False).sum()),
        "percentual": round(
            100 * mask.fillna(False).mean(), 3
        ),
    }
    for name, mask in masks.items()
])

_project_print("\n3. CONSISTÊNCIA FÍSICA E OPERACIONAL")
_project_display(audit03_operational)

# Distribuição do Operating Time:
_project_print("\nDistribuição do Operating Time:")
_project_display(
    operating.value_counts(
        dropna=False
    ).head(20).rename_axis(
        "Operating Time"
    ).reset_index(name="horas")
)


# ------------------------------------------------------------
# 5. COMPARAÇÃO: VARIÁVEIS ORIGINAIS VS. TRATADAS
# ------------------------------------------------------------

load_treated = pd.to_numeric(
    audit03_df["gross_load_mw_treated"],
    errors="coerce"
)

co2_treated = pd.to_numeric(
    audit03_df["co2_mass_short_tons_treated"],
    errors="coerce"
)

load_current = pd.to_numeric(
    audit03_df["load_current_mw"],
    errors="coerce"
)

def compare_columns(original, treated, name):
    both = original.notna() & treated.notna()

    different = (
        both &
        (~np.isclose(
            original.fillna(0).to_numpy(),
            treated.fillna(0).to_numpy(),
            rtol=1e-7,
            atol=1e-9
        ))
    )

    return {
        "comparacao": name,
        "original_ausente_tratado_presente": int(
            (original.isna() & treated.notna()).sum()
        ),
        "original_presente_tratado_ausente": int(
            (original.notna() & treated.isna()).sum()
        ),
        "ambos_presentes_diferentes": int(
            different.sum()
        ),
        "ambos_ausentes": int(
            (original.isna() & treated.isna()).sum()
        ),
    }

audit03_treated = pd.DataFrame([
    compare_columns(
        load_original,
        load_treated,
        "Gross Load vs gross_load_mw_treated"
    ),
    compare_columns(
        co2_original,
        co2_treated,
        "CO2 Mass vs co2_mass_short_tons_treated"
    ),
    compare_columns(
        load_treated,
        load_current,
        "gross_load_mw_treated vs load_current_mw"
    ),
])

_project_print("\n4. COMPARAÇÃO ENTRE VARIÁVEIS")
_project_display(audit03_treated)


# ------------------------------------------------------------
# 6. CONCORDÂNCIA DIAGNÓSTICA COM FLAGS DA FASE 1
# ------------------------------------------------------------

flags = [
    "is_on",
    "is_off",
    "is_partial_operation",
]

flag_table = pd.DataFrame({
    "flag": flags,
    "horas_true": [
        int(audit03_df[c].fillna(False).astype(bool).sum())
        for c in flags
    ]
})

flag_overlap = (
    audit03_df[flags]
    .fillna(False)
    .astype(bool)
    .sum(axis=1)
)

flag_table_extra = pd.DataFrame([{
    "horas_sem_flag": int((flag_overlap == 0).sum()),
    "horas_com_multiplas_flags": int(
        (flag_overlap > 1).sum()
    ),
}])

_project_print("\n5. FLAGS OPERACIONAIS DA FASE 1")
_project_display(flag_table)
_project_display(flag_table_extra)

# Não redefinimos nenhuma flag nesta célula.


# ------------------------------------------------------------
# 7. LISTAGEM DOS REGISTROS QUE EXIGEM INSPEÇÃO
# ------------------------------------------------------------

issue_matrix = pd.DataFrame({
    name: mask.fillna(False).astype(bool).to_numpy()
    for name, mask in masks.items()
})

issue_matrix["timestamp_invalido"] = (
    audit03_df["timestamp"].isna().to_numpy()
)

issue_matrix["timestamp_duplicado"] = (
    audit03_df["timestamp"].duplicated(
        keep=False
    ).to_numpy()
)

issue_matrix["fora_de_2025"] = (
    (~audit03_df["timestamp"].isin(expected_index))
    .fillna(False).to_numpy()
)

issue_matrix["flag_sobreposta"] = (
    (flag_overlap > 1).to_numpy()
)

issue_matrix["flag_ausente"] = (
    (flag_overlap == 0).to_numpy()
)

issue_count = issue_matrix.sum(axis=1)

audit03_issues = audit03_df.loc[
    issue_count > 0,
    [
        "timestamp",
        "Operating Time",
        "Gross Load (MW)",
        "CO2 Mass (short tons)",
        "gross_load_mw_treated",
        "co2_mass_short_tons_treated",
        "is_on",
        "is_off",
        "is_partial_operation",
    ]
].copy()

audit03_issues["numero_alertas"] = (
    issue_count.loc[issue_count > 0].to_numpy()
)

_project_print("\n6. REGISTROS COM ALERTAS DIAGNÓSTICOS")
_project_print(f"Total de registros sinalizados: {len(audit03_issues):,}")

if len(audit03_issues):
    _project_display(audit03_issues.head(20))
else:
    _project_print("Nenhum registro sinalizado.")


# ------------------------------------------------------------
# 8. VISUALIZAÇÕES DIAGNÓSTICAS
# ------------------------------------------------------------

# Série temporal: geração original vs tratada
fig, ax = plt.subplots(figsize=(15, 4))

ax.plot(
    audit03_df["timestamp"],
    load_treated,
    linewidth=0.7,
    label="Geração tratada",
    alpha=0.8
)

ax.scatter(
    audit03_df["timestamp"],
    load_original,
    s=2,
    label="Geração original observada",
    alpha=0.5
)

ax.set_title("Iatan 2 — Geração horária em 2025")
ax.set_ylabel("Potência média horária (MW)")
ax.set_xlabel("Data")
ax.legend()
ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()


# Disponibilidade mensal dos dados originais
monthly = audit03_df.assign(
    mes=audit03_df["timestamp"].dt.month,
    geracao_disponivel=load_original.notna(),
    co2_disponivel=co2_original.notna()
)

monthly_availability = (
    monthly.groupby("mes")[
        ["geracao_disponivel", "co2_disponivel"]
    ]
    .mean()
    .mul(100)
)

ax = monthly_availability.plot(
    kind="bar",
    figsize=(12, 4)
)

ax.set_title(
    "Disponibilidade mensal das medições originais — 2025"
)
ax.set_ylabel("Horas com valor registrado (%)")
ax.set_xlabel("Mês")
ax.set_ylim(0, 105)
ax.grid(axis="y", alpha=0.2)
plt.tight_layout()
plt.show()


# ------------------------------------------------------------
# 9. RESUMO E EXPORTAÇÃO DO RELATÓRIO
# ------------------------------------------------------------

audit03_summary = {
    "cell": "03/36",
    "facility_id": "6065",
    "unit_id": "2",
    "year": 2025,
    "rows": int(N),
    "expected_hours": int(len(expected_index)),
    "missing_hours": int(len(missing_hours)),
    "duplicate_timestamps": duplicate_count,
    "invalid_timestamps": invalid_ts_count,
    "out_of_year_timestamps": int(len(outside_hours)),
    "missing_original_load": int(load_original.isna().sum()),
    "missing_original_co2": int(co2_original.isna().sum()),
    "records_with_diagnostic_alerts": int(
        len(audit03_issues)
    ),
    "data_modified": False,
    "on_off_classification_performed": False,
    "model_trained": False,
}

# Usa o diretório da execução criado na Célula 01,
# sem criar um novo experimento.
if "RUN_DIR" in globals():
    audit03_report_dir = Path(RUN_DIR) / "reports"
elif "run_dir" in globals():
    audit03_report_dir = Path(run_dir) / "reports"
else:
    raise RuntimeError(
        "Diretório da execução não encontrado. "
        "Confira a variável definida na Célula 01."
    )

audit03_report_dir.mkdir(
    parents=True,
    exist_ok=True
)

report_path = (
    audit03_report_dir / "cell03_integrity_audit.json"
)

with open(report_path, "w", encoding="utf-8") as f:
    _project_dump(
        audit03_summary,
        f,
        indent=2,
        ensure_ascii=False
    )

_project_print("\n7. RESUMO FINAL")
_project_display(pd.DataFrame([audit03_summary]).T.rename(
    columns={0: "resultado"}
))

_project_print(f"\nRelatório: {report_path}")

_project_print(
    "\nCÉLULA 03 CONCLUÍDA — "
    "auditoria diagnóstica; "
    "nenhum dado alterado; "
    "nenhum modelo treinado."
)


In [ ]:

# ============================================================
# FASE 2 — CÉLULA 04/36
# IDENTIFICAÇÃO E VALIDAÇÃO DOS ESTADOS OPERACIONAIS ON/OFF
# ============================================================
#
# OBJETIVO
# Classificar as horas da Iatan 2 (2025) em:
#   ON           : operação confirmada
#   OFF          : desligamento confirmado
#   INDETERMINADO: evidência insuficiente ou contraditória
#
# A operação parcial é um atributo adicional de ON.
#
# ENTRADA
# df_fase2 — Célula 02
# audit03_operational — diagnóstico da Célula 03
#
# SAÍDAS
# df_fase2_operational
# audit04_summary
# audit04_concordance
# audit04_special_cases
#
# RESTRIÇÕES
# - Não modificar df_fase2.
# - Não excluir horas.
# - Não treinar HMM.
# - Não construir ciclos (Célula 05).
# - Não utilizar CO2 como feature de classificação.
# - Não preencher valores ausentes.
# ============================================================

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
from IPython.display import display

_project_print("=" * 72)
_project_print("FASE 2 — CÉLULA 04/36 | CLASSIFICAÇÃO OPERACIONAL")
_project_print("=" * 72)

# ------------------------------------------------------------
# 1. PRÉ-REQUISITOS
# ------------------------------------------------------------

if "df_fase2" not in globals():
    raise RuntimeError(
        "Execute a Célula 02 antes desta célula."
    )

required_cols = [
    "timestamp",
    "Operating Time",
    "Gross Load (MW)",
    "CO2 Mass (short tons)",
    "is_on",
    "is_off",
    "is_partial_operation",
]

missing = [
    col for col in required_cols
    if col not in df_fase2.columns
]

if missing:
    raise ValueError(
        f"Colunas obrigatórias ausentes: {missing}"
    )

df_fase2_operational = (
    df_fase2
    .copy(deep=True)
    .sort_values("timestamp")
    .reset_index(drop=True)
)

EPS = 1e-9

operating = pd.to_numeric(
    df_fase2_operational["Operating Time"],
    errors="coerce"
)

load = pd.to_numeric(
    df_fase2_operational["Gross Load (MW)"],
    errors="coerce"
)

co2 = pd.to_numeric(
    df_fase2_operational["CO2 Mass (short tons)"],
    errors="coerce"
)

# ------------------------------------------------------------
# 2. DEFINIÇÃO DA REGRA OPERACIONAL
# ------------------------------------------------------------
#
# Operating Time é a evidência primária de operação.
#
# ON:
#   0 < Operating Time <= 1
#
# OFF:
#   Operating Time = 0 e ausência de evidência positiva
#   de geração ou emissão.
#
# INDETERMINADO:
#   Operating Time ausente/inválido;
#   ou Operating Time = 0 com geração/CO2 positivo.
#
# CO2 não é usado para determinar ON; apenas para
# identificar possíveis contradições quando a hora
# foi registrada como OFF.
# ------------------------------------------------------------

valid_operating = (
    operating.notna()
    & np.isfinite(operating)
    & operating.ge(-EPS)
    & operating.le(1 + EPS)
)

on_mask = (
    valid_operating
    & operating.gt(EPS)
)

positive_load = load.gt(EPS).fillna(False)
positive_co2 = co2.gt(EPS).fillna(False)

off_mask = (
    valid_operating
    & operating.abs().le(EPS)
    & ~positive_load
    & ~positive_co2
)

unknown_mask = ~(on_mask | off_mask)

df_fase2_operational["operational_status_fase2"] = np.select(
    [on_mask, off_mask],
    ["ON", "OFF"],
    default="INDETERMINADO"
)

# Operação parcial: 0 < Operating Time < 1.
partial_mask = (
    on_mask
    & operating.lt(1 - EPS)
)

df_fase2_operational["partial_operation_fase2"] = (
    partial_mask.astype(bool)
)

# Evidência complementar, sem alterar o estado.
df_fase2_operational["on_zero_gross_load"] = (
    on_mask
    & load.notna()
    & load.abs().le(EPS)
)

df_fase2_operational["on_zero_load_positive_co2"] = (
    df_fase2_operational["on_zero_gross_load"]
    & positive_co2
)

df_fase2_operational["operational_conflict_fase2"] = (
    unknown_mask
)

# ------------------------------------------------------------
# 3. CONTAGEM E CONSISTÊNCIA DA CLASSIFICAÇÃO
# ------------------------------------------------------------

counts = (
    df_fase2_operational["operational_status_fase2"]
    .value_counts()
    .reindex(
        ["ON", "OFF", "INDETERMINADO"],
        fill_value=0
    )
)

assert int(counts.sum()) == len(df_fase2_operational)

assert not (
    df_fase2_operational["partial_operation_fase2"]
    & ~on_mask
).any()

audit04_summary = pd.DataFrame({
    "estado": counts.index,
    "horas": counts.values,
    "percentual": (
        counts.values
        / len(df_fase2_operational)
        * 100
    ).round(3)
})

_project_print("\n1. CLASSIFICAÇÃO OPERACIONAL")
_project_display(audit04_summary)

_project_print(
    "\nHoras de operação parcial:",
    int(partial_mask.sum())
)

_project_print(
    "Horas ON com geração bruta zero:",
    int(
        df_fase2_operational[
            "on_zero_gross_load"
        ].sum()
    )
)

_project_print(
    "Horas ON com geração zero e CO2 positivo:",
    int(
        df_fase2_operational[
            "on_zero_load_positive_co2"
        ].sum()
    )
)

# ------------------------------------------------------------
# 4. CONCORDÂNCIA COM AS FLAGS DA FASE 1
# ------------------------------------------------------------

audit04_concordance = pd.crosstab(
    df_fase2_operational["operational_status_fase2"],
    df_fase2_operational["is_on"],
    margins=True
)

audit04_off_concordance = pd.crosstab(
    df_fase2_operational["operational_status_fase2"],
    df_fase2_operational["is_off"],
    margins=True
)

_project_print("\n2. CONCORDÂNCIA COM is_on")
_project_display(audit04_concordance)

_project_print("\n3. CONCORDÂNCIA COM is_off")
_project_display(audit04_off_concordance)

# Comparação da operação parcial.
audit04_partial_concordance = pd.crosstab(
    df_fase2_operational["partial_operation_fase2"],
    df_fase2_operational["is_partial_operation"],
    margins=True
)

_project_print("\n4. CONCORDÂNCIA DA OPERAÇÃO PARCIAL")
_project_display(audit04_partial_concordance)

# ------------------------------------------------------------
# 5. CASOS ESPECIAIS
# ------------------------------------------------------------

special_mask = (
    df_fase2_operational["on_zero_gross_load"]
    | df_fase2_operational["partial_operation_fase2"]
    | df_fase2_operational["operational_conflict_fase2"]
)

special_cols = [
    "timestamp",
    "Operating Time",
    "Gross Load (MW)",
    "CO2 Mass (short tons)",
    "operational_status_fase2",
    "partial_operation_fase2",
    "on_zero_gross_load",
    "on_zero_load_positive_co2",
    "operational_conflict_fase2",
]

audit04_special_cases = (
    df_fase2_operational.loc[
        special_mask,
        special_cols
    ]
    .copy()
)

_project_print("\n5. CASOS OPERACIONAIS ESPECIAIS")
_project_print(
    f"Registros selecionados: "
    f"{len(audit04_special_cases):,}"
)

_project_display(audit04_special_cases.head(25))

# ------------------------------------------------------------
# 6. PERFIL OPERACIONAL POR ESTADO
# ------------------------------------------------------------

audit04_profile = (
    df_fase2_operational
    .groupby("operational_status_fase2")
    .agg(
        horas=("timestamp", "size"),
        operating_time_medio=("Operating Time", "mean"),
        geracao_media_mw=("Gross Load (MW)", "mean"),
        geracao_mediana_mw=("Gross Load (MW)", "median"),
        co2_medio_short_tons=("CO2 Mass (short tons)", "mean"),
        horas_parciais=("partial_operation_fase2", "sum"),
        horas_geracao_zero=("on_zero_gross_load", "sum"),
    )
    .reset_index()
)

_project_print("\n6. PERFIL POR ESTADO OPERACIONAL")
_project_display(audit04_profile)

# ------------------------------------------------------------
# 7. VISUALIZAÇÕES
# ------------------------------------------------------------

fig, axes = plt.subplots(
    1, 2,
    figsize=(13, 4)
)

colors = {
    "ON": "#0D9488",
    "OFF": "#94A3B8",
    "INDETERMINADO": "#D97706",
}

axes[0].bar(
    audit04_summary["estado"],
    audit04_summary["horas"],
    color=[
        colors[s]
        for s in audit04_summary["estado"]
    ]
)

axes[0].set_title(
    "Distribuição dos estados operacionais"
)
axes[0].set_ylabel("Horas")
axes[0].grid(axis="y", alpha=0.2)

for i, value in enumerate(audit04_summary["horas"]):
    axes[0].text(
        i,
        value + 50,
        f"{value:,}",
        ha="center",
        fontsize=10
    )

# Horas ON/OFF por mês.
monthly_states = (
    df_fase2_operational
    .assign(
        month=lambda d: pd.to_datetime(
            d["timestamp"]
        ).dt.month
    )
    .groupby([
        "month",
        "operational_status_fase2"
    ])
    .size()
    .unstack(fill_value=0)
    .reindex(
        columns=[
            "ON",
            "OFF",
            "INDETERMINADO"
        ],
        fill_value=0
    )
)

monthly_states.plot(
    kind="bar",
    stacked=True,
    ax=axes[1],
    color=[
        colors[c]
        for c in monthly_states.columns
    ]
)

axes[1].set_title(
    "Distribuição mensal dos estados"
)
axes[1].set_xlabel("Mês")
axes[1].set_ylabel("Horas")
axes[1].legend(title="Estado")
axes[1].grid(axis="y", alpha=0.2)

plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# 8. VERIFICAÇÕES FINAIS
# ------------------------------------------------------------

assert len(df_fase2_operational) == len(df_fase2)

assert (
    df_fase2_operational["timestamp"]
    .nunique()
    == 8760
)

assert not (
    on_mask & off_mask
).any()

assert (
    df_fase2_operational["operational_status_fase2"]
    .notna()
    .all()
)

_project_print("\n7. VERIFICAÇÕES FINAIS")
_project_print("Integridade das 8.760 horas: OK")
_project_print("Estados mutuamente exclusivos: OK")
_project_print("Operação parcial subordinada a ON: OK")
_project_print("Base original preservada: OK")

# ------------------------------------------------------------
# 9. REGISTRO DA CLASSIFICAÇÃO
# ------------------------------------------------------------

audit04_report = {
    "cell": "04/36",
    "facility_id": "6065",
    "unit_id": "2",
    "year": 2025,
    "classification_rule": {
        "ON": "0 < Operating Time <= 1",
        "OFF": (
            "Operating Time = 0 sem geração "
            "ou CO2 positivo"
        ),
        "INDETERMINADO": (
            "Operating Time ausente, inválido "
            "ou contraditório"
        ),
        "PARTIAL": "0 < Operating Time < 1",
    },
    "counts": {
        str(k): int(v)
        for k, v in counts.items()
    },
    "partial_hours": int(partial_mask.sum()),
    "on_zero_gross_load_hours": int(
        df_fase2_operational[
            "on_zero_gross_load"
        ].sum()
    ),
    "on_zero_load_positive_co2_hours": int(
        df_fase2_operational[
            "on_zero_load_positive_co2"
        ].sum()
    ),
    "data_original_modified": False,
    "hmm_trained": False,
}

if "RUN_DIR" in globals():
    report_dir = Path(RUN_DIR) / "reports"
elif "run_dir" in globals():
    report_dir = Path(run_dir) / "reports"
else:
    raise RuntimeError(
        "Diretório RUN_DIR não encontrado. "
        "Execute a Célula 01."
    )

report_dir.mkdir(parents=True, exist_ok=True)

report_path = (
    report_dir
    / "cell04_operational_classification.json"
)

with open(
    report_path,
    "w",
    encoding="utf-8"
) as f:
    _project_dump(
        audit04_report,
        f,
        ensure_ascii=False,
        indent=2
    )

_project_print(f"\nRelatório: {report_path}")

_project_print(
    "\nCÉLULA 04 CONCLUÍDA — "
    "classificação operacional realizada; "
    "nenhum HMM treinado; "
    "nenhum ciclo construído."
)


In [ ]:

# ============================================================
# FASE 2 — CÉLULA 05/36
# SEGMENTAÇÃO E CARACTERIZAÇÃO DOS CICLOS OPERACIONAIS
# ============================================================
#
# OBJETIVO
# Identificar ciclos contínuos ON/OFF da Iatan 2 em 2025,
# caracterizar duração, partidas, paradas e censura.
#
# ENTRADA
# df_fase2_operational — Célula 04.
#
# SAÍDAS
# df_fase2_cycles
# audit05_cycles
# audit05_on_cycles
# audit05_off_cycles
# audit05_summary
#
# RESTRIÇÕES
# - Não modificar as bases das células anteriores.
# - Não alterar a classificação ON/OFF aprovada.
# - Não excluir horas ON com geração zero.
# - Não treinar HMM.
# - Não realizar divisão treino/validação/teste.
# - Não construir logs de Process Mining nesta célula.
# ============================================================

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
from IPython.display import display

_project_print("=" * 72)
_project_print("FASE 2 — CÉLULA 05/36 | CICLOS OPERACIONAIS")
_project_print("=" * 72)

# ------------------------------------------------------------
# 1. PRÉ-REQUISITOS E INTEGRIDADE
# ------------------------------------------------------------

if "df_fase2_operational" not in globals():
    raise RuntimeError(
        "Execute a Célula 04 antes desta célula."
    )

required = [
    "timestamp",
    "operational_status_fase2",
    "Operating Time",
    "Gross Load (MW)",
    "CO2 Mass (short tons)",
    "partial_operation_fase2",
    "on_zero_gross_load",
]

missing = [
    c for c in required
    if c not in df_fase2_operational.columns
]

if missing:
    raise ValueError(f"Colunas ausentes: {missing}")

df_fase2_cycles = (
    df_fase2_operational
    .copy(deep=True)
    .sort_values("timestamp")
    .reset_index(drop=True)
)

df_fase2_cycles["timestamp"] = pd.to_datetime(
    df_fase2_cycles["timestamp"],
    errors="raise"
)

if df_fase2_cycles["timestamp"].duplicated().any():
    raise ValueError("Existem timestamps duplicados.")

if df_fase2_cycles["timestamp"].isna().any():
    raise ValueError("Existem timestamps inválidos.")

states = df_fase2_cycles["operational_status_fase2"]

if not states.isin(
    ["ON", "OFF", "INDETERMINADO"]
).all():
    raise ValueError("Estado operacional desconhecido.")

# ------------------------------------------------------------
# 2. IDENTIFICAÇÃO DOS LIMITES DOS CICLOS
# ------------------------------------------------------------
#
# Novo segmento quando:
# - muda o estado operacional;
# - existe descontinuidade temporal;
# - começa a série.
#
# INDETERMINADO interrompe a continuidade operacional.
# ------------------------------------------------------------

timestamps = df_fase2_cycles["timestamp"]

time_gap = timestamps.diff().ne(
    pd.Timedelta(hours=1)
)

state_change = states.ne(states.shift(1))

new_segment = (
    time_gap | state_change
)

new_segment.iloc[0] = True

df_fase2_cycles["segment_id"] = (
    new_segment.cumsum().astype(int)
)

# Identificadores distintos para ciclos ON e OFF.
segment_type = (
    df_fase2_cycles
    .groupby("segment_id")[
        "operational_status_fase2"
    ]
    .first()
)

on_segment_ids = segment_type[
    segment_type.eq("ON")
].index.tolist()

off_segment_ids = segment_type[
    segment_type.eq("OFF")
].index.tolist()

on_map = {
    sid: f"ON_{i:04d}"
    for i, sid in enumerate(on_segment_ids, start=1)
}

off_map = {
    sid: f"OFF_{i:04d}"
    for i, sid in enumerate(off_segment_ids, start=1)
}

df_fase2_cycles["cycle_id"] = (
    df_fase2_cycles["segment_id"]
    .map({**on_map, **off_map})
)

# INDETERMINADO permanece sem cycle_id.
df_fase2_cycles["cycle_type"] = states

# ------------------------------------------------------------
# 3. POSIÇÃO TEMPORAL DENTRO DO CICLO
# ------------------------------------------------------------

df_fase2_cycles["cycle_hour_index"] = (
    df_fase2_cycles
    .groupby("segment_id")
    .cumcount()
    .add(1)
)

df_fase2_cycles["cycle_elapsed_hours"] = (
    df_fase2_cycles["cycle_hour_index"] - 1
)

# O índice representa horas observadas desde o início
# do segmento, não o tempo efetivo de operação.

# ------------------------------------------------------------
# 4. IDENTIFICAÇÃO DE PARTIDAS E PARADAS
# ------------------------------------------------------------
#
# Partida observada:
# OFF na hora anterior -> ON na hora atual.
#
# Parada observada:
# ON na hora anterior -> OFF na hora atual.
#
# Transições atravessando lacunas não são confirmadas.
# ------------------------------------------------------------

previous_state = states.shift(1)

continuous_previous = (
    timestamps.diff().eq(pd.Timedelta(hours=1))
)

startup_observed = (
    states.eq("ON")
    & previous_state.eq("OFF")
    & continuous_previous
)

shutdown_observed = (
    states.eq("OFF")
    & previous_state.eq("ON")
    & continuous_previous
)

df_fase2_cycles["startup_observed_fase2"] = (
    startup_observed
)

df_fase2_cycles["shutdown_observed_fase2"] = (
    shutdown_observed
)

# ------------------------------------------------------------
# 5. CARACTERIZAÇÃO DOS CICLOS
# ------------------------------------------------------------

load = pd.to_numeric(
    df_fase2_cycles["Gross Load (MW)"],
    errors="coerce"
)

co2 = pd.to_numeric(
    df_fase2_cycles["CO2 Mass (short tons)"],
    errors="coerce"
)

operating = pd.to_numeric(
    df_fase2_cycles["Operating Time"],
    errors="coerce"
)

df_fase2_cycles["_audit05_load"] = load
df_fase2_cycles["_audit05_co2"] = co2
df_fase2_cycles["_audit05_operating"] = operating

audit05_cycles = (
    df_fase2_cycles
    .groupby("segment_id")
    .agg(
        cycle_id=("cycle_id", "first"),
        cycle_type=("cycle_type", "first"),
        start=("timestamp", "min"),
        end=("timestamp", "max"),
        duration_hours=("timestamp", "size"),
        operating_hours=("_audit05_operating", "sum"),
        mean_load_mw=("_audit05_load", "mean"),
        median_load_mw=("_audit05_load", "median"),
        max_load_mw=("_audit05_load", "max"),
        total_co2_short_tons=("_audit05_co2", "sum"),
        partial_hours=("partial_operation_fase2", "sum"),
        zero_load_on_hours=("on_zero_gross_load", "sum"),
    )
    .reset_index()
)

# Para ciclos OFF, a ausência de CO2 original não deve
# ser interpretada como medição de emissão igual a zero.
audit05_cycles.loc[
    audit05_cycles["cycle_type"].eq("OFF"),
    "total_co2_short_tons"
] = np.nan

# ------------------------------------------------------------
# 6. CENSURA NAS FRONTEIRAS DO PERÍODO
# ------------------------------------------------------------
#
# Um ciclo que começa na primeira hora do arquivo pode
# ter iniciado antes do período observado.
#
# Um ciclo que termina na última hora pode continuar
# no período seguinte.
#
# Ciclos junto a lacunas também são considerados
# incompletos, quando aplicável.
# ------------------------------------------------------------

first_timestamp = timestamps.min()
last_timestamp = timestamps.max()

segment_starts = df_fase2_cycles.groupby(
    "segment_id"
)["timestamp"].min()

segment_ends = df_fase2_cycles.groupby(
    "segment_id"
)["timestamp"].max()

audit05_cycles["left_censored"] = (
    audit05_cycles["start"].eq(first_timestamp)
)

audit05_cycles["right_censored"] = (
    audit05_cycles["end"].eq(last_timestamp)
)

# Identifica limites criados por lacunas.
previous_end = audit05_cycles["end"].shift(1)
next_start = audit05_cycles["start"].shift(-1)

gap_before = (
    audit05_cycles["start"].sub(previous_end)
    .ne(pd.Timedelta(hours=1))
    & previous_end.notna()
)

gap_after = (
    next_start.sub(audit05_cycles["end"])
    .ne(pd.Timedelta(hours=1))
    & next_start.notna()
)

audit05_cycles["left_censored"] |= gap_before
audit05_cycles["right_censored"] |= gap_after

audit05_cycles["complete_cycle"] = (
    ~audit05_cycles["left_censored"]
    & ~audit05_cycles["right_censored"]
)

# Nota:
# Um segmento ON precedido e seguido de OFF é completo.
# Um segmento OFF também pode ser completo, mas isso não
# significa que sua duração seja tempo de operação.

# ------------------------------------------------------------
# 7. TABELAS ON E OFF
# ------------------------------------------------------------

audit05_on_cycles = audit05_cycles.loc[
    audit05_cycles["cycle_type"].eq("ON")
].copy()

audit05_off_cycles = audit05_cycles.loc[
    audit05_cycles["cycle_type"].eq("OFF")
].copy()

# Propaga indicadores de censura para a base horária.
cycle_censor_map = audit05_cycles.set_index(
    "segment_id"
)[
    ["left_censored", "right_censored", "complete_cycle"]
]

df_fase2_cycles = df_fase2_cycles.join(
    cycle_censor_map,
    on="segment_id"
)

# Remove colunas temporárias de auditoria.
df_fase2_cycles = df_fase2_cycles.drop(
    columns=[
        "_audit05_load",
        "_audit05_co2",
        "_audit05_operating",
    ]
)

# ------------------------------------------------------------
# 8. RESUMO DOS CICLOS
# ------------------------------------------------------------

def duration_stats(frame):
    if frame.empty:
        return {
            "n": 0,
            "median_h": None,
            "mean_h": None,
            "p90_h": None,
            "max_h": None,
        }

    d = frame["duration_hours"]

    return {
        "n": int(len(frame)),
        "median_h": float(d.median()),
        "mean_h": float(d.mean()),
        "p90_h": float(d.quantile(0.90)),
        "max_h": int(d.max()),
    }

on_stats = duration_stats(audit05_on_cycles)
off_stats = duration_stats(audit05_off_cycles)

audit05_summary = {
    "facility_id": "6065",
    "unit_id": "2",
    "year": 2025,
    "total_hours": int(len(df_fase2_cycles)),
    "on_hours": int(states.eq("ON").sum()),
    "off_hours": int(states.eq("OFF").sum()),
    "unknown_hours": int(states.eq("INDETERMINADO").sum()),
    "on_cycles": int(len(audit05_on_cycles)),
    "off_cycles": int(len(audit05_off_cycles)),
    "observed_startups": int(startup_observed.sum()),
    "observed_shutdowns": int(shutdown_observed.sum()),
    "complete_on_cycles": int(
        audit05_on_cycles["complete_cycle"].sum()
    ),
    "censored_on_cycles": int(
        (~audit05_on_cycles["complete_cycle"]).sum()
    ),
    "on_duration": on_stats,
    "off_duration": off_stats,
}

_project_print("\n1. RESUMO DOS CICLOS")
_project_display(
    pd.DataFrame(
        [
            {
                "tipo": "ON",
                **on_stats,
            },
            {
                "tipo": "OFF",
                **off_stats,
            },
        ]
    )
)

_project_print("\n2. PARTIDAS, PARADAS E CENSURA")
_project_display(
    pd.DataFrame([{
        "ciclos_ON": audit05_summary["on_cycles"],
        "ciclos_OFF": audit05_summary["off_cycles"],
        "partidas_observadas": audit05_summary["observed_startups"],
        "paradas_observadas": audit05_summary["observed_shutdowns"],
        "ciclos_ON_completos": audit05_summary["complete_on_cycles"],
        "ciclos_ON_censurados": audit05_summary["censored_on_cycles"],
    }])
)

_project_print("\n3. AMOSTRA DOS CICLOS ON")
_project_display(audit05_on_cycles.head(15))

_project_print("\n4. CICLOS ON MAIS LONGOS")
_project_display(
    audit05_on_cycles.nlargest(
        10, "duration_hours"
    )[
        [
            "cycle_id",
            "start",
            "end",
            "duration_hours",
            "operating_hours",
            "mean_load_mw",
            "partial_hours",
            "zero_load_on_hours",
            "complete_cycle",
        ]
    ]
)

# ------------------------------------------------------------
# 9. VISUALIZAÇÕES DOS CICLOS
# ------------------------------------------------------------

fig, axes = plt.subplots(
    1, 2,
    figsize=(14, 4)
)

if len(audit05_on_cycles):
    axes[0].hist(
        audit05_on_cycles["duration_hours"],
        bins=min(30, max(5, len(audit05_on_cycles))),
        edgecolor="white",
        alpha=0.85
    )

axes[0].set_title("Duração dos ciclos ON")
axes[0].set_xlabel("Duração observada (horas)")
axes[0].set_ylabel("Frequência")
axes[0].grid(alpha=0.2)

if len(audit05_off_cycles):
    axes[1].hist(
        audit05_off_cycles["duration_hours"],
        bins=min(30, max(5, len(audit05_off_cycles))),
        edgecolor="white",
        alpha=0.85
    )

axes[1].set_title("Duração dos ciclos OFF")
axes[1].set_xlabel("Duração observada (horas)")
axes[1].set_ylabel("Frequência")
axes[1].grid(alpha=0.2)

plt.tight_layout()
plt.show()

# Ciclos ON ao longo do ano.
fig, ax = plt.subplots(figsize=(15, 5))

for _, cycle in audit05_on_cycles.iterrows():
    ax.hlines(
        y=cycle["segment_id"],
        xmin=cycle["start"],
        xmax=cycle["end"] + pd.Timedelta(hours=1),
        linewidth=2.2,
        alpha=0.8
    )

ax.set_title("Iatan 2 — Distribuição temporal dos ciclos ON")
ax.set_xlabel("Data")
ax.set_ylabel("Identificador sequencial do segmento")
ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# 10. VERIFICAÇÕES DE CONSISTÊNCIA
# ------------------------------------------------------------

assert len(df_fase2_cycles) == 8760

assert df_fase2_cycles["timestamp"].is_unique

assert (
    audit05_cycles["duration_hours"].sum()
    == len(df_fase2_cycles)
)

assert (
    audit05_on_cycles["duration_hours"].sum()
    == states.eq("ON").sum()
)

assert (
    audit05_off_cycles["duration_hours"].sum()
    == states.eq("OFF").sum()
)

assert not (
    df_fase2_cycles.loc[
        states.eq("ON"), "cycle_id"
    ].isna()
).any()

assert not (
    df_fase2_cycles.loc[
        states.eq("OFF"), "cycle_id"
    ].isna()
).any()

assert (
    df_fase2_cycles["cycle_hour_index"]
    .ge(1)
    .all()
)

_project_print("\n5. VERIFICAÇÕES FINAIS")
_project_print("Total de horas preservado: OK")
_project_print("Ciclos mutuamente exclusivos: OK")
_project_print("Horas ON/OFF reconciliadas: OK")
_project_print("Identificadores de ciclos: OK")
_project_print("Base original preservada: OK")

# ------------------------------------------------------------
# 11. REGISTRO DO RELATÓRIO
# ------------------------------------------------------------

if "RUN_DIR" in globals():
    report_dir = Path(RUN_DIR) / "reports"
elif "run_dir" in globals():
    report_dir = Path(run_dir) / "reports"
else:
    raise RuntimeError(
        "Diretório de execução não encontrado."
    )

report_dir.mkdir(parents=True, exist_ok=True)

report_path = (
    report_dir / "cell05_operational_cycles.json"
)

with open(
    report_path,
    "w",
    encoding="utf-8"
) as f:
    _project_dump(
        audit05_summary,
        f,
        ensure_ascii=False,
        indent=2
    )

_project_print(f"\nRelatório: {report_path}")

_project_print(
    "\nCÉLULA 05 CONCLUÍDA — "
    "ciclos identificados e caracterizados; "
    "nenhum HMM treinado; "
    "nenhum log de processos construído."
)


## 06–08 — Divisão cronológica e matrizes HMM

O corte preserva ciclos completos. O padronizador aprende no treino, e a matriz fornece um comprimento por ciclo ao HMM para reiniciar o estado inicial entre sequências. A divisão e a seleção por validação seguem o princípio de separar ajuste e avaliação. Percentuais, atributos e critérios de corte são escolhas da entrega. [Cawley e Talbot (2010)](https://jmlr.org/papers/v11/cawley10a.html) · [API GaussianHMM](https://hmmlearn.readthedocs.io/en/stable/api.html#hmmlearn.hmm.GaussianHMM)


In [ ]:

# ============================================================
# FASE 2 — CÉLULA 06/36
# DIVISÃO CRONOLÓGICA: TREINO, VALIDAÇÃO E TESTE
# ============================================================
#
# OBJETIVO
# Criar partições cronológicas para a análise HMM,
# preservando integralmente os ciclos ON.
#
# ENTRADA
# df_fase2_cycles, audit05_on_cycles — Célula 05.
#
# SAÍDAS
# df_fase2_split
# audit06_on_cycles
# audit06_partitions
# audit06_summary
#
# RESTRIÇÕES
# - Não modificar os objetos das células anteriores.
# - Não dividir ciclos ON entre partições.
# - Não embaralhar observações.
# - Não utilizar CO2 para selecionar fronteiras.
# - Não ajustar scaler ou HMM.
# - Não consultar resultados de teste para seleção.
# ============================================================

import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display

_project_print("=" * 72)
_project_print("FASE 2 — CÉLULA 06/36 | DIVISÃO CRONOLÓGICA")
_project_print("=" * 72)

# ------------------------------------------------------------
# 1. PRÉ-REQUISITOS
# ------------------------------------------------------------

if "df_fase2_cycles" not in globals():
    raise RuntimeError(
        "Execute a Célula 05 antes desta célula."
    )

if "audit05_on_cycles" not in globals():
    raise RuntimeError(
        "Tabela de ciclos ON não encontrada."
    )

required = [
    "timestamp",
    "cycle_id",
    "segment_id",
    "operational_status_fase2",
    "cycle_hour_index",
]

missing = [
    c for c in required
    if c not in df_fase2_cycles.columns
]

if missing:
    raise ValueError(f"Colunas ausentes: {missing}")

df_fase2_split = (
    df_fase2_cycles
    .copy(deep=True)
    .sort_values("timestamp")
    .reset_index(drop=True)
)

df_fase2_split["timestamp"] = pd.to_datetime(
    df_fase2_split["timestamp"],
    errors="raise"
)

if not df_fase2_split["timestamp"].is_unique:
    raise ValueError("Timestamps duplicados.")

if not df_fase2_split["timestamp"].is_monotonic_increasing:
    raise ValueError("Timestamps fora de ordem.")

if len(df_fase2_split) != 8760:
    raise ValueError(
        "Esperadas 8.760 observações em 2025."
    )

# ------------------------------------------------------------
# 2. PREPARAÇÃO DOS CICLOS ON
# ------------------------------------------------------------

audit06_on_cycles = (
    audit05_on_cycles[
        [
            "cycle_id",
            "segment_id",
            "start",
            "end",
            "duration_hours",
            "complete_cycle",
        ]
    ]
    .copy()
    .sort_values("start")
    .reset_index(drop=True)
)

if audit06_on_cycles.empty:
    raise ValueError("Nenhum ciclo ON encontrado.")

if audit06_on_cycles["cycle_id"].isna().any():
    raise ValueError("Existem ciclos ON sem identificador.")

if audit06_on_cycles["cycle_id"].duplicated().any():
    raise ValueError("Identificadores de ciclos repetidos.")

audit06_on_cycles["duration_hours"] = (
    audit06_on_cycles["duration_hours"].astype(int)
)

total_on_hours = int(
    audit06_on_cycles["duration_hours"].sum()
)

if total_on_hours != int(
    df_fase2_split[
        "operational_status_fase2"
    ].eq("ON").sum()
):
    raise AssertionError(
        "Horas ON não reconciliadas."
    )

# ------------------------------------------------------------
# 3. DEFINIÇÃO DAS PROPORÇÕES-ALVO
# ------------------------------------------------------------

TARGET_TRAIN = 0.70
TARGET_VALIDATION = 0.15
TARGET_TEST = 0.15

assert np.isclose(
    TARGET_TRAIN + TARGET_VALIDATION + TARGET_TEST,
    1.0
)

n_cycles = len(audit06_on_cycles)

if n_cycles < 3:
    raise ValueError(
        "São necessários ao menos três ciclos ON "
        "para esta estratégia."
    )

target_train_hours = (
    total_on_hours * TARGET_TRAIN
)

target_train_val_hours = (
    total_on_hours *
    (TARGET_TRAIN + TARGET_VALIDATION)
)

_project_print("\n1. METAS DE DIVISÃO")
_project_print(f"Ciclos ON: {n_cycles}")
_project_print(f"Horas ON: {total_on_hours:,}")
_project_print(
    "Metas: "
    f"{TARGET_TRAIN:.0%} treino, "
    f"{TARGET_VALIDATION:.0%} validação, "
    f"{TARGET_TEST:.0%} teste"
)

# ------------------------------------------------------------
# 4. ESCOLHA DAS FRONTEIRAS ENTRE CICLOS
# ------------------------------------------------------------
#
# Selecionamos dois pontos de corte ENTRE ciclos ON.
#
# Critério:
# minimizar a diferença entre as proporções-alvo
# e as proporções efetivas de horas ON.
#
# Não utilizamos geração, CO2 ou resultados do HMM.
# ------------------------------------------------------------

durations = (
    audit06_on_cycles["duration_hours"]
    .to_numpy(dtype=int)
)

cumulative = np.cumsum(durations)

best = None

for i in range(1, n_cycles - 1):
    for j in range(i + 1, n_cycles):

        train_h = int(cumulative[i - 1])

        val_h = int(
            cumulative[j - 1] - cumulative[i - 1]
        )

        test_h = int(
            cumulative[-1] - cumulative[j - 1]
        )

        proportions = np.array(
            [train_h, val_h, test_h],
            dtype=float
        ) / total_on_hours

        targets = np.array([
            TARGET_TRAIN,
            TARGET_VALIDATION,
            TARGET_TEST
        ])

        error = float(
            np.sum((proportions - targets) ** 2)
        )

        candidate = (
            error,
            i,
            j,
            train_h,
            val_h,
            test_h,
        )

        if best is None or candidate < best:
            best = candidate

_, split_i, split_j, train_h, val_h, test_h = best

audit06_on_cycles["partition"] = (
    ["train"] * split_i
    + ["validation"] * (split_j - split_i)
    + ["test"] * (n_cycles - split_j)
)

# ------------------------------------------------------------
# 5. PROPAGAÇÃO PARA A BASE HORÁRIA
# ------------------------------------------------------------
#
# Cada hora ON recebe a partição do seu ciclo.
#
# Horas OFF recebem a partição segundo os limites
# temporais estabelecidos entre ciclos.
# ------------------------------------------------------------

train_end = audit06_on_cycles.iloc[
    split_i - 1
]["end"]

validation_end = audit06_on_cycles.iloc[
    split_j - 1
]["end"]

df_fase2_split["partition"] = np.select(
    [
        df_fase2_split["timestamp"].le(train_end),
        df_fase2_split["timestamp"].le(validation_end),
    ],
    [
        "train",
        "validation",
    ],
    default="test"
)

# Verificação explícita dos ciclos ON.
cycle_partition_map = (
    audit06_on_cycles
    .set_index("cycle_id")["partition"]
)

on_mask = (
    df_fase2_split["operational_status_fase2"]
    .eq("ON")
)

expected_on_partition = (
    df_fase2_split.loc[
        on_mask, "cycle_id"
    ].map(cycle_partition_map)
)

actual_on_partition = (
    df_fase2_split.loc[
        on_mask, "partition"
    ]
)

if not np.array_equal(
    expected_on_partition.to_numpy(),
    actual_on_partition.to_numpy()
):
    raise AssertionError(
        "Um ciclo ON foi dividido indevidamente."
    )

# ------------------------------------------------------------
# 6. AUDITORIA DAS PARTIÇÕES
# ------------------------------------------------------------

audit06_partitions = (
    df_fase2_split
    .groupby("partition")
    .agg(
        total_hours=("timestamp", "size"),
        start=("timestamp", "min"),
        end=("timestamp", "max"),
        on_hours=(
            "operational_status_fase2",
            lambda s: int(s.eq("ON").sum())
        ),
        off_hours=(
            "operational_status_fase2",
            lambda s: int(s.eq("OFF").sum())
        ),
        unknown_hours=(
            "operational_status_fase2",
            lambda s: int(
                s.eq("INDETERMINADO").sum()
            )
        ),
    )
    .reindex(["train", "validation", "test"])
    .reset_index()
)

audit06_partitions["on_share_pct"] = (
    100 *
    audit06_partitions["on_hours"]
    / total_on_hours
).round(3)

audit06_partitions["all_hours_share_pct"] = (
    100 *
    audit06_partitions["total_hours"]
    / len(df_fase2_split)
).round(3)

cycle_counts = (
    audit06_on_cycles["partition"]
    .value_counts()
)

audit06_partitions["on_cycles"] = (
    audit06_partitions["partition"]
    .map(cycle_counts)
    .fillna(0)
    .astype(int)
)

_project_print("\n2. PARTIÇÕES CRONOLÓGICAS")
_project_display(audit06_partitions)

_project_print("\n3. CICLOS ON POR PARTIÇÃO")
_project_display(
    audit06_on_cycles[
        [
            "cycle_id",
            "start",
            "end",
            "duration_hours",
            "complete_cycle",
            "partition",
        ]
    ]
)

# ------------------------------------------------------------
# 7. VERIFICAÇÕES DE ISOLAMENTO TEMPORAL
# ------------------------------------------------------------

grouped = {
    name: df_fase2_split.loc[
        df_fase2_split["partition"].eq(name)
    ]
    for name in ["train", "validation", "test"]
}

for name, frame in grouped.items():
    if frame.empty:
        raise AssertionError(
            f"Partição vazia: {name}"
        )

assert (
    grouped["train"]["timestamp"].max()
    < grouped["validation"]["timestamp"].min()
)

assert (
    grouped["validation"]["timestamp"].max()
    < grouped["test"]["timestamp"].min()
)

on_partition_count = (
    df_fase2_split.loc[on_mask]
    .groupby("cycle_id")["partition"]
    .nunique()
)

assert on_partition_count.eq(1).all()

assert (
    audit06_partitions["on_hours"].sum()
    == total_on_hours
)

assert (
    audit06_partitions["total_hours"].sum()
    == 8760
)

# ------------------------------------------------------------
# 8. VISUALIZAÇÕES
# ------------------------------------------------------------

colors = {
    "train": "#0D9488",
    "validation": "#E69F00",
    "test": "#64748B",
}

fig, axes = plt.subplots(
    1, 2,
    figsize=(14, 4)
)

axes[0].bar(
    audit06_partitions["partition"],
    audit06_partitions["on_hours"],
    color=[
        colors[p]
        for p in audit06_partitions["partition"]
    ]
)

axes[0].set_title("Horas ON por partição")
axes[0].set_ylabel("Horas")
axes[0].grid(axis="y", alpha=0.2)

for idx, value in enumerate(
    audit06_partitions["on_hours"]
):
    axes[0].text(
        idx,
        value + 25,
        f"{value:,}",
        ha="center"
    )

axes[1].bar(
    audit06_partitions["partition"],
    audit06_partitions["on_cycles"],
    color=[
        colors[p]
        for p in audit06_partitions["partition"]
    ]
)

axes[1].set_title("Ciclos ON por partição")
axes[1].set_ylabel("Número de ciclos")
axes[1].grid(axis="y", alpha=0.2)

plt.tight_layout()
plt.show()

# Distribuição temporal dos ciclos por partição.
fig, ax = plt.subplots(figsize=(15, 5))

for idx, cycle in audit06_on_cycles.iterrows():
    ax.hlines(
        y=idx + 1,
        xmin=cycle["start"],
        xmax=cycle["end"] + pd.Timedelta(hours=1),
        color=colors[cycle["partition"]],
        linewidth=2.5
    )

ax.axvline(
    train_end,
    color=colors["train"],
    linestyle="--",
    label="Fim do treino"
)

ax.axvline(
    validation_end,
    color=colors["validation"],
    linestyle="--",
    label="Fim da validação"
)

ax.set_title(
    "Ciclos ON e divisão cronológica — Iatan 2, 2025"
)
ax.set_xlabel("Data")
ax.set_ylabel("Ciclo ON (ordem cronológica)")
ax.legend()
ax.grid(alpha=0.2)

plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# 9. REGISTRO METODOLÓGICO
# ------------------------------------------------------------

audit06_summary = {
    "cell": "06/36",
    "facility_id": "6065",
    "unit_id": "2",
    "year": 2025,
    "strategy": (
        "chronological_cycle_aware_split"
    ),
    "target_proportions_on_hours": {
        "train": TARGET_TRAIN,
        "validation": TARGET_VALIDATION,
        "test": TARGET_TEST,
    },
    "actual_on_hours": {
        "train": int(train_h),
        "validation": int(val_h),
        "test": int(test_h),
    },
    "on_cycles": {
        k: int(v)
        for k, v in cycle_counts.items()
    },
    "train_end": str(train_end),
    "validation_end": str(validation_end),
    "test_end": str(
        df_fase2_split["timestamp"].max()
    ),
    "split_cycle_indices": {
        "i": int(split_i),
        "j": int(split_j),
    },
    "selection_uses_co2": False,
    "hmm_trained": False,
}

if "RUN_DIR" in globals():
    report_dir = Path(RUN_DIR) / "reports"
elif "run_dir" in globals():
    report_dir = Path(run_dir) / "reports"
else:
    raise RuntimeError(
        "Diretório de execução não encontrado. "
        "Execute a Célula 01."
    )

report_dir.mkdir(parents=True, exist_ok=True)

report_path = (
    report_dir
    / "cell06_chronological_split.json"
)

with open(
    report_path,
    "w",
    encoding="utf-8"
) as f:
    _project_dump(
        audit06_summary,
        f,
        ensure_ascii=False,
        indent=2
    )

_project_print("\n4. VERIFICAÇÕES FINAIS")
_project_print("Ordem cronológica: OK")
_project_print("Ciclos ON íntegros: OK")
_project_print("Partições sem sobreposição: OK")
_project_print("Horas ON reconciliadas: OK")
_project_print("Teste isolado para seleção futura: OK")

_project_print(f"\nRelatório: {report_path}")

_project_print(
    "\nCÉLULA 06 CONCLUÍDA — "
    "partições cronológicas definidas; "
    "nenhum scaler ajustado; "
    "nenhum HMM treinado."
)


In [ ]:

# ============================================================
# FASE 2 — CÉLULA 07/36
# DIAGNÓSTICO DAS VARIÁVEIS CANDIDATAS AO HMM
# ============================================================
#
# OBJETIVO
# Avaliar variáveis operacionais para identificação de
# regimes ocultos da Iatan 2 em 2025.
#
# ENTRADA
# df_fase2_split — Célula 06.
#
# SAÍDAS
# df_fase2_hmm_features
# audit07_feature_catalog
# audit07_feature_quality
# audit07_feature_correlations
# audit07_candidate_models
# audit07_summary
#
# RESTRIÇÕES
# - Não alterar as partições.
# - Não utilizar CO2 como feature.
# - Não ajustar StandardScaler.
# - Não treinar HMM.
# - Não selecionar K.
# - Não modificar os dados das células anteriores.
# - Não usar observações futuras para criar features.
# ============================================================

import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display

_project_print("=" * 72)
_project_print("FASE 2 — CÉLULA 07/36 | VARIÁVEIS CANDIDATAS AO HMM")
_project_print("=" * 72)

# ------------------------------------------------------------
# 1. PRÉ-REQUISITOS
# ------------------------------------------------------------

if "df_fase2_split" not in globals():
    raise RuntimeError(
        "Execute a Célula 06 antes desta célula."
    )

required_cols = [
    "timestamp",
    "partition",
    "cycle_id",
    "operational_status_fase2",
    "load_current_mw",
    "delta_load_1h",
    "abs_delta_load_1h",
    "Operating Time",
    "Gross Load (MW)",
    "CO2 Mass (short tons)",
]

missing = [
    c for c in required_cols
    if c not in df_fase2_split.columns
]

if missing:
    raise ValueError(
        f"Colunas obrigatórias ausentes: {missing}"
    )

df_fase2_hmm_features = (
    df_fase2_split
    .copy(deep=True)
    .sort_values("timestamp")
    .reset_index(drop=True)
)

df_fase2_hmm_features["timestamp"] = pd.to_datetime(
    df_fase2_hmm_features["timestamp"],
    errors="raise"
)

if not df_fase2_hmm_features["timestamp"].is_unique:
    raise ValueError("Timestamps duplicados.")

if len(df_fase2_hmm_features) != 8760:
    raise ValueError("Esperadas 8.760 horas.")

on_mask = (
    df_fase2_hmm_features[
        "operational_status_fase2"
    ].eq("ON")
)

df_on = df_fase2_hmm_features.loc[
    on_mask
].copy()

_project_print(f"\nHoras totais: {len(df_fase2_hmm_features):,}")
_project_print(f"Horas ON: {len(df_on):,}")
_project_print(
    "Ciclos ON:",
    df_on["cycle_id"].nunique()
)

# ------------------------------------------------------------
# 2. CATÁLOGO METODOLÓGICO DAS VARIÁVEIS
# ------------------------------------------------------------

feature_catalog = [
    {
        "feature": "load_current_mw",
        "descricao": "Nível de carga elétrica na hora t",
        "papel": "candidata",
        "disponibilidade": "t",
        "uso_hmm": True,
    },
    {
        "feature": "delta_load_1h",
        "descricao": "Variação da carga entre t e t-1",
        "papel": "candidata",
        "disponibilidade": "t",
        "uso_hmm": True,
    },
    {
        "feature": "abs_delta_load_1h",
        "descricao": "Magnitude absoluta da variação horária",
        "papel": "candidata",
        "disponibilidade": "t",
        "uso_hmm": True,
    },
    {
        "feature": "Operating Time",
        "descricao": "Fração da hora em operação",
        "papel": "diagnostico",
        "disponibilidade": "t",
        "uso_hmm": False,
    },
    {
        "feature": "CO2 Mass (short tons)",
        "descricao": "Emissões horárias observadas",
        "papel": "interpretacao_posterior",
        "disponibilidade": "t",
        "uso_hmm": False,
    },
]

audit07_feature_catalog = pd.DataFrame(
    feature_catalog
)

_project_print("\n1. CATÁLOGO DAS VARIÁVEIS")
_project_display(audit07_feature_catalog)

# ------------------------------------------------------------
# 3. QUALIDADE DAS VARIÁVEIS POR PARTIÇÃO
# ------------------------------------------------------------

candidate_features = [
    "load_current_mw",
    "delta_load_1h",
    "abs_delta_load_1h",
]

quality_rows = []

for partition in ["train", "validation", "test"]:

    part = df_on.loc[
        df_on["partition"].eq(partition)
    ]

    for feature in candidate_features:

        values = pd.to_numeric(
            part[feature],
            errors="coerce"
        )

        finite = values.replace(
            [np.inf, -np.inf],
            np.nan
        )

        quality_rows.append({
            "partition": partition,
            "feature": feature,
            "hours": int(len(part)),
            "missing": int(values.isna().sum()),
            "infinite": int(np.isinf(values).sum()),
            "mean": float(finite.mean()),
            "std": float(finite.std()),
            "min": float(finite.min()),
            "median": float(finite.median()),
            "p05": float(finite.quantile(0.05)),
            "p95": float(finite.quantile(0.95)),
            "max": float(finite.max()),
        })

audit07_feature_quality = pd.DataFrame(
    quality_rows
)

_project_print("\n2. QUALIDADE DAS VARIÁVEIS POR PARTIÇÃO")
_project_display(audit07_feature_quality)

# ------------------------------------------------------------
# 4. CONSISTÊNCIA DAS VARIAÇÕES DE CARGA
# ------------------------------------------------------------
#
# Verificamos se abs_delta_load_1h corresponde ao
# valor absoluto de delta_load_1h.
#
# A comparação não altera as variáveis.
# ------------------------------------------------------------

delta = pd.to_numeric(
    df_on["delta_load_1h"],
    errors="coerce"
)

abs_delta = pd.to_numeric(
    df_on["abs_delta_load_1h"],
    errors="coerce"
)

both_finite = (
    np.isfinite(delta)
    & np.isfinite(abs_delta)
)

delta_mismatch = (
    both_finite
    & ~np.isclose(
        delta.abs(),
        abs_delta,
        rtol=1e-7,
        atol=1e-8
    )
)

_project_print("\n3. CONSISTÊNCIA DAS RAMPAS")
_project_print(
    "Divergências entre |delta_load_1h| "
    "e abs_delta_load_1h:",
    int(delta_mismatch.sum())
)

# ------------------------------------------------------------
# 5. CORRELAÇÃO — SOMENTE TREINAMENTO
# ------------------------------------------------------------
#
# A correlação utilizada para orientar a análise é
# calculada somente sobre as horas ON de treinamento.
# ------------------------------------------------------------

train_on = df_on.loc[
    df_on["partition"].eq("train")
].copy()

audit07_feature_correlations = (
    train_on[candidate_features]
    .apply(pd.to_numeric, errors="coerce")
    .corr(method="pearson")
)

_project_print("\n4. CORRELAÇÃO ENTRE VARIÁVEIS — TREINO")
_project_display(audit07_feature_correlations)

# ------------------------------------------------------------
# 6. MODELOS CANDIDATOS
# ------------------------------------------------------------

HMM_CANDIDATE_FEATURES = {
    "M1_LOAD": [
        "load_current_mw",
    ],
    "M2_LOAD_DELTA": [
        "load_current_mw",
        "delta_load_1h",
    ],
    "M3_LOAD_ABSDELTA": [
        "load_current_mw",
        "abs_delta_load_1h",
    ],
}

model_rows = []

for model_name, features in HMM_CANDIDATE_FEATURES.items():

    x = (
        train_on[features]
        .apply(pd.to_numeric, errors="coerce")
    )

    valid_rows = np.isfinite(
        x.to_numpy(dtype=float)
    ).all(axis=1)

    model_rows.append({
        "model": model_name,
        "features": ", ".join(features),
        "n_features": len(features),
        "train_on_hours": len(train_on),
        "complete_train_hours": int(
            valid_rows.sum()
        ),
        "missing_train_hours": int(
            (~valid_rows).sum()
        ),
        "complete_pct": round(
            100 * valid_rows.mean(), 3
        ),
    })

audit07_candidate_models = pd.DataFrame(
    model_rows
)

_project_print("\n5. CONFIGURAÇÕES CANDIDATAS")
_project_display(audit07_candidate_models)

# ------------------------------------------------------------
# 7. VISUALIZAÇÕES — DISTRIBUIÇÕES
# ------------------------------------------------------------

fig, axes = plt.subplots(
    1, 3,
    figsize=(16, 4)
)

for ax, feature in zip(
    axes,
    candidate_features
):
    values = pd.to_numeric(
        train_on[feature],
        errors="coerce"
    ).replace(
        [np.inf, -np.inf],
        np.nan
    ).dropna()

    ax.hist(
        values,
        bins=45,
        alpha=0.85,
        edgecolor="white"
    )

    ax.set_title(feature)
    ax.set_xlabel("Valor")
    ax.set_ylabel("Frequência")
    ax.grid(alpha=0.2)

fig.suptitle(
    "Distribuições das variáveis candidatas — treinamento",
    y=1.04
)

plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# 8. VISUALIZAÇÃO — CARGA E RAMPA
# ------------------------------------------------------------

fig, ax = plt.subplots(
    figsize=(10, 5)
)

ax.scatter(
    train_on["load_current_mw"],
    train_on["delta_load_1h"],
    s=8,
    alpha=0.25
)

ax.axhline(
    0,
    color="black",
    linewidth=0.8
)

ax.set_title(
    "Relação entre carga e rampa — treinamento"
)
ax.set_xlabel("Carga atual (MW)")
ax.set_ylabel("Variação horária da carga (MW/h)")
ax.grid(alpha=0.2)

plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# 9. VISUALIZAÇÃO — CORRELAÇÕES
# ------------------------------------------------------------

corr = audit07_feature_correlations

fig, ax = plt.subplots(
    figsize=(6, 5)
)

im = ax.imshow(
    corr.to_numpy(),
    vmin=-1,
    vmax=1,
    cmap="coolwarm"
)

ax.set_xticks(
    range(len(corr.columns))
)

ax.set_yticks(
    range(len(corr.index))
)

ax.set_xticklabels(
    corr.columns,
    rotation=35,
    ha="right"
)

ax.set_yticklabels(
    corr.index
)

for i in range(len(corr.index)):
    for j in range(len(corr.columns)):
        ax.text(
            j, i,
            f"{corr.iloc[i, j]:.2f}",
            ha="center",
            va="center",
            fontsize=9
        )

fig.colorbar(
    im,
    ax=ax,
    label="Correlação de Pearson"
)

ax.set_title(
    "Correlação das variáveis — treinamento"
)

plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# 10. DIAGNÓSTICO DA DISPONIBILIDADE CAUSAL
# ------------------------------------------------------------
#
# Todas as variáveis candidatas devem ser calculáveis
# com dados disponíveis até a hora t.
#
# Não utilizamos:
# - CO2 futuro;
# - estado HMM futuro;
# - duração final do ciclo;
# - variantes completas de processos;
# - estatísticas calculadas sobre o teste.
# ------------------------------------------------------------

causal_audit = pd.DataFrame([
    {
        "feature": "load_current_mw",
        "available_at_t": True,
        "uses_future": False,
        "note": "Carga observada na hora t",
    },
    {
        "feature": "delta_load_1h",
        "available_at_t": True,
        "uses_future": False,
        "note": "Carga t menos carga t-1",
    },
    {
        "feature": "abs_delta_load_1h",
        "available_at_t": True,
        "uses_future": False,
        "note": "Magnitude da diferença t/t-1",
    },
])

_project_print("\n6. AUDITORIA DE DISPONIBILIDADE TEMPORAL")
_project_display(causal_audit)

# ------------------------------------------------------------
# 11. VERIFICAÇÕES FINAIS
# ------------------------------------------------------------

assert len(df_fase2_hmm_features) == 8760

assert (
    df_fase2_hmm_features["partition"]
    .equals(df_fase2_split["partition"])
)

assert (
    df_fase2_hmm_features["cycle_id"]
    .equals(df_fase2_split["cycle_id"])
)

assert all(
    "CO2" not in feature.upper()
    for features in HMM_CANDIDATE_FEATURES.values()
    for feature in features
)

assert (
    audit07_candidate_models[
        "complete_train_hours"
    ].gt(0).all()
)

# ------------------------------------------------------------
# 12. RELATÓRIO
# ------------------------------------------------------------

audit07_summary = {
    "cell": "07/36",
    "facility_id": "6065",
    "unit_id": "2",
    "year": 2025,
    "train_on_hours": int(len(train_on)),
    "candidate_features": candidate_features,
    "candidate_models": HMM_CANDIDATE_FEATURES,
    "co2_used_as_feature": False,
    "scaler_fitted": False,
    "hmm_trained": False,
    "partition_modified": False,
    "data_original_modified": False,
    "ramp_absolute_mismatches": int(
        delta_mismatch.sum()
    ),
}

if "RUN_DIR" in globals():
    report_dir = Path(RUN_DIR) / "reports"
elif "run_dir" in globals():
    report_dir = Path(run_dir) / "reports"
else:
    raise RuntimeError(
        "Diretório de execução não encontrado. "
        "Execute a Célula 01."
    )

report_dir.mkdir(
    parents=True,
    exist_ok=True
)

report_path = (
    report_dir
    / "cell07_hmm_feature_diagnostics.json"
)

with open(
    report_path,
    "w",
    encoding="utf-8"
) as f:
    _project_dump(
        audit07_summary,
        f,
        ensure_ascii=False,
        indent=2
    )

_project_print("\n7. VERIFICAÇÕES FINAIS")
_project_print("Partições preservadas: OK")
_project_print("Ciclos preservados: OK")
_project_print("CO2 excluído das entradas HMM: OK")
_project_print("Nenhum scaler ajustado: OK")
_project_print("Nenhum HMM treinado: OK")

_project_print(f"\nRelatório: {report_path}")

_project_print(
    "\nCÉLULA 07 CONCLUÍDA — "
    "variáveis candidatas diagnosticadas; "
    "configurações HMM registradas; "
    "nenhum modelo treinado."
)


In [ ]:

# ============================================================
# FASE 2 — CÉLULA 08/36
# PADRONIZAÇÃO E PREPARAÇÃO DAS SEQUÊNCIAS HMM
# ============================================================
#
# OBJETIVO
# Preparar as matrizes de observações para Gaussian HMM,
# com padronização ajustada exclusivamente no treinamento
# e sequências independentes por ciclo ON.
#
# ENTRADAS
# df_fase2_hmm_features — Célula 07
# HMM_CANDIDATE_FEATURES — Célula 07
#
# SAÍDAS
# hmm08_data
# hmm08_scalers
# audit08_sequences
# audit08_scalers
# audit08_startup_ramps
# audit08_summary
#
# RESTRIÇÕES
# - Não modificar os objetos das células anteriores.
# - Não alterar partições ou ciclos.
# - Não usar CO2 nas matrizes HMM.
# - Não ajustar scaler com validação ou teste.
# - Não treinar HMM.
# - Não selecionar K.
# - Não excluir observações ou ciclos.
# ============================================================

import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from IPython.display import display

_project_print("=" * 72)
_project_print("FASE 2 — CÉLULA 08/36 | PREPARAÇÃO DAS SEQUÊNCIAS HMM")
_project_print("=" * 72)

# ------------------------------------------------------------
# 1. PRÉ-REQUISITOS
# ------------------------------------------------------------

if "df_fase2_hmm_features" not in globals():
    raise RuntimeError(
        "Execute a Célula 07 antes desta célula."
    )

if "HMM_CANDIDATE_FEATURES" not in globals():
    raise RuntimeError(
        "Configurações candidatas não encontradas."
    )

required = [
    "timestamp",
    "partition",
    "cycle_id",
    "cycle_hour_index",
    "operational_status_fase2",
    "load_current_mw",
    "delta_load_1h",
    "abs_delta_load_1h",
]

missing = [
    c for c in required
    if c not in df_fase2_hmm_features.columns
]

if missing:
    raise ValueError(f"Colunas ausentes: {missing}")

df08 = (
    df_fase2_hmm_features
    .copy(deep=True)
    .sort_values("timestamp")
    .reset_index(drop=True)
)

df08["timestamp"] = pd.to_datetime(
    df08["timestamp"],
    errors="raise"
)

if len(df08) != 8760:
    raise ValueError("Esperadas 8.760 horas.")

if not df08["timestamp"].is_unique:
    raise ValueError("Timestamps duplicados.")

PARTITIONS = ["train", "validation", "test"]

on_mask = df08["operational_status_fase2"].eq("ON")

df08_on = df08.loc[on_mask].copy()

if df08_on["cycle_id"].isna().any():
    raise ValueError("Existem horas ON sem cycle_id.")

if not df08_on["partition"].isin(PARTITIONS).all():
    raise ValueError("Partição desconhecida.")

# ------------------------------------------------------------
# 2. AUDITORIA DAS SEQUÊNCIAS ON
# ------------------------------------------------------------
#
# Cada cycle_id deve:
# - pertencer a uma única partição;
# - ter timestamps horários consecutivos;
# - começar com cycle_hour_index = 1.
#
# Os ciclos são unidades independentes para o HMM.
# ------------------------------------------------------------

sequence_rows = []

for cycle_id, group in df08_on.groupby(
    "cycle_id", sort=False
):
    group = group.sort_values("timestamp")

    partitions = group["partition"].unique()

    if len(partitions) != 1:
        raise AssertionError(
            f"Ciclo dividido entre partições: {cycle_id}"
        )

    diffs = group["timestamp"].diff().dropna()

    if not diffs.eq(pd.Timedelta(hours=1)).all():
        raise AssertionError(
            f"Descontinuidade no ciclo {cycle_id}"
        )

    if int(group["cycle_hour_index"].iloc[0]) != 1:
        raise AssertionError(
            f"Índice inicial inválido: {cycle_id}"
        )

    if not np.array_equal(
        group["cycle_hour_index"].to_numpy(),
        np.arange(1, len(group) + 1)
    ):
        raise AssertionError(
            f"Índices horários inválidos: {cycle_id}"
        )

    sequence_rows.append({
        "cycle_id": str(cycle_id),
        "partition": str(partitions[0]),
        "start": group["timestamp"].iloc[0],
        "end": group["timestamp"].iloc[-1],
        "length": int(len(group)),
    })

audit08_sequences = (
    pd.DataFrame(sequence_rows)
    .sort_values("start")
    .reset_index(drop=True)
)

_project_print("\n1. SEQUÊNCIAS ON POR PARTIÇÃO")

sequence_summary = (
    audit08_sequences
    .groupby("partition")
    .agg(
        cycles=("cycle_id", "size"),
        on_hours=("length", "sum"),
        min_length=("length", "min"),
        median_length=("length", "median"),
        max_length=("length", "max"),
    )
    .reindex(PARTITIONS)
    .reset_index()
)

_project_display(sequence_summary)

# ------------------------------------------------------------
# 3. DIAGNÓSTICO DAS RAMPAS DE PARTIDA
# ------------------------------------------------------------
#
# delta_load_1h pode refletir a passagem OFF -> ON.
# Esse valor é conhecido na hora t e será preservado.
#
# O HMM, porém, reinicia sua sequência probabilística
# no início de cada ciclo ON.
# ------------------------------------------------------------

startup_rows = df08_on.loc[
    df08_on["cycle_hour_index"].eq(1),
    [
        "cycle_id",
        "partition",
        "timestamp",
        "load_current_mw",
        "delta_load_1h",
        "abs_delta_load_1h",
    ]
].copy()

startup_rows["previous_timestamp"] = (
    startup_rows["timestamp"]
    - pd.Timedelta(hours=1)
)

previous_lookup = (
    df08.set_index("timestamp")[
        [
            "operational_status_fase2",
            "load_current_mw",
        ]
    ]
    .rename(columns={
        "operational_status_fase2": "previous_status",
        "load_current_mw": "previous_load_mw",
    })
)

startup_rows = startup_rows.join(
    previous_lookup,
    on="previous_timestamp"
)

startup_rows["previous_off"] = (
    startup_rows["previous_status"].eq("OFF")
)

startup_rows["startup_delta_consistent"] = (
    np.isclose(
        startup_rows["delta_load_1h"],
        startup_rows["load_current_mw"]
        - startup_rows["previous_load_mw"],
        rtol=1e-7,
        atol=1e-8,
    )
)

audit08_startup_ramps = startup_rows.copy()

_project_print("\n2. RAMPAS NA PRIMEIRA HORA DOS CICLOS ON")
_project_display(
    audit08_startup_ramps.head(15)
)

_project_print(
    "\nPrimeiras horas ON:",
    len(audit08_startup_ramps)
)

_project_print(
    "Precedidas por OFF:",
    int(
        audit08_startup_ramps["previous_off"].sum()
    )
)

_project_print(
    "Rampas consistentes com a hora anterior:",
    int(
        audit08_startup_ramps[
            "startup_delta_consistent"
        ].sum()
    )
)

# ------------------------------------------------------------
# 4. ORGANIZAÇÃO DAS MATRIZES POR MODELO
# ------------------------------------------------------------
#
# Para cada configuração:
#
# 1. Ajustar StandardScaler somente com train.
# 2. Transformar validation e test sem refit.
# 3. Concatenar ciclos ON em ordem cronológica.
# 4. Registrar lengths para hmmlearn.
#
# O hmmlearn utilizará:
# model.fit(X_train, lengths=train_lengths)
#
# Não executar fit nesta célula.
# ------------------------------------------------------------

hmm08_data = {}
hmm08_scalers = {}

scaler_rows = []

for model_name, features in HMM_CANDIDATE_FEATURES.items():

    _project_print(f"\nPreparando {model_name}: {features}")

    features = list(features)

    if any("CO2" in f.upper() for f in features):
        raise ValueError(
            "CO2 não pode ser entrada do HMM."
        )

    # --------------------------------------------------------
    # 4.1. Ajuste do scaler no treinamento
    # --------------------------------------------------------

    train_frame = df08_on.loc[
        df08_on["partition"].eq("train")
    ]

    X_train_raw = (
        train_frame[features]
        .apply(pd.to_numeric, errors="coerce")
        .to_numpy(dtype=float)
    )

    if not np.isfinite(X_train_raw).all():
        raise ValueError(
            f"Valores não finitos no treino: {model_name}"
        )

    scaler = StandardScaler()
    scaler.fit(X_train_raw)

    hmm08_scalers[model_name] = scaler

    for feature, mean, scale in zip(
        features,
        scaler.mean_,
        scaler.scale_,
    ):
        scaler_rows.append({
            "model": model_name,
            "feature": feature,
            "train_mean": float(mean),
            "train_scale": float(scale),
            "fitted_on": "train_only",
        })

    # --------------------------------------------------------
    # 4.2. Matrizes padronizadas por partição
    # --------------------------------------------------------

    model_data = {
        "features": features,
        "scaler": scaler,
    }

    for partition in PARTITIONS:

        part = (
            df08_on.loc[
                df08_on["partition"].eq(partition)
            ]
            .sort_values("timestamp")
            .copy()
        )

        X_raw = (
            part[features]
            .apply(pd.to_numeric, errors="coerce")
            .to_numpy(dtype=float)
        )

        if not np.isfinite(X_raw).all():
            raise ValueError(
                f"Valores não finitos em "
                f"{model_name}/{partition}"
            )

        X_scaled = scaler.transform(X_raw)

        if not np.isfinite(X_scaled).all():
            raise ValueError(
                f"Padronização inválida: "
                f"{model_name}/{partition}"
            )

        lengths = (
            part.groupby(
                "cycle_id",
                sort=False
            )
            .size()
            .to_numpy(dtype=int)
        )

        cycle_ids = (
            part["cycle_id"]
            .drop_duplicates()
            .astype(str)
            .tolist()
        )

        if int(lengths.sum()) != len(X_scaled):
            raise AssertionError(
                f"Comprimentos incompatíveis: "
                f"{model_name}/{partition}"
            )

        if np.any(lengths <= 0):
            raise AssertionError(
                "Comprimentos inválidos."
            )

        model_data[partition] = {
            "X_raw": X_raw,
            "X": X_scaled,
            "lengths": lengths,
            "cycle_ids": cycle_ids,
            "timestamps": (
                part["timestamp"]
                .to_numpy()
            ),
            "row_indices": (
                part.index.to_numpy()
            ),
        }

        _project_print(
            f"  {partition:10s} | "
            f"X={X_scaled.shape} | "
            f"ciclos={len(lengths):2d} | "
            f"soma_lengths={lengths.sum()}"
        )

    hmm08_data[model_name] = model_data

audit08_scalers = pd.DataFrame(
    scaler_rows
)

_project_print("\n3. PARÂMETROS DOS SCALERS")
_project_display(audit08_scalers)

# ------------------------------------------------------------
# 5. AUDITORIA DE RECONCILIAÇÃO
# ------------------------------------------------------------

expected_hours = {
    "train": 4504,
    "validation": 675,
    "test": 1070,
}

expected_cycles = {
    "train": 27,
    "validation": 6,
    "test": 5,
}

reconciliation_rows = []

for model_name, model_data in hmm08_data.items():

    for partition in PARTITIONS:

        entry = model_data[partition]

        n_hours = int(entry["X"].shape[0])
        n_cycles = int(len(entry["lengths"]))

        if n_hours != expected_hours[partition]:
            raise AssertionError(
                f"Horas inesperadas: "
                f"{model_name}/{partition}"
            )

        if n_cycles != expected_cycles[partition]:
            raise AssertionError(
                f"Ciclos inesperados: "
                f"{model_name}/{partition}"
            )

        if entry["X"].shape[1] != len(
            model_data["features"]
        ):
            raise AssertionError(
                "Número de features incompatível."
            )

        reconciliation_rows.append({
            "model": model_name,
            "partition": partition,
            "hours": n_hours,
            "cycles": n_cycles,
            "n_features": len(
                model_data["features"]
            ),
            "finite": bool(
                np.isfinite(entry["X"]).all()
            ),
            "lengths_ok": bool(
                entry["lengths"].sum() == n_hours
            ),
        })

audit08_reconciliation = pd.DataFrame(
    reconciliation_rows
)

_project_print("\n4. RECONCILIAÇÃO DAS MATRIZES")
_project_display(audit08_reconciliation)

# ------------------------------------------------------------
# 6. VERIFICAÇÃO DOS PARÂMETROS DE TREINAMENTO
# ------------------------------------------------------------

_project_print("\n5. VERIFICAÇÃO DA PADRONIZAÇÃO")

for model_name, model_data in hmm08_data.items():

    X = model_data["train"]["X"]

    means = X.mean(axis=0)
    stds = X.std(axis=0, ddof=0)

    _project_print(f"\n{model_name}")
    _project_print(
        "Médias padronizadas:",
        np.round(means, 6)
    )
    _project_print(
        "Desvios-padrão:",
        np.round(stds, 6)
    )

    if not np.allclose(
        means,
        0,
        atol=1e-8
    ):
        raise AssertionError(
            f"Média não centralizada: {model_name}"
        )

    if not np.allclose(
        stds,
        1,
        atol=1e-8
    ):
        raise AssertionError(
            f"Desvio-padrão inválido: {model_name}"
        )

# ------------------------------------------------------------
# 7. VISUALIZAÇÃO — DISTRIBUIÇÕES PADRONIZADAS
# ------------------------------------------------------------

fig, axes = plt.subplots(
    1, 3,
    figsize=(16, 4)
)

for ax, model_name in zip(
    axes,
    HMM_CANDIDATE_FEATURES.keys()
):

    entry = hmm08_data[model_name]["train"]

    X = entry["X"]

    for j, feature in enumerate(
        hmm08_data[model_name]["features"]
    ):
        ax.hist(
            X[:, j],
            bins=45,
            alpha=0.5,
            label=feature
        )

    ax.set_title(model_name)
    ax.set_xlabel("Valor padronizado")
    ax.set_ylabel("Frequência")
    ax.legend(fontsize=8)
    ax.grid(alpha=0.2)

plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# 8. RELATÓRIO DE REPRODUTIBILIDADE
# ------------------------------------------------------------

audit08_summary = {
    "cell": "08/36",
    "facility_id": "6065",
    "unit_id": "2",
    "year": 2025,
    "models": list(
        HMM_CANDIDATE_FEATURES.keys()
    ),
    "features": HMM_CANDIDATE_FEATURES,
    "scaler": "StandardScaler",
    "scaler_fit_partition": "train",
    "sequences_defined_by": "cycle_id",
    "startup_ramps_preserved": True,
    "co2_used_as_feature": False,
    "hmm_trained": False,
    "k_selected": False,
    "reconciliation": (
        audit08_reconciliation
        .to_dict(orient="records")
    ),
    "scaler_parameters": (
        audit08_scalers
        .to_dict(orient="records")
    ),
}

if "RUN_DIR" in globals():
    report_dir = Path(RUN_DIR) / "reports"
elif "run_dir" in globals():
    report_dir = Path(run_dir) / "reports"
else:
    raise RuntimeError(
        "Diretório de execução não encontrado."
    )

report_dir.mkdir(
    parents=True,
    exist_ok=True
)

report_path = (
    report_dir
    / "cell08_hmm_sequences_scaling.json"
)

with open(
    report_path,
    "w",
    encoding="utf-8"
) as f:
    _project_dump(
        audit08_summary,
        f,
        ensure_ascii=False,
        indent=2
    )

_project_print("\n6. VERIFICAÇÕES FINAIS")
_project_print("Scalers ajustados somente no treinamento: OK")
_project_print("Sequências ON independentes: OK")
_project_print("Comprimentos reconciliados: OK")
_project_print("CO2 excluído das entradas HMM: OK")
_project_print("Nenhum HMM treinado: OK")

_project_print(f"\nRelatório: {report_path}")

_project_print(
    "\nCÉLULA 08 CONCLUÍDA — "
    "matrizes e sequências preparadas; "
    "scalers ajustados somente no treino; "
    "nenhum HMM treinado."
)


## 09–12 — Treino, estabilidade e escolha do HMM

O HMM gaussiano usa carga e variações de carga como emissões do modelo estatístico. A forma do HMM e os algoritmos de ajuste e decodificação seguem Rabiner. Moradbeikie et al., §§3.2, 4.3–4.5 e 5.1, dão um exemplo de sensores → HMM → processo em pasteurização sintética. A prioridade da família, os limiares, K, as sementes e os pesos do ranking pertencem a Iatan. AIC/BIC são diagnósticos dentro de uma família de observações. O uso de horas como tamanho amostral no BIC é uma convenção aproximada. A estabilidade compara estados entre sementes por ARI. [Rabiner (1989)](https://doi.org/10.1109/5.18626) · [Moradbeikie et al. (2025)](https://doi.org/10.3390/systems13110935) · [hmmlearn](https://hmmlearn.readthedocs.io/en/stable/api.html) · [ARI em scikit-learn](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.adjusted_rand_score.html)


In [ ]:

# ============================================================
# FASE 2 — CÉLULA 09/36
# TREINAMENTO DOS MODELOS GAUSSIAN HMM CANDIDATOS
# ============================================================
#
# OBJETIVO
# Treinar HMMs com:
#   - M1_LOAD
#   - M2_LOAD_DELTA
#   - M3_LOAD_ABSDELTA
#   - K = 2,...,8
#   - 5 sementes por configuração
#
# ENTRADA
# hmm08_data — Célula 08
#
# SAÍDAS
# hmm09_models
# audit09_training
# audit09_summary
#
# RESTRIÇÕES
# - Treinar exclusivamente com train.
# - Respeitar os comprimentos dos ciclos ON.
# - Não utilizar CO2.
# - Não utilizar validation ou test durante o ajuste.
# - Não selecionar K.
# - Não modificar os scalers.
# - Não eliminar ciclos curtos.
# ============================================================

import json
import time
import warnings

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display

# ------------------------------------------------------------
# 1. DEPENDÊNCIA
# ------------------------------------------------------------

try:
    from hmmlearn.hmm import GaussianHMM
except ImportError:
    %pip -q install hmmlearn
    from hmmlearn.hmm import GaussianHMM

print("=" * 72)
print("FASE 2 — CÉLULA 09/36 | TREINAMENTO GAUSSIAN HMM")
print("=" * 72)

# ------------------------------------------------------------
# 2. PRÉ-REQUISITOS
# ------------------------------------------------------------

if "hmm08_data" not in globals():
    raise RuntimeError(
        "Execute a Célula 08 antes desta célula."
    )

MODEL_NAMES = [
    "M1_LOAD",
    "M2_LOAD_DELTA",
    "M3_LOAD_ABSDELTA",
]

for name in MODEL_NAMES:
    if name not in hmm08_data:
        raise ValueError(
            f"Configuração ausente: {name}"
        )

K_VALUES = list(range(2, 9))

SEEDS = [11, 23, 42, 67, 101]

N_ITER = 300
TOL = 1e-3
MIN_COVAR = 1e-3
COVARIANCE_TYPE = "diag"

# Limiar exclusivamente diagnóstico.
MIN_STATE_OCCUPANCY = 0.005

EXPECTED_FITS = (
    len(MODEL_NAMES)
    * len(K_VALUES)
    * len(SEEDS)
)

print("\nCONFIGURAÇÃO EXPERIMENTAL")
print("Modelos:", MODEL_NAMES)
print("K:", K_VALUES)
print("Sementes:", SEEDS)
print("Ajustes previstos:", EXPECTED_FITS)
print("Máximo de iterações:", N_ITER)
print("Covariância:", COVARIANCE_TYPE)

# ------------------------------------------------------------
# 3. VALIDAÇÃO DAS MATRIZES DE TREINAMENTO
# ------------------------------------------------------------

for name in MODEL_NAMES:

    entry = hmm08_data[name]["train"]

    X = np.asarray(
        entry["X"],
        dtype=float
    )

    lengths = np.asarray(
        entry["lengths"],
        dtype=int
    )

    if X.ndim != 2:
        raise ValueError(
            f"Matriz inválida: {name}"
        )

    if not np.isfinite(X).all():
        raise ValueError(
            f"Valores não finitos: {name}"
        )

    if len(X) != int(lengths.sum()):
        raise ValueError(
            f"Comprimentos incompatíveis: {name}"
        )

    if np.any(lengths <= 0):
        raise ValueError(
            f"Sequência vazia: {name}"
        )

    if len(X) != 4504:
        raise ValueError(
            f"Horas de treino inesperadas: {name}"
        )

    if len(lengths) != 27:
        raise ValueError(
            f"Número de ciclos inesperado: {name}"
        )

print("\nMatrizes de treinamento verificadas.")

# ------------------------------------------------------------
# 4. FUNÇÕES DE DIAGNÓSTICO
# ------------------------------------------------------------

def hmm09_parameter_count(k, d):
    """
    Número de parâmetros livres de GaussianHMM
    com covariância diagonal.

    Probabilidades iniciais: k - 1
    Transições: k * (k - 1)
    Médias: k * d
    Variâncias diagonais: k * d
    """
    return (
        (k - 1)
        + k * (k - 1)
        + 2 * k * d
    )


def hmm09_information_criteria(loglik, n_params, n_obs):
    """
    AIC e BIC calculados sobre o conjunto de treino.
    BIC utiliza número de observações horárias como
    aproximação convencional; a dependência temporal
    do HMM deve ser considerada na interpretação.
    """
    aic = 2 * n_params - 2 * loglik
    bic = (
        np.log(n_obs) * n_params
        - 2 * loglik
    )
    return float(aic), float(bic)


def hmm09_diagnostics(model, X, lengths, k):
    """
    Diagnóstico retrospectivo, somente em treino.
    Os estados Viterbi aqui não são features causais.
    """
    states = model.predict(
        X,
        lengths=lengths
    )

    counts = np.bincount(
        states,
        minlength=k
    )

    occupancy = counts / len(states)

    transmat = np.asarray(
        model.transmat_,
        dtype=float
    )

    covars = np.asarray(
        model.covars_,
        dtype=float
    )

    finite_parameters = all([
        np.isfinite(model.startprob_).all(),
        np.isfinite(transmat).all(),
        np.isfinite(model.means_).all(),
        np.isfinite(covars).all(),
    ])

    valid_startprob = np.allclose(
        model.startprob_.sum(),
        1.0,
        atol=1e-6
    )

    valid_transitions = np.allclose(
        transmat.sum(axis=1),
        1.0,
        atol=1e-6
    )

    min_occupancy = float(
        occupancy.min()
    )

    n_rare_states = int(
        (occupancy < MIN_STATE_OCCUPANCY).sum()
    )

    return {
        "min_occupancy": min_occupancy,
        "max_occupancy": float(
            occupancy.max()
        ),
        "rare_states": n_rare_states,
        "empty_states": int(
            (counts == 0).sum()
        ),
        "min_transition_probability": float(
            transmat.min()
        ),
        "max_transition_probability": float(
            transmat.max()
        ),
        "min_covariance": float(
            covars.min()
        ),
        "finite_parameters": bool(
            finite_parameters
        ),
        "valid_startprob": bool(
            valid_startprob
        ),
        "valid_transitions": bool(
            valid_transitions
        ),
    }

# ------------------------------------------------------------
# 5. TREINAMENTO DOS 105 CANDIDATOS
# ------------------------------------------------------------
#
# Cada combinação é treinada do zero.
# Nenhum modelo previamente ajustado é carregado.
#
# Modelos válidos ficam disponíveis em hmm09_models.
# Falhas são registradas, não ocultadas.
# ------------------------------------------------------------

hmm09_models = {}
training_rows = []

total_start = time.perf_counter()
fit_counter = 0

for model_name in MODEL_NAMES:

    entry = hmm08_data[model_name]["train"]

    X_train = np.asarray(
        entry["X"],
        dtype=float
    )

    lengths_train = np.asarray(
        entry["lengths"],
        dtype=int
    )

    n_obs, n_features = X_train.shape

    print("\n" + "-" * 72)
    print(
        f"CONFIGURAÇÃO: {model_name} "
        f"| Features: {n_features}"
    )
    print("-" * 72)

    for k in K_VALUES:

        for seed in SEEDS:

            fit_counter += 1

            key = (
                model_name,
                int(k),
                int(seed)
            )

            row = {
                "model": model_name,
                "k": int(k),
                "seed": int(seed),
                "n_obs": int(n_obs),
                "n_features": int(n_features),
                "n_sequences": int(
                    len(lengths_train)
                ),
                "success": False,
                "converged": False,
                "iterations": np.nan,
                "loglik_train": np.nan,
                "loglik_per_hour": np.nan,
                "aic_train": np.nan,
                "bic_train": np.nan,
                "n_parameters": np.nan,
                "min_occupancy": np.nan,
                "max_occupancy": np.nan,
                "rare_states": np.nan,
                "empty_states": np.nan,
                "min_transition_probability": np.nan,
                "max_transition_probability": np.nan,
                "min_covariance": np.nan,
                "finite_parameters": False,
                "valid_startprob": False,
                "valid_transitions": False,
                "runtime_seconds": np.nan,
                "warnings": "",
                "error": "",
            }

            start = time.perf_counter()

            try:

                hmm = GaussianHMM(
                    n_components=k,
                    covariance_type=COVARIANCE_TYPE,
                    n_iter=N_ITER,
                    tol=TOL,
                    min_covar=MIN_COVAR,
                    random_state=seed,
                    implementation="log",
                    verbose=False,
                )

                with warnings.catch_warnings(
                    record=True
                ) as caught:

                    warnings.simplefilter("always")

                    hmm.fit(
                        X_train,
                        lengths=lengths_train
                    )

                    loglik = float(
                        hmm.score(
                            X_train,
                            lengths=lengths_train
                        )
                    )

                    diagnostics = hmm09_diagnostics(
                        hmm,
                        X_train,
                        lengths_train,
                        k
                    )

                n_parameters = hmm09_parameter_count(
                    k,
                    n_features
                )

                aic, bic = hmm09_information_criteria(
                    loglik,
                    n_parameters,
                    n_obs
                )

                converged = bool(
                    hmm.monitor_.converged
                )

                iterations = int(
                    hmm.monitor_.iter
                )

                warning_text = " | ".join(
                    str(w.message)
                    for w in caught
                )

                row.update({
                    "success": True,
                    "converged": converged,
                    "iterations": iterations,
                    "loglik_train": loglik,
                    "loglik_per_hour": (
                        loglik / n_obs
                    ),
                    "aic_train": aic,
                    "bic_train": bic,
                    "n_parameters": n_parameters,
                    "warnings": warning_text,
                    **diagnostics,
                })

                hmm09_models[key] = hmm

            except Exception as exc:

                row["error"] = (
                    f"{type(exc).__name__}: {exc}"
                )

            row["runtime_seconds"] = float(
                time.perf_counter() - start
            )

            training_rows.append(row)

            status = (
                "OK"
                if row["success"]
                else "FALHA"
            )

            print(
                f"[{fit_counter:03d}/{EXPECTED_FITS}] "
                f"{model_name:18s} "
                f"K={k} seed={seed:3d} "
                f"{status:5s} "
                f"LL={row['loglik_train']:.2f} "
                f"conv={row['converged']} "
                f"t={row['runtime_seconds']:.1f}s"
            )

audit09_training = pd.DataFrame(
    training_rows
)

total_runtime = (
    time.perf_counter() - total_start
)

# ------------------------------------------------------------
# 6. VERIFICAÇÕES DOS RESULTADOS
# ------------------------------------------------------------

if len(audit09_training) != EXPECTED_FITS:
    raise AssertionError(
        "Quantidade de tentativas incompatível."
    )

if audit09_training[
    ["model", "k", "seed"]
].duplicated().any():
    raise AssertionError(
        "Combinações repetidas."
    )

successful = audit09_training.loc[
    audit09_training["success"]
].copy()

failed = audit09_training.loc[
    ~audit09_training["success"]
].copy()

if successful.empty:
    raise RuntimeError(
        "Nenhum modelo HMM foi treinado com sucesso."
    )

# ------------------------------------------------------------
# 7. RESUMO POR CONFIGURAÇÃO E K
# ------------------------------------------------------------

audit09_by_k = (
    audit09_training
    .groupby(["model", "k"])
    .agg(
        attempted=("seed", "size"),
        successful=("success", "sum"),
        converged=("converged", "sum"),
        mean_loglik=("loglik_train", "mean"),
        best_loglik=("loglik_train", "max"),
        mean_aic=("aic_train", "mean"),
        mean_bic=("bic_train", "mean"),
        min_occupancy=("min_occupancy", "min"),
        max_empty_states=("empty_states", "max"),
        mean_runtime_seconds=(
            "runtime_seconds",
            "mean"
        ),
    )
    .reset_index()
)

print("\n1. RESUMO DOS TREINAMENTOS POR K")
display(audit09_by_k)

# ------------------------------------------------------------
# 8. DIAGNÓSTICO DE CONVERGÊNCIA E FALHAS
# ------------------------------------------------------------

convergence_summary = (
    audit09_training
    .groupby("model")
    .agg(
        attempts=("seed", "size"),
        successful=("success", "sum"),
        converged=("converged", "sum"),
        failures=(
            "success",
            lambda x: int((~x).sum())
        ),
        mean_runtime_seconds=(
            "runtime_seconds",
            "mean"
        ),
    )
    .reset_index()
)

print("\n2. CONVERGÊNCIA POR CONFIGURAÇÃO")
display(convergence_summary)

if not failed.empty:

    print("\n3. FALHAS REGISTRADAS")

    display(
        failed[
            [
                "model",
                "k",
                "seed",
                "error",
                "warnings",
            ]
        ]
    )

# ------------------------------------------------------------
# 9. VISUALIZAÇÕES DIAGNÓSTICAS
# ------------------------------------------------------------
#
# Os gráficos são exploratórios.
# Nenhuma seleção de K será feita nesta célula.
# ------------------------------------------------------------

fig, axes = plt.subplots(
    1, 3,
    figsize=(17, 4)
)

for ax, model_name in zip(
    axes,
    MODEL_NAMES
):

    subset = audit09_training.loc[
        audit09_training["model"].eq(model_name)
        & audit09_training["success"]
    ]

    for seed in SEEDS:

        seed_data = subset.loc[
            subset["seed"].eq(seed)
        ].sort_values("k")

        ax.plot(
            seed_data["k"],
            seed_data["loglik_per_hour"],
            marker="o",
            alpha=0.7,
            label=f"Seed {seed}"
        )

    ax.set_title(model_name)
    ax.set_xlabel("Número de estados K")
    ax.set_ylabel("Log-verossimilhança/hora")
    ax.grid(alpha=0.2)

axes[-1].legend(
    fontsize=8,
    loc="best"
)

fig.suptitle(
    "Diagnóstico de ajuste HMM — somente treinamento",
    y=1.04
)

plt.tight_layout()
plt.show()

# Convergência por número de estados.
convergence_plot = (
    audit09_training
    .groupby("k")["converged"]
    .mean()
    .mul(100)
)

fig, ax = plt.subplots(
    figsize=(8, 4)
)

ax.bar(
    convergence_plot.index.astype(str),
    convergence_plot.values
)

ax.set_title(
    "Proporção de ajustes convergidos por K"
)
ax.set_xlabel("Número de estados K")
ax.set_ylabel("Convergência (%)")
ax.set_ylim(0, 105)
ax.grid(axis="y", alpha=0.2)

plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# 10. REGISTRO DOS RESULTADOS
# ------------------------------------------------------------
#
# Não exportamos modelos definitivos nesta etapa.
# Mantemos os objetos em memória para comparação
# e seleção nas próximas células.
# ------------------------------------------------------------

audit09_summary = {
    "cell": "09/36",
    "facility_id": "6065",
    "unit_id": "2",
    "year": 2025,
    "model_names": MODEL_NAMES,
    "k_values": K_VALUES,
    "seeds": SEEDS,
    "n_iter": N_ITER,
    "tol": TOL,
    "min_covar": MIN_COVAR,
    "covariance_type": COVARIANCE_TYPE,
    "expected_fits": int(EXPECTED_FITS),
    "attempted_fits": int(
        len(audit09_training)
    ),
    "successful_fits": int(
        audit09_training["success"].sum()
    ),
    "converged_fits": int(
        audit09_training["converged"].sum()
    ),
    "failed_fits": int(
        (~audit09_training["success"]).sum()
    ),
    "total_runtime_seconds": float(
        total_runtime
    ),
    "fit_partition": "train",
    "validation_used": False,
    "test_used": False,
    "co2_used": False,
    "k_selected": False,
}

if "RUN_DIR" in globals():
    report_dir = Path(RUN_DIR) / "reports"
elif "run_dir" in globals():
    report_dir = Path(run_dir) / "reports"
else:
    raise RuntimeError(
        "Diretório de execução não encontrado."
    )

report_dir.mkdir(
    parents=True,
    exist_ok=True
)

training_path = (
    report_dir / "cell09_hmm_training_results.csv"
)

summary_path = (
    report_dir / "cell09_hmm_training_summary.json"
)

audit09_training.to_csv(
    training_path,
    index=False
)

with open(
    summary_path,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        audit09_summary,
        f,
        ensure_ascii=False,
        indent=2
    )

print("\n4. VERIFICAÇÕES FINAIS")
print("Tentativas registradas:", EXPECTED_FITS)
print(
    "Ajustes bem-sucedidos:",
    audit09_summary["successful_fits"]
)
print(
    "Convergências:",
    audit09_summary["converged_fits"]
)
print(
    "Falhas:",
    audit09_summary["failed_fits"]
)
print(
    "Tempo total:",
    round(total_runtime, 1),
    "segundos"
)

print(f"\nResultados: {training_path}")
print(f"Relatório: {summary_path}")

print(
    "\nCÉLULA 09 CONCLUÍDA — "
    "HMMs candidatos treinados exclusivamente no treino; "
    "nenhum K selecionado; "
    "validação e teste não utilizados."
)


In [ ]:

# ============================================================
# FASE 2 — CÉLULA 10/36
# COMPARAÇÃO DOS MODELOS HMM E DO NÚMERO DE ESTADOS
# ============================================================
#
# OBJETIVO
# Comparar os HMMs treinados na Célula 09 usando:
#   1. AIC/BIC de treinamento;
#   2. log-verossimilhança na validação;
#   3. ocupação dos estados;
#   4. degeneração numérica;
#   5. dispersão entre sementes.
#
# ENTRADAS
#   hmm08_data
#   hmm09_models
#   audit09_training
#
# SAÍDAS
#   audit10_candidates
#   audit10_by_k
#   audit10_best_by_configuration
#   audit10_summary
#
# RESTRIÇÕES
#   - Não treinar novos HMMs.
#   - Não utilizar o conjunto de teste.
#   - Não selecionar o modelo definitivo.
#   - Não modificar os modelos ou scalers.
#   - Não comparar diretamente LL de dimensões diferentes.
# ============================================================

import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display

_project_print("=" * 72)
_project_print("FASE 2 — CÉLULA 10/36 | COMPARAÇÃO DOS MODELOS HMM")
_project_print("=" * 72)

# ------------------------------------------------------------
# 1. PRÉ-REQUISITOS
# ------------------------------------------------------------

required_objects = [
    "hmm08_data",
    "hmm09_models",
    "audit09_training",
]

missing_objects = [
    name for name in required_objects
    if name not in globals()
]

if missing_objects:
    raise RuntimeError(
        "Execute as Células 08 e 09. "
        f"Objetos ausentes: {missing_objects}"
    )

MODEL_NAMES_10 = [
    "M1_LOAD",
    "M2_LOAD_DELTA",
    "M3_LOAD_ABSDELTA",
]

K_VALUES_10 = list(range(2, 9))
SEEDS_10 = [11, 23, 42, 67, 101]

MIN_OCCUPANCY_10 = 0.005

# Limiar diagnóstico, não regra de exclusão automática.
COVARIANCE_WARNING_10 = 1e-6

# ------------------------------------------------------------
# 2. VERIFICAÇÃO DO INVENTÁRIO
# ------------------------------------------------------------

audit10_input = audit09_training.copy(deep=True)

expected_combinations = {
    (m, k, s)
    for m in MODEL_NAMES_10
    for k in K_VALUES_10
    for s in SEEDS_10
}

observed_combinations = set(
    zip(
        audit10_input["model"],
        audit10_input["k"],
        audit10_input["seed"],
    )
)

if observed_combinations != expected_combinations:
    raise AssertionError(
        "Inventário da Célula 09 incompatível."
    )

if len(audit10_input) != 105:
    raise AssertionError(
        "Esperadas 105 tentativas de treinamento."
    )

_project_print("\nInventário confirmado: 105 ajustes registrados.")

# ------------------------------------------------------------
# 3. AVALIAÇÃO NA VALIDAÇÃO
# ------------------------------------------------------------
#
# A validação é avaliada usando:
#
#   model.score(X_validation, lengths_validation)
#
# Não há fit nem atualização de parâmetros.
#
# A log-verossimilhança por hora permite comparar
# diferentes K dentro da mesma configuração.
# ------------------------------------------------------------

candidate_rows = []

for row in audit10_input.itertuples(index=False):

    model_name = row.model
    k = int(row.k)
    seed = int(row.seed)

    key = (model_name, k, seed)

    result = row._asdict()

    result.update({
        "loglik_validation": np.nan,
        "loglik_validation_per_hour": np.nan,
        "validation_success": False,
        "validation_error": "",
        "covariance_warning": False,
        "occupancy_warning": False,
        "parameter_warning": False,
        "optimization_decrease_warning": False,
        "last_em_improvement": np.nan,
    })

    if not bool(row.success):
        result["validation_error"] = (
            "Ajuste não concluído na Célula 09."
        )
        candidate_rows.append(result)
        continue

    if key not in hmm09_models:
        result["validation_error"] = (
            "Modelo treinado não encontrado em memória."
        )
        candidate_rows.append(result)
        continue

    model = hmm09_models[key]

    entry = hmm08_data[model_name]["validation"]

    X_val = np.asarray(
        entry["X"],
        dtype=float
    )

    lengths_val = np.asarray(
        entry["lengths"],
        dtype=int
    )

    if X_val.ndim != 2:
        raise ValueError(
            f"Matriz de validação inválida: {model_name}"
        )

    if len(X_val) != lengths_val.sum():
        raise AssertionError(
            f"Comprimentos inválidos: {model_name}"
        )

    if not np.isfinite(X_val).all():
        raise ValueError(
            f"Validação contém valores não finitos: {model_name}"
        )

    try:

        with warnings.catch_warnings():
            warnings.simplefilter("ignore")

            ll_val = float(
                model.score(
                    X_val,
                    lengths=lengths_val
                )
            )

        if not np.isfinite(ll_val):
            raise ValueError(
                "Log-verossimilhança não finita."
            )

        result["loglik_validation"] = ll_val

        result["loglik_validation_per_hour"] = (
            ll_val / len(X_val)
        )

        result["validation_success"] = True

    except Exception as exc:

        result["validation_error"] = (
            f"{type(exc).__name__}: {exc}"
        )

    # --------------------------------------------------------
    # 3.1. Diagnósticos dos parâmetros treinados
    # --------------------------------------------------------

    covariance_values = np.asarray(
        model.covars_,
        dtype=float
    )

    result["covariance_warning"] = bool(
        np.any(
            covariance_values
            <= COVARIANCE_WARNING_10
        )
    )

    result["occupancy_warning"] = bool(
        result["min_occupancy"]
        < MIN_OCCUPANCY_10
    )

    result["parameter_warning"] = bool(
        not (
            result["finite_parameters"]
            and result["valid_startprob"]
            and result["valid_transitions"]
        )
    )

    # --------------------------------------------------------
    # 3.2. Histórico da otimização
    # --------------------------------------------------------
    #
    # O hmmlearn pode indicar converged=True mesmo após
    # uma pequena redução na função objetivo.
    #
    # Verificamos o histórico armazenado pelo monitor.
    # --------------------------------------------------------

    history = np.asarray(
        list(model.monitor_.history),
        dtype=float
    )

    if len(history) >= 2:

        improvements = np.diff(history)

        result["last_em_improvement"] = float(
            improvements[-1]
        )

        result["optimization_decrease_warning"] = bool(
            np.any(improvements < -1e-8)
        )

    candidate_rows.append(result)

audit10_candidates = pd.DataFrame(
    candidate_rows
)

_project_print("\n1. AVALIAÇÃO INDIVIDUAL DOS 105 AJUSTES")

_project_display(
    audit10_candidates[
        [
            "model",
            "k",
            "seed",
            "loglik_train",
            "loglik_validation",
            "loglik_validation_per_hour",
            "bic_train",
            "min_occupancy",
            "empty_states",
            "optimization_decrease_warning",
            "validation_success",
        ]
    ].head(20)
)

# ------------------------------------------------------------
# 4. RESUMO POR CONFIGURAÇÃO E K
# ------------------------------------------------------------

audit10_by_k = (
    audit10_candidates
    .groupby(["model", "k"])
    .agg(
        attempts=("seed", "size"),
        validation_successes=(
            "validation_success",
            "sum"
        ),
        mean_val_ll_hour=(
            "loglik_validation_per_hour",
            "mean"
        ),
        median_val_ll_hour=(
            "loglik_validation_per_hour",
            "median"
        ),
        std_val_ll_hour=(
            "loglik_validation_per_hour",
            "std"
        ),
        best_val_ll_hour=(
            "loglik_validation_per_hour",
            "max"
        ),
        mean_bic_train=(
            "bic_train",
            "mean"
        ),
        best_bic_train=(
            "bic_train",
            "min"
        ),
        mean_aic_train=(
            "aic_train",
            "mean"
        ),
        min_occupancy=(
            "min_occupancy",
            "min"
        ),
        max_empty_states=(
            "empty_states",
            "max"
        ),
        covariance_warnings=(
            "covariance_warning",
            "sum"
        ),
        occupancy_warnings=(
            "occupancy_warning",
            "sum"
        ),
        optimization_warnings=(
            "optimization_decrease_warning",
            "sum"
        ),
        parameter_warnings=(
            "parameter_warning",
            "sum"
        ),
    )
    .reset_index()
)

_project_print("\n2. COMPARAÇÃO CONSOLIDADA POR K")

_project_display(
    audit10_by_k.round(4)
)

# ------------------------------------------------------------
# 5. DESTAQUES EXPLORATÓRIOS POR CONFIGURAÇÃO
# ------------------------------------------------------------
#
# Apenas para inspeção.
# A seleção formal ocorrerá na Célula 12.
#
# A melhor LL de validação isoladamente não define
# a configuração vencedora.
# ------------------------------------------------------------

valid_candidates = audit10_candidates.loc[
    audit10_candidates["validation_success"]
].copy()

if valid_candidates.empty:
    raise RuntimeError(
        "Nenhum HMM pôde ser avaliado na validação."
    )

audit10_best_by_configuration = (
    valid_candidates
    .sort_values(
        [
            "model",
            "loglik_validation_per_hour",
        ],
        ascending=[True, False]
    )
    .groupby("model", as_index=False)
    .head(1)
    .reset_index(drop=True)
)

_project_print("\n3. MELHOR LL DE VALIDAÇÃO EM CADA CONFIGURAÇÃO")
_project_print("Destaque exploratório — não é seleção definitiva.")

_project_display(
    audit10_best_by_configuration[
        [
            "model",
            "k",
            "seed",
            "loglik_validation_per_hour",
            "bic_train",
            "min_occupancy",
            "optimization_decrease_warning",
        ]
    ]
)

# ------------------------------------------------------------
# 6. GRÁFICO — LL DE VALIDAÇÃO POR K
# ------------------------------------------------------------

fig, axes = plt.subplots(
    1, 3,
    figsize=(17, 4.5)
)

for ax, model_name in zip(
    axes,
    MODEL_NAMES_10
):

    subset = audit10_by_k.loc[
        audit10_by_k["model"].eq(model_name)
    ].sort_values("k")

    ax.errorbar(
        subset["k"],
        subset["mean_val_ll_hour"],
        yerr=subset["std_val_ll_hour"],
        marker="o",
        capsize=4,
        label="Média ± DP"
    )

    ax.plot(
        subset["k"],
        subset["best_val_ll_hour"],
        linestyle="--",
        marker="s",
        alpha=0.7,
        label="Melhor semente"
    )

    ax.set_title(model_name)
    ax.set_xlabel("Número de estados K")
    ax.set_ylabel(
        "Log-verossimilhança/hora — validação"
    )
    ax.set_xticks(K_VALUES_10)
    ax.grid(alpha=0.25)
    ax.legend(fontsize=8)

fig.suptitle(
    "Desempenho na validação — comparação dentro de cada configuração",
    y=1.04
)

plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# 7. GRÁFICO — AIC E BIC DE TREINAMENTO
# ------------------------------------------------------------

fig, axes = plt.subplots(
    1, 3,
    figsize=(17, 4.5)
)

for ax, model_name in zip(
    axes,
    MODEL_NAMES_10
):

    subset = audit10_by_k.loc[
        audit10_by_k["model"].eq(model_name)
    ].sort_values("k")

    ax.plot(
        subset["k"],
        subset["best_bic_train"],
        marker="o",
        label="Menor BIC"
    )

    ax.plot(
        subset["k"],
        subset["mean_bic_train"],
        marker="s",
        linestyle="--",
        label="BIC médio"
    )

    ax.plot(
        subset["k"],
        subset["mean_aic_train"],
        marker="^",
        linestyle=":",
        label="AIC médio"
    )

    ax.set_title(model_name)
    ax.set_xlabel("Número de estados K")
    ax.set_ylabel("Critério de informação")
    ax.set_xticks(K_VALUES_10)
    ax.grid(alpha=0.25)
    ax.legend(fontsize=8)

fig.suptitle(
    "Critérios de informação — treinamento",
    y=1.04
)

plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# 8. GRÁFICO — OCUPAÇÃO MÍNIMA DOS ESTADOS
# ------------------------------------------------------------

fig, axes = plt.subplots(
    1, 3,
    figsize=(17, 4.5)
)

for ax, model_name in zip(
    axes,
    MODEL_NAMES_10
):

    subset = audit10_by_k.loc[
        audit10_by_k["model"].eq(model_name)
    ].sort_values("k")

    ax.plot(
        subset["k"],
        100 * subset["min_occupancy"],
        marker="o"
    )

    ax.axhline(
        100 * MIN_OCCUPANCY_10,
        linestyle="--",
        color="red",
        label="Limiar diagnóstico"
    )

    ax.set_title(model_name)
    ax.set_xlabel("Número de estados K")
    ax.set_ylabel(
        "Menor ocupação observada (%)"
    )
    ax.set_xticks(K_VALUES_10)
    ax.grid(alpha=0.25)
    ax.legend(fontsize=8)

fig.suptitle(
    "Diagnóstico de ocupação dos estados — treinamento",
    y=1.04
)

plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# 9. GRÁFICO — DISPERSÃO ENTRE SEMENTES
# ------------------------------------------------------------

fig, axes = plt.subplots(
    1, 3,
    figsize=(17, 4.5)
)

for ax, model_name in zip(
    axes,
    MODEL_NAMES_10
):

    subset = audit10_candidates.loc[
        audit10_candidates["model"].eq(model_name)
        & audit10_candidates["validation_success"]
    ]

    values = [
        subset.loc[
            subset["k"].eq(k),
            "loglik_validation_per_hour"
        ].dropna().to_numpy()
        for k in K_VALUES_10
    ]

    ax.boxplot(
        values,
        tick_labels=[
            str(k) for k in K_VALUES_10
        ],
        showmeans=True
    )

    ax.set_title(model_name)
    ax.set_xlabel("Número de estados K")
    ax.set_ylabel(
        "Log-verossimilhança/hora — validação"
    )
    ax.grid(axis="y", alpha=0.25)

fig.suptitle(
    "Sensibilidade à inicialização — validação",
    y=1.04
)

plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# 10. RESUMO DOS ALERTAS
# ------------------------------------------------------------

warning_columns = [
    "covariance_warning",
    "occupancy_warning",
    "parameter_warning",
    "optimization_decrease_warning",
]

audit10_warning_summary = pd.DataFrame({
    "diagnostic": warning_columns,
    "affected_fits": [
        int(
            audit10_candidates[column].sum()
        )
        for column in warning_columns
    ],
})

_project_print("\n4. ALERTAS DIAGNÓSTICOS")
_project_display(audit10_warning_summary)

# ------------------------------------------------------------
# 11. EXPORTAÇÃO DOS RELATÓRIOS
# ------------------------------------------------------------

if "RUN_DIR" in globals():
    report_dir = Path(RUN_DIR) / "reports"
elif "run_dir" in globals():
    report_dir = Path(run_dir) / "reports"
else:
    raise RuntimeError(
        "Diretório da execução não encontrado."
    )

report_dir.mkdir(
    parents=True,
    exist_ok=True
)

candidates_path = (
    report_dir / "cell10_hmm_candidates.csv"
)

by_k_path = (
    report_dir / "cell10_hmm_comparison_by_k.csv"
)

summary_path = (
    report_dir / "cell10_hmm_comparison_summary.json"
)

_project_csv(audit10_candidates, 
    candidates_path,
    index=False
)

_project_csv(audit10_by_k, 
    by_k_path,
    index=False
)

audit10_summary = {
    "cell": "10/36",
    "facility_id": "6065",
    "unit_id": "2",
    "year": 2025,
    "candidate_fits": int(
        len(audit10_candidates)
    ),
    "validation_successes": int(
        audit10_candidates[
            "validation_success"
        ].sum()
    ),
    "model_configurations": MODEL_NAMES_10,
    "k_values": K_VALUES_10,
    "validation_used": True,
    "test_used": False,
    "new_models_trained": False,
    "final_model_selected": False,
    "diagnostic_warnings": {
        column: int(
            audit10_candidates[column].sum()
        )
        for column in warning_columns
    },
    "notes": [
        "LL, AIC e BIC não são diretamente comparáveis "
        "entre diferentes conjuntos de variáveis.",
        "O BIC utiliza horas como tamanho amostral "
        "convencional para sequências HMM.",
        "A escolha final requer análise de estabilidade "
        "e interpretação operacional.",
    ],
}

with open(
    summary_path,
    "w",
    encoding="utf-8"
) as f:
    _project_dump(
        audit10_summary,
        f,
        ensure_ascii=False,
        indent=2
    )

_project_print("\n5. ARQUIVOS GERADOS")
_project_print(candidates_path)
_project_print(by_k_path)
_project_print(summary_path)

_project_print("\n6. VERIFICAÇÕES FINAIS")
_project_print("Novos modelos treinados: NÃO")
_project_print("Validação utilizada: SIM")
_project_print("Teste utilizado: NÃO")
_project_print("Modelo definitivo selecionado: NÃO")

_project_print(
    "\nCÉLULA 10 CONCLUÍDA — "
    "comparação de desempenho, complexidade e "
    "diagnósticos realizada; seleção ainda pendente."
)


In [ ]:

# ============================================================
# FASE 2 — CÉLULA 11/36
# ESTABILIDADE, SENSIBILIDADE E DEGENERAÇÃO DOS HMMs
# ============================================================
#
# OBJETIVO:
#   1. Comparar estados entre sementes.
#   2. Medir estabilidade com ARI e concordância alinhada.
#   3. Examinar fragmentação temporal dos estados.
#   4. Avaliar persistência e ocupação dos regimes.
#   5. Investigar covariâncias e problemas de otimização.
#   6. Comparar estabilidade entre treino e validação.
#
# ENTRADAS:
#   hmm08_data
#   hmm09_models
#   audit10_candidates
#   audit10_by_k
#
# SAÍDAS:
#   audit11_individual
#   audit11_pairwise
#   audit11_by_k
#   audit11_summary
#
# RESTRIÇÕES:
#   - Nenhum novo treinamento.
#   - Nenhum uso do conjunto de teste.
#   - Nenhuma seleção definitiva.
#   - Nenhuma alteração nos modelos existentes.
#   - Nenhuma utilização de CO2.
# ============================================================

import json
import itertools
import warnings

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.optimize import linear_sum_assignment
from sklearn.metrics import (
    adjusted_rand_score,
    confusion_matrix,
)

from IPython.display import display

_project_print("=" * 72)
_project_print("FASE 2 — CÉLULA 11/36 | ESTABILIDADE DOS HMMs")
_project_print("=" * 72)

# ------------------------------------------------------------
# 1. PRÉ-REQUISITOS
# ------------------------------------------------------------

required = [
    "hmm08_data",
    "hmm09_models",
    "audit10_candidates",
    "audit10_by_k",
]

missing = [
    name for name in required
    if name not in globals()
]

if missing:
    raise RuntimeError(
        f"Objetos ausentes: {missing}. "
        "Execute as células anteriores."
    )

MODEL_NAMES_11 = [
    "M1_LOAD",
    "M2_LOAD_DELTA",
    "M3_LOAD_ABSDELTA",
]

K_VALUES_11 = list(range(2, 9))
SEEDS_11 = [11, 23, 42, 67, 101]

# Limiares diagnósticos, não regras de seleção.
MIN_OCCUPANCY_11 = 0.005
LOW_COVARIANCE_11 = 1e-6
LOW_ARI_11 = 0.70

# ------------------------------------------------------------
# 2. FUNÇÕES DE ESTABILIDADE
# ------------------------------------------------------------

def align_state_labels(reference, candidate, k):
    """
    Alinha rótulos dos estados por máxima concordância.

    Utiliza o algoritmo húngaro.
    A concordância é descritiva, não uma métrica causal.
    """

    cm = confusion_matrix(
        reference,
        candidate,
        labels=np.arange(k)
    )

    rows, cols = linear_sum_assignment(-cm)

    mapping = {
        int(col): int(row)
        for row, col in zip(rows, cols)
    }

    aligned = np.array(
        [mapping[int(s)] for s in candidate],
        dtype=int
    )

    agreement = float(
        np.mean(reference == aligned)
    )

    return aligned, agreement


def temporal_diagnostics(states, lengths, k):
    """
    Calcula mudanças de estado e duração de trechos
    dentro das sequências ON.

    As fronteiras dos ciclos não são contabilizadas
    como transições entre estados.
    """

    states = np.asarray(states, dtype=int)
    lengths = np.asarray(lengths, dtype=int)

    if lengths.sum() != len(states):
        raise ValueError(
            "Comprimentos das sequências incompatíveis."
        )

    changes = 0
    possible_transitions = 0
    run_lengths = []

    cursor = 0

    for length in lengths:

        seq = states[cursor:cursor + length]
        cursor += length

        if len(seq) == 0:
            continue

        change_positions = (
            np.flatnonzero(np.diff(seq) != 0) + 1
        )

        boundaries = np.r_[
            0,
            change_positions,
            len(seq)
        ]

        durations = np.diff(boundaries)

        run_lengths.extend(durations.tolist())

        changes += len(change_positions)

        possible_transitions += max(
            len(seq) - 1,
            0
        )

    counts = np.bincount(
        states,
        minlength=k
    )

    occupancy = counts / len(states)

    return {
        "state_changes": int(changes),
        "possible_transitions": int(
            possible_transitions
        ),
        "change_rate": (
            float(changes / possible_transitions)
            if possible_transitions > 0
            else np.nan
        ),
        "median_run_hours": (
            float(np.median(run_lengths))
            if run_lengths
            else np.nan
        ),
        "mean_run_hours": (
            float(np.mean(run_lengths))
            if run_lengths
            else np.nan
        ),
        "p10_run_hours": (
            float(np.percentile(run_lengths, 10))
            if run_lengths
            else np.nan
        ),
        "min_occupancy": float(
            occupancy.min()
        ),
        "max_occupancy": float(
            occupancy.max()
        ),
        "empty_states": int(
            np.sum(counts == 0)
        ),
        "rare_states": int(
            np.sum(
                occupancy < MIN_OCCUPANCY_11
            )
        ),
    }


def covariance_diagnostics(model):
    """
    Analisa as covariâncias do GaussianHMM.

    Para covariance_type='diag', hmmlearn.covars_
    pode devolver matrizes diagonais expandidas.
    Utilizamos os elementos diagonais.
    """

    covars = np.asarray(
        model.covars_,
        dtype=float
    )

    if covars.ndim == 3:
        variances = np.diagonal(
            covars,
            axis1=1,
            axis2=2
        )
    elif covars.ndim == 2:
        variances = covars
    else:
        raise ValueError(
            f"Formato inesperado de covariância: "
            f"{covars.shape}"
        )

    return {
        "min_variance": float(
            np.min(variances)
        ),
        "median_variance": float(
            np.median(variances)
        ),
        "max_variance": float(
            np.max(variances)
        ),
        "low_variance_count": int(
            np.sum(
                variances <= LOW_COVARIANCE_11
            )
        ),
        "nonpositive_variance_count": int(
            np.sum(variances <= 0)
        ),
        "nonfinite_variance_count": int(
            np.sum(~np.isfinite(variances))
        ),
    }


def optimization_diagnostics(model):
    """
    Inspeciona o histórico de log-verossimilhança
    registrado pelo monitor do hmmlearn.
    """

    history = np.asarray(
        list(model.monitor_.history),
        dtype=float
    )

    if len(history) < 2:
        return {
            "em_iterations_recorded": int(
                len(history)
            ),
            "negative_steps": 0,
            "largest_em_decrease": np.nan,
            "last_em_improvement": np.nan,
        }

    differences = np.diff(history)

    negative = differences[
        differences < -1e-8
    ]

    return {
        "em_iterations_recorded": int(
            len(history)
        ),
        "negative_steps": int(
            len(negative)
        ),
        "largest_em_decrease": (
            float(np.min(negative))
            if len(negative)
            else 0.0
        ),
        "last_em_improvement": float(
            differences[-1]
        ),
    }


# ------------------------------------------------------------
# 3. INFERÊNCIA RETROSPECTIVA PARA DIAGNÓSTICO
# ------------------------------------------------------------
#
# Utilizamos Viterbi somente para comparar os modelos.
#
# Esta inferência pode usar informações futuras dentro
# de cada sequência e NÃO será utilizada diretamente
# como feature causal para previsão t+1.
#
# A inferência causal será desenvolvida na Célula 13.
# ------------------------------------------------------------

state_cache_11 = {}
individual_rows = []

for model_name in MODEL_NAMES_11:

    for partition in ["train", "validation"]:

        entry = hmm08_data[model_name][partition]

        X = np.asarray(
            entry["X"],
            dtype=float
        )

        lengths = np.asarray(
            entry["lengths"],
            dtype=int
        )

        if len(X) != lengths.sum():
            raise AssertionError(
                f"Sequências incompatíveis: "
                f"{model_name}/{partition}"
            )

        for k in K_VALUES_11:

            for seed in SEEDS_11:

                key = (model_name, k, seed)

                if key not in hmm09_models:
                    continue

                model = hmm09_models[key]

                try:

                    with warnings.catch_warnings():
                        warnings.simplefilter("ignore")

                        states = model.predict(
                            X,
                            lengths=lengths
                        )

                    state_cache_11[
                        (model_name, k, seed, partition)
                    ] = states

                    temporal = temporal_diagnostics(
                        states,
                        lengths,
                        k
                    )

                    covariance = covariance_diagnostics(
                        model
                    )

                    optimization = optimization_diagnostics(
                        model
                    )

                    individual_rows.append({
                        "model": model_name,
                        "k": k,
                        "seed": seed,
                        "partition": partition,
                        "n_hours": len(X),
                        "n_sequences": len(lengths),
                        **temporal,
                        **covariance,
                        **optimization,
                    })

                except Exception as exc:

                    _project_print(
                        "ERRO:",
                        model_name,
                        k,
                        seed,
                        partition,
                        str(exc)
                    )

audit11_individual = pd.DataFrame(
    individual_rows
)

_project_print("\n1. DIAGNÓSTICOS INDIVIDUAIS")

_project_display(
    audit11_individual[
        [
            "model",
            "k",
            "seed",
            "partition",
            "change_rate",
            "median_run_hours",
            "min_occupancy",
            "min_variance",
            "negative_steps",
        ]
    ].head(20)
)

# ------------------------------------------------------------
# 4. COMPARAÇÃO ENTRE SEMENTES
# ------------------------------------------------------------
#
# ARI é invariante à permutação dos rótulos.
#
# A concordância alinhada permite visualizar
# a similaridade após corrigir a numeração arbitrária
# dos estados ocultos.
#
# São 10 pares de sementes para cada configuração/K.
# ------------------------------------------------------------

pairwise_rows = []

for model_name in MODEL_NAMES_11:

    for k in K_VALUES_11:

        for partition in ["train", "validation"]:

            for seed_a, seed_b in itertools.combinations(
                SEEDS_11,
                2
            ):

                key_a = (
                    model_name,
                    k,
                    seed_a,
                    partition
                )

                key_b = (
                    model_name,
                    k,
                    seed_b,
                    partition
                )

                if (
                    key_a not in state_cache_11
                    or key_b not in state_cache_11
                ):
                    continue

                states_a = state_cache_11[key_a]
                states_b = state_cache_11[key_b]

                ari = float(
                    adjusted_rand_score(
                        states_a,
                        states_b
                    )
                )

                _, agreement = align_state_labels(
                    states_a,
                    states_b,
                    k
                )

                pairwise_rows.append({
                    "model": model_name,
                    "k": k,
                    "partition": partition,
                    "seed_a": seed_a,
                    "seed_b": seed_b,
                    "ari": ari,
                    "aligned_agreement": agreement,
                    "low_ari_warning": bool(
                        ari < LOW_ARI_11
                    ),
                })

audit11_pairwise = pd.DataFrame(
    pairwise_rows
)

expected_pairs = (
    len(MODEL_NAMES_11)
    * len(K_VALUES_11)
    * 2
    * 10
)

_project_print("\n2. COMPARAÇÕES ENTRE SEMENTES")
_project_print("Pares esperados:", expected_pairs)
_project_print("Pares calculados:", len(audit11_pairwise))

_project_display(
    audit11_pairwise.head(20)
)

# ------------------------------------------------------------
# 5. RESUMO DA ESTABILIDADE POR CONFIGURAÇÃO E K
# ------------------------------------------------------------

stability_summary = (
    audit11_pairwise
    .groupby(
        ["model", "k", "partition"]
    )
    .agg(
        mean_ari=("ari", "mean"),
        median_ari=("ari", "median"),
        min_ari=("ari", "min"),
        std_ari=("ari", "std"),
        mean_aligned_agreement=(
            "aligned_agreement",
            "mean"
        ),
        low_ari_pairs=(
            "low_ari_warning",
            "sum"
        ),
    )
    .reset_index()
)

temporal_summary = (
    audit11_individual
    .groupby(
        ["model", "k", "partition"]
    )
    .agg(
        mean_change_rate=(
            "change_rate",
            "mean"
        ),
        median_run_hours=(
            "median_run_hours",
            "median"
        ),
        mean_min_occupancy=(
            "min_occupancy",
            "mean"
        ),
        min_variance=(
            "min_variance",
            "min"
        ),
        total_negative_steps=(
            "negative_steps",
            "sum"
        ),
    )
    .reset_index()
)

audit11_by_k = stability_summary.merge(
    temporal_summary,
    on=["model", "k", "partition"],
    how="left",
    validate="one_to_one"
)

_project_print("\n3. RESUMO DE ESTABILIDADE POR K")
_project_display(
    audit11_by_k.round(4)
)

# ------------------------------------------------------------
# 6. GRÁFICO — ESTABILIDADE ENTRE SEMENTES
# ------------------------------------------------------------

fig, axes = plt.subplots(
    1, 3,
    figsize=(17, 4.5)
)

for ax, model_name in zip(
    axes,
    MODEL_NAMES_11
):

    subset = audit11_by_k.loc[
        audit11_by_k["model"].eq(model_name)
    ]

    for partition, linestyle in [
        ("train", "-"),
        ("validation", "--"),
    ]:

        data = subset.loc[
            subset["partition"].eq(partition)
        ].sort_values("k")

        ax.plot(
            data["k"],
            data["mean_ari"],
            marker="o",
            linestyle=linestyle,
            label=partition
        )

    ax.axhline(
        LOW_ARI_11,
        color="red",
        linestyle=":",
        label="Referência ARI"
    )

    ax.set_title(model_name)
    ax.set_xlabel("Número de estados K")
    ax.set_ylabel("ARI médio entre sementes")
    ax.set_ylim(-0.1, 1.05)
    ax.set_xticks(K_VALUES_11)
    ax.grid(alpha=0.25)
    ax.legend(fontsize=8)

fig.suptitle(
    "Estabilidade dos estados HMM entre inicializações",
    y=1.04
)

plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# 7. GRÁFICO — FRAGMENTAÇÃO TEMPORAL
# ------------------------------------------------------------

fig, axes = plt.subplots(
    1, 3,
    figsize=(17, 4.5)
)

for ax, model_name in zip(
    axes,
    MODEL_NAMES_11
):

    subset = audit11_by_k.loc[
        audit11_by_k["model"].eq(model_name)
    ]

    for partition, linestyle in [
        ("train", "-"),
        ("validation", "--"),
    ]:

        data = subset.loc[
            subset["partition"].eq(partition)
        ].sort_values("k")

        ax.plot(
            data["k"],
            100 * data["mean_change_rate"],
            marker="o",
            linestyle=linestyle,
            label=partition
        )

    ax.set_title(model_name)
    ax.set_xlabel("Número de estados K")
    ax.set_ylabel(
        "Mudanças de estado / transições (%)"
    )
    ax.set_xticks(K_VALUES_11)
    ax.grid(alpha=0.25)
    ax.legend(fontsize=8)

fig.suptitle(
    "Fragmentação temporal dos regimes HMM",
    y=1.04
)

plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# 8. GRÁFICO — DURAÇÃO DOS REGIMES
# ------------------------------------------------------------

fig, axes = plt.subplots(
    1, 3,
    figsize=(17, 4.5)
)

for ax, model_name in zip(
    axes,
    MODEL_NAMES_11
):

    subset = audit11_by_k.loc[
        audit11_by_k["model"].eq(model_name)
    ]

    for partition, linestyle in [
        ("train", "-"),
        ("validation", "--"),
    ]:

        data = subset.loc[
            subset["partition"].eq(partition)
        ].sort_values("k")

        ax.plot(
            data["k"],
            data["median_run_hours"],
            marker="o",
            linestyle=linestyle,
            label=partition
        )

    ax.set_title(model_name)
    ax.set_xlabel("Número de estados K")
    ax.set_ylabel(
        "Duração mediana dos trechos (horas)"
    )
    ax.set_xticks(K_VALUES_11)
    ax.grid(alpha=0.25)
    ax.legend(fontsize=8)

fig.suptitle(
    "Persistência temporal dos estados operacionais",
    y=1.04
)

plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# 9. GRÁFICO — VARIÂNCIA MÍNIMA
# ------------------------------------------------------------

fig, axes = plt.subplots(
    1, 3,
    figsize=(17, 4.5)
)

for ax, model_name in zip(
    axes,
    MODEL_NAMES_11
):

    subset = audit11_by_k.loc[
        audit11_by_k["model"].eq(model_name)
        & audit11_by_k["partition"].eq("train")
    ].sort_values("k")

    ax.plot(
        subset["k"],
        subset["min_variance"],
        marker="o"
    )

    ax.axhline(
        LOW_COVARIANCE_11,
        color="red",
        linestyle="--",
        label="Limiar diagnóstico"
    )

    ax.set_yscale("symlog", linthresh=1e-8)
    ax.set_title(model_name)
    ax.set_xlabel("Número de estados K")
    ax.set_ylabel("Menor variância")
    ax.set_xticks(K_VALUES_11)
    ax.grid(alpha=0.25)
    ax.legend(fontsize=8)

fig.suptitle(
    "Diagnóstico de degeneração das covariâncias",
    y=1.04
)

plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# 10. ANÁLISE COMPARATIVA TREINO/VALIDAÇÃO
# ------------------------------------------------------------

train_stability = audit11_by_k.loc[
    audit11_by_k["partition"].eq("train")
].drop(columns="partition").add_suffix("_train")

val_stability = audit11_by_k.loc[
    audit11_by_k["partition"].eq("validation")
].drop(columns="partition").add_suffix("_validation")

audit11_train_val = train_stability.merge(
    val_stability,
    left_on=["model_train", "k_train"],
    right_on=["model_validation", "k_validation"],
    how="inner",
    validate="one_to_one"
)

audit11_train_val["ari_gap"] = (
    audit11_train_val["mean_ari_train"]
    - audit11_train_val["mean_ari_validation"]
)

audit11_train_val["fragmentation_gap"] = (
    audit11_train_val["mean_change_rate_validation"]
    - audit11_train_val["mean_change_rate_train"]
)

_project_print("\n4. COMPARAÇÃO TREINO × VALIDAÇÃO")

_project_display(
    audit11_train_val[
        [
            "model_train",
            "k_train",
            "mean_ari_train",
            "mean_ari_validation",
            "ari_gap",
            "mean_change_rate_train",
            "mean_change_rate_validation",
            "fragmentation_gap",
        ]
    ].round(4)
)

# ------------------------------------------------------------
# 11. SÍNTESE DIAGNÓSTICA
# ------------------------------------------------------------
#
# A tabela não produz ranking definitivo.
# Apenas consolida evidências para a Célula 12.
# ------------------------------------------------------------

audit11_selection_evidence = (
    audit10_by_k.merge(
        audit11_by_k.loc[
            audit11_by_k["partition"].eq("validation")
        ].drop(columns="partition"),
        on=["model", "k"],
        how="left",
        validate="one_to_one"
    )
)

_project_print("\n5. EVIDÊNCIAS PARA SELEÇÃO POSTERIOR")

_project_display(
    audit11_selection_evidence[
        [
            "model",
            "k",
            "mean_val_ll_hour",
            "std_val_ll_hour",
            "best_bic_train",
            "mean_ari",
            "min_ari",
            "mean_change_rate",
            "median_run_hours",
            "min_variance",
        ]
    ].round(4)
)

# ------------------------------------------------------------
# 12. EXPORTAÇÃO DOS DIAGNÓSTICOS
# ------------------------------------------------------------

if "RUN_DIR" in globals():
    report_dir = Path(RUN_DIR) / "reports"
elif "run_dir" in globals():
    report_dir = Path(run_dir) / "reports"
else:
    raise RuntimeError(
        "Diretório de execução não encontrado."
    )

report_dir.mkdir(
    parents=True,
    exist_ok=True
)

output_files_11 = {
    "individual": (
        report_dir / "cell11_individual_diagnostics.csv"
    ),
    "pairwise": (
        report_dir / "cell11_pairwise_stability.csv"
    ),
    "by_k": (
        report_dir / "cell11_stability_by_k.csv"
    ),
    "selection_evidence": (
        report_dir / "cell11_selection_evidence.csv"
    ),
    "summary": (
        report_dir / "cell11_summary.json"
    ),
}

_project_csv(audit11_individual, 
    output_files_11["individual"],
    index=False
)

_project_csv(audit11_pairwise, 
    output_files_11["pairwise"],
    index=False
)

_project_csv(audit11_by_k, 
    output_files_11["by_k"],
    index=False
)

_project_csv(audit11_selection_evidence, 
    output_files_11["selection_evidence"],
    index=False
)

audit11_summary = {
    "cell": "11/36",
    "facility_id": "6065",
    "unit_id": "2",
    "year": 2025,
    "models": MODEL_NAMES_11,
    "k_values": K_VALUES_11,
    "seeds": SEEDS_11,
    "pairwise_comparisons": int(
        len(audit11_pairwise)
    ),
    "individual_diagnostics": int(
        len(audit11_individual)
    ),
    "low_ari_pairs": int(
        audit11_pairwise[
            "low_ari_warning"
        ].sum()
    ),
    "models_with_low_variance": int(
        audit11_individual.loc[
            audit11_individual["partition"].eq("train"),
            "low_variance_count"
        ].gt(0).sum()
    ),
    "validation_used": True,
    "test_used": False,
    "new_models_trained": False,
    "final_model_selected": False,
    "notes": [
        "ARI mede estabilidade de partições de estados "
        "e não interpretação física.",
        "Viterbi é usado apenas para diagnóstico.",
        "Inferência causal será tratada na Célula 13.",
        "Nenhuma decisão definitiva foi tomada.",
    ],
}

with open(
    output_files_11["summary"],
    "w",
    encoding="utf-8"
) as f:
    _project_dump(
        audit11_summary,
        f,
        ensure_ascii=False,
        indent=2
    )

_project_print("\n6. RELATÓRIOS GERADOS")

for name, path in output_files_11.items():
    _project_print(f"{name}: {path}")

_project_print("\n7. VERIFICAÇÕES FINAIS")
_project_print("Novos HMMs treinados: NÃO")
_project_print("Validação utilizada: SIM")
_project_print("Teste utilizado: NÃO")
_project_print("Modelo definitivo selecionado: NÃO")

_project_print(
    "\nCÉLULA 11 CONCLUÍDA — "
    "estabilidade, fragmentação e degeneração "
    "avaliadas; seleção reservada à Célula 12."
)


In [ ]:

# ============================================================
# FASE 2 — CÉLULA 12/36
# SELEÇÃO FORMAL E CONGELAMENTO DO HMM — VERSÃO CONSOLIDADA
# ============================================================
#
# PROJETO:
# Iatan | Facility 6065 | Unit 2 | Ano 2025
#
# OBJETIVOS:
# 1. Recuperar as evidências das Células 10 e 11.
# 2. Aplicar critérios explícitos de admissibilidade.
# 3. Selecionar a família e o número de estados K.
# 4. Selecionar a semente pela validação.
# 5. Recuperar HMM e scaler já treinados.
# 6. Congelar e verificar os artefatos.
# 7. Registrar a decisão para as próximas células.
#
# NÃO REALIZA:
# - Novo treinamento.
# - Reajuste do scaler.
# - Inferência no conjunto de teste.
# - Mineração de processos.
# - Modelagem de emissões.
#
# SAÍDAS PRINCIPAIS:
# HMM_SELECTED
# SCALER_SELECTED
# HMM_SELECTION_12
# cell12_hmm_selected.pkl
# cell12_scaler_selected.pkl
# cell12_selection_decision.json
# ============================================================

import json
import pickle
import hashlib
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display

_project_print("=" * 72)
_project_print("FASE 2 — CÉLULA 12/36 | SELEÇÃO E CONGELAMENTO DO HMM")
_project_print("=" * 72)

# ------------------------------------------------------------
# 1. VERIFICAÇÃO DOS OBJETOS
# ------------------------------------------------------------

required_objects = [
    "audit10_candidates",
    "audit11_selection_evidence",
    "hmm09_models",
    "hmm08_scalers",
    "hmm08_data",
]

missing_objects = [
    name for name in required_objects
    if name not in globals()
]

if missing_objects:
    raise RuntimeError(
        "Execute as Células 08 a 11. "
        f"Objetos ausentes: {missing_objects}"
    )

if not isinstance(hmm09_models, dict):
    raise TypeError("hmm09_models deve ser um dicionário.")

if not isinstance(hmm08_scalers, dict):
    raise TypeError("hmm08_scalers deve ser um dicionário.")

# Diretório de execução da Célula 01.
if "RUN_DIR" in globals():
    run_path = Path(RUN_DIR)
elif "run_dir" in globals():
    run_path = Path(run_dir)
else:
    raise RuntimeError(
        "Diretório da execução não encontrado. "
        "Recupere RUN_DIR da Célula 01."
    )

report_dir = run_path / "reports"
model_dir = run_path / "models"

report_dir.mkdir(parents=True, exist_ok=True)
model_dir.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 2. RECUPERAÇÃO DAS EVIDÊNCIAS
# ------------------------------------------------------------

evidence = audit11_selection_evidence.copy(deep=True)
individual = audit10_candidates.copy(deep=True)

required_evidence_columns = [
    "model",
    "k",
    "mean_val_ll_hour",
    "mean_ari",
    "mean_change_rate",
    "min_variance",
]

required_individual_columns = [
    "model",
    "k",
    "seed",
    "validation_success",
    "loglik_validation_per_hour",
    "min_occupancy",
    "parameter_warning",
    "optimization_decrease_warning",
]

for name, df, columns in [
    ("evidence", evidence, required_evidence_columns),
    ("individual", individual, required_individual_columns),
]:
    missing = [c for c in columns if c not in df.columns]
    if missing:
        raise ValueError(
            f"{name}: colunas ausentes: {missing}"
        )

if evidence.duplicated(["model", "k"]).any():
    raise AssertionError(
        "Evidências duplicadas por configuração e K."
    )

_project_print("\n1. INVENTÁRIO")
_project_print(f"Configurações/K: {len(evidence)}")
_project_print(f"Ajustes individuais: {len(individual)}")
_project_print(f"Modelos disponíveis: {len(hmm09_models)}")

# ------------------------------------------------------------
# 3. CRITÉRIOS DE ADMISSIBILIDADE
# ------------------------------------------------------------
#
# Os limiares são decisões metodológicas explícitas,
# não limites físicos universais.
#
# A ocupação é avaliada individualmente nas sementes
# e também no resumo por configuração/K.
# ------------------------------------------------------------

MIN_MEAN_ARI = 0.70
MIN_OCCUPANCY = 0.005
MIN_VARIANCE = 1e-6
MAX_CHANGE_RATE = 0.30

# Prioridade científica:
# carga + rampa assinada é a representação preferida
# para identificar processos operacionais.
MODEL_PRIORITY = [
    "M2_LOAD_DELTA",
    "M1_LOAD",
    "M3_LOAD_ABSDELTA",
]

selection = evidence.copy()

# Recuperar ocupação mínima observada por K.
if "audit11_individual" in globals():

    val_occ = audit11_individual.loc[
        audit11_individual["partition"].eq("validation")
    ].copy()

    occupancy = (
        val_occ.groupby(["model", "k"], as_index=False)
        .agg(
            worst_min_occupancy=(
                "min_occupancy",
                "min"
            )
        )
    )

else:

    occupancy = (
        individual.groupby(["model", "k"], as_index=False)
        .agg(
            worst_min_occupancy=(
                "min_occupancy",
                "min"
            )
        )
    )

selection = selection.merge(
    occupancy,
    on=["model", "k"],
    how="left",
    validate="one_to_one",
)

selection["stable"] = (
    selection["mean_ari"] >= MIN_MEAN_ARI
)

selection["covariance_ok"] = (
    selection["min_variance"] >= MIN_VARIANCE
)

selection["fragmentation_ok"] = (
    selection["mean_change_rate"] <= MAX_CHANGE_RATE
)

selection["occupancy_ok"] = (
    selection["worst_min_occupancy"] >= MIN_OCCUPANCY
)

selection["admissible"] = (
    selection["stable"]
    & selection["covariance_ok"]
    & selection["fragmentation_ok"]
    & selection["occupancy_ok"]
)

_project_print("\n2. ADMISSIBILIDADE DOS CANDIDATOS")

_project_display(
    selection[
        [
            "model",
            "k",
            "mean_val_ll_hour",
            "mean_ari",
            "mean_change_rate",
            "worst_min_occupancy",
            "admissible",
        ]
    ].round(4)
)

# ------------------------------------------------------------
# 4. RANQUEAMENTO DENTRO DE CADA FAMÍLIA
# ------------------------------------------------------------
#
# Nunca comparar diretamente LL entre famílias com
# diferentes dimensões observadas.
#
# A pontuação combina:
# - desempenho na validação: 40%
# - estabilidade: 30%
# - fragmentação: 20%
# - parcimônia: 10%
#
# Pesos documentados como regra operacional de seleção.
# ------------------------------------------------------------

admissible = selection.loc[
    selection["admissible"]
].copy()

if admissible.empty:
    raise RuntimeError(
        "Nenhuma configuração admissível. "
        "Não é permitido selecionar um HMM "
        "ignorando silenciosamente os critérios."
    )

admissible["ll_rank"] = (
    admissible.groupby("model")[
        "mean_val_ll_hour"
    ].rank(
        ascending=False,
        method="min"
    )
)

admissible["stability_rank"] = (
    admissible.groupby("model")[
        "mean_ari"
    ].rank(
        ascending=False,
        method="min"
    )
)

admissible["fragmentation_rank"] = (
    admissible.groupby("model")[
        "mean_change_rate"
    ].rank(
        ascending=True,
        method="min"
    )
)

admissible["parsimony_rank"] = (
    admissible.groupby("model")[
        "k"
    ].rank(
        ascending=True,
        method="min"
    )
)

admissible["selection_score"] = (
    0.40 * admissible["ll_rank"]
    + 0.30 * admissible["stability_rank"]
    + 0.20 * admissible["fragmentation_rank"]
    + 0.10 * admissible["parsimony_rank"]
)

winners = (
    admissible.sort_values(
        ["model", "selection_score", "k"],
        ascending=[True, True, True]
    )
    .groupby("model", as_index=False)
    .first()
)

_project_print("\n3. MELHOR CANDIDATO DE CADA FAMÍLIA")

_project_display(
    winners[
        [
            "model",
            "k",
            "mean_val_ll_hour",
            "mean_ari",
            "mean_change_rate",
            "selection_score",
        ]
    ].round(4)
)

# ------------------------------------------------------------
# 5. SELEÇÃO DA FAMÍLIA E DE K
# ------------------------------------------------------------

available_families = set(winners["model"])

selected_family = next(
    (
        family for family in MODEL_PRIORITY
        if family in available_families
    ),
    None
)

if selected_family is None:
    raise RuntimeError(
        "Nenhuma família prioritária admissível."
    )

winner = winners.loc[
    winners["model"].eq(selected_family)
].iloc[0]

selected_k = int(winner["k"])

_project_print("\n4. CONFIGURAÇÃO SELECIONADA")
_project_print(f"Família: {selected_family}")
_project_print(f"K: {selected_k}")
_project_print(
    "Fundamento: adequação operacional, "
    "estabilidade e validação."
)

# ------------------------------------------------------------
# 6. SELEÇÃO DA SEMENTE
# ------------------------------------------------------------
#
# Somente candidatos:
# - com validação bem-sucedida;
# - sem problemas de parâmetros;
# - sem redução de LL durante EM;
# - com ocupação mínima adequada.
#
# Covariâncias pequenas são analisadas no diagnóstico
# específico da Célula 11 e não excluídas automaticamente.
# ------------------------------------------------------------

seed_candidates = individual.loc[
    individual["model"].eq(selected_family)
    & individual["k"].eq(selected_k)
    & individual["validation_success"].eq(True)
].copy()

if seed_candidates.empty:
    raise RuntimeError(
        "Não existem sementes válidas para o candidato."
    )

seed_candidates["eligible"] = (
    ~seed_candidates["parameter_warning"].astype(bool)
    & ~seed_candidates[
        "optimization_decrease_warning"
    ].astype(bool)
    & seed_candidates["min_occupancy"].ge(
        MIN_OCCUPANCY
    )
    & np.isfinite(
        seed_candidates["loglik_validation_per_hour"]
    )
)

seed_candidates = seed_candidates.sort_values(
    ["loglik_validation_per_hour", "seed"],
    ascending=[False, True]
)

_project_print("\n5. COMPARAÇÃO DAS SEMENTES")

_project_display(
    seed_candidates[
        [
            "seed",
            "loglik_validation_per_hour",
            "min_occupancy",
            "covariance_warning",
            "optimization_decrease_warning",
            "eligible",
        ]
    ].round(6)
)

eligible_seeds = seed_candidates.loc[
    seed_candidates["eligible"]
].copy()

if eligible_seeds.empty:
    raise RuntimeError(
        "Nenhuma semente atende aos critérios. "
        "Revisão metodológica necessária."
    )

selected_seed = int(
    eligible_seeds.iloc[0]["seed"]
)

selected_seed_metrics = eligible_seeds.iloc[0]

# ------------------------------------------------------------
# 7. RECUPERAÇÃO DO HMM E DO SCALER
# ------------------------------------------------------------

selected_key = (
    selected_family,
    selected_k,
    selected_seed,
)

if selected_key not in hmm09_models:
    raise KeyError(
        f"HMM treinado não encontrado: {selected_key}"
    )

if selected_family not in hmm08_scalers:
    raise KeyError(
        f"Scaler treinado não encontrado: {selected_family}"
    )

HMM_SELECTED = hmm09_models[selected_key]
SCALER_SELECTED = hmm08_scalers[selected_family]

# Verificação estrutural.
assert HMM_SELECTED.n_components == selected_k
assert hasattr(HMM_SELECTED, "score")
assert hasattr(HMM_SELECTED, "predict")
assert hasattr(SCALER_SELECTED, "transform")
assert hasattr(SCALER_SELECTED, "n_features_in_")

if not np.isfinite(
    np.asarray(HMM_SELECTED.means_)
).all():
    raise AssertionError(
        "HMM selecionado contém médias não finitas."
    )

if not np.isfinite(
    np.asarray(HMM_SELECTED.transmat_)
).all():
    raise AssertionError(
        "HMM selecionado contém transições não finitas."
    )

if not np.allclose(
    HMM_SELECTED.transmat_.sum(axis=1),
    1.0,
    atol=1e-5
):
    raise AssertionError(
        "Matriz de transição inválida."
    )

if HMM_SELECTED.n_features != SCALER_SELECTED.n_features_in_:
    raise AssertionError(
        "Dimensões incompatíveis entre HMM e scaler."
    )

_project_print("\n6. OBJETOS RECUPERADOS")
_project_print(f"Chave HMM: {selected_key}")
_project_print(f"Estados: {HMM_SELECTED.n_components}")
_project_print(f"Variáveis: {HMM_SELECTED.n_features}")
_project_print("Scaler: recuperado do treinamento original")

# ------------------------------------------------------------
# 8. CONGELAMENTO DOS ARTEFATOS
# ------------------------------------------------------------

model_path = (
    model_dir / "cell12_hmm_selected.pkl"
)

scaler_path = (
    model_dir / "cell12_scaler_selected.pkl"
)

with open(model_path, "wb") as f:
    pickle.dump(
        HMM_SELECTED,
        f,
        protocol=pickle.HIGHEST_PROTOCOL
    )

with open(scaler_path, "wb") as f:
    pickle.dump(
        SCALER_SELECTED,
        f,
        protocol=pickle.HIGHEST_PROTOCOL
    )

# ------------------------------------------------------------
# 9. VERIFICAÇÃO DA SERIALIZAÇÃO
# ------------------------------------------------------------

with open(model_path, "rb") as f:
    restored_hmm = pickle.load(f)

with open(scaler_path, "rb") as f:
    restored_scaler = pickle.load(f)

assert restored_hmm.n_components == selected_k

assert (
    restored_scaler.n_features_in_
    == SCALER_SELECTED.n_features_in_
)

np.testing.assert_allclose(
    restored_hmm.means_,
    HMM_SELECTED.means_,
    rtol=0,
    atol=0
)

np.testing.assert_allclose(
    restored_hmm.transmat_,
    HMM_SELECTED.transmat_,
    rtol=0,
    atol=0
)

def sha256_file(path):
    digest = hashlib.sha256()

    with open(path, "rb") as f:
        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            digest.update(chunk)

    return digest.hexdigest()

model_hash = sha256_file(model_path)
scaler_hash = sha256_file(scaler_path)

# ------------------------------------------------------------
# 10. METADADOS DO MODELO SELECIONADO
# ------------------------------------------------------------

HMM_SELECTION_12 = {
    "facility_name": "Iatan",
    "facility_id": "6065",
    "unit_id": "2",
    "year": 2025,
    "selected_model": selected_family,
    "selected_k": selected_k,
    "selected_seed": selected_seed,
    "selection_partition": "validation",
    "selection_method": (
        "admissibility + within-family ordinal ranking "
        "+ operational family priority + seed validation"
    ),
    "selection_criteria": {
        "min_mean_ari": MIN_MEAN_ARI,
        "min_occupancy": MIN_OCCUPANCY,
        "min_variance": MIN_VARIANCE,
        "max_change_rate": MAX_CHANGE_RATE,
        "family_priority": MODEL_PRIORITY,
        "ranking_weights": {
            "validation": 0.40,
            "stability": 0.30,
            "fragmentation": 0.20,
            "parsimony": 0.10,
        },
    },
    "selected_evidence": {
        "mean_validation_ll_hour": float(
            winner["mean_val_ll_hour"]
        ),
        "mean_ari": float(
            winner["mean_ari"]
        ),
        "mean_change_rate": float(
            winner["mean_change_rate"]
        ),
        "median_run_hours": float(
            winner["median_run_hours"]
        ),
        "selected_seed_validation_ll_hour": float(
            selected_seed_metrics[
                "loglik_validation_per_hour"
            ]
        ),
        "selected_seed_min_occupancy": float(
            selected_seed_metrics["min_occupancy"]
        ),
    },
    "artifacts": {
        "hmm": str(model_path),
        "scaler": str(scaler_path),
        "hmm_sha256": model_hash,
        "scaler_sha256": scaler_hash,
    },
    "new_models_trained": False,
    "scaler_refitted": False,
    "test_used": False,
    "frozen": True,
}

# ------------------------------------------------------------
# 11. EXPORTAÇÃO DOS RELATÓRIOS
# ------------------------------------------------------------

selection_path = (
    report_dir / "cell12_admissibility.csv"
)

winners_path = (
    report_dir / "cell12_family_winners.csv"
)

seeds_path = (
    report_dir / "cell12_seed_comparison.csv"
)

decision_path = (
    report_dir / "cell12_selection_decision.json"
)

_project_csv(selection, 
    selection_path,
    index=False
)

_project_csv(winners, 
    winners_path,
    index=False
)

_project_csv(seed_candidates, 
    seeds_path,
    index=False
)

with open(
    decision_path,
    "w",
    encoding="utf-8"
) as f:
    _project_dump(
        HMM_SELECTION_12,
        f,
        ensure_ascii=False,
        indent=2
    )

# ------------------------------------------------------------
# 12. RESULTADO FINAL
# ------------------------------------------------------------

_project_print("\n" + "=" * 72)
_project_print("RESULTADO FINAL — CÉLULA 12/36")
_project_print("=" * 72)

_project_print(f"Unidade: Iatan | 6065 | 2 | 2025")
_project_print(f"Configuração: {selected_family}")
_project_print(f"Número de estados: {selected_k}")
_project_print(f"Semente selecionada: {selected_seed}")

_project_print(
    "LL/hora na validação: "
    f"{selected_seed_metrics['loglik_validation_per_hour']:.6f}"
)

_project_print(
    "ARI médio: "
    f"{winner['mean_ari']:.4f}"
)

_project_print(
    "Fragmentação: "
    f"{100 * winner['mean_change_rate']:.2f}%"
)

_project_print(f"\nHMM: {model_path}")
_project_print(f"Scaler: {scaler_path}")
_project_print(f"Manifesto: {decision_path}")

_project_print("\nVerificações:")
_project_print("  HMM recuperado: OK")
_project_print("  Scaler recuperado: OK")
_project_print("  Serialização: OK")
_project_print("  Hashes SHA-256: OK")
_project_print("  Novo treinamento: NÃO")
_project_print("  Reajuste do scaler: NÃO")
_project_print("  Conjunto de teste utilizado: NÃO")
_project_print("  Modelo congelado: SIM")

_project_print(
    "\nCÉLULA 12 CONCLUÍDA — "
    "HMM selecionado e congelado para as próximas etapas."
)


In [ ]:
# V2-A — Auditoria multicritério de HMMs, SEM alterar a seleção original.
# Executar após a célula original 11 e antes da 12.
from pathlib import Path
import numpy as np
import pandas as pd
import json
from IPython.display import display

for obj in ("audit10_candidates", "audit11_selection_evidence", "hmm09_models", "RUN_DIR"):
    if obj not in globals():
        raise RuntimeError(f"Execute as células anteriores: {obj} não encontrado.")

v2_dir = Path(RUN_DIR) / "reports" / "v2_scientific_audit"
v2_dir.mkdir(parents=True, exist_ok=True)

candidate_v2 = audit10_candidates.copy()
warning_cols_v2 = [c for c in (
    "covariance_warning", "occupancy_warning",
    "parameter_warning", "optimization_decrease_warning"
) if c in candidate_v2.columns]
candidate_v2["warning_count_v2"] = (
    candidate_v2[warning_cols_v2].fillna(False).astype(int).sum(axis=1)
)
candidate_v2["numerically_clean_v2"] = candidate_v2["warning_count_v2"].eq(0)

diagnostics_v2 = []
for key, model in hmm09_models.items():
    monitor = getattr(model, "monitor_", None)
    history = list(getattr(monitor, "history", []))
    delta = float(history[-1] - history[-2]) if len(history) >= 2 else np.nan
    # monitor_.converged pode também ser True no limite de iterações;
    # por isso registramos separadamente o término por limite.
    diagnostics_v2.append({
        "model": key[0], "k": int(key[1]), "seed": int(key[2]),
        "monitor_converged": bool(getattr(monitor, "converged", False)),
        "iterations": getattr(monitor, "iter", None),
        "max_iterations": getattr(monitor, "n_iter", None),
        "hit_iteration_limit": (
            getattr(monitor, "iter", None) == getattr(monitor, "n_iter", None)
        ),
        "last_ll_change": delta,
        "negative_last_ll_change": bool(np.isfinite(delta) and delta < -1e-8),
    })
diag_v2 = pd.DataFrame(diagnostics_v2)
audit_hmm_v2 = candidate_v2.merge(
    diag_v2, on=["model", "k", "seed"], how="left", validate="one_to_one"
)
if len(audit_hmm_v2) != len(candidate_v2):
    raise AssertionError("Perda de candidatos na auditoria.")

evidence_v2 = audit11_selection_evidence.copy()
# Métricas em dimensões diferentes NÃO são usadas para classificar famílias.
# LL/h pode ser inspecionada apenas dentro de uma mesma família/dimensão.
cols_v2 = [c for c in evidence_v2.columns if
           c in ("model", "k") or
           any(t in c.lower() for t in ("ari", "occup", "fragment", "warning", "change_rate"))]
comparison_v2 = evidence_v2[cols_v2].copy()
comparison_v2["comparison_scope"] = (
    "Entre famílias: estabilidade/ocupação/interpretabilidade; "
    "LL não comparável entre dimensões"
)
_project_csv(audit_hmm_v2, v2_dir / "hmm_105_fits_diagnostics.csv", index=False)
_project_csv(comparison_v2, v2_dir / "hmm_family_sensitivity.csv", index=False)
_project_print("AUDITORIA V2 — HMMs")
_project_print("Ajustes auditados:", len(audit_hmm_v2))
_project_print("Ajustes sem alertas registrados:", int(audit_hmm_v2["numerically_clean_v2"].sum()))
_project_print("Ajustes no limite de iterações:", int(audit_hmm_v2["hit_iteration_limit"].sum()))
_project_display(comparison_v2)
_project_print("Arquivos:", v2_dir)


## 13–17 — Filtragem e interpretação dos estados

Forward usa o prefixo para estimar a distribuição dos estados na hora corrente. Viterbi recupera o caminho mais provável usando a sequência disponível inteira e serve aqui ao diagnóstico retrospectivo. Os rótulos operacionais e as transições observadas interpretam padrões de Iatan. A numeração do estado é interna ao modelo congelado. [Rabiner (1989), §III](https://doi.org/10.1109/5.18626) · [hmmlearn, API de decodificação](https://hmmlearn.readthedocs.io/en/stable/api.html)


In [ ]:

# ============================================================
# FASE 2 — CÉLULA 13/36
# INFERÊNCIA VITERBI E FILTRAGEM CAUSAL FORWARD
# ============================================================
#
# OBJETIVO:
# 1. Aplicar o HMM congelado aos ciclos ON.
# 2. Inferir estados retrospectivos por Viterbi.
# 3. Calcular probabilidades filtradas P(S_t | X_1:t).
# 4. Reiniciar a inferência em cada ciclo ON.
# 5. Preservar as partições train/validation/test.
# 6. Verificar normalização e causalidade da filtragem.
#
# ENTRADAS:
#   hmm08_data
#   HMM_SELECTED
#   SCALER_SELECTED
#   HMM_SELECTION_12
#
# SAÍDAS:
#   hmm13_inference
#   audit13_summary
#   reports/cell13_inference_summary.json
#
# RESTRIÇÕES:
# - Nenhum novo treinamento.
# - Nenhum reajuste do scaler.
# - Nenhuma seleção de modelo.
# - Nenhuma utilização de CO2.
# - Nenhuma avaliação de desempenho no teste.
# ============================================================

import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.special import logsumexp
from IPython.display import display

_project_print("=" * 72)
_project_print("FASE 2 — CÉLULA 13/36 | INFERÊNCIA DOS ESTADOS HMM")
_project_print("=" * 72)

# ------------------------------------------------------------
# 1. PRÉ-REQUISITOS
# ------------------------------------------------------------

required_objects = [
    "hmm08_data",
    "HMM_SELECTED",
    "SCALER_SELECTED",
    "HMM_SELECTION_12",
]

missing_objects = [
    name for name in required_objects
    if name not in globals()
]

if missing_objects:
    raise RuntimeError(
        f"Objetos ausentes: {missing_objects}. "
        "Execute as Células 08 a 12."
    )

selected_family = HMM_SELECTION_12["selected_model"]
K = int(HMM_SELECTION_12["selected_k"])

if HMM_SELECTED.n_components != K:
    raise AssertionError(
        "Número de estados incompatível com a seleção."
    )

if selected_family not in hmm08_data:
    raise KeyError(
        f"Configuração ausente: {selected_family}"
    )

if "RUN_DIR" in globals():
    run_path = Path(RUN_DIR)
elif "run_dir" in globals():
    run_path = Path(run_dir)
else:
    raise RuntimeError(
        "Diretório da execução não encontrado."
    )

report_dir = run_path / "reports"
report_dir.mkdir(parents=True, exist_ok=True)

_project_print("\nMODELO CONGELADO")
_project_print("Família:", selected_family)
_project_print("Estados:", K)
_project_print("Semente:", HMM_SELECTION_12["selected_seed"])

# ------------------------------------------------------------
# 2. FUNÇÃO DE FILTRAGEM CAUSAL
# ------------------------------------------------------------
#
# alpha_t(j) = P(S_t=j | X_1, ..., X_t)
#
# A filtragem é realizada no domínio logarítmico
# para reduzir problemas de underflow numérico.
#
# Cada ciclo começa com startprob_.
# ------------------------------------------------------------

def hmm13_forward_filter(model, X):
    """
    Calcula probabilidades filtradas por hora.

    Retorna matriz [n_horas, K].

    Não utiliza observações futuras.
    """

    X = np.asarray(X, dtype=float)

    n_hours = len(X)
    k = model.n_components

    if n_hours == 0:
        return np.empty((0, k), dtype=float)

    # Emissões gaussianas:
    # log P(X_t | S_t=j)
    log_emission = model._compute_log_likelihood(X)

    log_start = np.log(
        np.maximum(model.startprob_, 1e-300)
    )

    log_transition = np.log(
        np.maximum(model.transmat_, 1e-300)
    )

    log_alpha = np.empty(
        (n_hours, k),
        dtype=float
    )

    # Primeira observação do ciclo.
    initial = log_start + log_emission[0]

    log_alpha[0] = (
        initial - logsumexp(initial)
    )

    # Atualização recursiva.
    for t in range(1, n_hours):

        prediction = logsumexp(
            log_alpha[t - 1][:, None]
            + log_transition,
            axis=0
        )

        update = prediction + log_emission[t]

        log_alpha[t] = (
            update - logsumexp(update)
        )

    probabilities = np.exp(log_alpha)

    return probabilities


# ------------------------------------------------------------
# 3. FUNÇÃO DE INFERÊNCIA POR CICLOS
# ------------------------------------------------------------

def hmm13_infer_partition(model, X, lengths):
    """
    Executa Viterbi e forward filtering
    separadamente em cada ciclo ON.

    Não mistura observações entre ciclos.
    """

    X = np.asarray(X, dtype=float)
    lengths = np.asarray(lengths, dtype=int)

    if X.ndim != 2:
        raise ValueError("Matriz X inválida.")

    if lengths.sum() != len(X):
        raise AssertionError(
            "Comprimentos incompatíveis."
        )

    if np.any(lengths <= 0):
        raise ValueError(
            "Sequência operacional vazia."
        )

    k = model.n_components

    viterbi_states = np.empty(
        len(X),
        dtype=int
    )

    filtered_probabilities = np.empty(
        (len(X), k),
        dtype=float
    )

    cycle_index = np.empty(
        len(X),
        dtype=int
    )

    cursor = 0

    for cycle_number, length in enumerate(
        lengths,
        start=1
    ):

        end = cursor + length

        X_cycle = X[cursor:end]

        # Viterbi: inferência retrospectiva.
        states = model.predict(
            X_cycle
        )

        # Forward: inferência causal.
        probabilities = hmm13_forward_filter(
            model,
            X_cycle
        )

        viterbi_states[cursor:end] = states

        filtered_probabilities[
            cursor:end
        ] = probabilities

        cycle_index[
            cursor:end
        ] = cycle_number

        cursor = end

    return {
        "viterbi_states": viterbi_states,
        "filtered_probabilities": filtered_probabilities,
        "cycle_index": cycle_index,
    }


# ------------------------------------------------------------
# 4. INFERÊNCIA NAS TRÊS PARTIÇÕES
# ------------------------------------------------------------
#
# O modelo permanece congelado.
#
# A aplicação ao teste não calcula métricas de
# desempenho nem altera qualquer decisão.
# ------------------------------------------------------------

PARTITIONS = [
    "train",
    "validation",
    "test",
]

hmm13_inference = {}

for partition in PARTITIONS:

    entry = hmm08_data[
        selected_family
    ][partition]

    X = np.asarray(
        entry["X"],
        dtype=float
    )

    lengths = np.asarray(
        entry["lengths"],
        dtype=int
    )

    result = hmm13_infer_partition(
        HMM_SELECTED,
        X,
        lengths
    )

    result["X"] = X
    result["lengths"] = lengths

    hmm13_inference[partition] = result

    _project_print(
        f"\n{partition.upper()}: "
        f"{len(X)} horas | "
        f"{len(lengths)} ciclos | "
        f"{K} estados"
    )

# ------------------------------------------------------------
# 5. VERIFICAÇÃO DAS PROBABILIDADES
# ------------------------------------------------------------

for partition, result in hmm13_inference.items():

    probs = result["filtered_probabilities"]
    states = result["viterbi_states"]

    if probs.shape[1] != K:
        raise AssertionError(
            f"Dimensão incorreta: {partition}"
        )

    if not np.isfinite(probs).all():
        raise AssertionError(
            f"Probabilidades não finitas: {partition}"
        )

    if np.any(probs < -1e-12):
        raise AssertionError(
            f"Probabilidades negativas: {partition}"
        )

    np.testing.assert_allclose(
        probs.sum(axis=1),
        1.0,
        rtol=0,
        atol=1e-8
    )

    if not np.isin(
        states,
        np.arange(K)
    ).all():
        raise AssertionError(
            f"Estados inválidos: {partition}"
        )

_project_print("\n1. NORMALIZAÇÃO DAS PROBABILIDADES: OK")

# ------------------------------------------------------------
# 6. TESTE COMPUTACIONAL DE CAUSALIDADE
# ------------------------------------------------------------
#
# Verifica que a probabilidade na hora t
# não muda quando removemos observações posteriores.
#
# É uma verificação da implementação forward,
# não uma demonstração de causalidade física.
# ------------------------------------------------------------

train_entry = hmm08_data[
    selected_family
]["train"]

X_train = np.asarray(
    train_entry["X"],
    dtype=float
)

lengths_train = np.asarray(
    train_entry["lengths"],
    dtype=int
)

# Selecionar primeiro ciclo com ao menos 10 horas.
eligible_cycles = np.where(
    lengths_train >= 10
)[0]

if len(eligible_cycles) == 0:
    raise RuntimeError(
        "Nenhum ciclo suficientemente longo "
        "para verificar a filtragem."
    )

cycle_position = int(
    eligible_cycles[0]
)

start = int(
    lengths_train[:cycle_position].sum()
)

length = int(
    lengths_train[cycle_position]
)

X_cycle = X_train[
    start:start + length
]

full_prob = hmm13_forward_filter(
    HMM_SELECTED,
    X_cycle
)

check_points = sorted(set([
    1,
    min(5, length),
    min(10, length),
    length,
]))

max_causal_difference = 0.0

for horizon in check_points:

    prefix_prob = hmm13_forward_filter(
        HMM_SELECTED,
        X_cycle[:horizon]
    )

    difference = np.max(
        np.abs(
            prefix_prob
            - full_prob[:horizon]
        )
    )

    max_causal_difference = max(
        max_causal_difference,
        float(difference)
    )

if max_causal_difference > 1e-10:
    raise AssertionError(
        "Filtragem apresentou dependência "
        "de observações futuras."
    )

_project_print("\n2. VERIFICAÇÃO DE CAUSALIDADE COMPUTACIONAL")
_project_print(
    "Diferença máxima:",
    f"{max_causal_difference:.3e}"
)
_project_print("Resultado: OK")

# ------------------------------------------------------------
# 7. COMPARAÇÃO DESCRITIVA VITERBI × FORWARD
# ------------------------------------------------------------
#
# O estado filtrado é o argmax das probabilidades
# disponíveis até a hora t.
#
# Não é necessariamente igual ao estado Viterbi,
# que utiliza a sequência completa.
# ------------------------------------------------------------

summary_rows = []

for partition, result in hmm13_inference.items():

    viterbi = result["viterbi_states"]
    probs = result["filtered_probabilities"]

    filtered_state = np.argmax(
        probs,
        axis=1
    )

    confidence = np.max(
        probs,
        axis=1
    )

    agreement = float(
        np.mean(
            viterbi == filtered_state
        )
    )

    entropy = -np.sum(
        probs * np.log(
            np.maximum(probs, 1e-300)
        ),
        axis=1
    )

    result["filtered_state"] = filtered_state
    result["filtered_confidence"] = confidence
    result["filtered_entropy"] = entropy

    summary_rows.append({
        "partition": partition,
        "n_hours": int(len(viterbi)),
        "n_cycles": int(len(result["lengths"])),
        "agreement_viterbi_forward": agreement,
        "mean_filtered_confidence": float(
            confidence.mean()
        ),
        "median_filtered_confidence": float(
            np.median(confidence)
        ),
        "mean_filtered_entropy": float(
            entropy.mean()
        ),
    })

audit13_summary = pd.DataFrame(
    summary_rows
)

_project_print("\n3. RESUMO DESCRITIVO DA INFERÊNCIA")
_project_display(
    audit13_summary.round(4)
)

# ------------------------------------------------------------
# 8. VISUALIZAÇÃO DE UM CICLO OPERACIONAL
# ------------------------------------------------------------
#
# Exibe estados Viterbi e filtrados para um ciclo
# do treinamento, sem utilizar CO2.
# ------------------------------------------------------------

train_result = hmm13_inference["train"]

cycle_number = cycle_position + 1
cycle_mask = (
    train_result["cycle_index"] == cycle_number
)

X_plot = train_result["X"][
    cycle_mask
]

viterbi_plot = train_result[
    "viterbi_states"
][cycle_mask]

filtered_plot = train_result[
    "filtered_state"
][cycle_mask]

confidence_plot = train_result[
    "filtered_confidence"
][cycle_mask]

hours = np.arange(
    len(X_plot)
)

fig, axes = plt.subplots(
    3, 1,
    figsize=(14, 9),
    sharex=True
)

axes[0].plot(
    hours,
    X_plot[:, 0],
    linewidth=1.3
)

axes[0].set_ylabel(
    "Carga padronizada"
)
axes[0].set_title(
    f"Ciclo ON {cycle_number} — treinamento"
)
axes[0].grid(alpha=0.2)

axes[1].step(
    hours,
    viterbi_plot,
    where="mid",
    label="Viterbi retrospectivo"
)

axes[1].step(
    hours,
    filtered_plot,
    where="mid",
    alpha=0.65,
    linestyle="--",
    label="Forward causal"
)

axes[1].set_ylabel("Estado HMM")
axes[1].set_yticks(range(K))
axes[1].legend()
axes[1].grid(alpha=0.2)

axes[2].plot(
    hours,
    confidence_plot
)

axes[2].set_ylabel(
    "Confiança filtrada"
)
axes[2].set_xlabel(
    "Horas desde o início do ciclo"
)
axes[2].set_ylim(0, 1.05)
axes[2].grid(alpha=0.2)

plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# 9. EXPORTAÇÃO DO RELATÓRIO
# ------------------------------------------------------------
#
# Os resultados permanecem em memória.
# A exportação completa das features e logs
# será realizada nas Células 32–36.
# ------------------------------------------------------------

report = {
    "cell": "13/36",
    "facility_id": "6065",
    "unit_id": "2",
    "year": 2025,
    "selected_model": selected_family,
    "selected_k": K,
    "selected_seed": int(
        HMM_SELECTION_12["selected_seed"]
    ),
    "partitions": {
        row["partition"]: {
            "n_hours": row["n_hours"],
            "n_cycles": row["n_cycles"],
            "agreement_viterbi_forward": (
                row["agreement_viterbi_forward"]
            ),
            "mean_filtered_confidence": (
                row["mean_filtered_confidence"]
            ),
            "mean_filtered_entropy": (
                row["mean_filtered_entropy"]
            ),
        }
        for row in summary_rows
    },
    "max_causal_prefix_difference": (
        max_causal_difference
    ),
    "new_models_trained": False,
    "scaler_refitted": False,
    "test_used_for_model_selection": False,
    "co2_used": False,
    "final_features_exported": False,
}

report_path = (
    report_dir / "cell13_inference_summary.json"
)

with open(
    report_path,
    "w",
    encoding="utf-8"
) as f:
    _project_dump(
        report,
        f,
        ensure_ascii=False,
        indent=2
    )

_project_print("\n4. RELATÓRIO GERADO")
_project_print(report_path)

_project_print("\n5. VERIFICAÇÕES FINAIS")
_project_print("Novo treinamento: NÃO")
_project_print("Scaler reajustado: NÃO")
_project_print("Modelo alterado: NÃO")
_project_print("CO2 utilizado: NÃO")
_project_print("Teste utilizado para seleção: NÃO")
_project_print("Features finais exportadas: NÃO")

_project_print(
    "\nCÉLULA 13 CONCLUÍDA — "
    "Viterbi retrospectivo e filtragem causal "
    "calculados por ciclo ON."
)


In [ ]:
# V2-B — Diagnóstico do HMM selecionado e transparência da decisão.
# Executar após a célula original 12. Não retreina e não muda HMM_SELECTED.
import numpy as np
import pandas as pd
import json
from IPython.display import display

for obj in ("HMM_SELECTION_12", "HMM_SELECTED", "audit_hmm_v2", "v2_dir"):
    if obj not in globals():
        raise RuntimeError(f"Objeto ausente: {obj}")
selected_key_v2 = (
    HMM_SELECTION_12["selected_model"],
    int(HMM_SELECTION_12["selected_k"]),
    int(HMM_SELECTION_12["selected_seed"]),
)
selected_row_v2 = audit_hmm_v2.loc[
    (audit_hmm_v2["model"] == selected_key_v2[0]) &
    (audit_hmm_v2["k"] == selected_key_v2[1]) &
    (audit_hmm_v2["seed"] == selected_key_v2[2])
]
if len(selected_row_v2) != 1:
    raise AssertionError("Modelo selecionado não corresponde a um único ajuste.")
# Para covariance_type='diag', versões de hmmlearn podem expor matrizes diagonais.
cov_v2 = np.asarray(HMM_SELECTED.covars_, dtype=float)
if cov_v2.ndim == 3:
    cov_diag_v2 = np.diagonal(cov_v2, axis1=1, axis2=2)
elif cov_v2.ndim == 2:
    cov_diag_v2 = cov_v2
else:
    raise ValueError(f"Forma de covariância inesperada: {cov_v2.shape}")
summary_v2 = {
    "model": selected_key_v2[0],
    "k": selected_key_v2[1],
    "seed": selected_key_v2[2],
    "min_covariance_diagonal": float(cov_diag_v2.min()),
    "max_covariance_diagonal": float(cov_diag_v2.max()),
    "transition_rows_sum_to_one": bool(np.allclose(HMM_SELECTED.transmat_.sum(axis=1), 1)),
    "selection_original_preserved": True,
    "selection_family_priority_is_hypothesis": True,
    "test_used_for_model_selection_v2": False,
}
summary_v2.update(selected_row_v2.iloc[0][[
    "warning_count_v2", "hit_iteration_limit", "last_ll_change"
]].to_dict())
with open(v2_dir / "selected_hmm_diagnostic.json", "w", encoding="utf-8") as f:
    _project_dump(summary_v2, f, indent=2, ensure_ascii=False,
              default=lambda x: x.item() if hasattr(x, "item") else str(x))
_project_display(pd.DataFrame([summary_v2]).T.rename(columns={0:"resultado"}))
if summary_v2["warning_count_v2"] or summary_v2["hit_iteration_limit"]:
    _project_print("ATENÇÃO: inspecione os alertas do HMM escolhido antes de afirmar convergência robusta.")


In [ ]:

# ============================================================
# FASE 2 — CÉLULA 14/36
# CARACTERIZAÇÃO FÍSICA E INTERPRETAÇÃO DOS ESTADOS HMM
# ============================================================
#
# UNIDADE:
# Iatan | Facility 6065 | Unit 2 | 2025
#
# MODELO CONGELADO:
# M2_LOAD_DELTA | K=5 | Seed=11
#
# OBJETIVOS:
# 1. Recuperar carga e rampa na escala física.
# 2. Caracterizar os cinco estados do HMM.
# 3. Avaliar ocupação, duração e transições.
# 4. Comparar Viterbi e forward.
# 5. Propor rótulos operacionais descritivos.
# 6. Verificar consistência na validação.
# 7. Exportar perfis e documentação.
#
# NÃO REALIZA:
# - Novo treinamento ou seleção de HMM.
# - Reajuste do scaler.
# - Utilização de CO2.
# - Avaliação do conjunto de teste.
# - Alteração dos estados inferidos.
# ============================================================

from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display

_project_print("=" * 72)
_project_print("FASE 2 — CÉLULA 14/36 | PERFIL DOS ESTADOS HMM")
_project_print("=" * 72)

# ------------------------------------------------------------
# 1. VERIFICAÇÃO DOS PRÉ-REQUISITOS
# ------------------------------------------------------------

required = [
    "HMM_SELECTED",
    "SCALER_SELECTED",
    "HMM_SELECTION_12",
    "hmm13_inference",
]

missing = [x for x in required if x not in globals()]

if missing:
    raise RuntimeError(
        f"Objetos ausentes: {missing}. "
        "Execute as Células 12 e 13."
    )

MODEL_NAME = HMM_SELECTION_12["selected_model"]
K = int(HMM_SELECTION_12["selected_k"])

if MODEL_NAME != "M2_LOAD_DELTA":
    raise AssertionError(
        "Esta caracterização exige M2_LOAD_DELTA."
    )

if K != 5:
    raise AssertionError(
        "Número de estados incompatível."
    )

if HMM_SELECTED.n_components != K:
    raise AssertionError(
        "HMM incompatível com a seleção."
    )

if SCALER_SELECTED.n_features_in_ != 2:
    raise AssertionError(
        "Esperadas duas variáveis: carga e rampa."
    )

if "RUN_DIR" in globals():
    RUN_PATH = Path(RUN_DIR)
elif "run_dir" in globals():
    RUN_PATH = Path(run_dir)
else:
    raise RuntimeError(
        "Diretório da execução não encontrado."
    )

REPORT_DIR_14 = RUN_PATH / "reports"
FIGURE_DIR_14 = RUN_PATH / "figures"

REPORT_DIR_14.mkdir(parents=True, exist_ok=True)
FIGURE_DIR_14.mkdir(parents=True, exist_ok=True)

PARTITIONS_14 = ["train", "validation"]

_project_print("\nMODELO")
_project_print("Configuração:", MODEL_NAME)
_project_print("Estados:", K)
_project_print("Semente:", HMM_SELECTION_12["selected_seed"])

# ------------------------------------------------------------
# 2. RECUPERAÇÃO DAS VARIÁVEIS FÍSICAS
# ------------------------------------------------------------
#
# Ordem esperada no M2:
# coluna 0 = carga
# coluna 1 = delta de carga com sinal
#
# A inversão do scaler recupera a escala original
# das features fornecidas ao HMM.
#
# IMPORTANTE:
# MW e MW/h somente são válidos se essas forem
# as unidades originais da Célula 07.
# ------------------------------------------------------------

hmm14_data = {}

for partition in PARTITIONS_14:

    result = hmm13_inference[partition]

    X_scaled = np.asarray(
        result["X"],
        dtype=float
    )

    X_original = SCALER_SELECTED.inverse_transform(
        X_scaled
    )

    if X_original.shape[1] != 2:
        raise AssertionError(
            "Número inesperado de variáveis."
        )

    load_mw = X_original[:, 0]
    delta_mw = X_original[:, 1]

    viterbi = np.asarray(
        result["viterbi_states"],
        dtype=int
    )

    filtered = np.asarray(
        result["filtered_state"],
        dtype=int
    )

    confidence = np.asarray(
        result["filtered_confidence"],
        dtype=float
    )

    cycle_index = np.asarray(
        result["cycle_index"],
        dtype=int
    )

    n = len(load_mw)

    if not all(
        len(a) == n
        for a in [
            delta_mw,
            viterbi,
            filtered,
            confidence,
            cycle_index,
        ]
    ):
        raise AssertionError(
            f"Comprimentos incompatíveis: {partition}"
        )

    df = pd.DataFrame({
        "partition": partition,
        "hour_index": np.arange(n),
        "cycle_index": cycle_index,
        "load_mw": load_mw,
        "delta_mw": delta_mw,
        "state_viterbi": viterbi,
        "state_forward": filtered,
        "confidence_forward": confidence,
    })

    hmm14_data[partition] = df

_project_print("\n1. DADOS FÍSICOS RECUPERADOS")

for partition, df in hmm14_data.items():

    _project_print(
        f"{partition}: "
        f"{len(df)} horas | "
        f"{df['cycle_index'].nunique()} ciclos"
    )

    _project_display(
        df[
            [
                "load_mw",
                "delta_mw",
                "state_viterbi",
                "state_forward",
            ]
        ].describe().round(3)
    )

# ------------------------------------------------------------
# 3. VERIFICAÇÃO DA CONSISTÊNCIA DAS RAMPAS
# ------------------------------------------------------------
#
# A diferença entre horas consecutivas é calculada
# apenas dentro de cada ciclo ON.
#
# A primeira hora de cada ciclo pode ter uma
# convenção específica de delta definida na Célula 07.
# Por isso, ela não entra na comparação.
# ------------------------------------------------------------

ramp_checks = []

for partition, df in hmm14_data.items():

    observed = []
    reconstructed = []

    for _, cycle in df.groupby(
        "cycle_index",
        sort=False
    ):

        if len(cycle) < 2:
            continue

        observed.extend(
            cycle["delta_mw"].iloc[1:].to_numpy()
        )

        reconstructed.extend(
            np.diff(
                cycle["load_mw"].to_numpy()
            )
        )

    observed = np.asarray(observed)
    reconstructed = np.asarray(reconstructed)

    if len(observed):

        abs_error = np.abs(
            observed - reconstructed
        )

        ramp_checks.append({
            "partition": partition,
            "n_compared": len(observed),
            "mae": float(abs_error.mean()),
            "max_error": float(abs_error.max()),
        })

audit14_ramp_consistency = pd.DataFrame(
    ramp_checks
)

_project_print("\n2. CONSISTÊNCIA DAS RAMPAS")
_project_display(
    audit14_ramp_consistency.round(6)
)

# ------------------------------------------------------------
# 4. PERFIL FÍSICO POR ESTADO
# ------------------------------------------------------------

def hmm14_state_profile(df, state_column):

    rows = []

    for state in range(K):

        subset = df.loc[
            df[state_column] == state
        ]

        if subset.empty:

            rows.append({
                "state": state,
                "n_hours": 0,
                "occupancy": 0.0,
                "load_mean": np.nan,
                "load_median": np.nan,
                "load_q10": np.nan,
                "load_q90": np.nan,
                "delta_mean": np.nan,
                "delta_median": np.nan,
                "delta_q10": np.nan,
                "delta_q90": np.nan,
                "ramp_up_share": np.nan,
                "ramp_down_share": np.nan,
                "near_zero_ramp_share": np.nan,
            })

            continue

        load = subset["load_mw"]
        delta = subset["delta_mw"]

        # Tolerância operacional para rampa quase nula:
        # 1% da amplitude observada no treinamento.
        threshold = RAMP_THRESHOLD_14

        rows.append({
            "state": state,
            "n_hours": len(subset),
            "occupancy": len(subset) / len(df),
            "load_mean": float(load.mean()),
            "load_median": float(load.median()),
            "load_q10": float(load.quantile(0.10)),
            "load_q90": float(load.quantile(0.90)),
            "delta_mean": float(delta.mean()),
            "delta_median": float(delta.median()),
            "delta_q10": float(delta.quantile(0.10)),
            "delta_q90": float(delta.quantile(0.90)),
            "ramp_up_share": float(
                (delta > threshold).mean()
            ),
            "ramp_down_share": float(
                (delta < -threshold).mean()
            ),
            "near_zero_ramp_share": float(
                (delta.abs() <= threshold).mean()
            ),
        })

    return pd.DataFrame(rows)


train_load = hmm14_data["train"]["load_mw"]

LOAD_RANGE_14 = float(
    train_load.max() - train_load.min()
)

RAMP_THRESHOLD_14 = max(
    0.01 * LOAD_RANGE_14,
    1e-9
)

_project_print(
    "\nLimiar descritivo para rampa quase nula:",
    f"{RAMP_THRESHOLD_14:.4f} MW/h"
)

profiles = {}

for partition in PARTITIONS_14:

    df = hmm14_data[partition]

    for method, column in [
        ("viterbi", "state_viterbi"),
        ("forward", "state_forward"),
    ]:

        profiles[(partition, method)] = (
            hmm14_state_profile(
                df,
                column
            )
        )

profile_train = profiles[
    ("train", "viterbi")
]

profile_validation = profiles[
    ("validation", "viterbi")
]

_project_print("\n3. PERFIL DOS ESTADOS — TREINAMENTO / VITERBI")

_project_display(
    profile_train.round(3)
)

_project_print("\n4. PERFIL DOS ESTADOS — VALIDAÇÃO / VITERBI")

_project_display(
    profile_validation.round(3)
)

# ------------------------------------------------------------
# 5. DURAÇÃO DOS EPISÓDIOS POR ESTADO
# ------------------------------------------------------------
#
# Cada episódio corresponde a uma sequência contínua
# do mesmo estado dentro de um ciclo ON.
# ------------------------------------------------------------

def hmm14_state_runs(df, state_column):

    runs = []

    for cycle_id, cycle in df.groupby(
        "cycle_index",
        sort=False
    ):

        states = cycle[
            state_column
        ].to_numpy()

        if len(states) == 0:
            continue

        boundaries = np.r_[
            0,
            np.flatnonzero(
                states[1:] != states[:-1]
            ) + 1,
            len(states)
        ]

        for start, end in zip(
            boundaries[:-1],
            boundaries[1:]
        ):

            runs.append({
                "cycle_index": int(cycle_id),
                "state": int(states[start]),
                "duration_hours": int(end - start),
                "start_in_cycle": int(start),
                "end_in_cycle": int(end - 1),
            })

    return pd.DataFrame(runs)


runs_train = hmm14_state_runs(
    hmm14_data["train"],
    "state_viterbi"
)

duration_profile = (
    runs_train.groupby("state")
    .agg(
        n_episodes=("duration_hours", "size"),
        mean_duration=("duration_hours", "mean"),
        median_duration=("duration_hours", "median"),
        q90_duration=(
            "duration_hours",
            lambda x: x.quantile(0.90)
        ),
        max_duration=("duration_hours", "max"),
    )
    .reset_index()
)

_project_print("\n5. DURAÇÃO DOS EPISÓDIOS — TREINAMENTO")
_project_display(
    duration_profile.round(2)
)

# ------------------------------------------------------------
# 6. TRANSIÇÕES OBSERVADAS ENTRE ESTADOS
# ------------------------------------------------------------
#
# Transições contadas somente dentro dos ciclos ON.
# Não incluir passagem de um ciclo ao próximo.
# ------------------------------------------------------------

transition_counts = np.zeros(
    (K, K),
    dtype=int
)

for _, cycle in hmm14_data["train"].groupby(
    "cycle_index",
    sort=False
):

    states = cycle[
        "state_viterbi"
    ].to_numpy()

    for origin, destination in zip(
        states[:-1],
        states[1:]
    ):

        transition_counts[
            origin,
            destination
        ] += 1

transition_df = pd.DataFrame(
    transition_counts,
    index=[
        f"S{i}" for i in range(K)
    ],
    columns=[
        f"S{i}" for i in range(K)
    ]
)

_project_print("\n6. MATRIZ DE TRANSIÇÕES OBSERVADAS")
_project_display(transition_df)

# ------------------------------------------------------------
# 7. PROPOSTA DE RÓTULOS OPERACIONAIS
# ------------------------------------------------------------
#
# Rótulos descritivos, não diagnósticos físicos.
#
# Critérios:
# - Nível relativo de carga no treinamento.
# - Predominância de rampa positiva ou negativa.
# - Estabilidade relativa.
#
# Os IDs originais do HMM não são alterados.
# ------------------------------------------------------------

load_rank = (
    profile_train.sort_values(
        "load_median"
    )["state"]
    .tolist()
)

relative_load = {}

for rank, state in enumerate(load_rank):

    if rank == 0:
        label = "carga mínima"
    elif rank == 1:
        label = "carga baixa"
    elif rank == 2:
        label = "carga intermediária"
    elif rank == 3:
        label = "carga elevada"
    else:
        label = "carga máxima"

    relative_load[state] = label

operational_labels = {}

for _, row in profile_train.iterrows():

    state = int(row["state"])

    if row["n_hours"] == 0:

        operational_labels[state] = (
            f"S{state} — não observado"
        )
        continue

    up = row["ramp_up_share"]
    down = row["ramp_down_share"]
    stable = row["near_zero_ramp_share"]

    if up >= 0.60:
        behavior = "rampa ascendente"

    elif down >= 0.60:
        behavior = "rampa descendente"

    elif stable >= 0.60:
        behavior = "carga relativamente estável"

    else:
        behavior = "operação variável"

    operational_labels[state] = (
        f"S{state} — "
        f"{relative_load[state]}, "
        f"{behavior}"
    )

label_table = pd.DataFrame({
    "state": list(operational_labels.keys()),
    "operational_label": list(
        operational_labels.values()
    )
})

_project_print("\n7. RÓTULOS OPERACIONAIS PROPOSTOS")
_project_display(label_table)

# ------------------------------------------------------------
# 8. VISUALIZAÇÕES DOS PERFIS
# ------------------------------------------------------------

train_df = hmm14_data["train"]

fig, axes = plt.subplots(
    2, 2,
    figsize=(15, 11)
)

# 8.1 Carga por estado
load_groups = [
    train_df.loc[
        train_df["state_viterbi"] == state,
        "load_mw"
    ].to_numpy()
    for state in range(K)
]

axes[0, 0].boxplot(
    load_groups,
    tick_labels=[
        f"S{i}" for i in range(K)
    ],
    showfliers=False
)

axes[0, 0].set_title(
    "Distribuição da carga por estado"
)
axes[0, 0].set_ylabel("Carga (MW)")
axes[0, 0].grid(alpha=0.2)

# 8.2 Rampas por estado
delta_groups = [
    train_df.loc[
        train_df["state_viterbi"] == state,
        "delta_mw"
    ].to_numpy()
    for state in range(K)
]

axes[0, 1].boxplot(
    delta_groups,
    tick_labels=[
        f"S{i}" for i in range(K)
    ],
    showfliers=False
)

axes[0, 1].axhline(
    0,
    linestyle="--",
    linewidth=1
)

axes[0, 1].set_title(
    "Distribuição das rampas por estado"
)
axes[0, 1].set_ylabel("Rampa (MW/h)")
axes[0, 1].grid(alpha=0.2)

# 8.3 Ocupação dos estados
axes[1, 0].bar(
    profile_train["state"],
    100 * profile_train["occupancy"]
)

axes[1, 0].set_title(
    "Ocupação dos estados — treinamento"
)
axes[1, 0].set_xlabel("Estado HMM")
axes[1, 0].set_ylabel("Horas ON (%)")
axes[1, 0].set_xticks(range(K))
axes[1, 0].grid(
    axis="y",
    alpha=0.2
)

# 8.4 Duração mediana dos episódios
axes[1, 1].bar(
    duration_profile["state"],
    duration_profile["median_duration"]
)

axes[1, 1].set_title(
    "Duração mediana por estado"
)
axes[1, 1].set_xlabel("Estado HMM")
axes[1, 1].set_ylabel("Horas")
axes[1, 1].set_xticks(range(K))
axes[1, 1].grid(
    axis="y",
    alpha=0.2
)

plt.tight_layout()

figure_path = (
    FIGURE_DIR_14
    / "cell14_state_profiles.png"
)

plt.savefig(
    figure_path,
    dpi=180,
    bbox_inches="tight"
)

plt.show()

# ------------------------------------------------------------
# 9. COMPARAÇÃO TREINAMENTO × VALIDAÇÃO
# ------------------------------------------------------------
#
# Os estados mantêm os mesmos IDs do HMM congelado.
# Nenhum rótulo é reajustado pela validação.
# ------------------------------------------------------------

comparison = (
    profile_train[
        [
            "state",
            "occupancy",
            "load_median",
            "delta_median",
        ]
    ]
    .merge(
        profile_validation[
            [
                "state",
                "occupancy",
                "load_median",
                "delta_median",
            ]
        ],
        on="state",
        suffixes=("_train", "_validation"),
        validate="one_to_one",
    )
)

comparison["occupancy_gap_pp"] = (
    100 * (
        comparison["occupancy_validation"]
        - comparison["occupancy_train"]
    )
)

comparison["load_median_gap_mw"] = (
    comparison["load_median_validation"]
    - comparison["load_median_train"]
)

comparison["delta_median_gap_mw"] = (
    comparison["delta_median_validation"]
    - comparison["delta_median_train"]
)

_project_print("\n8. CONSISTÊNCIA TREINAMENTO × VALIDAÇÃO")

_project_display(
    comparison.round(3)
)

# ------------------------------------------------------------
# 10. EXPORTAÇÃO DOS RESULTADOS
# ------------------------------------------------------------

_project_csv(profile_train, 
    REPORT_DIR_14
    / "cell14_state_profile_train.csv",
    index=False
)

_project_csv(profile_validation, 
    REPORT_DIR_14
    / "cell14_state_profile_validation.csv",
    index=False
)

_project_csv(duration_profile, 
    REPORT_DIR_14
    / "cell14_state_durations.csv",
    index=False
)

_project_csv(transition_df, 
    REPORT_DIR_14
    / "cell14_transition_counts.csv"
)

_project_csv(label_table, 
    REPORT_DIR_14
    / "cell14_operational_labels.csv",
    index=False
)

_project_csv(comparison, 
    REPORT_DIR_14
    / "cell14_train_validation_comparison.csv",
    index=False
)

_project_csv(audit14_ramp_consistency, 
    REPORT_DIR_14
    / "cell14_ramp_consistency.csv",
    index=False
)

# ------------------------------------------------------------
# 11. MANIFESTO METODOLÓGICO
# ------------------------------------------------------------

manifest = {
    "cell": "14/36",
    "facility": "Iatan",
    "facility_id": "6065",
    "unit_id": "2",
    "year": 2025,
    "selected_model": MODEL_NAME,
    "selected_k": K,
    "selected_seed": int(
        HMM_SELECTION_12["selected_seed"]
    ),
    "partitions_used": PARTITIONS_14,
    "state_label_source": "train",
    "state_label_method": (
        "relative load median + ramp direction shares"
    ),
    "ramp_threshold_mw_per_hour": (
        RAMP_THRESHOLD_14
    ),
    "state_labels": {
        str(k): v
        for k, v in operational_labels.items()
    },
    "co2_used": False,
    "test_used": False,
    "hmm_refitted": False,
    "scaler_refitted": False,
    "state_ids_modified": False,
    "labels_are_descriptive": True,
}

manifest_path = (
    REPORT_DIR_14
    / "cell14_state_interpretation.json"
)

with open(
    manifest_path,
    "w",
    encoding="utf-8"
) as f:
    _project_dump(
        manifest,
        f,
        ensure_ascii=False,
        indent=2
    )

# ------------------------------------------------------------
# 12. RESULTADO FINAL
# ------------------------------------------------------------

_project_print("\n" + "=" * 72)
_project_print("RESULTADO FINAL — CÉLULA 14/36")
_project_print("=" * 72)

_project_print("Modelo:", MODEL_NAME)
_project_print("Estados:", K)
_project_print("Semente:", HMM_SELECTION_12["selected_seed"])

_project_print("\nRótulos descritivos:")
for state, label in operational_labels.items():
    _project_print(f"  {state}: {label}")

_project_print("\nArquivos:")
_project_print("Perfis, durações e transições: salvos")
_project_print("Figura:", figure_path)
_project_print("Manifesto:", manifest_path)

_project_print("\nVerificações:")
_project_print("HMM reajustado: NÃO")
_project_print("Scaler reajustado: NÃO")
_project_print("CO2 utilizado: NÃO")
_project_print("Teste utilizado: NÃO")
_project_print("IDs dos estados modificados: NÃO")

_project_print(
    "\nCÉLULA 14 CONCLUÍDA — "
    "Perfis físicos e rótulos operacionais "
    "descritivos gerados."
)


In [ ]:

# ============================================================
# FASE 2 — CÉLULA 15/36
# VISUALIZAÇÃO TEMPORAL DOS REGIMES OPERACIONAIS
# E INVESTIGAÇÃO DO ESTADO S4
# ============================================================
#
# PROJETO:
# Iatan | Facility 6065 | Unit 2 | Ano 2025
#
# OBJETIVOS:
# 1. Visualizar diferentes ciclos ON completos.
# 2. Investigar a posição temporal do estado S4.
# 3. Verificar horas com geração nula nos ciclos ON.
# 4. Caracterizar entradas e saídas do estado S4.
# 5. Comparar treinamento e validação sem reajustes.
# 6. Documentar evidências para os futuros event logs.
#
# RESTRIÇÕES:
# - Não utilizar CO2.
# - Não reajustar o HMM ou o scaler.
# - Não alterar as classificações já inferidas.
# - Não utilizar o teste para decisões metodológicas.
# - Não declarar partida térmica sem evidência CEMS.
#
# SAÍDAS:
# hmm15_cycle_summary
# hmm15_s4_episodes
# hmm15_s4_transitions
# hmm15_s4_diagnostics
# Figuras e relatórios em RUN_DIR.
# ============================================================

from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display

_project_print("=" * 76)
_project_print("FASE 2 — CÉLULA 15/36 | ANÁLISE TEMPORAL DOS REGIMES")
_project_print("=" * 76)

# ============================================================
# 1. PRÉ-REQUISITOS E CONFIGURAÇÃO
# ============================================================

required_15 = [
    "HMM_SELECTED",
    "HMM_SELECTION_12",
    "hmm13_inference",
    "hmm14_data",
    "operational_labels",
]

missing_15 = [
    name for name in required_15
    if name not in globals()
]

if missing_15:
    raise RuntimeError(
        "Execute as Células 12 a 14. "
        f"Objetos ausentes: {missing_15}"
    )

MODEL_15 = HMM_SELECTION_12["selected_model"]
K_15 = int(HMM_SELECTION_12["selected_k"])
S4_15 = 4

assert MODEL_15 == "M2_LOAD_DELTA"
assert K_15 == 5
assert HMM_SELECTED.n_components == K_15

if "RUN_DIR" in globals():
    RUN_PATH_15 = Path(RUN_DIR)
elif "run_dir" in globals():
    RUN_PATH_15 = Path(run_dir)
else:
    raise RuntimeError("RUN_DIR não encontrado.")

REPORT_DIR_15 = RUN_PATH_15 / "reports"
FIGURE_DIR_15 = RUN_PATH_15 / "figures"

REPORT_DIR_15.mkdir(parents=True, exist_ok=True)
FIGURE_DIR_15.mkdir(parents=True, exist_ok=True)

PARTITIONS_15 = ["train", "validation"]

# Tolerância numérica para geração zero.
# Não é um limiar de potência mínima operacional.
ZERO_TOL_15 = 1e-6

# Quantidade máxima de ciclos mostrados por partição.
MAX_PLOTS_PER_PARTITION_15 = 3

_project_print("\nModelo:", MODEL_15)
_project_print("K:", K_15)
_project_print("Semente:", HMM_SELECTION_12["selected_seed"])
_project_print("Estado investigado: S4")
_project_print("Partições:", PARTITIONS_15)

# ============================================================
# 2. PREPARAÇÃO DOS CICLOS
# ============================================================
#
# Utiliza os dados físicos reconstruídos na Célula 14.
# Os IDs dos estados e dos ciclos são preservados.
# ============================================================

cycle_tables_15 = {}
cycle_rows_15 = []

for partition in PARTITIONS_15:

    df = hmm14_data[partition].copy()

    required_columns = [
        "cycle_index",
        "load_mw",
        "delta_mw",
        "state_viterbi",
        "state_forward",
        "confidence_forward",
    ]

    missing_columns = [
        c for c in required_columns
        if c not in df.columns
    ]

    if missing_columns:
        raise ValueError(
            f"{partition}: colunas ausentes: "
            f"{missing_columns}"
        )

    if not np.isfinite(
        df[["load_mw", "delta_mw"]].to_numpy()
    ).all():
        raise ValueError(
            f"{partition}: dados físicos não finitos."
        )

    df["zero_generation"] = (
        df["load_mw"].abs() <= ZERO_TOL_15
    )

    df["is_s4"] = (
        df["state_viterbi"] == S4_15
    )

    df["hour_in_cycle"] = (
        df.groupby("cycle_index", sort=False)
        .cumcount()
    )

    df["cycle_length"] = (
        df.groupby("cycle_index")["cycle_index"]
        .transform("size")
    )

    df["relative_position"] = np.where(
        df["cycle_length"] > 1,
        df["hour_in_cycle"]
        / (df["cycle_length"] - 1),
        0.0
    )

    cycle_tables_15[partition] = df

    for cycle_id, cycle in df.groupby(
        "cycle_index", sort=False
    ):

        states = cycle["state_viterbi"].to_numpy()
        loads = cycle["load_mw"].to_numpy()

        s4_mask = states == S4_15
        zero_mask = (
            np.abs(loads) <= ZERO_TOL_15
        )

        positive_idx = np.flatnonzero(
            loads > ZERO_TOL_15
        )

        first_positive = (
            int(positive_idx[0])
            if len(positive_idx)
            else None
        )

        s4_idx = np.flatnonzero(s4_mask)

        first_s4 = (
            int(s4_idx[0])
            if len(s4_idx)
            else None
        )

        last_s4 = (
            int(s4_idx[-1])
            if len(s4_idx)
            else None
        )

        cycle_rows_15.append({
            "partition": partition,
            "cycle_index": int(cycle_id),
            "n_hours": int(len(cycle)),
            "n_s4_hours": int(s4_mask.sum()),
            "n_zero_generation_hours": int(
                zero_mask.sum()
            ),
            "s4_at_start": bool(
                states[0] == S4_15
            ),
            "s4_at_end": bool(
                states[-1] == S4_15
            ),
            "first_s4_hour": first_s4,
            "last_s4_hour": last_s4,
            "first_positive_generation_hour": (
                first_positive
            ),
            "max_load_mw": float(
                np.max(loads)
            ),
            "mean_load_mw": float(
                np.mean(loads)
            ),
            "n_state_changes": int(
                np.sum(states[1:] != states[:-1])
            ),
        })

hmm15_cycle_summary = pd.DataFrame(
    cycle_rows_15
)

_project_print("\n1. RESUMO DOS CICLOS")
_project_display(
    hmm15_cycle_summary.round(3)
)

# ============================================================
# 3. EPISÓDIOS CONTÍNUOS DO ESTADO S4
# ============================================================
#
# Distingue:
# - S4 no início do ciclo;
# - S4 no interior;
# - S4 no final.
#
# Um episódio pode tocar mais de uma fronteira,
# especialmente em ciclos muito curtos.
# ============================================================

s4_episodes_15 = []

for partition, df in cycle_tables_15.items():

    for cycle_id, cycle in df.groupby(
        "cycle_index", sort=False
    ):

        states = cycle[
            "state_viterbi"
        ].to_numpy()

        loads = cycle[
            "load_mw"
        ].to_numpy()

        n = len(states)
        mask = states == S4_15

        if not mask.any():
            continue

        boundaries = np.r_[
            0,
            np.flatnonzero(
                mask[1:] != mask[:-1]
            ) + 1,
            n
        ]

        for start, end in zip(
            boundaries[:-1],
            boundaries[1:]
        ):

            if not mask[start]:
                continue

            at_start = start == 0
            at_end = end == n

            if at_start and at_end:
                position = "whole_cycle"
            elif at_start:
                position = "start"
            elif at_end:
                position = "end"
            else:
                position = "interior"

            next_state = (
                int(states[end])
                if end < n
                else None
            )

            previous_state = (
                int(states[start - 1])
                if start > 0
                else None
            )

            episode_loads = loads[start:end]

            s4_episodes_15.append({
                "partition": partition,
                "cycle_index": int(cycle_id),
                "start_hour": int(start),
                "end_hour_inclusive": int(end - 1),
                "duration_hours": int(end - start),
                "position": position,
                "previous_state": previous_state,
                "next_state": next_state,
                "mean_load_mw": float(
                    episode_loads.mean()
                ),
                "max_abs_load_mw": float(
                    np.max(np.abs(episode_loads))
                ),
                "zero_generation_share": float(
                    np.mean(
                        np.abs(episode_loads)
                        <= ZERO_TOL_15
                    )
                ),
            })

hmm15_s4_episodes = pd.DataFrame(
    s4_episodes_15
)

if hmm15_s4_episodes.empty:
    raise RuntimeError(
        "Nenhum episódio S4 identificado. "
        "Verifique os IDs dos estados."
    )

_project_print("\n2. EPISÓDIOS DO ESTADO S4")
_project_display(
    hmm15_s4_episodes.round(4)
)

_project_print("\nDistribuição dos episódios por posição:")
_project_display(
    pd.crosstab(
        hmm15_s4_episodes["partition"],
        hmm15_s4_episodes["position"]
    )
)

# ============================================================
# 4. RELAÇÃO ENTRE S4 E GERAÇÃO ZERO
# ============================================================
#
# Verifica as duas direções:
#
# P(Geração=0 | S4)
# P(S4 | Geração=0)
#
# São frequências descritivas, não probabilidades
# de causalidade física.
# ============================================================

diagnostics_15 = []

for partition, df in cycle_tables_15.items():

    s4 = df["is_s4"].to_numpy()
    zero = df["zero_generation"].to_numpy()

    n_s4 = int(s4.sum())
    n_zero = int(zero.sum())
    n_both = int(np.sum(s4 & zero))

    diagnostics_15.append({
        "partition": partition,
        "n_hours": int(len(df)),
        "n_s4_hours": n_s4,
        "n_zero_generation_hours": n_zero,
        "n_s4_and_zero_hours": n_both,
        "p_zero_given_s4": (
            n_both / n_s4
            if n_s4 else np.nan
        ),
        "p_s4_given_zero": (
            n_both / n_zero
            if n_zero else np.nan
        ),
        "s4_with_positive_generation": int(
            np.sum(
                s4 & (df["load_mw"] > ZERO_TOL_15)
            )
        ),
        "zero_generation_outside_s4": int(
            np.sum(zero & ~s4)
        ),
    })

hmm15_s4_diagnostics = pd.DataFrame(
    diagnostics_15
)

_project_print("\n3. RELAÇÃO S4 × GERAÇÃO ZERO")
_project_display(
    hmm15_s4_diagnostics.round(5)
)

# ============================================================
# 5. TRANSIÇÕES ENVOLVENDO S4
# ============================================================
#
# Considera apenas mudanças de estado dentro
# de cada ciclo ON, sem autopermanências.
# ============================================================

transition_rows_15 = []

for partition, df in cycle_tables_15.items():

    for cycle_id, cycle in df.groupby(
        "cycle_index", sort=False
    ):

        states = cycle[
            "state_viterbi"
        ].to_numpy()

        loads = cycle[
            "load_mw"
        ].to_numpy()

        for t in range(1, len(states)):

            previous = int(states[t - 1])
            current = int(states[t])

            if previous == current:
                continue

            if (
                previous != S4_15
                and current != S4_15
            ):
                continue

            transition_rows_15.append({
                "partition": partition,
                "cycle_index": int(cycle_id),
                "hour_in_cycle": int(t),
                "origin": previous,
                "destination": current,
                "load_before_mw": float(
                    loads[t - 1]
                ),
                "load_after_mw": float(
                    loads[t]
                ),
                "delta_load_mw": float(
                    loads[t] - loads[t - 1]
                ),
            })

hmm15_s4_transitions = pd.DataFrame(
    transition_rows_15
)

_project_print("\n4. TRANSIÇÕES ENVOLVENDO S4")

if not hmm15_s4_transitions.empty:

    transition_summary_15 = (
        hmm15_s4_transitions
        .groupby(
            ["partition", "origin", "destination"],
            as_index=False
        )
        .agg(
            n_transitions=(
                "cycle_index", "size"
            ),
            median_load_after_mw=(
                "load_after_mw", "median"
            ),
            median_delta_load_mw=(
                "delta_load_mw", "median"
            ),
        )
    )

    _project_display(
        transition_summary_15.round(3)
    )

else:
    transition_summary_15 = pd.DataFrame()
    _project_print("Nenhuma transição envolvendo S4.")

# ============================================================
# 6. SELEÇÃO DE CICLOS REPRESENTATIVOS
# ============================================================
#
# Seleção determinística:
# - primeiro ciclo;
# - ciclo com maior número de horas S4;
# - ciclo mais longo.
#
# Remove duplicações.
# ============================================================

selected_cycles_15 = {}

for partition in PARTITIONS_15:

    summary = hmm15_cycle_summary.loc[
        hmm15_cycle_summary[
            "partition"
        ].eq(partition)
    ].copy()

    first_cycle = int(
        summary.iloc[0]["cycle_index"]
    )

    max_s4_cycle = int(
        summary.sort_values(
            ["n_s4_hours", "cycle_index"],
            ascending=[False, True]
        ).iloc[0]["cycle_index"]
    )

    longest_cycle = int(
        summary.sort_values(
            ["n_hours", "cycle_index"],
            ascending=[False, True]
        ).iloc[0]["cycle_index"]
    )

    selected = list(dict.fromkeys([
        first_cycle,
        max_s4_cycle,
        longest_cycle,
    ]))

    selected_cycles_15[partition] = (
        selected[:MAX_PLOTS_PER_PARTITION_15]
    )

_project_print("\n5. CICLOS SELECIONADOS PARA VISUALIZAÇÃO")
_project_print(selected_cycles_15)

# ============================================================
# 7. VISUALIZAÇÃO TEMPORAL DOS CICLOS
# ============================================================
#
# Cada figura apresenta:
# A. Carga física e horas com geração zero.
# B. Estados Viterbi e forward.
# C. Probabilidades filtradas por estado.
#
# As regiões S4 são destacadas.
# ============================================================

def plot_cycle_15(partition, cycle_id):

    df = cycle_tables_15[partition]

    cycle = df.loc[
        df["cycle_index"] == cycle_id
    ].copy()

    cycle = cycle.reset_index(drop=True)

    n = len(cycle)
    x = np.arange(n)

    original_result = hmm13_inference[
        partition
    ]

    original_cycle_mask = (
        original_result["cycle_index"]
        == cycle_id
    )

    probs = original_result[
        "filtered_probabilities"
    ][original_cycle_mask]

    if len(probs) != n:
        raise AssertionError(
            "Probabilidades desalinhadas "
            f"no ciclo {cycle_id}."
        )

    fig, axes = plt.subplots(
        3, 1,
        figsize=(15, 10),
        sharex=True,
        gridspec_kw={
            "height_ratios": [2, 1.2, 1.6]
        }
    )

    # A. Carga
    axes[0].plot(
        x,
        cycle["load_mw"],
        linewidth=1.5,
        label="Carga elétrica"
    )

    axes[0].scatter(
        x[
            cycle["zero_generation"].to_numpy()
        ],
        cycle.loc[
            cycle["zero_generation"],
            "load_mw"
        ],
        s=22,
        label="Geração zero",
        zorder=3
    )

    axes[0].set_ylabel("Carga (MW)")
    axes[0].set_title(
        f"{partition.upper()} | Ciclo ON {cycle_id}"
        f" | {n} horas"
    )
    axes[0].legend(loc="best")
    axes[0].grid(alpha=0.2)

    # B. Estados
    axes[1].step(
        x,
        cycle["state_viterbi"],
        where="mid",
        label="Viterbi retrospectivo",
        linewidth=1.5
    )

    axes[1].step(
        x,
        cycle["state_forward"],
        where="mid",
        linestyle="--",
        label="Forward causal",
        alpha=0.75
    )

    axes[1].set_ylabel("Estado HMM")
    axes[1].set_yticks(range(K_15))
    axes[1].legend(loc="best")
    axes[1].grid(alpha=0.2)

    # C. Probabilidades filtradas
    for state in range(K_15):

        axes[2].plot(
            x,
            probs[:, state],
            label=f"S{state}",
            linewidth=1.1
        )

    axes[2].set_ylabel(
        "Probabilidade filtrada"
    )
    axes[2].set_xlabel(
        "Horas desde o início do ciclo ON"
    )
    axes[2].set_ylim(-0.02, 1.02)
    axes[2].legend(
        ncol=K_15,
        loc="upper center"
    )
    axes[2].grid(alpha=0.2)

    # Destacar intervalos S4.
    mask = (
        cycle["state_viterbi"].to_numpy()
        == S4_15
    )

    boundaries = np.r_[
        0,
        np.flatnonzero(
            mask[1:] != mask[:-1]
        ) + 1,
        n
    ]

    for start, end in zip(
        boundaries[:-1],
        boundaries[1:]
    ):

        if not mask[start]:
            continue

        for ax in axes:

            ax.axvspan(
                start - 0.5,
                end - 0.5,
                alpha=0.12
            )

    plt.tight_layout()

    path = (
        FIGURE_DIR_15
        / (
            f"cell15_{partition}_"
            f"cycle_{cycle_id:03d}.png"
        )
    )

    plt.savefig(
        path,
        dpi=170,
        bbox_inches="tight"
    )

    plt.show()

    return path


figure_paths_15 = []

for partition, cycle_ids in (
    selected_cycles_15.items()
):

    for cycle_id in cycle_ids:

        path = plot_cycle_15(
            partition,
            cycle_id
        )

        figure_paths_15.append(
            str(path)
        )

# ============================================================
# 8. DISTRIBUIÇÃO TEMPORAL DO S4
# ============================================================
#
# Compara a localização dos episódios S4
# ao longo dos ciclos.
#
# A posição é relativa ao comprimento de cada ciclo,
# não ao horário do calendário.
# ============================================================

fig, axes = plt.subplots(
    1, 2,
    figsize=(14, 5),
    sharey=True
)

for ax, partition in zip(
    axes,
    PARTITIONS_15
):

    episodes = hmm15_s4_episodes.loc[
        hmm15_s4_episodes[
            "partition"
        ].eq(partition)
    ]

    cycle_lengths = (
        hmm15_cycle_summary.loc[
            hmm15_cycle_summary[
                "partition"
            ].eq(partition),
            ["cycle_index", "n_hours"]
        ]
    )

    episodes = episodes.merge(
        cycle_lengths,
        on="cycle_index",
        how="left",
        validate="many_to_one"
    )

    for _, row in episodes.iterrows():

        length = int(row["n_hours"])

        start = (
            row["start_hour"] / length
        )

        end = (
            (row["end_hour_inclusive"] + 1)
            / length
        )

        ax.plot(
            [start, end],
            [
                row["cycle_index"],
                row["cycle_index"]
            ],
            linewidth=5
        )

    ax.set_title(
        f"S4 ao longo dos ciclos — {partition}"
    )

    ax.set_xlabel(
        "Posição relativa no ciclo"
    )

    ax.set_xlim(0, 1)
    ax.grid(alpha=0.2)

axes[0].set_ylabel("ID do ciclo ON")

plt.tight_layout()

s4_timeline_path_15 = (
    FIGURE_DIR_15
    / "cell15_s4_relative_timeline.png"
)

plt.savefig(
    s4_timeline_path_15,
    dpi=180,
    bbox_inches="tight"
)

plt.show()

# ============================================================
# 9. SÍNTESE DAS EVIDÊNCIAS
# ============================================================

synthesis_rows_15 = []

for partition in PARTITIONS_15:

    cycles = hmm15_cycle_summary.loc[
        hmm15_cycle_summary[
            "partition"
        ].eq(partition)
    ]

    episodes = hmm15_s4_episodes.loc[
        hmm15_s4_episodes[
            "partition"
        ].eq(partition)
    ]

    diag = hmm15_s4_diagnostics.loc[
        hmm15_s4_diagnostics[
            "partition"
        ].eq(partition)
    ].iloc[0]

    synthesis_rows_15.append({
        "partition": partition,
        "n_cycles": int(len(cycles)),
        "cycles_starting_s4": int(
            cycles["s4_at_start"].sum()
        ),
        "cycles_ending_s4": int(
            cycles["s4_at_end"].sum()
        ),
        "s4_start_episodes": int(
            episodes["position"].isin(
                ["start", "whole_cycle"]
            ).sum()
        ),
        "s4_interior_episodes": int(
            episodes["position"].eq(
                "interior"
            ).sum()
        ),
        "s4_end_episodes": int(
            episodes["position"].isin(
                ["end", "whole_cycle"]
            ).sum()
        ),
        "p_zero_given_s4": float(
            diag["p_zero_given_s4"]
        ),
        "p_s4_given_zero": float(
            diag["p_s4_given_zero"]
        ),
        "median_s4_duration_hours": float(
            episodes["duration_hours"].median()
        ),
    })

audit15_synthesis = pd.DataFrame(
    synthesis_rows_15
)

_project_print("\n6. SÍNTESE DA INVESTIGAÇÃO S4")
_project_display(
    audit15_synthesis.round(4)
)

# ============================================================
# 10. EXPORTAÇÃO DOS RESULTADOS
# ============================================================

_project_csv(hmm15_cycle_summary, 
    REPORT_DIR_15 / "cell15_cycle_summary.csv",
    index=False
)

_project_csv(hmm15_s4_episodes, 
    REPORT_DIR_15 / "cell15_s4_episodes.csv",
    index=False
)

_project_csv(hmm15_s4_transitions, 
    REPORT_DIR_15 / "cell15_s4_transitions.csv",
    index=False
)

_project_csv(hmm15_s4_diagnostics, 
    REPORT_DIR_15 / "cell15_s4_diagnostics.csv",
    index=False
)

_project_csv(audit15_synthesis, 
    REPORT_DIR_15 / "cell15_s4_synthesis.csv",
    index=False
)

manifest_15 = {
    "cell": "15/36",
    "facility": "Iatan",
    "facility_id": "6065",
    "unit_id": "2",
    "year": 2025,
    "model": MODEL_15,
    "k": K_15,
    "seed": int(
        HMM_SELECTION_12["selected_seed"]
    ),
    "partitions": PARTITIONS_15,
    "investigated_state": S4_15,
    "zero_generation_tolerance_mw": (
        ZERO_TOL_15
    ),
    "hypothesis": (
        "S4 may represent a pre-generation "
        "or startup-related operational phase"
    ),
    "hypothesis_confirmed": False,
    "co2_used": False,
    "test_used": False,
    "hmm_refitted": False,
    "scaler_refitted": False,
    "state_ids_modified": False,
    "figures": figure_paths_15 + [
        str(s4_timeline_path_15)
    ],
}

manifest_path_15 = (
    REPORT_DIR_15
    / "cell15_temporal_regimes_manifest.json"
)

with open(
    manifest_path_15,
    "w",
    encoding="utf-8"
) as f:

    _project_dump(
        manifest_15,
        f,
        ensure_ascii=False,
        indent=2
    )

# ============================================================
# 11. RESULTADO FINAL
# ============================================================

_project_print("\n" + "=" * 76)
_project_print("RESULTADO FINAL — CÉLULA 15/36")
_project_print("=" * 76)

_project_print("Modelo:", MODEL_15)
_project_print("Estados:", K_15)
_project_print("Semente:", HMM_SELECTION_12["selected_seed"])

_project_print("\nDiagnóstico S4:")
_project_display(
    audit15_synthesis.round(4)
)

_project_print("\nArquivos:")
_project_print("Relatórios:", REPORT_DIR_15)
_project_print("Figuras:", FIGURE_DIR_15)
_project_print("Manifesto:", manifest_path_15)

_project_print("\nVerificações:")
_project_print("HMM reajustado: NÃO")
_project_print("Scaler reajustado: NÃO")
_project_print("CO2 utilizado: NÃO")
_project_print("Teste utilizado: NÃO")
_project_print("Estados modificados: NÃO")

_project_print(
    "\nCÉLULA 15 CONCLUÍDA — "
    "Análise temporal e investigação "
    "operacional do estado S4 realizadas."
)


In [ ]:

# ============================================================
# FASE 2 — CÉLULA 16/36
# MATRIZES DE TRANSIÇÃO E PERSISTÊNCIA DOS REGIMES HMM
# ============================================================
#
# OBJETIVOS
# 1. Recuperar as inferências congeladas da Célula 13.
# 2. Quantificar transições entre estados HMM.
# 3. Construir matrizes de contagem e probabilidade.
# 4. Comparar transições empíricas com o HMM treinado.
# 5. Medir persistência e duração dos episódios.
# 6. Comparar treinamento e validação.
# 7. Investigar a estrutura de transição de S4.
# 8. Gerar gráficos e relatórios reproduzíveis.
#
# RESTRIÇÕES
# - Não reajustar o HMM.
# - Não reajustar o scaler.
# - Não usar CO2.
# - Não usar teste para análise ou seleção.
# - Não conectar ciclos ON diferentes.
# - Não alterar estados previamente inferidos.
#
# ============================================================

import os
import json
import pickle
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display

# ------------------------------------------------------------
# 1. CONFIGURAÇÃO
# ------------------------------------------------------------

MODEL_NAME = "M2_LOAD_DELTA"
K = 5
SEED = int(HMM_SELECTION_12["selected_seed"])

STATES = list(range(K))
PARTITIONS = ["train", "validation"]

# Pasta da execução existente.
if "RUN_DIR" not in globals() or not Path(RUN_DIR).is_dir():
    raise RuntimeError("Execute a configuração da Fase 2 para recuperar RUN_DIR.")
ROOT16 = Path(RUN_DIR)

REPORT_DIR16 = ROOT16 / "reports"
FIGURE_DIR16 = ROOT16 / "figures"

REPORT_DIR16.mkdir(parents=True, exist_ok=True)
FIGURE_DIR16.mkdir(parents=True, exist_ok=True)

_project_print("=" * 76)
_project_print("FASE 2 — CÉLULA 16/36 | TRANSIÇÕES E PERSISTÊNCIA")
_project_print("=" * 76)
_project_print(f"Modelo: {MODEL_NAME}")
_project_print(f"K: {K}")
_project_print(f"Semente: {SEED}")
_project_print(f"Partições: {PARTITIONS}")

# ------------------------------------------------------------
# 2. RECUPERAR HMM CONGELADO
# ------------------------------------------------------------

model_candidates16 = [
    ROOT16 / "models" / "cell12_hmm_selected.pkl"
]

hmm16 = None

for path16 in model_candidates16:
    if path16.exists():
        with open(path16, "rb") as f:
            hmm16 = pickle.load(f)
        break

if hmm16 is None:
    for variable16 in [
        "hmm12_selected_model",
        "hmm12_model",
        "selected_hmm",
        "hmm_selected"
    ]:
        if variable16 in globals():
            hmm16 = globals()[variable16]
            break

if hmm16 is None:
    raise RuntimeError(
        "HMM congelado não localizado. "
        "Execute a Célula 12 antes desta célula."
    )

A16 = np.asarray(hmm16.transmat_, dtype=float)

assert A16.shape == (K, K), (
    f"Matriz HMM incompatível: {A16.shape}"
)

assert np.allclose(
    A16.sum(axis=1), 1.0, atol=1e-6
), "Matriz de transição HMM não normalizada."

# ------------------------------------------------------------
# 3. RECUPERAR INFERÊNCIAS DA CÉLULA 13
# ------------------------------------------------------------
#
# Não recalculamos estados.
# A inferência causal deve ter sido realizada anteriormente.
#
# Como a estrutura exata das variáveis pode variar entre
# versões do notebook, procuramos objetos conhecidos e
# validamos explicitamente o formato.
# ------------------------------------------------------------

def find_inference16():

    names = [
        "hmm13_inference",
        "hmm13_results",
        "hmm13_data",
        "inference13",
        "results13",
    ]

    for name in names:
        if name in globals():
            obj = globals()[name]
            if isinstance(obj, dict):
                if all(p in obj for p in PARTITIONS):
                    return obj, name

    raise RuntimeError(
        "Não foi encontrado um dicionário de inferência "
        "da Célula 13 com train e validation. "
        "Verifique o nome do objeto produzido naquela célula."
    )

inference16, inference_source16 = find_inference16()

_project_print(f"\nInferências recuperadas de: {inference_source16}")

# ------------------------------------------------------------
# 4. EXTRAÇÃO ESTRITA DAS SEQUÊNCIAS
# ------------------------------------------------------------

def get_field16(obj, names):

    if isinstance(obj, dict):
        for name in names:
            if name in obj:
                return obj[name]

    if isinstance(obj, pd.DataFrame):
        for name in names:
            if name in obj.columns:
                return obj[name].to_numpy()

    return None


def extract_sequences16(partition_obj):

    # Aceita lista de ciclos ou dicionário com ciclos.
    if isinstance(partition_obj, dict):

        for key in ["cycles", "sequences", "cycle_results"]:
            if key in partition_obj:
                partition_obj = partition_obj[key]
                break

        else:
            # Possível dicionário com arrays concatenados
            states = get_field16(
                partition_obj,
                [
                    "filtered_state",
                    "forward_state",
                    "state_forward",
                    "causal_state",
                    "state_filtered",
                    "hmm_state_filtered",
                ]
            )

            lengths = get_field16(
                partition_obj,
                ["lengths", "cycle_lengths"]
            )

            if states is not None and lengths is not None:

                states = np.asarray(states)
                lengths = np.asarray(lengths, dtype=int)

                assert lengths.sum() == len(states)

                offsets = np.cumsum(
                    np.r_[0, lengths]
                )

                return [
                    states[offsets[i]:offsets[i + 1]]
                    for i in range(len(lengths))
                ]

    if isinstance(partition_obj, (list, tuple)):

        sequences = []

        for cycle in partition_obj:

            states = get_field16(
                cycle,
                [
                    "filtered_state",
                    "forward_state",
                    "state_forward",
                    "causal_state",
                    "state_filtered",
                    "hmm_state_filtered",
                ]
            )

            if states is None:
                raise RuntimeError(
                    "Um ciclo não contém os estados causais."
                )

            states = np.asarray(states, dtype=int)
            sequences.append(states)

        return sequences

    raise RuntimeError(
        "Formato das inferências da Célula 13 não reconhecido."
    )


sequences16 = {}

for partition in PARTITIONS:

    sequences16[partition] = extract_sequences16(
        inference16[partition]
    )

    for seq in sequences16[partition]:

        assert len(seq) >= 1

        assert np.isin(
            seq, STATES
        ).all(), "Estado fora do intervalo 0..4."

    n_hours = sum(
        len(s) for s in sequences16[partition]
    )

    _project_print(
        f"{partition.upper()}: "
        f"{len(sequences16[partition])} ciclos | "
        f"{n_hours} horas"
    )

# Consistência com a Célula 13.
expected16 = {
    "train": (4504, 27),
    "validation": (675, 6)
}

for partition, (n_expected, c_expected) in expected16.items():

    n_observed = sum(
        len(s) for s in sequences16[partition]
    )

    c_observed = len(sequences16[partition])

    assert n_observed == n_expected, (
        f"{partition}: horas divergentes."
    )

    assert c_observed == c_expected, (
        f"{partition}: ciclos divergentes."
    )

# ------------------------------------------------------------
# 5. MATRIZES EMPÍRICAS DE TRANSIÇÃO
# ------------------------------------------------------------

def transition_counts16(sequences, k):

    counts = np.zeros(
        (k, k), dtype=np.int64
    )

    for seq in sequences:

        if len(seq) < 2:
            continue

        np.add.at(
            counts,
            (seq[:-1], seq[1:]),
            1
        )

    return counts


def transition_probabilities16(counts):

    totals = counts.sum(axis=1, keepdims=True)

    probabilities = np.divide(
        counts,
        totals,
        out=np.full(
            counts.shape, np.nan, dtype=float
        ),
        where=totals > 0
    )

    return probabilities


counts16 = {}
probs16 = {}

for partition in PARTITIONS:

    counts16[partition] = transition_counts16(
        sequences16[partition], K
    )

    probs16[partition] = transition_probabilities16(
        counts16[partition]
    )

    expected_transitions = sum(
        max(len(s) - 1, 0)
        for s in sequences16[partition]
    )

    assert counts16[partition].sum() == expected_transitions

    _project_print(
        f"\n{partition.upper()} | "
        f"{expected_transitions} transições internas"
    )

    _project_print("Contagens:")
    _project_display(
        pd.DataFrame(
            counts16[partition],
            index=[f"S{i}" for i in STATES],
            columns=[f"S{i}" for i in STATES]
        )
    )

    _project_print("Probabilidades condicionais:")
    _project_display(
        pd.DataFrame(
            probs16[partition],
            index=[f"S{i}" for i in STATES],
            columns=[f"S{i}" for i in STATES]
        ).round(4)
    )

# ------------------------------------------------------------
# 6. MATRIZ DE TRANSIÇÃO ESTIMADA PELO HMM
# ------------------------------------------------------------

_project_print("\nMATRIZ PARAMÉTRICA DO HMM")

_project_display(
    pd.DataFrame(
        A16,
        index=[f"S{i}" for i in STATES],
        columns=[f"S{i}" for i in STATES]
    ).round(4)
)

# Atenção:
# A matriz paramétrica é estimada no treinamento.
# As matrizes empíricas são calculadas a partir dos estados
# mais prováveis do filtro causal.
# Portanto, não precisam coincidir exatamente.

# ------------------------------------------------------------
# 7. EPISÓDIOS E PERSISTÊNCIA
# ------------------------------------------------------------

episode_rows16 = []

for partition in PARTITIONS:

    for cycle_idx, seq in enumerate(
        sequences16[partition], start=1
    ):

        changes = np.flatnonzero(
            np.diff(seq) != 0
        ) + 1

        boundaries = np.r_[
            0, changes, len(seq)
        ]

        for j in range(len(boundaries) - 1):

            start = int(boundaries[j])
            end = int(boundaries[j + 1])

            state = int(seq[start])

            episode_rows16.append({
                "partition": partition,
                "cycle_index": cycle_idx,
                "state": state,
                "start_hour": start,
                "end_hour_inclusive": end - 1,
                "duration_hours": end - start,
                "is_cycle_start": start == 0,
                "is_cycle_end": end == len(seq),
                "is_complete_episode": (
                    start > 0 and end < len(seq)
                )
            })

episodes16 = pd.DataFrame(episode_rows16)

duration_summary16 = (
    episodes16
    .groupby(["partition", "state"])
    .agg(
        n_episodes=("duration_hours", "size"),
        mean_duration=("duration_hours", "mean"),
        median_duration=("duration_hours", "median"),
        p90_duration=(
            "duration_hours",
            lambda x: np.percentile(x, 90)
        ),
        max_duration=("duration_hours", "max"),
        complete_episodes=(
            "is_complete_episode", "sum"
        )
    )
    .reset_index()
)

_project_print("\nPERSISTÊNCIA DOS REGIMES")

_project_display(duration_summary16.round(3))

# ------------------------------------------------------------
# 8. OCUPAÇÃO DOS ESTADOS
# ------------------------------------------------------------

occupancy_rows16 = []

for partition in PARTITIONS:

    all_states = np.concatenate(
        sequences16[partition]
    )

    n = len(all_states)

    for state in STATES:

        count = int(
            np.sum(all_states == state)
        )

        occupancy_rows16.append({
            "partition": partition,
            "state": state,
            "hours": count,
            "occupancy": count / n
        })

occupancy16 = pd.DataFrame(occupancy_rows16)

_project_print("\nOCUPAÇÃO DOS ESTADOS")

_project_display(occupancy16.round(4))

# ------------------------------------------------------------
# 9. TRANSIÇÕES EFETIVAS — SEM AUTOTRANSIÇÕES
# ------------------------------------------------------------

effective_rows16 = []

for partition in PARTITIONS:

    matrix = counts16[partition]

    for i in STATES:
        for j in STATES:

            if i == j:
                continue

            n = int(matrix[i, j])

            if n > 0:

                effective_rows16.append({
                    "partition": partition,
                    "origin": i,
                    "destination": j,
                    "n_transitions": n
                })

effective16 = pd.DataFrame(effective_rows16)

_project_print("\nTRANSIÇÕES EFETIVAS")

if not effective16.empty:

    _project_display(
        effective16.sort_values(
            ["partition", "n_transitions"],
            ascending=[True, False]
        )
    )

# ------------------------------------------------------------
# 10. DIAGNÓSTICO ESPECÍFICO DO ESTADO S4
# ------------------------------------------------------------

s4_rows16 = []

for partition in PARTITIONS:

    matrix = counts16[partition]

    s4_out = {
        int(j): int(matrix[4, j])
        for j in STATES if j != 4
    }

    s4_in = {
        int(i): int(matrix[i, 4])
        for i in STATES if i != 4
    }

    s4_episodes = episodes16[
        (episodes16["partition"] == partition) &
        (episodes16["state"] == 4)
    ]

    s4_rows16.append({
        "partition": partition,
        "n_s4_episodes": len(s4_episodes),
        "median_duration_hours": (
            s4_episodes["duration_hours"].median()
        ),
        "s4_to_s3": s4_out[3],
        "s4_to_other": sum(
            value for state, value in s4_out.items()
            if state != 3
        ),
        "other_to_s4": sum(s4_in.values()),
        "s4_self_transitions": int(matrix[4, 4])
    })

s4_summary16 = pd.DataFrame(s4_rows16)

_project_print("\nESTRUTURA DE TRANSIÇÃO DO S4")

_project_display(s4_summary16)

# ------------------------------------------------------------
# 11. COMPARAÇÃO TRAIN × VALIDATION
# ------------------------------------------------------------

comparison_rows16 = []

for i in STATES:

    for j in STATES:

        p_train = probs16["train"][i, j]
        p_val = probs16["validation"][i, j]

        comparison_rows16.append({
            "origin": i,
            "destination": j,
            "p_train": p_train,
            "p_validation": p_val,
            "difference_validation_minus_train": (
                p_val - p_train
            )
        })

comparison16 = pd.DataFrame(comparison_rows16)

_project_print("\nCOMPARAÇÃO DAS PROBABILIDADES")

_project_display(
    comparison16.sort_values(
        "difference_validation_minus_train",
        key=lambda s: s.abs(),
        ascending=False
    ).head(15).round(4)
)

# ------------------------------------------------------------
# 12. VISUALIZAÇÃO DAS MATRIZES
# ------------------------------------------------------------

fig, axes = plt.subplots(
    1, 3, figsize=(17, 5),
    constrained_layout=True
)

matrices = [
    ("HMM treinado", A16),
    ("Treinamento — empírico", probs16["train"]),
    ("Validação — empírico", probs16["validation"])
]

for ax, (title, matrix) in zip(axes, matrices):

    image = ax.imshow(
        np.ma.masked_invalid(matrix),
        vmin=0,
        vmax=1,
        cmap="Blues"
    )

    ax.set_title(title)
    ax.set_xlabel("Estado destino")
    ax.set_ylabel("Estado origem")

    ax.set_xticks(STATES)
    ax.set_yticks(STATES)

    ax.set_xticklabels([f"S{i}" for i in STATES])
    ax.set_yticklabels([f"S{i}" for i in STATES])

    for i in STATES:
        for j in STATES:

            value = matrix[i, j]

            if np.isfinite(value):

                ax.text(
                    j, i,
                    f"{value:.2f}",
                    ha="center",
                    va="center",
                    color=(
                        "white" if value > 0.55
                        else "black"
                    ),
                    fontsize=9
                )

fig.colorbar(
    image,
    ax=axes,
    shrink=0.75,
    label="Probabilidade"
)

plt.savefig(
    FIGURE_DIR16 / "cell16_transition_matrices.png",
    dpi=180,
    bbox_inches="tight"
)

plt.show()

# ------------------------------------------------------------
# 13. VISUALIZAÇÃO DAS TRANSIÇÕES EFETIVAS
# ------------------------------------------------------------

fig, axes = plt.subplots(
    1, 2, figsize=(14, 5),
    constrained_layout=True
)

for ax, partition in zip(axes, PARTITIONS):

    matrix = counts16[partition].astype(float)

    np.fill_diagonal(matrix, 0)

    image = ax.imshow(
        matrix,
        cmap="Blues"
    )

    ax.set_title(
        f"Transições efetivas — {partition}"
    )

    ax.set_xlabel("Destino")
    ax.set_ylabel("Origem")

    ax.set_xticks(STATES)
    ax.set_yticks(STATES)

    ax.set_xticklabels([f"S{i}" for i in STATES])
    ax.set_yticklabels([f"S{i}" for i in STATES])

    for i in STATES:
        for j in STATES:

            if matrix[i, j] > 0:

                ax.text(
                    j, i,
                    str(int(matrix[i, j])),
                    ha="center",
                    va="center"
                )

    fig.colorbar(
        image,
        ax=ax,
        shrink=0.75
    )

plt.savefig(
    FIGURE_DIR16 / "cell16_effective_transitions.png",
    dpi=180,
    bbox_inches="tight"
)

plt.show()

# ------------------------------------------------------------
# 14. VISUALIZAÇÃO DAS DURAÇÕES
# ------------------------------------------------------------

fig, ax = plt.subplots(
    figsize=(11, 5)
)

data_plot = [
    episodes16[
        (episodes16["partition"] == "train") &
        (episodes16["state"] == state)
    ]["duration_hours"].to_numpy()
    for state in STATES
]

ax.boxplot(
    data_plot,
    tick_labels=[f"S{i}" for i in STATES],
    showfliers=False
)

ax.set_title(
    "Distribuição das durações dos episódios — treinamento"
)

ax.set_xlabel("Estado HMM")
ax.set_ylabel("Duração do episódio (horas)")

ax.grid(alpha=0.25)

plt.tight_layout()

plt.savefig(
    FIGURE_DIR16 / "cell16_episode_durations.png",
    dpi=180
)

plt.show()

# ------------------------------------------------------------
# 15. GRÁFICO DE OCUPAÇÃO
# ------------------------------------------------------------

occupancy_pivot16 = occupancy16.pivot(
    index="state",
    columns="partition",
    values="occupancy"
)

ax = occupancy_pivot16.plot.bar(
    figsize=(10, 5)
)

ax.set_title(
    "Ocupação dos estados — treinamento e validação"
)

ax.set_xlabel("Estado HMM")
ax.set_ylabel("Proporção de horas")

ax.set_xticklabels(
    [f"S{i}" for i in STATES],
    rotation=0
)

ax.grid(axis="y", alpha=0.25)

plt.tight_layout()

plt.savefig(
    FIGURE_DIR16 / "cell16_state_occupancy.png",
    dpi=180
)

plt.show()

# ------------------------------------------------------------
# 16. EXPORTAÇÃO DOS RESULTADOS
# ------------------------------------------------------------

for partition in PARTITIONS:

    _project_csv(pd.DataFrame(
        counts16[partition],
        index=[f"S{i}" for i in STATES],
        columns=[f"S{i}" for i in STATES]
    ), 
        REPORT_DIR16 /
        f"cell16_transition_counts_{partition}.csv"
    )

    _project_csv(pd.DataFrame(
        probs16[partition],
        index=[f"S{i}" for i in STATES],
        columns=[f"S{i}" for i in STATES]
    ), 
        REPORT_DIR16 /
        f"cell16_transition_probabilities_{partition}.csv"
    )

_project_csv(episodes16, 
    REPORT_DIR16 / "cell16_episodes.csv",
    index=False
)

_project_csv(duration_summary16, 
    REPORT_DIR16 / "cell16_duration_summary.csv",
    index=False
)

_project_csv(occupancy16, 
    REPORT_DIR16 / "cell16_occupancy.csv",
    index=False
)

_project_csv(comparison16, 
    REPORT_DIR16 / "cell16_train_validation_comparison.csv",
    index=False
)

_project_csv(s4_summary16, 
    REPORT_DIR16 / "cell16_s4_transition_summary.csv",
    index=False
)

# ------------------------------------------------------------
# 17. MANIFESTO
# ------------------------------------------------------------

manifest16 = {
    "cell": 16,
    "phase": 2,
    "model": MODEL_NAME,
    "k": K,
    "seed": SEED,
    "partitions": PARTITIONS,
    "inference_source": inference_source16,
    "n_cycles": {
        p: len(sequences16[p])
        for p in PARTITIONS
    },
    "n_hours": {
        p: int(sum(len(s) for s in sequences16[p]))
        for p in PARTITIONS
    },
    "n_transitions": {
        p: int(counts16[p].sum())
        for p in PARTITIONS
    },
    "method": {
        "states": "forward_filter_argmax",
        "transitions": "within_cycle_only",
        "self_transitions_included": True,
        "episodes": "consecutive_equal_states",
        "hmm_retrained": False,
        "scaler_refitted": False,
        "co2_used": False,
        "test_used": False
    }
}

manifest_path16 = (
    REPORT_DIR16 /
    "cell16_transition_persistence_manifest.json"
)

with open(
    manifest_path16,
    "w",
    encoding="utf-8"
) as f:

    _project_dump(
        manifest16,
        f,
        ensure_ascii=False,
        indent=2
    )

# ------------------------------------------------------------
# 18. OBJETOS DISPONÍVEIS PARA AS PRÓXIMAS CÉLULAS
# ------------------------------------------------------------

hmm16_transition_counts = counts16
hmm16_transition_probabilities = probs16
hmm16_episodes = episodes16
hmm16_duration_summary = duration_summary16
hmm16_occupancy = occupancy16
hmm16_effective_transitions = effective16
hmm16_s4_summary = s4_summary16
hmm16_comparison = comparison16

# ------------------------------------------------------------
# 19. RESUMO FINAL
# ------------------------------------------------------------

_project_print("\n" + "=" * 76)
_project_print("RESULTADO FINAL — CÉLULA 16/36")
_project_print("=" * 76)

_project_print(f"Modelo: {MODEL_NAME}")
_project_print(f"K: {K}")
_project_print(f"Semente: {SEED}")

for partition in PARTITIONS:

    _project_print(
        f"\n{partition.upper()}: "
        f"{len(sequences16[partition])} ciclos | "
        f"{sum(len(s) for s in sequences16[partition])} horas | "
        f"{counts16[partition].sum()} transições"
    )

_project_print("\nDiagnóstico do estado S4:")
_project_display(s4_summary16)

_project_print("\nVerificações metodológicas:")
_project_print("HMM reajustado: NÃO")
_project_print("Scaler reajustado: NÃO")
_project_print("CO2 utilizado: NÃO")
_project_print("Teste utilizado: NÃO")
_project_print("Estados modificados: NÃO")
_project_print("Transições entre ciclos diferentes: NÃO")

_project_print(f"\nRelatórios: {REPORT_DIR16}")
_project_print(f"Figuras: {FIGURE_DIR16}")
_project_print(f"Manifesto: {manifest_path16}")

_project_print(
    "\nCÉLULA 16 CONCLUÍDA — "
    "Estrutura dinâmica e persistência dos regimes calculadas."
)


In [ ]:

# ============================================================
# FASE 2 — CÉLULA 17/36
# ROBUSTEZ, ESTABILIDADE E DIAGNÓSTICO DO ESTADO S4
# ============================================================
#
# PROJETO:
# Modelagem de regimes operacionais de uma unidade
# termelétrica com HMM e Process Mining.
#
# UNIDADE:
# Iatan — Facility 6065 — Unit 2 — Ano 2025
#
# MODELO CONGELADO:
# M2_LOAD_DELTA | K=5 | seed=11
#
# OBJETIVOS:
#  1. recuperar a inferência da Célula 13;
#  2. verificar sua integridade estrutural;
#  3. comparar Viterbi retrospectivo e forward causal;
#  4. medir estabilidade, transições e fragmentação;
#  5. investigar a posição temporal do estado S4;
#  6. avaliar S4 como possível regime de pré-geração;
#  7. verificar robustez entre treino e validação;
#  8. gerar gráficos, tabelas e relatórios.
#
# RESTRIÇÕES METODOLÓGICAS:
#  - não retreinar o HMM;
#  - não reajustar o scaler;
#  - não utilizar CO2;
#  - não utilizar dados de teste;
#  - não alterar estados inferidos;
#  - não usar informação futura na filtragem causal;
#  - não atribuir causalidade física ao estado S4.
#
# ============================================================

# ============================================================
# 17.1 — IMPORTAÇÕES E CONFIGURAÇÃO
# ============================================================

import os
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display

MODEL_NAME = "M2_LOAD_DELTA"
K = 5
SEED = int(HMM_SELECTION_12["selected_seed"])

PARTITIONS = ("train", "validation")
STATE_NAMES = [f"S{i}" for i in range(K)]

S4 = 4

# Limiares descritivos, não usados para alterar o HMM.
HIGH_CONFIDENCE = 0.90
EARLY_CYCLE_FRACTION = 0.20

# Para avaliar se S4 ocorre antes da primeira hora
# de geração positiva, é necessário recuperar a
# carga física em MW. Esta célula não inventa MW
# a partir das variáveis padronizadas.

# ============================================================
# 17.2 — DIRETÓRIOS E OBJETOS OBRIGATÓRIOS
# ============================================================

if "hmm13_inference" not in globals():
    raise RuntimeError(
        "Objeto hmm13_inference não encontrado. "
        "Execute primeiro a Célula 13."
    )

if not isinstance(hmm13_inference, dict):
    raise TypeError(
        "hmm13_inference deve ser um dicionário."
    )

missing_partitions = [
    p for p in PARTITIONS
    if p not in hmm13_inference
]

if missing_partitions:
    raise KeyError(
        f"Partições ausentes: {missing_partitions}"
    )

if "RUN_DIR" in globals():
    base17 = Path(RUN_DIR)
elif "run_dir" in globals():
    base17 = Path(run_dir)
else:
    raise RuntimeError(
        "RUN_DIR ou run_dir não encontrado. "
        "Execute a configuração inicial do notebook."
    )

REPORT_DIR17 = base17 / "reports"
FIGURE_DIR17 = base17 / "figures"

REPORT_DIR17.mkdir(
    parents=True,
    exist_ok=True
)

FIGURE_DIR17.mkdir(
    parents=True,
    exist_ok=True
)

_project_print("=" * 76)
_project_print("FASE 2 — CÉLULA 17/36")
_project_print("ROBUSTEZ E DIAGNÓSTICO DO ESTADO S4")
_project_print("=" * 76)

_project_print(f"Modelo: {MODEL_NAME}")
_project_print(f"K: {K}")
_project_print(f"Semente: {SEED}")
_project_print(f"Relatórios: {REPORT_DIR17}")
_project_print(f"Figuras: {FIGURE_DIR17}")

# ============================================================
# 17.3 — RECUPERAÇÃO CORRIGIDA DA CÉLULA 13
# ============================================================
#
# Estrutura real:
#
# hmm13_inference[partition] = {
#     "viterbi_states": ndarray (N,),
#     "filtered_probabilities": ndarray (N,K),
#     "cycle_index": ndarray (N,),
#     "X": ndarray (N,2),
#     "lengths": ndarray (n_cycles,),
#     "filtered_state": ndarray (N,),
#     "filtered_confidence": ndarray (N,),
#     "filtered_entropy": ndarray (N,)
# }
#
# Não reconstruímos inferência nem estados.
# ============================================================

def prepare_partition17(partition):

    source = hmm13_inference[partition]

    required = [
        "viterbi_states",
        "filtered_probabilities",
        "cycle_index",
        "lengths",
        "filtered_state"
    ]

    missing = [
        key for key in required
        if key not in source
    ]

    if missing:
        raise KeyError(
            f"{partition}: campos ausentes: {missing}"
        )

    viterbi = np.asarray(
        source["viterbi_states"]
    ).reshape(-1)

    filtered = np.asarray(
        source["filtered_state"]
    ).reshape(-1)

    probabilities = np.asarray(
        source["filtered_probabilities"],
        dtype=float
    )

    original_cycle_index = np.asarray(
        source["cycle_index"]
    ).reshape(-1)

    lengths = np.asarray(
        source["lengths"],
        dtype=int
    ).reshape(-1)

    n = len(viterbi)

    # --------------------------------------------------------
    # Verificações estruturais
    # --------------------------------------------------------

    if len(filtered) != n:
        raise ValueError(
            f"{partition}: comprimento forward incompatível."
        )

    if len(original_cycle_index) != n:
        raise ValueError(
            f"{partition}: cycle_index incompatível."
        )

    if probabilities.shape != (n, K):
        raise ValueError(
            f"{partition}: matriz de probabilidades "
            f"{probabilities.shape}; esperado {(n,K)}."
        )

    if np.any(lengths <= 0):
        raise ValueError(
            f"{partition}: comprimentos de ciclos inválidos."
        )

    if lengths.sum() != n:
        raise ValueError(
            f"{partition}: soma(lengths)={lengths.sum()} "
            f"difere de N={n}."
        )

    if not np.all(np.isfinite(probabilities)):
        raise ValueError(
            f"{partition}: probabilidades não finitas."
        )

    if np.any(probabilities < -1e-10):
        raise ValueError(
            f"{partition}: probabilidades negativas."
        )

    if not np.allclose(
        probabilities.sum(axis=1),
        1.0,
        atol=1e-6
    ):
        raise ValueError(
            f"{partition}: probabilidades não normalizadas."
        )

    if not np.all(
        np.isin(viterbi, np.arange(K))
    ):
        raise ValueError(
            f"{partition}: estados Viterbi inválidos."
        )

    if not np.all(
        np.isin(filtered, np.arange(K))
    ):
        raise ValueError(
            f"{partition}: estados forward inválidos."
        )

    # --------------------------------------------------------
    # Reconstrução das fronteiras dos ciclos
    # --------------------------------------------------------

    cycle_id = np.repeat(
        np.arange(1, len(lengths) + 1),
        lengths
    )

    expected_boundaries = np.cumsum(
        lengths
    )[:-1]

    observed_boundaries = (
        np.flatnonzero(
            original_cycle_index[1:] !=
            original_cycle_index[:-1]
        ) + 1
    )

    if not np.array_equal(
        expected_boundaries,
        observed_boundaries
    ):
        raise ValueError(
            f"{partition}: cycle_index e lengths "
            "não concordam quanto às fronteiras dos ciclos."
        )

    # --------------------------------------------------------
    # Construção do DataFrame
    # --------------------------------------------------------

    df = pd.DataFrame({
        "partition": partition,
        "cycle_id": cycle_id,
        "original_cycle_index": original_cycle_index,
        "state_viterbi": viterbi.astype(int),
        "state_filtered": filtered.astype(int)
    })

    for state in range(K):
        df[f"p_s{state}"] = probabilities[:, state]

    calculated_confidence = probabilities.max(
        axis=1
    )

    safe_prob = np.clip(
        probabilities,
        1e-15,
        1.0
    )

    calculated_entropy = -np.sum(
        safe_prob * np.log(safe_prob),
        axis=1
    )

    # Usamos os resultados efetivamente gerados
    # pela Célula 13 e verificamos consistência.

    if "filtered_confidence" in source:

        confidence = np.asarray(
            source["filtered_confidence"]
        ).reshape(-1)

        if len(confidence) != n:
            raise ValueError(
                f"{partition}: confidence incompatível."
            )

        if not np.allclose(
            confidence,
            calculated_confidence,
            atol=1e-6
        ):
            raise ValueError(
                f"{partition}: confiança inconsistente "
                "com probabilidades filtradas."
            )

        df["filtered_confidence"] = confidence

    else:
        df["filtered_confidence"] = (
            calculated_confidence
        )

    if "filtered_entropy" in source:

        entropy = np.asarray(
            source["filtered_entropy"]
        ).reshape(-1)

        if len(entropy) != n:
            raise ValueError(
                f"{partition}: entropy incompatível."
            )

        # A Célula 13 pode utilizar entropia
        # normalizada ou entropia em nats.
        # Preservamos o valor original.

        df["filtered_entropy"] = entropy

    else:
        df["filtered_entropy"] = (
            calculated_entropy
        )

    df["cycle_position"] = (
        df.groupby("cycle_id", sort=False)
        .cumcount()
    )

    df["cycle_length"] = (
        df.groupby("cycle_id", sort=False)["cycle_id"]
        .transform("size")
    )

    df["relative_position"] = np.where(
        df["cycle_length"] > 1,
        df["cycle_position"] /
        (df["cycle_length"] - 1),
        0.0
    )

    df["agreement"] = (
        df["state_viterbi"] ==
        df["state_filtered"]
    )

    df["is_s4"] = (
        df["state_filtered"] == S4
    )

    return df


data17 = {
    partition: prepare_partition17(partition)
    for partition in PARTITIONS
}

_project_print("\n17.3 — INFERÊNCIA RECUPERADA")

for partition, df in data17.items():

    _project_print(
        f"{partition.upper()}: "
        f"{len(df)} horas | "
        f"{df['cycle_id'].nunique()} ciclos | "
        f"{df['state_filtered'].nunique()} estados"
    )

# ============================================================
# 17.4 — ROBUSTEZ VITERBI × FORWARD
# ============================================================

robustness_rows17 = []

for partition, df in data17.items():

    agreement = df["agreement"].mean()

    mean_conf = df[
        "filtered_confidence"
    ].mean()

    median_conf = df[
        "filtered_confidence"
    ].median()

    low_conf = (
        df["filtered_confidence"] <
        HIGH_CONFIDENCE
    ).mean()

    s4_agreement = np.nan

    s4_union = (
        (df["state_viterbi"] == S4) |
        (df["state_filtered"] == S4)
    )

    if s4_union.any():

        s4_agreement = (
            (
                (df["state_viterbi"] == S4) &
                (df["state_filtered"] == S4)
            ).sum()
            /
            s4_union.sum()
        )

    robustness_rows17.append({
        "partition": partition,
        "n_hours": len(df),
        "n_cycles": df["cycle_id"].nunique(),
        "agreement_viterbi_forward": agreement,
        "mean_confidence": mean_conf,
        "median_confidence": median_conf,
        "share_confidence_below_090": low_conf,
        "s4_jaccard_viterbi_forward": s4_agreement
    })

audit17_robustness = pd.DataFrame(
    robustness_rows17
)

_project_print("\n17.4 — ROBUSTEZ DA INFERÊNCIA")

_project_display(
    audit17_robustness.round(4)
)

# ============================================================
# 17.5 — EPISÓDIOS CONTÍNUOS POR ESTADO
# ============================================================
#
# Uma mudança de estado é identificada somente
# dentro de um mesmo ciclo ON.
#
# O início de outro ciclo sempre inicia novo episódio.
# ============================================================

def build_episodes17(df, state_column):

    records = []

    for cycle_id, cycle_df in df.groupby(
        "cycle_id",
        sort=False
    ):

        states = cycle_df[
            state_column
        ].to_numpy()

        positions = cycle_df[
            "cycle_position"
        ].to_numpy()

        starts = np.r_[
            0,
            np.flatnonzero(
                states[1:] != states[:-1]
            ) + 1
        ]

        ends = np.r_[
            starts[1:],
            len(states)
        ]

        for start, end in zip(starts, ends):

            records.append({
                "partition": cycle_df[
                    "partition"
                ].iloc[0],
                "cycle_id": int(cycle_id),
                "state": int(states[start]),
                "start_hour": int(positions[start]),
                "end_hour": int(positions[end-1]),
                "duration_hours": int(end-start),
                "cycle_length": len(states),
                "relative_start": (
                    float(
                        cycle_df[
                            "relative_position"
                        ].iloc[start]
                    )
                )
            })

    return pd.DataFrame(records)


episodes17 = {}

for partition, df in data17.items():

    episodes17[partition] = build_episodes17(
        df,
        "state_filtered"
    )

audit17_episodes = pd.concat(
    episodes17.values(),
    ignore_index=True
)

episode_summary17 = (
    audit17_episodes
    .groupby(
        ["partition", "state"],
        as_index=False
    )
    .agg(
        n_episodes=("duration_hours", "size"),
        mean_duration=("duration_hours", "mean"),
        median_duration=("duration_hours", "median"),
        max_duration=("duration_hours", "max"),
        mean_relative_start=("relative_start", "mean")
    )
)

_project_print("\n17.5 — EPISÓDIOS DOS ESTADOS")

_project_display(
    episode_summary17.round(4)
)

# ============================================================
# 17.6 — TRANSIÇÕES E FRAGMENTAÇÃO
# ============================================================

transition_rows17 = []
fragmentation_rows17 = []

for partition, df in data17.items():

    transition_matrix = np.zeros(
        (K, K),
        dtype=int
    )

    for cycle_id, cycle_df in df.groupby(
        "cycle_id",
        sort=False
    ):

        states = cycle_df[
            "state_filtered"
        ].to_numpy()

        for origin, destination in zip(
            states[:-1],
            states[1:]
        ):

            if origin != destination:
                transition_matrix[
                    origin,
                    destination
                ] += 1

    n_transitions = int(
        transition_matrix.sum()
    )

    possible_transitions = (
        len(df) -
        df["cycle_id"].nunique()
    )

    fragmentation_rows17.append({
        "partition": partition,
        "n_hours": len(df),
        "n_cycles": df["cycle_id"].nunique(),
        "n_effective_transitions": n_transitions,
        "transition_rate": (
            n_transitions /
            possible_transitions
            if possible_transitions > 0
            else np.nan
        ),
        "episodes_per_100_hours": (
            len(episodes17[partition]) /
            len(df) * 100
        )
    })

    for origin in range(K):
        for destination in range(K):

            transition_rows17.append({
                "partition": partition,
                "origin": origin,
                "destination": destination,
                "count": int(
                    transition_matrix[
                        origin,
                        destination
                    ]
                )
            })

audit17_transitions = pd.DataFrame(
    transition_rows17
)

audit17_fragmentation = pd.DataFrame(
    fragmentation_rows17
)

_project_print("\n17.6 — FRAGMENTAÇÃO OPERACIONAL")

_project_display(
    audit17_fragmentation.round(4)
)

# ============================================================
# 17.7 — DIAGNÓSTICO ESPECÍFICO DO ESTADO S4
# ============================================================
#
# Questões:
#
#  - Em quantos ciclos S4 aparece?
#  - Qual é sua duração?
#  - Em que posição do ciclo aparece?
#  - Está concentrado no início dos ciclos?
#  - Há episódios posteriores de S4?
#
# IMPORTANTE:
# A posição inicial não prova aquecimento de caldeira.
# ============================================================

s4_cycle_rows17 = []

for partition, df in data17.items():

    for cycle_id, cycle_df in df.groupby(
        "cycle_id",
        sort=False
    ):

        mask = (
            cycle_df["state_filtered"] == S4
        )

        n_s4 = int(mask.sum())

        if n_s4 > 0:

            first_position = int(
                cycle_df.loc[
                    mask,
                    "cycle_position"
                ].min()
            )

            last_position = int(
                cycle_df.loc[
                    mask,
                    "cycle_position"
                ].max()
            )

            mean_relative = float(
                cycle_df.loc[
                    mask,
                    "relative_position"
                ].mean()
            )

            share_early = float(
                (
                    cycle_df.loc[
                        mask,
                        "relative_position"
                    ] <= EARLY_CYCLE_FRACTION
                ).mean()
            )

            starts_in_s4 = bool(
                cycle_df[
                    "state_filtered"
                ].iloc[0] == S4
            )

        else:

            first_position = np.nan
            last_position = np.nan
            mean_relative = np.nan
            share_early = np.nan
            starts_in_s4 = False

        s4_cycle_rows17.append({
            "partition": partition,
            "cycle_id": int(cycle_id),
            "cycle_length": len(cycle_df),
            "s4_hours": n_s4,
            "s4_share": n_s4 / len(cycle_df),
            "s4_present": n_s4 > 0,
            "starts_in_s4": starts_in_s4,
            "first_s4_hour": first_position,
            "last_s4_hour": last_position,
            "mean_relative_position_s4": mean_relative,
            "share_s4_in_first_20pct": share_early
        })

audit17_s4_cycles = pd.DataFrame(
    s4_cycle_rows17
)

s4_summary_rows17 = []

for partition in PARTITIONS:

    d = audit17_s4_cycles[
        audit17_s4_cycles[
            "partition"
        ] == partition
    ]

    e = audit17_episodes[
        (audit17_episodes["partition"] == partition) &
        (audit17_episodes["state"] == S4)
    ]

    present = d[d["s4_present"]]

    s4_summary_rows17.append({
        "partition": partition,
        "n_cycles": len(d),
        "cycles_with_s4": int(
            d["s4_present"].sum()
        ),
        "share_cycles_with_s4": (
            d["s4_present"].mean()
        ),
        "cycles_starting_in_s4": int(
            d["starts_in_s4"].sum()
        ),
        "s4_total_hours": int(
            d["s4_hours"].sum()
        ),
        "s4_hour_share": (
            d["s4_hours"].sum() /
            d["cycle_length"].sum()
        ),
        "n_s4_episodes": len(e),
        "median_s4_episode_hours": (
            e["duration_hours"].median()
            if len(e) else np.nan
        ),
        "median_first_s4_hour": (
            present["first_s4_hour"].median()
            if len(present) else np.nan
        ),
        "mean_s4_relative_position": (
            e["relative_start"].mean()
            if len(e) else np.nan
        )
    })

audit17_s4_summary = pd.DataFrame(
    s4_summary_rows17
)

_project_print("\n17.7 — DIAGNÓSTICO DO ESTADO S4")

_project_display(
    audit17_s4_summary.round(4)
)

_project_print("\nS4 POR CICLO")

_project_display(
    audit17_s4_cycles.round(4)
)

# ============================================================
# 17.8 — PERSISTÊNCIA E PROBABILIDADE DE S4
# ============================================================

probability_rows17 = []

for partition, df in data17.items():

    for state in range(K):

        mask = (
            df["state_filtered"] == state
        )

        probability_rows17.append({
            "partition": partition,
            "state": state,
            "n_hours": int(mask.sum()),
            "occupancy": float(mask.mean()),
            "mean_probability_when_selected": (
                df.loc[
                    mask,
                    f"p_s{state}"
                ].mean()
                if mask.any() else np.nan
            ),
            "mean_confidence_when_selected": (
                df.loc[
                    mask,
                    "filtered_confidence"
                ].mean()
                if mask.any() else np.nan
            ),
            "share_low_confidence_when_selected": (
                (
                    df.loc[
                        mask,
                        "filtered_confidence"
                    ] < HIGH_CONFIDENCE
                ).mean()
                if mask.any() else np.nan
            )
        })

audit17_state_probabilities = pd.DataFrame(
    probability_rows17
)

_project_print("\n17.8 — CONFIANÇA POR ESTADO")

_project_display(
    audit17_state_probabilities.round(4)
)

# ============================================================
# 17.9 — GRÁFICO 1: COMPARAÇÃO DA OCUPAÇÃO
# ============================================================

fig, ax = plt.subplots(
    figsize=(10, 5)
)

x = np.arange(K)
width = 0.36

for i, partition in enumerate(PARTITIONS):

    df = data17[partition]

    occupancy = (
        df["state_filtered"]
        .value_counts(normalize=True)
        .reindex(range(K), fill_value=0)
        .to_numpy()
    )

    offset = (
        -width/2 if i == 0
        else width/2
    )

    ax.bar(
        x + offset,
        occupancy,
        width,
        label=partition
    )

ax.set_xticks(x)
ax.set_xticklabels(STATE_NAMES)
ax.set_ylabel("Proporção das horas ON")
ax.set_xlabel("Estado HMM")
ax.set_title(
    "Ocupação dos estados — treino × validação"
)
ax.legend()
ax.grid(
    axis="y",
    alpha=0.25
)

plt.tight_layout()

fig.savefig(
    FIGURE_DIR17 /
    "cell17_state_occupancy.png",
    dpi=180,
    bbox_inches="tight"
)

plt.show()

# ============================================================
# 17.10 — GRÁFICO 2: DURAÇÃO DOS EPISÓDIOS
# ============================================================

fig, axes = plt.subplots(
    1,
    2,
    figsize=(14, 5),
    sharey=True
)

for ax, partition in zip(
    axes,
    PARTITIONS
):

    ep = episodes17[partition]

    values = [
        ep.loc[
            ep["state"] == state,
            "duration_hours"
        ].to_numpy()
        for state in range(K)
    ]

    # Evitar erro se um estado não tiver episódios.

    valid_states = [
        state
        for state in range(K)
        if len(values[state]) > 0
    ]

    valid_values = [
        values[state]
        for state in valid_states
    ]

    if valid_values:

        ax.boxplot(
            valid_values,
            positions=[
                state + 1
                for state in valid_states
            ],
            showfliers=False
        )

    ax.set_xticks(
        np.arange(1, K+1)
    )

    ax.set_xticklabels(
        STATE_NAMES
    )

    ax.set_title(
        partition.upper()
    )

    ax.set_xlabel(
        "Estado HMM"
    )

    ax.grid(
        axis="y",
        alpha=0.25
    )

axes[0].set_ylabel(
    "Duração do episódio (horas)"
)

fig.suptitle(
    "Persistência dos regimes operacionais"
)

plt.tight_layout()

fig.savefig(
    FIGURE_DIR17 /
    "cell17_episode_durations.png",
    dpi=180,
    bbox_inches="tight"
)

plt.show()

# ============================================================
# 17.11 — GRÁFICO 3: POSIÇÃO DE S4 NOS CICLOS
# ============================================================
#
# Cada ponto representa uma hora classificada como S4.
# O eixo X mostra a posição relativa dentro do ciclo.
# O eixo Y mostra o ciclo ON.
# ============================================================

fig, axes = plt.subplots(
    1,
    2,
    figsize=(15, 6)
)

for ax, partition in zip(
    axes,
    PARTITIONS
):

    df = data17[partition]

    s4_df = df[
        df["state_filtered"] == S4
    ]

    if not s4_df.empty:

        ax.scatter(
            s4_df["relative_position"],
            s4_df["cycle_id"],
            s=16,
            alpha=0.65
        )

    ax.axvline(
        EARLY_CYCLE_FRACTION,
        linestyle="--",
        color="gray",
        label="Primeiros 20% do ciclo"
    )

    ax.set_xlim(
        -0.02,
        1.02
    )

    ax.set_xlabel(
        "Posição relativa no ciclo ON"
    )

    ax.set_ylabel(
        "ID do ciclo ON"
    )

    ax.set_title(
        partition.upper()
    )

    ax.grid(
        alpha=0.2
    )

    ax.legend()

fig.suptitle(
    "Distribuição temporal do estado S4"
)

plt.tight_layout()

fig.savefig(
    FIGURE_DIR17 /
    "cell17_s4_relative_position.png",
    dpi=180,
    bbox_inches="tight"
)

plt.show()

# ============================================================
# 17.12 — GRÁFICO 4: S4 NO INÍCIO DOS CICLOS
# ============================================================

fig, axes = plt.subplots(
    1,
    2,
    figsize=(14, 5),
    sharey=True
)

for ax, partition in zip(
    axes,
    PARTITIONS
):

    d = audit17_s4_cycles[
        audit17_s4_cycles[
            "partition"
        ] == partition
    ]

    ax.bar(
        d["cycle_id"],
        d["s4_share"]
    )

    ax.set_xlabel(
        "Ciclo ON"
    )

    ax.set_ylabel(
        "Proporção de horas em S4"
    )

    ax.set_title(
        partition.upper()
    )

    ax.grid(
        axis="y",
        alpha=0.25
    )

fig.suptitle(
    "Participação de S4 em cada ciclo operacional"
)

plt.tight_layout()

fig.savefig(
    FIGURE_DIR17 /
    "cell17_s4_cycle_share.png",
    dpi=180,
    bbox_inches="tight"
)

plt.show()

# ============================================================
# 17.13 — GRÁFICO 5: MATRIZ DE TRANSIÇÕES EFETIVAS
# ============================================================

fig, axes = plt.subplots(
    1,
    2,
    figsize=(13, 5)
)

for ax, partition in zip(
    axes,
    PARTITIONS
):

    d = audit17_transitions[
        audit17_transitions[
            "partition"
        ] == partition
    ]

    matrix = (
        d.pivot(
            index="origin",
            columns="destination",
            values="count"
        )
        .reindex(
            index=range(K),
            columns=range(K),
            fill_value=0
        )
        .to_numpy()
    )

    image = ax.imshow(
        matrix,
        cmap="Blues"
    )

    ax.set_xticks(
        range(K)
    )

    ax.set_yticks(
        range(K)
    )

    ax.set_xticklabels(
        STATE_NAMES
    )

    ax.set_yticklabels(
        STATE_NAMES
    )

    ax.set_xlabel(
        "Destino"
    )

    ax.set_ylabel(
        "Origem"
    )

    ax.set_title(
        partition.upper()
    )

    for i in range(K):
        for j in range(K):

            value = matrix[i, j]

            if value > 0:

                ax.text(
                    j,
                    i,
                    str(value),
                    ha="center",
                    va="center",
                    color=(
                        "white"
                        if value > matrix.max()/2
                        else "black"
                    )
                )

    fig.colorbar(
        image,
        ax=ax,
        fraction=0.046,
        pad=0.04
    )

fig.suptitle(
    "Transições efetivas entre regimes"
)

plt.tight_layout()

fig.savefig(
    FIGURE_DIR17 /
    "cell17_effective_transitions.png",
    dpi=180,
    bbox_inches="tight"
)

plt.show()

# ============================================================
# 17.14 — COMPARAÇÃO TRAIN × VALIDATION
# ============================================================

comparison_rows17 = []

for state in range(K):

    train_df = data17["train"]
    val_df = data17["validation"]

    train_occupancy = (
        train_df["state_filtered"] == state
    ).mean()

    val_occupancy = (
        val_df["state_filtered"] == state
    ).mean()

    train_ep = episodes17["train"]
    val_ep = episodes17["validation"]

    train_durations = train_ep.loc[
        train_ep["state"] == state,
        "duration_hours"
    ]

    val_durations = val_ep.loc[
        val_ep["state"] == state,
        "duration_hours"
    ]

    comparison_rows17.append({
        "state": state,
        "train_occupancy": train_occupancy,
        "validation_occupancy": val_occupancy,
        "occupancy_gap": (
            val_occupancy - train_occupancy
        ),
        "train_median_duration": (
            train_durations.median()
            if len(train_durations)
            else np.nan
        ),
        "validation_median_duration": (
            val_durations.median()
            if len(val_durations)
            else np.nan
        )
    })

audit17_train_val = pd.DataFrame(
    comparison_rows17
)

_project_print("\n17.14 — ROBUSTEZ ENTRE PARTIÇÕES")

_project_display(
    audit17_train_val.round(4)
)

# ============================================================
# 17.15 — INTERPRETAÇÃO METODOLÓGICA DE S4
# ============================================================
#
# A hipótese "S4 = pré-geração" exige duas verificações:
#
#  1. associação temporal com início dos ciclos;
#  2. associação com geração elétrica zero.
#
# A primeira pode ser analisada nesta célula.
#
# A segunda NÃO pode ser confirmada apenas com
# X padronizado sem recuperar a carga em MW.
#
# A existência de emissões de CO2 durante geração
# zero também requer análise observacional posterior.
# ============================================================

_project_print("\n17.15 — INTERPRETAÇÃO DO ESTADO S4")

for partition in PARTITIONS:

    row = audit17_s4_summary.loc[
        audit17_s4_summary[
            "partition"
        ] == partition
    ].iloc[0]

    _project_print(f"\n{partition.upper()}")

    _project_print(
        "Ciclos com S4:",
        int(row["cycles_with_s4"]),
        "/",
        int(row["n_cycles"])
    )

    _project_print(
        "Ciclos iniciados em S4:",
        int(row["cycles_starting_in_s4"])
    )

    _project_print(
        "Horas classificadas como S4:",
        int(row["s4_total_hours"])
    )

    _project_print(
        "Participação horária de S4:",
        f"{row['s4_hour_share']:.2%}"
    )

    _project_print(
        "Duração mediana dos episódios S4:",
        row["median_s4_episode_hours"]
    )

_project_print("\nINTERPRETAÇÃO CIENTÍFICA")

_project_print(
    "S4 é um estado latente inferido a partir "
    "das variáveis de carga e rampa."
)

_project_print(
    "Sua concentração no início dos ciclos "
    "pode sustentar a hipótese de um regime "
    "de pré-geração."
)

_project_print(
    "Essa associação não comprova, isoladamente, "
    "aquecimento da caldeira, consumo de combustível "
    "ou emissão de CO2."
)

_project_print(
    "A verificação física deverá utilizar os "
    "registros operacionais e de emissões "
    "em etapa observacional específica."
)

# ============================================================
# 17.16 — EXPORTAÇÃO DOS RELATÓRIOS
# ============================================================

reports17 = {
    "robustness": audit17_robustness,
    "episodes": audit17_episodes,
    "episode_summary": episode_summary17,
    "fragmentation": audit17_fragmentation,
    "transitions": audit17_transitions,
    "s4_cycles": audit17_s4_cycles,
    "s4_summary": audit17_s4_summary,
    "state_probabilities": audit17_state_probabilities,
    "train_val_comparison": audit17_train_val
}

for name, df in reports17.items():

    path = (
        REPORT_DIR17 /
        f"cell17_{name}.csv"
    )

    _project_csv(df, 
        path,
        index=False
    )

summary17 = {
    "cell": 17,
    "phase": 2,
    "model": MODEL_NAME,
    "k": K,
    "seed": SEED,
    "partitions": list(PARTITIONS),
    "hours": {
        p: int(len(data17[p]))
        for p in PARTITIONS
    },
    "cycles": {
        p: int(
            data17[p]["cycle_id"].nunique()
        )
        for p in PARTITIONS
    },
    "s4_diagnostic": {
        p: {
            "hours": int(
                (
                    data17[p][
                        "state_filtered"
                    ] == S4
                ).sum()
            ),
            "occupancy": float(
                (
                    data17[p][
                        "state_filtered"
                    ] == S4
                ).mean()
            )
        }
        for p in PARTITIONS
    },
    "methodology": {
        "new_training": False,
        "scaler_refitted": False,
        "test_used": False,
        "co2_used": False,
        "states_modified": False,
        "causal_filtering_recomputed": False,
        "physical_interpretation_confirmed": False
    }
}

summary_path17 = (
    REPORT_DIR17 /
    "cell17_robustness_summary.json"
)

with open(
    summary_path17,
    "w",
    encoding="utf-8"
) as file:

    _project_dump(
        summary17,
        file,
        indent=2,
        ensure_ascii=False
    )

# ============================================================
# 17.17 — DISPONIBILIZAÇÃO PARA PRÓXIMAS CÉLULAS
# ============================================================

hmm17_data = data17
hmm17_episodes = audit17_episodes
hmm17_transitions = audit17_transitions
hmm17_s4_diagnostic = audit17_s4_cycles

hmm17_reports = reports17

# ============================================================
# 17.18 — VERIFICAÇÕES FINAIS
# ============================================================

assert set(hmm17_data.keys()) == {
    "train",
    "validation"
}

assert all(
    len(hmm17_data[p]) ==
    len(
        hmm13_inference[p][
            "viterbi_states"
        ]
    )
    for p in PARTITIONS
)

assert all(
    np.array_equal(
        hmm17_data[p][
            "state_filtered"
        ].to_numpy(),
        np.asarray(
            hmm13_inference[p][
                "filtered_state"
            ]
        )
    )
    for p in PARTITIONS
)

_project_print("\n" + "=" * 76)
_project_print("RESULTADO FINAL — CÉLULA 17/36")
_project_print("=" * 76)

_project_print(f"Modelo: {MODEL_NAME}")
_project_print(f"K: {K}")
_project_print(f"Semente: {SEED}")

_project_print("\nPARTIÇÕES ANALISADAS")

for p in PARTITIONS:

    _project_print(
        f"{p.upper()}: "
        f"{len(hmm17_data[p])} horas | "
        f"{hmm17_data[p]['cycle_id'].nunique()} ciclos"
    )

_project_print("\nVERIFICAÇÕES")

_project_print("Inferência original preservada: SIM")
_project_print("Modelo retreinado: NÃO")
_project_print("Scaler reajustado: NÃO")
_project_print("CO2 utilizado: NÃO")
_project_print("Teste utilizado: NÃO")
_project_print("Estados modificados: NÃO")
_project_print("Interpretação física definitiva: NÃO")

_project_print("\nRELATÓRIOS")
_project_print(REPORT_DIR17)

_project_print("\nFIGURAS")
_project_print(FIGURE_DIR17)

_project_print("\nRESUMO JSON")
_project_print(summary_path17)

_project_print("\nCÉLULA 17 CONCLUÍDA.")
_project_print(
    "Próxima etapa: Célula 18/36 — "
    "semântica dos casos e atividades "
    "para Process Mining."
)


## 18–21 — Construção e qualidade do log de eventos

A correlação associa horas ao ciclo ON. A abstração reúne sequências de estados ou rampas em episódios. São definidos logs de regime, rampa e combinação, com validação de ordem, cobertura, variantes e granularidade. A Aula 04, pp. 29–31, e Diba et al., §2, sustentam a distinção entre as três operações. O limiar de rampa e as atividades são do estudo. [Diba et al. (2020)](https://doi.org/10.1002/widm.1346)


In [ ]:

# ============================================================
# FASE 2 — CÉLULA 18/36
# DEFINIÇÃO FORMAL DOS CASOS, ATIVIDADES E EVENTOS
# ============================================================
#
# OBJETIVOS
# 1. Recuperar a inferência causal da Célula 13.
# 2. Formalizar os casos como ciclos ON.
# 3. Definir as atividades de regime e de rampa.
# 4. Definir a semântica de episódios e transições.
# 5. Criar o contrato metodológico dos event logs.
# 6. Validar as regras sem construir os logs definitivos.
#
# NÃO REALIZA:
# - novo treinamento;
# - seleção de HMM;
# - mineração de processos;
# - análise de emissões;
# - construção definitiva dos event logs;
# - utilização do teste para decisões metodológicas.
# ============================================================

import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
from IPython.display import display

_project_print("=" * 76)
_project_print("FASE 2 — CÉLULA 18/36 | SEMÂNTICA DOS EVENT LOGS")
_project_print("=" * 76)

# ------------------------------------------------------------
# 18.1 — CONFIGURAÇÃO E PRÉ-CONDIÇÕES
# ------------------------------------------------------------

assert "hmm13_inference" in globals(), (
    "Execute a Célula 13 antes da Célula 18."
)

assert isinstance(hmm13_inference, dict)

MODEL_NAME = "M2_LOAD_DELTA"
N_STATES = 5
SEED = int(HMM_SELECTION_12["selected_seed"])

PARTITIONS_DEVELOPMENT = ["train", "validation"]

for partition in PARTITIONS_DEVELOPMENT:
    assert partition in hmm13_inference, (
        f"Partição ausente: {partition}"
    )

if "RUN_DIR" in globals():
    BASE_DIR18 = Path(RUN_DIR)
elif "run_dir" in globals():
    BASE_DIR18 = Path(run_dir)
else:
    raise RuntimeError(
        "RUN_DIR não encontrado. "
        "Recupere o diretório da execução anterior."
    )

REPORT_DIR18 = BASE_DIR18 / "reports"
REPORT_DIR18.mkdir(parents=True, exist_ok=True)

_project_print(f"Modelo: {MODEL_NAME}")
_project_print(f"K: {N_STATES}")
_project_print(f"Semente: {SEED}")
_project_print(f"Diretório: {BASE_DIR18}")

# ------------------------------------------------------------
# 18.2 — CONTRATO DOS CASOS
# ------------------------------------------------------------

CASE_DEFINITION18 = {
    "case_type": "ON_operational_cycle",
    "case_id_format": "{partition}_ON_{cycle_number:04d}",
    "case_start": "first_observation_of_ON_cycle",
    "case_end": "last_observation_of_ON_cycle",
    "case_boundary_rule": (
        "Events cannot cross operational-cycle boundaries."
    ),
    "partition_rule": (
        "Cases from train, validation and test are independent."
    ),
    "timestamp_rule": (
        "Use observed hourly timestamps when available. "
        "Never fabricate calendar timestamps."
    ),
    "short_cycles": (
        "Retain short cycles; flag them for later sensitivity analysis."
    ),
    "right_censoring": (
        "A cycle ending at a partition boundary is not "
        "automatically a physical shutdown."
    ),
}

# ------------------------------------------------------------
# 18.3 — ATIVIDADES DE REGIME
# ------------------------------------------------------------

# IDs HMM são preservados.
# Rótulos físicos são provisórios, não fatos confirmados.

REGIME_ACTIVITIES18 = {
    0: {
        "activity": "REGIME_S0",
        "description": "Regime de carga elevada",
        "physical_status": "provisional",
    },
    1: {
        "activity": "REGIME_S1",
        "description": "Regime de carga intermediária inferior",
        "physical_status": "provisional",
    },
    2: {
        "activity": "REGIME_S2",
        "description": "Regime de carga elevada ou transição",
        "physical_status": "provisional",
    },
    3: {
        "activity": "REGIME_S3",
        "description": "Regime de carga variável",
        "physical_status": "provisional",
    },
    4: {
        "activity": "REGIME_S4",
        "description": (
            "Regime inicial de ciclo ON, "
            "frequentemente com geração zero"
        ),
        "physical_status": "provisional",
    },
}

# ------------------------------------------------------------
# 18.4 — ATIVIDADES DE RAMPA
# ------------------------------------------------------------

# Os limites de rampa devem ser definidos com base
# exclusivamente nos dados de treinamento.
#
# Nesta célula estabelecemos o procedimento, mas não
# calculamos limiares sem recuperar a carga original
# em MW e os deltas horários correspondentes.

RAMP_ACTIVITIES18 = {
    "RAMP_UP": {
        "description": "Aumento relevante da carga elétrica",
        "condition": "delta_mw > positive_threshold",
    },
    "RAMP_DOWN": {
        "description": "Redução relevante da carga elétrica",
        "condition": "delta_mw < negative_threshold",
    },
    "STABLE": {
        "description": "Variação dentro da faixa de estabilidade",
        "condition": (
            "negative_threshold <= delta_mw "
            "<= positive_threshold"
        ),
    },
    "RAMP_UNDEFINED": {
        "description": (
            "Delta indisponível ou primeira observação "
            "sem histórico válido"
        ),
        "condition": "delta_mw is missing",
    },
}

RAMP_THRESHOLD_POLICY18 = {
    "source_partition": "train",
    "unit": "MW/h",
    "positive_threshold": None,
    "negative_threshold": None,
    "method": (
        "Determine robust thresholds using the distribution "
        "of signed hourly load changes in training only."
    ),
    "status": "pending_cell19",
    "leakage_policy": (
        "Never estimate thresholds from validation or test."
    ),
}

# ------------------------------------------------------------
# 18.5 — DEFINIÇÃO DE EVENTO
# ------------------------------------------------------------

EVENT_DEFINITION18 = {
    "regime_event": (
        "Maximal consecutive run of the same filtered HMM "
        "state within one ON cycle."
    ),
    "ramp_event": (
        "Maximal consecutive run of the same ramp category "
        "within one ON cycle."
    ),
    "combined_event": (
        "Maximal consecutive run of the same joint "
        "(regime, ramp) activity within one ON cycle."
    ),
    "start_timestamp": (
        "Timestamp of the first hourly observation in the run."
    ),
    "end_timestamp": (
        "Timestamp of the last hourly observation in the run."
    ),
    "duration_hours": (
        "Number of consecutive hourly observations in the run."
    ),
    "transition": (
        "Change between consecutive activities within a case."
    ),
    "state_source": "filtered_state",
    "reference_state_source": "viterbi_states",
    "confidence_source": "filtered_confidence",
    "time_availability": (
        "An event is only known to have ended after "
        "the next activity has been observed."
    ),
}

# ------------------------------------------------------------
# 18.6 — ESQUEMA DOS TRÊS LOGS
# ------------------------------------------------------------

EVENT_LOG_SCHEMAS18 = {
    "regime": {
        "case_column": "case_id",
        "activity_column": "activity",
        "timestamp_column": "start_timestamp",
        "mandatory_columns": [
            "case_id",
            "partition",
            "cycle_id",
            "activity",
            "state_id",
            "start_index",
            "end_index",
            "duration_hours",
            "start_timestamp",
            "end_timestamp",
        ],
    },
    "ramp": {
        "case_column": "case_id",
        "activity_column": "activity",
        "timestamp_column": "start_timestamp",
        "mandatory_columns": [
            "case_id",
            "partition",
            "cycle_id",
            "activity",
            "start_index",
            "end_index",
            "duration_hours",
            "start_timestamp",
            "end_timestamp",
        ],
    },
    "combined": {
        "case_column": "case_id",
        "activity_column": "activity",
        "timestamp_column": "start_timestamp",
        "mandatory_columns": [
            "case_id",
            "partition",
            "cycle_id",
            "activity",
            "state_id",
            "ramp_activity",
            "start_index",
            "end_index",
            "duration_hours",
            "start_timestamp",
            "end_timestamp",
        ],
    },
}

# ------------------------------------------------------------
# 18.7 — RECUPERAÇÃO DA ESTRUTURA HMM
# ------------------------------------------------------------

def extract_partition18(partition):
    source = hmm13_inference[partition]

    required = [
        "filtered_state",
        "viterbi_states",
        "filtered_probabilities",
        "filtered_confidence",
        "lengths",
    ]

    missing = [key for key in required if key not in source]

    if missing:
        raise ValueError(
            f"{partition}: campos ausentes: {missing}"
        )

    states = np.asarray(
        source["filtered_state"], dtype=int
    )

    viterbi = np.asarray(
        source["viterbi_states"], dtype=int
    )

    probabilities = np.asarray(
        source["filtered_probabilities"], dtype=float
    )

    confidence = np.asarray(
        source["filtered_confidence"], dtype=float
    )

    lengths = np.asarray(
        source["lengths"], dtype=int
    )

    n = len(states)

    assert len(viterbi) == n
    assert len(confidence) == n
    assert probabilities.shape == (n, N_STATES)
    assert lengths.sum() == n
    assert np.all(lengths > 0)
    assert np.all((states >= 0) & (states < N_STATES))
    assert np.allclose(
        probabilities.sum(axis=1),
        1.0,
        atol=1e-6,
    )

    return {
        "states": states,
        "viterbi": viterbi,
        "probabilities": probabilities,
        "confidence": confidence,
        "lengths": lengths,
    }


data18 = {
    partition: extract_partition18(partition)
    for partition in PARTITIONS_DEVELOPMENT
}

# ------------------------------------------------------------
# 18.8 — IDENTIFICAÇÃO DOS CASOS
# ------------------------------------------------------------

case_rows18 = []
case_index18 = {}

for partition, data in data18.items():

    lengths = data["lengths"]
    offsets = np.r_[0, np.cumsum(lengths)]

    case_ids = np.empty(
        len(data["states"]), dtype=object
    )

    for cycle_number, (start, end) in enumerate(
        zip(offsets[:-1], offsets[1:]),
        start=1,
    ):

        case_id = (
            f"{partition}_ON_{cycle_number:04d}"
        )

        case_ids[start:end] = case_id

        case_rows18.append({
            "case_id": case_id,
            "partition": partition,
            "cycle_id": cycle_number,
            "start_index": int(start),
            "end_index_exclusive": int(end),
            "duration_hours": int(end - start),
            "initial_state": int(
                data["states"][start]
            ),
            "final_state": int(
                data["states"][end - 1]
            ),
        })

    case_index18[partition] = case_ids

cases18 = pd.DataFrame(case_rows18)

assert cases18["case_id"].is_unique

_project_print("\n18.8 — CASOS IDENTIFICADOS")

_project_display(
    cases18.groupby("partition").agg(
        n_cases=("case_id", "count"),
        total_hours=("duration_hours", "sum"),
        median_case_hours=("duration_hours", "median"),
        min_case_hours=("duration_hours", "min"),
        max_case_hours=("duration_hours", "max"),
    ).reset_index()
)

# ------------------------------------------------------------
# 18.9 — VALIDAÇÃO DA SEMÂNTICA DOS EPISÓDIOS
# ------------------------------------------------------------

# Esta função gera apenas um exemplo metodológico
# de segmentação. Os event logs completos serão
# construídos na Célula 19.

def segment_states18(states):

    states = np.asarray(states, dtype=int)

    if len(states) == 0:
        return []

    change_points = np.flatnonzero(
        states[1:] != states[:-1]
    ) + 1

    starts = np.r_[0, change_points]
    ends = np.r_[change_points, len(states)]

    episodes = []

    for start, end in zip(starts, ends):
        episodes.append({
            "state_id": int(states[start]),
            "activity": (
                REGIME_ACTIVITIES18[
                    int(states[start])
                ]["activity"]
            ),
            "start_index": int(start),
            "end_index": int(end - 1),
            "duration_hours": int(end - start),
        })

    return episodes


example_partition18 = "train"
example_data18 = data18[example_partition18]

example_length18 = int(
    example_data18["lengths"][0]
)

example_states18 = (
    example_data18["states"][:example_length18]
)

example_episodes18 = segment_states18(
    example_states18
)

example_df18 = pd.DataFrame(
    example_episodes18
)

assert (
    example_df18["duration_hours"].sum()
    == example_length18
)

_project_print("\n18.9 — EXEMPLO DE SEGMENTAÇÃO")
_project_print(
    "Caso:",
    case_index18["train"][0],
    "| Horas:",
    example_length18,
)

_project_display(example_df18)

# ------------------------------------------------------------
# 18.10 — VERIFICAÇÃO DAS REGRAS DE SEGMENTAÇÃO
# ------------------------------------------------------------

validation_rows18 = []

for partition, data in data18.items():

    states = data["states"]
    lengths = data["lengths"]

    offset = 0
    total_episodes = 0
    reconstructed_hours = 0

    for cycle_id, length in enumerate(
        lengths, start=1
    ):

        cycle_states = states[
            offset:offset + length
        ]

        episodes = segment_states18(
            cycle_states
        )

        total_episodes += len(episodes)

        reconstructed = sum(
            episode["duration_hours"]
            for episode in episodes
        )

        reconstructed_hours += reconstructed

        assert reconstructed == length

        if len(episodes) > 1:
            for left, right in zip(
                episodes[:-1],
                episodes[1:],
            ):
                assert (
                    left["state_id"]
                    != right["state_id"]
                )

                assert (
                    left["end_index"] + 1
                    == right["start_index"]
                )

        offset += length

    assert offset == len(states)

    validation_rows18.append({
        "partition": partition,
        "n_hours": len(states),
        "n_cases": len(lengths),
        "n_regime_episodes": total_episodes,
        "reconstructed_hours": reconstructed_hours,
        "coverage_ok": (
            reconstructed_hours == len(states)
        ),
    })

validation18 = pd.DataFrame(
    validation_rows18
)

_project_print("\n18.10 — VALIDAÇÃO DA SEGMENTAÇÃO")

_project_display(validation18)

assert validation18["coverage_ok"].all()

# ------------------------------------------------------------
# 18.11 — VISUALIZAÇÃO DA SEMÂNTICA
# ------------------------------------------------------------

fig, axes = plt.subplots(
    2, 1,
    figsize=(13, 6),
    sharex=True,
    gridspec_kw={"height_ratios": [1, 1.2]},
)

x = np.arange(example_length18)

axes[0].step(
    x,
    example_states18,
    where="post",
    linewidth=2,
)

axes[0].set_yticks(range(N_STATES))
axes[0].set_yticklabels(
    [f"S{i}" for i in range(N_STATES)]
)

axes[0].set_ylabel("Estado HMM")
axes[0].set_title(
    "Sequência horária causal — primeiro ciclo ON"
)
axes[0].grid(alpha=0.2)

for _, episode in example_df18.iterrows():

    start = int(episode["start_index"])
    end = int(episode["end_index"])

    axes[1].barh(
        y=0,
        width=end - start + 1,
        left=start - 0.5,
        height=0.5,
        edgecolor="black",
        alpha=0.75,
    )

    axes[1].text(
        (start + end) / 2,
        0,
        episode["activity"],
        ha="center",
        va="center",
        fontsize=9,
    )

axes[1].set_yticks([])
axes[1].set_xlabel(
    "Hora relativa ao início do ciclo ON"
)
axes[1].set_title(
    "Consolidação em episódios de regime"
)
axes[1].set_xlim(
    -0.5,
    example_length18 - 0.5
)
axes[1].grid(axis="x", alpha=0.2)

plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# 18.12 — CONTRATO METODOLÓGICO
# ------------------------------------------------------------

METHODOLOGY18 = {
    "phase": 2,
    "cell": 18,
    "model": MODEL_NAME,
    "k": N_STATES,
    "seed": SEED,
    "case_definition": CASE_DEFINITION18,
    "regime_activities": REGIME_ACTIVITIES18,
    "ramp_activities": RAMP_ACTIVITIES18,
    "ramp_threshold_policy": RAMP_THRESHOLD_POLICY18,
    "event_definition": EVENT_DEFINITION18,
    "event_log_schemas": EVENT_LOG_SCHEMAS18,
    "development_partitions": (
        PARTITIONS_DEVELOPMENT
    ),
    "test_policy": (
        "Test remains excluded from methodological "
        "definition and threshold calibration."
    ),
    "causal_policy": (
        "Use filtered states for predictive process "
        "features. Retrospective Viterbi is diagnostic."
    ),
    "carbon_policy": (
        "CO2 emissions are not used to define "
        "cases or activities."
    ),
}

report_path18 = (
    REPORT_DIR18
    / "cell18_event_log_semantics.json"
)

with open(
    report_path18,
    "w",
    encoding="utf-8",
) as file:

    _project_dump(
        METHODOLOGY18,
        file,
        indent=2,
        ensure_ascii=False,
    )

# ------------------------------------------------------------
# 18.13 — OBJETOS DISPONÍVEIS PARA A CÉLULA 19
# ------------------------------------------------------------

hmm18_semantics = METHODOLOGY18
hmm18_cases = cases18
hmm18_case_index = case_index18
hmm18_validation = validation18

_project_print("\n" + "=" * 76)
_project_print("RESULTADO — CÉLULA 18/36")
_project_print("=" * 76)

_project_print(f"Modelo: {MODEL_NAME}")
_project_print(f"Estados: {N_STATES}")
_project_print(f"Semente: {SEED}")

_project_print(
    "Casos de desenvolvimento:",
    len(hmm18_cases)
)

_project_print(
    "Atividades de regime:",
    len(REGIME_ACTIVITIES18)
)

_project_print(
    "Categorias de rampa:",
    len(RAMP_ACTIVITIES18)
)

_project_print(
    "Esquemas de logs:",
    list(EVENT_LOG_SCHEMAS18)
)

_project_print(
    "Cobertura dos episódios:",
    "OK" if validation18["coverage_ok"].all()
    else "FALHA"
)

_project_print("Novo treinamento: NÃO")
_project_print("CO2 utilizado: NÃO")
_project_print("Teste utilizado: NÃO")
_project_print("Logs definitivos construídos: NÃO")

_project_print(f"Relatório: {report_path18}")

_project_print(
    "\nCÉLULA 18 CONCLUÍDA — "
    "semântica formalizada para a Célula 19."
)


In [ ]:

# ================================================================
# FASE 2 — CÉLULA 19/36
# CONSTRUÇÃO DOS EVENT LOGS — VERSÃO CORRIGIDA
#
# Fontes:
#   hmm13_inference
#   hmm18_cases
#   hmm18_case_index
#   hmm18_semantics
#   hmm08_data
#   df_fase2_operational
#
# Não treina HMM.
# Não usa CO2 para inferir estados.
# Não utiliza teste na descoberta de processos.
# ================================================================

import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import display

_project_print("=" * 78)
_project_print("FASE 2 — CÉLULA 19/36 | CONSTRUÇÃO DOS EVENT LOGS")
_project_print("=" * 78)

PARTITIONS19 = ("train", "validation")
MODEL19 = "M2_LOAD_DELTA"
K19 = 5

required19 = [
    "hmm13_inference",
    "hmm18_cases",
    "hmm18_case_index",
    "hmm18_semantics",
    "hmm08_data",
    "df_fase2_operational",
]

missing19 = [
    name for name in required19
    if name not in globals()
]

if missing19:
    raise RuntimeError(
        "Objetos ausentes: " + ", ".join(missing19)
    )

# ------------------------------------------------
# 19.1 — Diretórios
# ------------------------------------------------

if "RUN_DIR" not in globals() or not Path(RUN_DIR).is_dir():
    raise RuntimeError("Execute a configuração da Fase 2 para recuperar RUN_DIR.")
base19 = Path(RUN_DIR)

reports19 = base19 / "reports"
figures19 = base19 / "figures"
logs_dir19 = base19 / "event_logs"

for folder in (reports19, figures19, logs_dir19):
    folder.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------
# 19.2 — Inspeção das estruturas disponíveis
# ------------------------------------------------

source08_19 = hmm08_data[MODEL19]

_project_print("\nESTRUTURA DA CÉLULA 08")

for partition in PARTITIONS19:
    obj = source08_19[partition]

    _project_print(f"\n{partition}: {type(obj).__name__}")

    if isinstance(obj, dict):
        for key, value in obj.items():
            _project_print(
                " ",
                key,
                type(value).__name__,
                getattr(value, "shape", "")
            )

_project_print("\nCOLUNAS DA BASE OPERACIONAL")
_project_print(list(df_fase2_operational.columns))

# ------------------------------------------------
# 19.3 — Identificação das colunas originais
# ------------------------------------------------

def find_column19(df, candidates, required=True):
    lookup = {
        str(c).strip().lower(): c
        for c in df.columns
    }

    for candidate in candidates:
        key = candidate.strip().lower()
        if key in lookup:
            return lookup[key]

    if required:
        raise RuntimeError(
            "Coluna não localizada. Candidatas: "
            + str(candidates)
            + "\nDisponíveis: "
            + str(list(df.columns))
        )

    return None


oper19 = df_fase2_operational.copy()

date_col19 = find_column19(
    oper19,
    ["Date", "date", "data"]
)

hour_col19 = find_column19(
    oper19,
    ["Hour", "hour", "hora"],
    required=False
)

timestamp_col19 = find_column19(
    oper19,
    ["timestamp", "datetime", "date_time",
     "datetime_utc", "timestamp_utc"],
    required=False
)

load_col19 = find_column19(
    oper19,
    [
        "Gross Load (MW)",
        "gross_load_mw",
        "gross_load",
        "load_mw",
        "carga_mw",
        "load"
    ]
)

# O campo Operating Time não define carga.
# Ele é apenas preservado quando disponível.

operating_col19 = find_column19(
    oper19,
    ["Operating Time", "operating_time"],
    required=False
)

# ------------------------------------------------
# 19.4 — Timestamp original
# ------------------------------------------------

if timestamp_col19 is not None:
    oper19["timestamp19"] = pd.to_datetime(
        oper19[timestamp_col19],
        errors="coerce"
    )

else:
    if hour_col19 is None:
        raise RuntimeError(
            "Não foi possível construir timestamp: "
            "faltam coluna horária e timestamp."
        )

    date_values19 = pd.to_datetime(
        oper19[date_col19],
        errors="coerce"
    )

    hour_values19 = pd.to_numeric(
        oper19[hour_col19],
        errors="coerce"
    )

    valid_hours19 = hour_values19.dropna()

    # EPA CAMD normalmente registra horas 0–23.
    # Não assumir 1–24 sem confirmação.

    if not valid_hours19.between(0, 23).all():
        raise RuntimeError(
            "Horas fora do intervalo 0–23. "
            "Verifique a convenção temporal da fonte."
        )

    oper19["timestamp19"] = (
        date_values19
        + pd.to_timedelta(hour_values19, unit="h")
    )

if oper19["timestamp19"].isna().any():
    raise RuntimeError(
        "Existem timestamps inválidos na base operacional."
    )

if oper19["timestamp19"].duplicated().any():
    raise RuntimeError(
        "Existem timestamps duplicados na base operacional."
    )

oper19 = (
    oper19
    .sort_values("timestamp19")
    .reset_index(drop=True)
)

oper19["load_mw19"] = pd.to_numeric(
    oper19[load_col19],
    errors="coerce"
)

_project_print("\nBASE OPERACIONAL")
_project_print("Registros:", len(oper19))
_project_print("Primeiro timestamp:", oper19["timestamp19"].min())
_project_print("Último timestamp:", oper19["timestamp19"].max())

# ------------------------------------------------
# 19.5 — Recuperação da matriz HMM
# ------------------------------------------------

def get_matrix19(partition):

    obj = source08_19[partition]

    if not isinstance(obj, dict):
        raise RuntimeError(
            f"Formato inesperado em hmm08_data[{partition}]"
        )

    if "X" not in obj:
        raise RuntimeError(
            f"Matriz X ausente na partição {partition}"
        )

    X = np.asarray(obj["X"], dtype=float)

    inferred_X = np.asarray(
        hmm13_inference[partition]["X"],
        dtype=float
    )

    if X.shape != inferred_X.shape:
        raise RuntimeError(
            f"Dimensões incompatíveis em {partition}: "
            f"{X.shape} != {inferred_X.shape}"
        )

    if not np.allclose(
        X,
        inferred_X,
        rtol=1e-8,
        atol=1e-10,
        equal_nan=True
    ):
        raise RuntimeError(
            f"Matriz HMM divergente em {partition}."
        )

    return X, obj


# ------------------------------------------------
# 19.6 — Alinhamento temporal
# ------------------------------------------------
#
# Não se deve assumir que as primeiras 4504
# linhas de df_fase2_operational sejam o treino.
#
# A função exige um índice ou timestamp explícito
# associado às observações da partição.
# ------------------------------------------------

def locate_hourly19(partition):

    X, obj = get_matrix19(partition)

    n = len(X)

    # Possíveis vetores temporais preservados
    # na preparação da Célula 08.

    time_keys = [
        "timestamps",
        "timestamp",
        "datetime",
        "dates",
        "time_index",
        "index"
    ]

    for key in time_keys:

        if key not in obj:
            continue

        values = obj[key]

        if len(values) != n:
            continue

        # Índices inteiros só são aceitos quando
        # correspondem diretamente à base original.

        if key == "index":
            index_values = np.asarray(values)

            if (
                np.issubdtype(
                    index_values.dtype,
                    np.integer
                )
                and np.all(index_values >= 0)
                and np.all(index_values < len(oper19))
                and len(np.unique(index_values)) == n
            ):
                frame = oper19.iloc[
                    index_values
                ].copy()

                frame = frame.reset_index(drop=True)

                return frame, "hmm08_data.index"

        # Timestamps explícitos

        try:
            timestamps = pd.to_datetime(
                values,
                errors="coerce"
            )
        except Exception:
            continue

        if pd.isna(timestamps).any():
            continue

        if pd.Index(timestamps).duplicated().any():
            continue

        lookup = oper19.set_index("timestamp19")

        if not pd.Index(timestamps).isin(
            lookup.index
        ).all():
            continue

        frame = lookup.loc[
            list(timestamps)
        ].reset_index()

        return frame, f"hmm08_data.{key}"

    # Alternativa: índices presentes na própria
    # inferência da Célula 13.

    inferred = hmm13_inference[partition]

    for key in time_keys:

        if key not in inferred:
            continue

        values = inferred[key]

        if len(values) != n:
            continue

        try:
            timestamps = pd.to_datetime(
                values,
                errors="coerce"
            )
        except Exception:
            continue

        if pd.isna(timestamps).any():
            continue

        lookup = oper19.set_index("timestamp19")

        if pd.Index(timestamps).isin(
            lookup.index
        ).all():
            frame = lookup.loc[
                list(timestamps)
            ].reset_index()

            return frame, f"hmm13_inference.{key}"

    raise RuntimeError(
        f"\nALINHAMENTO TEMPORAL NÃO COMPROVADO: {partition}\n"
        "A Célula 08 e a Célula 13 não expõem um vetor "
        "de timestamps ou índices originais reconhecido.\n"
        "Não é seguro atribuir horários por posição.\n"
        "Examine a estrutura da partição impressa acima "
        "e recupere os timestamps do DataFrame usado "
        "na construção da matriz X."
    )


# ------------------------------------------------
# 19.7 — Construção da tabela horária
# ------------------------------------------------

hourly19 = {}
alignment19 = {}

for partition in PARTITIONS19:

    frame, source_name = locate_hourly19(partition)

    inference = hmm13_inference[partition]

    n = len(inference["filtered_state"])

    if len(frame) != n:
        raise RuntimeError(
            f"Quantidade de linhas incompatível: {partition}"
        )

    frame = frame.reset_index(drop=True)

    frame["partition"] = partition

    frame["state"] = np.asarray(
        inference["filtered_state"],
        dtype=int
    )

    frame["confidence"] = np.asarray(
        inference["filtered_confidence"],
        dtype=float
    )

    frame["entropy"] = np.asarray(
        inference["filtered_entropy"],
        dtype=float
    )

    frame["cycle_index_hmm"] = np.asarray(
        inference["cycle_index"]
    )

    frame["case_id"] = np.asarray(
        hmm18_case_index[partition]
    )

    if frame["case_id"].isna().any():
        raise RuntimeError(
            f"case_id ausente em {partition}"
        )

    # Rampas somente dentro do mesmo caso.
    frame["delta_load_mw"] = (
        frame.groupby(
            "case_id",
            sort=False
        )["load_mw19"].diff()
    )

    # Primeira hora não tem rampa observável
    # dentro do ciclo.

    hourly19[partition] = frame

    alignment19[partition] = source_name

    _project_print(
        f"\n{partition.upper()}: "
        f"{n} horas | alinhamento: {source_name}"
    )

# ------------------------------------------------
# 19.8 — Validação contra os casos da Célula 18
# ------------------------------------------------

cases19 = hmm18_cases.copy()

required_case_columns19 = [
    "case_id",
    "partition",
    "start_index",
    "end_index_exclusive",
    "duration_hours"
]

for col in required_case_columns19:
    if col not in cases19.columns:
        raise RuntimeError(
            f"Coluna ausente em hmm18_cases: {col}"
        )

for partition in PARTITIONS19:

    frame = hourly19[partition]

    cases = cases19.loc[
        cases19["partition"] == partition
    ].copy()

    for _, case in cases.iterrows():

        start = int(case["start_index"])
        end = int(case["end_index_exclusive"])

        observed = frame.iloc[start:end]

        if len(observed) != int(case["duration_hours"]):
            raise RuntimeError(
                f"Duração inconsistente: {case['case_id']}"
            )

        if not (
            observed["case_id"] == case["case_id"]
        ).all():
            raise RuntimeError(
                f"Alinhamento de caso inconsistente: "
                f"{case['case_id']}"
            )

        times = observed["timestamp19"]

        if len(times) > 1:
            steps = times.diff().dropna()

            if not (
                steps == pd.Timedelta(hours=1)
            ).all():
                raise RuntimeError(
                    f"Lacuna temporal dentro do caso "
                    f"{case['case_id']}"
                )

# ------------------------------------------------
# 19.9 — Política de classificação de rampas
# ------------------------------------------------
#
# O limiar deve ser definido sem usar o teste.
# Se houver limiar documentado na Célula 18,
# recuperar; caso contrário, usar regra explícita
# de engenharia e registrar a escolha.
# ------------------------------------------------

RAMP_THRESHOLD_MW19 = 20.0

def ramp_activity19(delta):

    if pd.isna(delta):
        return "RAMP_UNDEFINED"

    if delta > RAMP_THRESHOLD_MW19:
        return "RAMP_UP"

    if delta < -RAMP_THRESHOLD_MW19:
        return "RAMP_DOWN"

    return "RAMP_STABLE"


for partition in PARTITIONS19:

    frame = hourly19[partition]

    frame["regime_activity"] = (
        "REGIME_S" + frame["state"].astype(str)
    )

    frame["ramp_activity"] = (
        frame["delta_load_mw"]
        .apply(ramp_activity19)
    )

    frame["combined_activity"] = (
        frame["regime_activity"]
        + "__"
        + frame["ramp_activity"]
    )

# ------------------------------------------------
# 19.10 — Consolidação causal em episódios
# ------------------------------------------------

def build_episodes19(frame, activity_column):

    records = []

    for case_id, group in frame.groupby(
        "case_id",
        sort=False
    ):

        group = group.reset_index(drop=True)

        activities = group[
            activity_column
        ].astype(str).to_numpy()

        start = 0

        for i in range(1, len(group) + 1):

            boundary = (
                i == len(group)
                or (
                    i < len(group)
                    and activities[i] != activities[start]
                )
            )

            if not boundary:
                continue

            segment = group.iloc[start:i]

            records.append({
                "case:concept:name": str(case_id),
                "concept:name": activities[start],
                "time:timestamp": segment[
                    "timestamp19"
                ].iloc[0],
                "end_timestamp": segment[
                    "timestamp19"
                ].iloc[-1],
                "partition": segment[
                    "partition"
                ].iloc[0],
                "start_hour_in_case": int(start),
                "end_hour_exclusive": int(i),
                "duration_hours": int(i - start),
                "mean_confidence": float(
                    segment["confidence"].mean()
                )
            })

            start = i

    return pd.DataFrame(records)


hmm19_event_logs = {}

for partition in PARTITIONS19:

    frame = hourly19[partition]

    hmm19_event_logs[partition] = {
        "regime": build_episodes19(
            frame,
            "regime_activity"
        ),
        "ramp": build_episodes19(
            frame,
            "ramp_activity"
        ),
        "combined": build_episodes19(
            frame,
            "combined_activity"
        )
    }

# ------------------------------------------------
# 19.11 — Auditoria dos logs
# ------------------------------------------------

audit_rows19 = []

for partition in PARTITIONS19:

    frame = hourly19[partition]

    for log_name, log in (
        hmm19_event_logs[partition].items()
    ):

        duration_total = int(
            log["duration_hours"].sum()
        )

        if duration_total != len(frame):
            raise RuntimeError(
                f"Cobertura incorreta: "
                f"{partition}/{log_name}"
            )

        if log["time:timestamp"].isna().any():
            raise RuntimeError(
                f"Timestamps ausentes: "
                f"{partition}/{log_name}"
            )

        audit_rows19.append({
            "partition": partition,
            "event_log": log_name,
            "n_cases": int(
                log["case:concept:name"].nunique()
            ),
            "n_events": len(log),
            "n_activities": int(
                log["concept:name"].nunique()
            ),
            "covered_hours": duration_total,
            "mean_episode_hours": float(
                log["duration_hours"].mean()
            ),
            "median_episode_hours": float(
                log["duration_hours"].median()
            )
        })

audit19_summary = pd.DataFrame(audit_rows19)

_project_print("\n" + "=" * 78)
_project_print("RESUMO DOS EVENT LOGS")
_project_print("=" * 78)

_project_display(audit19_summary.round(3))

# ------------------------------------------------
# 19.12 — Visualização das atividades
# ------------------------------------------------

fig, axes = plt.subplots(
    1, 3,
    figsize=(17, 4.5)
)

for ax, log_name in zip(
    axes,
    ("regime", "ramp", "combined")
):

    log = hmm19_event_logs["train"][log_name]

    counts = (
        log["concept:name"]
        .value_counts()
        .head(15)
        .sort_values()
    )

    counts.plot.barh(ax=ax)

    ax.set_title(
        f"Treinamento — {log_name}"
    )

    ax.set_xlabel("Número de episódios")
    ax.set_ylabel("Atividade")
    ax.grid(axis="x", alpha=0.2)

plt.tight_layout()

figure_path19 = (
    figures19
    / "cell19_activity_frequencies.png"
)

plt.savefig(
    figure_path19,
    dpi=180,
    bbox_inches="tight"
)

plt.show()

# ------------------------------------------------
# 19.13 — Exportação
# ------------------------------------------------

exported19 = []

for partition in PARTITIONS19:

    for log_name, log in (
        hmm19_event_logs[partition].items()
    ):

        path = (
            logs_dir19
            / f"cell19_{partition}_{log_name}.csv"
        )

        _project_csv(log, 
            path,
            index=False
        )

        exported19.append(str(path))

summary_path19 = (
    reports19
    / "cell19_event_log_summary.csv"
)

_project_csv(audit19_summary, 
    summary_path19,
    index=False
)

metadata19 = {
    "phase": 2,
    "cell": 19,
    "model": MODEL19,
    "k": K19,
    "partitions": list(PARTITIONS19),
    "alignment_sources": alignment19,
    "ramp_threshold_mw": RAMP_THRESHOLD_MW19,
    "event_logs": [
        "regime",
        "ramp",
        "combined"
    ],
    "test_used": False,
    "hmm_retrained": False,
    "co2_used_for_state_inference": False,
    "files": exported19
}

metadata_path19 = (
    reports19
    / "cell19_event_log_metadata.json"
)

with open(
    metadata_path19,
    "w",
    encoding="utf-8"
) as f:
    _project_dump(
        metadata19,
        f,
        indent=2,
        ensure_ascii=False
    )

# Objetos para a Célula 20
hmm19_hourly = hourly19
hmm19_logs = hmm19_event_logs

_project_print("\n" + "=" * 78)
_project_print("RESULTADO DA CÉLULA 19")
_project_print("=" * 78)

_project_print("Logs construídos: regime, ramp, combined")
_project_print("Partições: train, validation")
_project_print("Teste utilizado: NÃO")
_project_print("HMM retreinado: NÃO")
_project_print("CO2 utilizado para inferência: NÃO")
_project_print("Arquivos exportados:", len(exported19))
_project_print("Relatório:", summary_path19)
_project_print("Metadados:", metadata_path19)

_project_print("\nCÉLULA 19 CONCLUÍDA.")


In [ ]:

# ================================================================
# FASE 2 — CÉLULA 20/36
# VALIDAÇÃO FORMAL DOS EVENT LOGS
#
# Unidade: Iatan — Facility 6065 — Unit 2
# Ano: 2025
#
# OBJETIVOS
# 1. Validar integridade e cobertura dos Event Logs.
# 2. Verificar a reconstrução exata dos episódios.
# 3. Auditar timestamps, casos, atividades e durações.
# 4. Verificar coerência com a inferência causal do HMM.
# 5. Avaliar a distribuição das atividades.
# 6. Registrar a política de classificação de rampas.
# 7. Preparar logs validados para Process Mining.
#
# RESTRIÇÕES
# - Não treinar ou modificar o HMM.
# - Não utilizar CO2 na inferência.
# - Não utilizar o conjunto de teste.
# - Não selecionar modelos de processo nesta célula.
# ================================================================

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import display

_project_print("=" * 82)
_project_print("FASE 2 — CÉLULA 20/36 | VALIDAÇÃO FORMAL DOS EVENT LOGS")
_project_print("=" * 82)

# ================================================================
# 20.1 — CONFIGURAÇÃO
# ================================================================

PARTITIONS20 = ("train", "validation")
LOG_TYPES20 = ("regime", "ramp", "combined")

REQUIRED20 = [
    "hmm19_hourly",
    "hmm19_logs",
    "hmm18_cases",
    "hmm18_semantics",
    "hmm13_inference",
]

missing20 = [
    name for name in REQUIRED20
    if name not in globals()
]

if missing20:
    raise RuntimeError(
        "Execute as células anteriores. Objetos ausentes: "
        + ", ".join(missing20)
    )

if "RUN_DIR" not in globals() or not Path(RUN_DIR).is_dir():
    raise RuntimeError("Execute a configuração da Fase 2 para recuperar RUN_DIR.")
BASE20 = Path(RUN_DIR)

REPORTS20 = BASE20 / "reports"
FIGURES20 = BASE20 / "figures"
LOGS20 = BASE20 / "event_logs"

for folder in (REPORTS20, FIGURES20, LOGS20):
    folder.mkdir(parents=True, exist_ok=True)

# Política efetivamente utilizada na Célula 19.
# A validação formal não altera esse parâmetro.

RAMP_THRESHOLD20 = float(
    globals().get("RAMP_THRESHOLD_MW19", 20.0)
)

_project_print("\nConfiguração:")
_project_print("Partições:", PARTITIONS20)
_project_print("Logs:", LOG_TYPES20)
_project_print("Limiar de rampa utilizado:", RAMP_THRESHOLD20, "MW/h")
_project_print("Diretório:", BASE20)

# ================================================================
# 20.2 — ESTRUTURA DE AUDITORIA
# ================================================================

checks20 = []
details20 = []

def register20(
    partition,
    log_type,
    check,
    passed,
    detail=""
):
    checks20.append({
        "partition": str(partition),
        "event_log": str(log_type),
        "check": str(check),
        "passed": bool(passed),
        "detail": str(detail)
    })


def assert_check20(
    partition,
    log_type,
    check,
    condition,
    detail=""
):
    passed = bool(condition)

    register20(
        partition,
        log_type,
        check,
        passed,
        detail
    )

    return passed


REQUIRED_EVENT_COLUMNS20 = [
    "case:concept:name",
    "concept:name",
    "time:timestamp",
    "end_timestamp",
    "partition",
    "start_hour_in_case",
    "end_hour_exclusive",
    "duration_hours",
    "mean_confidence"
]

EXPECTED_HOURLY_COLUMNS20 = [
    "timestamp19",
    "load_mw19",
    "case_id",
    "state",
    "confidence",
    "entropy",
    "regime_activity",
    "ramp_activity",
    "combined_activity"
]

ACTIVITY_COLUMNS20 = {
    "regime": "regime_activity",
    "ramp": "ramp_activity",
    "combined": "combined_activity"
}

# ================================================================
# 20.3 — VERIFICAÇÃO DA BASE HORÁRIA
# ================================================================

_project_print("\n" + "=" * 82)
_project_print("20.3 — INTEGRIDADE DAS BASES HORÁRIAS")
_project_print("=" * 82)

hourly_stats20 = []

for partition in PARTITIONS20:

    frame = hmm19_hourly[partition].copy()

    missing_cols = [
        col for col in EXPECTED_HOURLY_COLUMNS20
        if col not in frame.columns
    ]

    if missing_cols:
        raise RuntimeError(
            f"{partition}: colunas ausentes: {missing_cols}"
        )

    n_hours = len(frame)

    inference = hmm13_inference[partition]

    expected_n = len(inference["filtered_state"])

    assert_check20(
        partition,
        "hourly",
        "hour_count_matches_hmm",
        n_hours == expected_n,
        f"observado={n_hours}; esperado={expected_n}"
    )

    assert_check20(
        partition,
        "hourly",
        "timestamps_non_null",
        frame["timestamp19"].notna().all()
    )

    assert_check20(
        partition,
        "hourly",
        "timestamps_unique",
        not frame["timestamp19"].duplicated().any()
    )

    assert_check20(
        partition,
        "hourly",
        "case_ids_non_null",
        frame["case_id"].notna().all()
    )

    states = np.asarray(
        inference["filtered_state"],
        dtype=int
    )

    assert_check20(
        partition,
        "hourly",
        "states_match_hmm13",
        np.array_equal(
            frame["state"].to_numpy(dtype=int),
            states
        )
    )

    confidence = np.asarray(
        inference["filtered_confidence"],
        dtype=float
    )

    assert_check20(
        partition,
        "hourly",
        "confidence_matches_hmm13",
        np.allclose(
            frame["confidence"].to_numpy(dtype=float),
            confidence,
            rtol=1e-10,
            atol=1e-12,
            equal_nan=False
        )
    )

    assert_check20(
        partition,
        "hourly",
        "confidence_in_unit_interval",
        (
            frame["confidence"].notna().all()
            and frame["confidence"].between(0, 1).all()
        )
    )

    assert_check20(
        partition,
        "hourly",
        "states_in_expected_range",
        frame["state"].isin(range(5)).all()
    )

    hourly_stats20.append({
        "partition": partition,
        "n_hours": n_hours,
        "n_cases": frame["case_id"].nunique(),
        "n_states": frame["state"].nunique(),
        "min_timestamp": str(
            frame["timestamp19"].min()
        ),
        "max_timestamp": str(
            frame["timestamp19"].max()
        )
    })

audit20_hourly = pd.DataFrame(hourly_stats20)

_project_display(audit20_hourly)

# ================================================================
# 20.4 — VALIDAÇÃO DOS CASOS OPERACIONAIS
# ================================================================

_project_print("\n" + "=" * 82)
_project_print("20.4 — VALIDAÇÃO DOS CASOS")
_project_print("=" * 82)

case_rows20 = []

for partition in PARTITIONS20:

    frame = hmm19_hourly[partition].reset_index(drop=True)

    cases = (
        hmm18_cases.loc[
            hmm18_cases["partition"] == partition
        ]
        .sort_values("start_index")
        .reset_index(drop=True)
    )

    expected_lengths = np.asarray(
        hmm13_inference[partition]["lengths"],
        dtype=int
    )

    observed_lengths = cases[
        "duration_hours"
    ].to_numpy(dtype=int)

    assert_check20(
        partition,
        "cases",
        "cycle_lengths_match_hmm13",
        np.array_equal(
            observed_lengths,
            expected_lengths
        )
    )

    assert_check20(
        partition,
        "cases",
        "cycle_hours_cover_partition",
        int(observed_lengths.sum()) == len(frame)
    )

    previous_end = 0

    for _, case in cases.iterrows():

        case_id = case["case_id"]
        start = int(case["start_index"])
        end = int(case["end_index_exclusive"])
        duration = int(case["duration_hours"])

        segment = frame.iloc[start:end]

        index_contiguous = start == previous_end

        same_case = (
            len(segment) == duration
            and segment["case_id"].eq(case_id).all()
        )

        timestamps = pd.to_datetime(
            segment["timestamp19"]
        )

        hourly_contiguous = (
            len(timestamps) <= 1
            or timestamps.diff().dropna().eq(
                pd.Timedelta(hours=1)
            ).all()
        )

        valid = (
            index_contiguous
            and same_case
            and hourly_contiguous
            and duration > 0
        )

        assert_check20(
            partition,
            "cases",
            f"case_{case_id}",
            valid,
            (
                f"start={start}; end={end}; "
                f"duration={duration}"
            )
        )

        case_rows20.append({
            "partition": partition,
            "case_id": str(case_id),
            "duration_hours": duration,
            "start_timestamp": (
                timestamps.iloc[0]
                if len(timestamps) else pd.NaT
            ),
            "end_timestamp": (
                timestamps.iloc[-1]
                if len(timestamps) else pd.NaT
            ),
            "valid": bool(valid)
        })

        previous_end = end

    assert_check20(
        partition,
        "cases",
        "final_index_matches_partition",
        previous_end == len(frame)
    )

audit20_cases = pd.DataFrame(case_rows20)

_project_display(
    audit20_cases.groupby("partition").agg(
        n_cases=("case_id", "count"),
        total_hours=("duration_hours", "sum"),
        min_duration=("duration_hours", "min"),
        median_duration=("duration_hours", "median"),
        max_duration=("duration_hours", "max"),
        all_valid=("valid", "all")
    ).reset_index()
)

# ================================================================
# 20.5 — VALIDAÇÃO FORMAL DOS EVENT LOGS
# ================================================================

_project_print("\n" + "=" * 82)
_project_print("20.5 — VALIDAÇÃO DOS EPISÓDIOS")
_project_print("=" * 82)

event_stats20 = []

for partition in PARTITIONS20:

    hourly = hmm19_hourly[partition].reset_index(
        drop=True
    )

    for log_type in LOG_TYPES20:

        log = hmm19_logs[partition][log_type].copy()

        missing_cols = [
            col for col in REQUIRED_EVENT_COLUMNS20
            if col not in log.columns
        ]

        if missing_cols:
            raise RuntimeError(
                f"{partition}/{log_type}: "
                f"colunas ausentes: {missing_cols}"
            )

        log["time:timestamp"] = pd.to_datetime(
            log["time:timestamp"]
        )

        log["end_timestamp"] = pd.to_datetime(
            log["end_timestamp"]
        )

        # -----------------------------------------
        # Estrutura geral
        # -----------------------------------------

        assert_check20(
            partition,
            log_type,
            "nonempty_log",
            len(log) > 0
        )

        assert_check20(
            partition,
            log_type,
            "activities_non_null",
            log["concept:name"].notna().all()
        )

        assert_check20(
            partition,
            log_type,
            "case_ids_non_null",
            log["case:concept:name"].notna().all()
        )

        assert_check20(
            partition,
            log_type,
            "timestamps_non_null",
            (
                log["time:timestamp"].notna().all()
                and log["end_timestamp"].notna().all()
            )
        )

        assert_check20(
            partition,
            log_type,
            "start_not_after_end",
            (
                log["time:timestamp"]
                <= log["end_timestamp"]
            ).all()
        )

        assert_check20(
            partition,
            log_type,
            "positive_duration",
            (log["duration_hours"] > 0).all()
        )

        assert_check20(
            partition,
            log_type,
            "duration_matches_indices",
            (
                log["end_hour_exclusive"]
                - log["start_hour_in_case"]
                == log["duration_hours"]
            ).all()
        )

        assert_check20(
            partition,
            log_type,
            "total_hour_coverage",
            int(log["duration_hours"].sum())
            == len(hourly)
        )

        assert_check20(
            partition,
            log_type,
            "partition_label_consistent",
            log["partition"].eq(partition).all()
        )

        # -----------------------------------------
        # Reconstrução exata das sequências
        # -----------------------------------------

        activity_column = ACTIVITY_COLUMNS20[
            log_type
        ]

        reconstructed = []
        all_cases_valid = True

        for case_id, group in hourly.groupby(
            "case_id",
            sort=False
        ):

            original_activities = group[
                activity_column
            ].astype(str).tolist()

            episodes = (
                log.loc[
                    log["case:concept:name"].astype(str)
                    == str(case_id)
                ]
                .sort_values("start_hour_in_case")
            )

            if episodes.empty:
                all_cases_valid = False
                continue

            expected_start = 0
            reconstructed_case = []

            for _, event in episodes.iterrows():

                start = int(
                    event["start_hour_in_case"]
                )

                end = int(
                    event["end_hour_exclusive"]
                )

                duration = int(
                    event["duration_hours"]
                )

                if start != expected_start:
                    all_cases_valid = False

                if end - start != duration:
                    all_cases_valid = False

                if end > len(group):
                    all_cases_valid = False
                    continue

                original_segment = group.iloc[
                    start:end
                ]

                if original_segment.empty:
                    all_cases_valid = False
                    continue

                expected_start_time = (
                    original_segment["timestamp19"].iloc[0]
                )

                expected_end_time = (
                    original_segment["timestamp19"].iloc[-1]
                )

                if (
                    pd.Timestamp(event["time:timestamp"])
                    != pd.Timestamp(expected_start_time)
                ):
                    all_cases_valid = False

                if (
                    pd.Timestamp(event["end_timestamp"])
                    != pd.Timestamp(expected_end_time)
                ):
                    all_cases_valid = False

                activity = str(
                    event["concept:name"]
                )

                if not original_segment[
                    activity_column
                ].astype(str).eq(activity).all():
                    all_cases_valid = False

                reconstructed_case.extend(
                    [activity] * duration
                )

                expected_start = end

            if expected_start != len(group):
                all_cases_valid = False

            if reconstructed_case != original_activities:
                all_cases_valid = False

            reconstructed.extend(reconstructed_case)

        assert_check20(
            partition,
            log_type,
            "exact_hourly_reconstruction",
            all_cases_valid,
            (
                f"horas_reconstruidas="
                f"{len(reconstructed)}"
            )
        )

        # -----------------------------------------
        # Ausência de episódios consecutivos
        # com mesma atividade no mesmo caso
        # -----------------------------------------

        no_redundant_episodes = True

        for case_id, group in log.groupby(
            "case:concept:name",
            sort=False
        ):

            ordered = group.sort_values(
                "start_hour_in_case"
            )

            activities = ordered[
                "concept:name"
            ].astype(str).to_numpy()

            if (
                len(activities) > 1
                and np.any(
                    activities[1:]
                    == activities[:-1]
                )
            ):
                no_redundant_episodes = False

        assert_check20(
            partition,
            log_type,
            "no_adjacent_duplicate_episodes",
            no_redundant_episodes
        )

        event_stats20.append({
            "partition": partition,
            "event_log": log_type,
            "n_cases": log[
                "case:concept:name"
            ].nunique(),
            "n_events": len(log),
            "n_activities": log[
                "concept:name"
            ].nunique(),
            "covered_hours": int(
                log["duration_hours"].sum()
            ),
            "median_duration_hours": float(
                log["duration_hours"].median()
            ),
            "exact_reconstruction": (
                bool(all_cases_valid)
            )
        })

audit20_events = pd.DataFrame(event_stats20)

_project_display(audit20_events)

# ================================================================
# 20.6 — COERÊNCIA DAS ATIVIDADES
# ================================================================

_project_print("\n" + "=" * 82)
_project_print("20.6 — COERÊNCIA SEMÂNTICA")
_project_print("=" * 82)

EXPECTED_REGIMES20 = {
    f"REGIME_S{i}" for i in range(5)
}

EXPECTED_RAMPS20 = {
    "RAMP_UP",
    "RAMP_DOWN",
    "RAMP_STABLE",
    "RAMP_UNDEFINED"
}

semantic_rows20 = []

for partition in PARTITIONS20:

    frame = hmm19_hourly[partition]

    regimes = set(
        frame["regime_activity"].unique()
    )

    ramps = set(
        frame["ramp_activity"].unique()
    )

    regime_ok = regimes.issubset(
        EXPECTED_REGIMES20
    )

    ramp_ok = ramps.issubset(
        EXPECTED_RAMPS20
    )

    assert_check20(
        partition,
        "semantics",
        "valid_regime_activities",
        regime_ok
    )

    assert_check20(
        partition,
        "semantics",
        "valid_ramp_activities",
        ramp_ok
    )

    # RAMP_UNDEFINED somente na primeira hora
    # de cada caso, conforme a política da Célula 19.

    first_hour = (
        frame.groupby("case_id", sort=False)
        .cumcount()
        .eq(0)
    )

    undefined = frame[
        "ramp_activity"
    ].eq("RAMP_UNDEFINED")

    undefined_ok = (
        np.array_equal(
            undefined.to_numpy(),
            first_hour.to_numpy()
        )
    )

    assert_check20(
        partition,
        "semantics",
        "undefined_ramp_only_at_case_start",
        undefined_ok,
        (
            f"undefined={int(undefined.sum())}; "
            f"cases={frame['case_id'].nunique()}"
        )
    )

    # Coerência da regra de rampas, sem alterar
    # os eventos previamente produzidos.

    delta = pd.to_numeric(
        frame["delta_load_mw"],
        errors="coerce"
    )

    expected_ramp = np.select(
        [
            delta.isna(),
            delta > RAMP_THRESHOLD20,
            delta < -RAMP_THRESHOLD20
        ],
        [
            "RAMP_UNDEFINED",
            "RAMP_UP",
            "RAMP_DOWN"
        ],
        default="RAMP_STABLE"
    )

    ramp_rule_ok = np.array_equal(
        frame["ramp_activity"].astype(str).to_numpy(),
        expected_ramp
    )

    assert_check20(
        partition,
        "semantics",
        "ramp_threshold_reproducible",
        ramp_rule_ok
    )

    semantic_rows20.append({
        "partition": partition,
        "n_regime_activities": len(regimes),
        "n_ramp_activities": len(ramps),
        "undefined_ramp_hours": int(
            undefined.sum()
        ),
        "n_cases": int(
            frame["case_id"].nunique()
        ),
        "ramp_threshold_mw": RAMP_THRESHOLD20,
        "semantic_checks_passed": bool(
            regime_ok
            and ramp_ok
            and undefined_ok
            and ramp_rule_ok
        )
    })

audit20_semantics = pd.DataFrame(
    semantic_rows20
)

_project_display(audit20_semantics)

# ================================================================
# 20.7 — ANÁLISE DESCRITIVA DAS ATIVIDADES
# ================================================================

activity_rows20 = []

for partition in PARTITIONS20:

    for log_type in LOG_TYPES20:

        log = hmm19_logs[partition][log_type]

        counts = (
            log["concept:name"]
            .value_counts()
        )

        for activity, count in counts.items():

            activity_rows20.append({
                "partition": partition,
                "event_log": log_type,
                "activity": activity,
                "n_episodes": int(count),
                "episode_share": float(
                    count / len(log)
                )
            })

audit20_activities = pd.DataFrame(
    activity_rows20
)

# Comparação de distribuição das atividades.
# Total variation distance:
# 0 = distribuições idênticas
# 1 = distribuições completamente distintas

distribution_rows20 = []

for log_type in LOG_TYPES20:

    subset = audit20_activities.loc[
        audit20_activities["event_log"] == log_type
    ]

    pivot = (
        subset.pivot_table(
            index="activity",
            columns="partition",
            values="episode_share",
            fill_value=0
        )
    )

    for partition in PARTITIONS20:
        if partition not in pivot.columns:
            pivot[partition] = 0.0

    tvd = 0.5 * np.abs(
        pivot["train"]
        - pivot["validation"]
    ).sum()

    distribution_rows20.append({
        "event_log": log_type,
        "total_variation_distance": float(tvd),
        "n_activities_train": int(
            audit20_events.loc[
                (audit20_events["partition"] == "train")
                & (audit20_events["event_log"] == log_type),
                "n_activities"
            ].iloc[0]
        ),
        "n_activities_validation": int(
            audit20_events.loc[
                (audit20_events["partition"] == "validation")
                & (audit20_events["event_log"] == log_type),
                "n_activities"
            ].iloc[0]
        )
    })

audit20_distribution = pd.DataFrame(
    distribution_rows20
)

_project_print("\nDISTÂNCIA ENTRE DISTRIBUIÇÕES")
_project_display(audit20_distribution.round(4))

# ================================================================
# 20.8 — VISUALIZAÇÕES
# ================================================================

fig, axes = plt.subplots(
    1, 3,
    figsize=(17, 4.8)
)

for ax, log_type in zip(
    axes,
    LOG_TYPES20
):

    subset = audit20_activities.loc[
        audit20_activities["event_log"] == log_type
    ]

    pivot = subset.pivot_table(
        index="activity",
        columns="partition",
        values="episode_share",
        fill_value=0
    )

    pivot = pivot.sort_values(
        "train",
        ascending=False
    ).head(12)

    pivot.plot.bar(
        ax=ax,
        width=0.8
    )

    ax.set_title(
        f"Distribuição das atividades — {log_type}"
    )

    ax.set_xlabel("Atividade")
    ax.set_ylabel("Proporção dos episódios")
    ax.tick_params(
        axis="x",
        rotation=75
    )

    ax.grid(
        axis="y",
        alpha=0.2
    )

plt.tight_layout()

FIGURE20 = (
    FIGURES20
    / "cell20_activity_distribution.png"
)

plt.savefig(
    FIGURE20,
    dpi=180,
    bbox_inches="tight"
)

plt.show()

# ================================================================
# 20.9 — CONSOLIDAÇÃO DA AUDITORIA
# ================================================================

audit20_checks = pd.DataFrame(
    checks20
)

failed20 = audit20_checks.loc[
    ~audit20_checks["passed"]
].copy()

summary20 = (
    audit20_checks.groupby(
        ["partition", "event_log"],
        as_index=False
    )
    .agg(
        n_checks=("passed", "size"),
        passed_checks=("passed", "sum")
    )
)

summary20["failed_checks"] = (
    summary20["n_checks"]
    - summary20["passed_checks"]
)

summary20["all_passed"] = (
    summary20["failed_checks"] == 0
)

_project_print("\n" + "=" * 82)
_project_print("RESULTADO DA AUDITORIA")
_project_print("=" * 82)

_project_display(summary20)

if len(failed20):
    _project_print("\nVERIFICAÇÕES NÃO APROVADAS")
    _project_display(failed20)
else:
    _project_print("\nTodas as verificações formais foram aprovadas.")

# ================================================================
# 20.10 — EXPORTAÇÃO DOS RELATÓRIOS
# ================================================================

reports_to_save20 = {
    "cell20_checks.csv": audit20_checks,
    "cell20_summary.csv": summary20,
    "cell20_cases.csv": audit20_cases,
    "cell20_events.csv": audit20_events,
    "cell20_semantics.csv": audit20_semantics,
    "cell20_activity_distribution.csv": audit20_activities,
    "cell20_distribution_distance.csv": audit20_distribution
}

for filename, df in reports_to_save20.items():

    _project_csv(df, 
        REPORTS20 / filename,
        index=False
    )

all_passed20 = bool(
    audit20_checks["passed"].all()
)

metadata20 = {
    "phase": 2,
    "cell": 20,
    "model": "M2_LOAD_DELTA",
    "k": 5,
    "seed": 11,
    "partitions": list(PARTITIONS20),
    "event_logs": list(LOG_TYPES20),
    "ramp_threshold_mw": RAMP_THRESHOLD20,
    "n_checks": int(
        len(audit20_checks)
    ),
    "n_failed_checks": int(
        len(failed20)
    ),
    "formal_validation_passed": all_passed20,
    "test_used": False,
    "hmm_retrained": False,
    "co2_used_for_inference": False,
    "process_model_discovered": False
}

METADATA20 = (
    REPORTS20
    / "cell20_validation_metadata.json"
)

with open(
    METADATA20,
    "w",
    encoding="utf-8"
) as f:
    _project_dump(
        metadata20,
        f,
        indent=2,
        ensure_ascii=False
    )

# ================================================================
# 20.11 — DISPONIBILIZAÇÃO PARA A CÉLULA 21
# ================================================================

hmm20_validated_logs = (
    hmm19_logs if all_passed20 else None
)

hmm20_validation_passed = all_passed20

_project_print("\n" + "=" * 82)
_project_print("FASE 2 — CÉLULA 20/36 | RESULTADO FINAL")
_project_print("=" * 82)

_project_print("Verificações:", len(audit20_checks))
_project_print("Aprovadas:", int(audit20_checks["passed"].sum()))
_project_print("Reprovadas:", len(failed20))
_project_print("Integridade formal:", "APROVADA" if all_passed20 else "REPROVADA")
_project_print("Limiar de rampa:", RAMP_THRESHOLD20, "MW/h")
_project_print("Teste utilizado: NÃO")
_project_print("HMM modificado: NÃO")
_project_print("CO2 usado na inferência: NÃO")
_project_print("Relatórios:", REPORTS20)
_project_print("Figura:", FIGURE20)

if all_passed20:
    _project_print(
        "\nCÉLULA 20 CONCLUÍDA — "
        "Event Logs formalmente validados."
    )
    _project_print(
        "Próxima etapa: Célula 21/36 — "
        "Variantes e complexidade das sequências."
    )
else:
    raise RuntimeError(
        "A validação formal encontrou inconsistências. "
        "Consulte audit20_checks antes de avançar."
    )


In [ ]:

# ============================================================
# FASE 2 — CÉLULA 21/36
# VARIANTES E COMPLEXIDADE DAS SEQUÊNCIAS OPERACIONAIS
# ============================================================
#
# OBJETIVOS
# 1. Recuperar os Event Logs da Célula 19.
# 2. Reconstruir uma sequência por caso e tipo de log.
# 3. Identificar variantes exatas e suas frequências.
# 4. Medir diversidade, entropia e complexidade.
# 5. Identificar padrões de transições e trigramas.
# 6. Comparar treinamento e validação.
# 7. Medir variantes inéditas na validação.
# 8. Gerar gráficos e relatórios reproduzíveis.
#
# RESTRIÇÕES
# - Não utilizar a partição test.
# - Não utilizar emissões de CO2.
# - Não modificar o HMM.
# - Não alterar os Event Logs originais.
# - Não ajustar parâmetros com dados de validação.
#
# ============================================================

import os
import json
import math
import hashlib
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display

_project_print("=" * 78)
_project_print("FASE 2 — CÉLULA 21/36 | VARIANTES E COMPLEXIDADE")
_project_print("=" * 78)

PARTITIONS21 = ("train", "validation")
LOG_TYPES21 = ("regime", "ramp", "combined")

TOP_N21 = 12
EPS21 = 1e-12

# ------------------------------------------------------------
# 21.1 — DIRETÓRIOS
# ------------------------------------------------------------

if "RUN_DIR" not in globals() or not Path(RUN_DIR).is_dir():
    raise RuntimeError("Execute a configuração da Fase 2 para recuperar RUN_DIR.")
ROOT21 = Path(RUN_DIR)

REPORT_DIR21 = ROOT21 / "reports"
FIGURE_DIR21 = ROOT21 / "figures"

REPORT_DIR21.mkdir(parents=True, exist_ok=True)
FIGURE_DIR21.mkdir(parents=True, exist_ok=True)

_project_print("Diretório:", ROOT21)


# ------------------------------------------------------------
# 21.2 — LOCALIZAÇÃO DOS EVENT LOGS
# ------------------------------------------------------------
#
# Preferência:
#   1. Objetos da Célula 19 em memória.
#   2. Arquivos exportados pela Célula 19.
#
# Não reconstruímos os logs a partir do HMM.
# ------------------------------------------------------------

def _candidate_memory21():

    preferred = [
        "hmm19_logs",
        "hmm19_event_logs",
        "event_logs19",
        "event_logs",
        "logs19",
    ]

    candidates = []

    for name in preferred:
        if name in globals():
            obj = globals()[name]
            if isinstance(obj, dict):
                candidates.append((name, obj))

    return candidates


def _resolve_nested21(obj, partition, log_type):

    # Estrutura: logs[partition][log_type]
    if partition in obj:
        sub = obj[partition]
        if isinstance(sub, dict) and log_type in sub:
            if isinstance(sub[log_type], pd.DataFrame):
                return sub[log_type]

    # Estrutura: logs[log_type][partition]
    if log_type in obj:
        sub = obj[log_type]
        if isinstance(sub, dict) and partition in sub:
            if isinstance(sub[partition], pd.DataFrame):
                return sub[partition]

    # Estrutura: logs[(partition, log_type)]
    for key in [
        (partition, log_type),
        (log_type, partition),
        f"{partition}_{log_type}",
        f"{log_type}_{partition}",
    ]:
        if key in obj and isinstance(obj[key], pd.DataFrame):
            return obj[key]

    return None


def _file_candidates21(partition, log_type):

    # Procurar somente na execução corrente.
    patterns = [
        f"*cell19*{partition}*{log_type}*.csv",
        f"*cell19*{log_type}*{partition}*.csv",
        f"*{partition}*{log_type}*event*log*.csv",
        f"*{log_type}*{partition}*event*log*.csv",
        f"*cell19*{partition}*{log_type}*.parquet",
        f"*cell19*{log_type}*{partition}*.parquet",
    ]

    found = []

    for pattern in patterns:
        found.extend(ROOT21.rglob(pattern))

    return sorted(
        set(found),
        key=lambda p: (
            0 if "cell19" in p.name.lower() else 1,
            len(p.parts),
            str(p),
        ),
    )


def _load_logs21():

    result = {}

    memory = _candidate_memory21()

    for partition in PARTITIONS21:

        result[partition] = {}

        for log_type in LOG_TYPES21:

            df = None
            source = None

            for name, obj in memory:

                candidate = _resolve_nested21(
                    obj, partition, log_type
                )

                if candidate is not None:
                    df = candidate.copy()
                    source = f"memória:{name}"
                    break

            if df is None:

                files = _file_candidates21(
                    partition, log_type
                )

                if files:
                    path = files[0]

                    if path.suffix == ".parquet":
                        df = pd.read_parquet(path)
                    else:
                        df = pd.read_csv(path)

                    source = str(path)

            if df is None:

                raise RuntimeError(
                    f"Event Log não encontrado: "
                    f"{partition}/{log_type}.\n"
                    "Verifique os objetos da Célula 19 "
                    "ou seus arquivos exportados.\n"
                    "Objetos dict encontrados: "
                    f"{[x[0] for x in memory]}"
                )

            result[partition][log_type] = df

            _project_print(
                f"{partition:10s} | "
                f"{log_type:8s} | "
                f"{len(df):5d} eventos | {source}"
            )

    return result


hmm21_logs = _load_logs21()


# ------------------------------------------------------------
# 21.3 — NORMALIZAÇÃO DO ESQUEMA
# ------------------------------------------------------------

def _find_col21(df, candidates, required=True):

    mapping = {
        str(c).lower().strip(): c
        for c in df.columns
    }

    for candidate in candidates:
        key = candidate.lower().strip()

        if key in mapping:
            return mapping[key]

    if required:
        raise RuntimeError(
            f"Coluna não encontrada entre {candidates}.\n"
            f"Disponíveis: {list(df.columns)}"
        )

    return None


def _standardize21(df, partition, log_type):

    case_col = _find_col21(
        df,
        [
            "case_id",
            "case:concept:name",
            "case",
            "trace_id",
        ],
    )

    activity_col = _find_col21(
        df,
        [
            "activity",
            "concept:name",
            "activity_name",
            "event_activity",
        ],
    )

    start_col = _find_col21(
        df,
        [
            "start_timestamp",
            "start_time",
            "start_datetime",
            "timestamp",
            "time:timestamp",
            "event_timestamp",
        ],
        required=False,
    )

    end_col = _find_col21(
        df,
        [
            "end_timestamp",
            "end_time",
            "end_datetime",
        ],
        required=False,
    )

    order_col = _find_col21(
        df,
        [
            "start_index",
            "start_hour_index",
            "hour_start",
            "start_position",
            "event_index",
            "event_order",
            "sequence_index",
            "position",
        ],
        required=False,
    )

    duration_col = _find_col21(
        df,
        [
            "duration_hours",
            "duration",
            "episode_hours",
            "n_hours",
        ],
        required=False,
    )

    out = pd.DataFrame({
        "case_id": df[case_col].astype(str),
        "activity": df[activity_col].astype(str),
        "_original_order": np.arange(len(df)),
    })

    if start_col is not None:
        out["start_timestamp"] = pd.to_datetime(
            df[start_col],
            errors="coerce",
        )

    if end_col is not None:
        out["end_timestamp"] = pd.to_datetime(
            df[end_col],
            errors="coerce",
        )

    if order_col is not None:
        out["event_order"] = pd.to_numeric(
            df[order_col],
            errors="coerce",
        )

    if duration_col is not None:
        out["duration_hours"] = pd.to_numeric(
            df[duration_col],
            errors="coerce",
        )

    if out["case_id"].isna().any():
        raise ValueError("case_id contém valores ausentes.")

    if out["activity"].isna().any():
        raise ValueError("activity contém valores ausentes.")

    if (
        "start_timestamp" in out
        and out["start_timestamp"].notna().all()
    ):
        sort_cols = [
            "case_id",
            "start_timestamp",
            "_original_order",
        ]

    elif (
        "event_order" in out
        and out["event_order"].notna().all()
    ):
        sort_cols = [
            "case_id",
            "event_order",
            "_original_order",
        ]

    else:
        raise RuntimeError(
            f"Não foi possível determinar a ordem "
            f"cronológica dos eventos em "
            f"{partition}/{log_type}.\n"
            "É necessária uma coluna de timestamp "
            "ou índice temporal explícito."
        )

    out = out.sort_values(
        sort_cols,
        kind="stable",
    ).reset_index(drop=True)

    out["partition"] = partition
    out["event_log"] = log_type

    return out


hmm21_standardized = {}

for partition in PARTITIONS21:

    hmm21_standardized[partition] = {}

    for log_type in LOG_TYPES21:

        hmm21_standardized[partition][log_type] = (
            _standardize21(
                hmm21_logs[partition][log_type],
                partition,
                log_type,
            )
        )


# ------------------------------------------------------------
# 21.4 — SEQUÊNCIAS POR CASO
# ------------------------------------------------------------

def _variant_id21(sequence):

    payload = _project_dumps(
        list(sequence),
        ensure_ascii=False,
        separators=(",", ":"),
    )

    return hashlib.sha256(
        payload.encode("utf-8")
    ).hexdigest()[:16]


def _build_cases21(df):

    records = []

    for case_id, group in df.groupby(
        "case_id", sort=False
    ):

        seq = tuple(group["activity"].tolist())

        if not seq:
            continue

        transitions = tuple(
            (seq[i], seq[i + 1])
            for i in range(len(seq) - 1)
        )

        trigrams = tuple(
            (seq[i], seq[i + 1], seq[i + 2])
            for i in range(len(seq) - 2)
        )

        record = {
            "case_id": case_id,
            "partition": group["partition"].iloc[0],
            "event_log": group["event_log"].iloc[0],
            "n_events": len(seq),
            "n_distinct_activities": len(set(seq)),
            "n_transitions": len(transitions),
            "n_distinct_transitions": len(set(transitions)),
            "n_trigrams": len(trigrams),
            "n_distinct_trigrams": len(set(trigrams)),
            "variant_id": _variant_id21(seq),
            "sequence": seq,
            "transitions": transitions,
            "trigrams": trigrams,
        }

        if "duration_hours" in group:
            record["covered_hours"] = float(
                group["duration_hours"].sum()
            )
        else:
            record["covered_hours"] = np.nan

        records.append(record)

    return pd.DataFrame(records)


case_frames21 = []

for partition in PARTITIONS21:
    for log_type in LOG_TYPES21:

        case_frames21.append(
            _build_cases21(
                hmm21_standardized[partition][log_type]
            )
        )

hmm21_cases = pd.concat(
    case_frames21,
    ignore_index=True,
)

_project_print("\nCasos reconstruídos:")
_project_display(
    hmm21_cases.groupby(
        ["partition", "event_log"]
    ).agg(
        n_cases=("case_id", "size"),
        mean_events=("n_events", "mean"),
        median_events=("n_events", "median"),
    ).reset_index().round(3)
)


# ------------------------------------------------------------
# 21.5 — VARIANTES EXATAS
# ------------------------------------------------------------

variant_records21 = []

for (partition, log_type), group in (
    hmm21_cases.groupby(["partition", "event_log"])
):

    counts = Counter(group["variant_id"])

    examples = (
        group.drop_duplicates("variant_id")
        .set_index("variant_id")["sequence"]
        .to_dict()
    )

    for variant_id, frequency in counts.items():

        sequence = examples[variant_id]

        variant_records21.append({
            "partition": partition,
            "event_log": log_type,
            "variant_id": variant_id,
            "frequency_cases": frequency,
            "n_events": len(sequence),
            "sequence_json": _project_dumps(
                list(sequence),
                ensure_ascii=False,
            ),
        })

hmm21_variants = pd.DataFrame(variant_records21)

hmm21_variants = hmm21_variants.sort_values(
    [
        "partition",
        "event_log",
        "frequency_cases",
        "variant_id",
    ],
    ascending=[True, True, False, True],
).reset_index(drop=True)


# ------------------------------------------------------------
# 21.6 — MÉTRICAS DE COMPLEXIDADE
# ------------------------------------------------------------

def _entropy21(values):

    counts = np.asarray(
        list(Counter(values).values()),
        dtype=float,
    )

    if len(counts) == 0:
        return np.nan

    p = counts / counts.sum()

    return float(
        -(p * np.log2(p + EPS21)).sum()
    )


def _metrics21(group):

    n_cases = len(group)

    variant_counts = Counter(group["variant_id"])

    n_variants = len(variant_counts)

    frequencies = sorted(
        variant_counts.values(),
        reverse=True,
    )

    top1 = frequencies[0] / n_cases
    top3 = sum(frequencies[:3]) / n_cases

    singleton_variants = sum(
        frequency == 1
        for frequency in frequencies
    )

    entropy = _entropy21(
        group["variant_id"].tolist()
    )

    max_entropy = (
        math.log2(n_variants)
        if n_variants > 1
        else 0.0
    )

    normalized_entropy = (
        entropy / max_entropy
        if max_entropy > 0
        else 0.0
    )

    all_activities = [
        activity
        for sequence in group["sequence"]
        for activity in sequence
    ]

    all_transitions = [
        pair
        for pairs in group["transitions"]
        for pair in pairs
    ]

    all_trigrams = [
        triple
        for triples in group["trigrams"]
        for triple in triples
    ]

    return {
        "n_cases": n_cases,
        "n_variants": n_variants,
        "variant_case_ratio": n_variants / n_cases,
        "singleton_variants": singleton_variants,
        "singleton_variant_ratio": (
            singleton_variants / n_variants
        ),
        "top1_case_coverage": top1,
        "top3_case_coverage": top3,
        "variant_entropy_bits": entropy,
        "normalized_variant_entropy": normalized_entropy,
        "mean_events_per_case": group["n_events"].mean(),
        "median_events_per_case": group["n_events"].median(),
        "max_events_per_case": group["n_events"].max(),
        "mean_distinct_activities_per_case": (
            group["n_distinct_activities"].mean()
        ),
        "n_distinct_activities_global": len(
            set(all_activities)
        ),
        "n_distinct_transitions_global": len(
            set(all_transitions)
        ),
        "n_distinct_trigrams_global": len(
            set(all_trigrams)
        ),
        "transition_entropy_bits": (
            _entropy21(all_transitions)
        ),
        "trigram_entropy_bits": (
            _entropy21(all_trigrams)
        ),
    }


summary_records21 = []

for (partition, log_type), group in (
    hmm21_cases.groupby(["partition", "event_log"])
):

    summary_records21.append({
        "partition": partition,
        "event_log": log_type,
        **_metrics21(group),
    })

hmm21_summary = pd.DataFrame(summary_records21)

_project_print("\n" + "=" * 78)
_project_print("21.6 — COMPLEXIDADE DAS SEQUÊNCIAS")
_project_print("=" * 78)

_project_display(
    hmm21_summary[
        [
            "partition",
            "event_log",
            "n_cases",
            "n_variants",
            "variant_case_ratio",
            "singleton_variant_ratio",
            "top1_case_coverage",
            "top3_case_coverage",
            "variant_entropy_bits",
            "normalized_variant_entropy",
            "median_events_per_case",
            "n_distinct_transitions_global",
            "n_distinct_trigrams_global",
        ]
    ].round(4)
)


# ------------------------------------------------------------
# 21.7 — GENERALIZAÇÃO: TRAIN -> VALIDATION
# ------------------------------------------------------------
#
# O vocabulário de variantes é definido em TRAIN.
# Validation é utilizada somente para avaliação.
#
# Atenção:
# Uma variante exata inédita não implica necessariamente
# comportamento operacional completamente novo.
# Por isso também medimos pares e trigramas inéditos.
# ------------------------------------------------------------

def _flatten21(group, column):

    return [
        item
        for sequence in group[column]
        for item in sequence
    ]


generalization_records21 = []

for log_type in LOG_TYPES21:

    train = hmm21_cases[
        (hmm21_cases["partition"] == "train")
        & (hmm21_cases["event_log"] == log_type)
    ]

    validation = hmm21_cases[
        (hmm21_cases["partition"] == "validation")
        & (hmm21_cases["event_log"] == log_type)
    ]

    train_variants = set(train["variant_id"])

    val_variants = set(validation["variant_id"])

    known_val_cases = int(
        validation["variant_id"]
        .isin(train_variants)
        .sum()
    )

    train_pairs = set(
        _flatten21(train, "transitions")
    )

    val_pairs_list = _flatten21(
        validation, "transitions"
    )

    train_trigrams = set(
        _flatten21(train, "trigrams")
    )

    val_trigrams_list = _flatten21(
        validation, "trigrams"
    )

    novel_pairs = [
        pair for pair in val_pairs_list
        if pair not in train_pairs
    ]

    novel_trigrams = [
        triple for triple in val_trigrams_list
        if triple not in train_trigrams
    ]

    generalization_records21.append({
        "event_log": log_type,
        "train_variants": len(train_variants),
        "validation_variants": len(val_variants),
        "shared_exact_variants": len(
            train_variants & val_variants
        ),
        "novel_validation_variants": len(
            val_variants - train_variants
        ),
        "known_validation_case_ratio": (
            known_val_cases / len(validation)
        ),
        "novel_validation_case_ratio": (
            1 - known_val_cases / len(validation)
        ),
        "train_distinct_pairs": len(train_pairs),
        "validation_distinct_pairs": len(
            set(val_pairs_list)
        ),
        "novel_validation_distinct_pairs": len(
            set(novel_pairs)
        ),
        "novel_validation_pair_occurrence_ratio": (
            len(novel_pairs) / len(val_pairs_list)
            if val_pairs_list else np.nan
        ),
        "train_distinct_trigrams": len(
            train_trigrams
        ),
        "validation_distinct_trigrams": len(
            set(val_trigrams_list)
        ),
        "novel_validation_distinct_trigrams": len(
            set(novel_trigrams)
        ),
        "novel_validation_trigram_occurrence_ratio": (
            len(novel_trigrams) / len(val_trigrams_list)
            if val_trigrams_list else np.nan
        ),
    })

hmm21_generalization = pd.DataFrame(
    generalization_records21
)

_project_print("\n" + "=" * 78)
_project_print("21.7 — GENERALIZAÇÃO PARA VALIDAÇÃO")
_project_print("=" * 78)

_project_display(
    hmm21_generalization.round(4)
)


# ------------------------------------------------------------
# 21.8 — PADRÕES LOCAIS: PARES E TRIGRAMAS
# ------------------------------------------------------------

pattern_records21 = []

for (partition, log_type), group in (
    hmm21_cases.groupby(["partition", "event_log"])
):

    for pattern_type, column in [
        ("pair", "transitions"),
        ("trigram", "trigrams"),
    ]:

        counts = Counter(
            _flatten21(group, column)
        )

        total = sum(counts.values())

        for pattern, frequency in counts.items():

            pattern_records21.append({
                "partition": partition,
                "event_log": log_type,
                "pattern_type": pattern_type,
                "pattern_json": _project_dumps(
                    list(pattern),
                    ensure_ascii=False,
                ),
                "frequency": frequency,
                "relative_frequency": (
                    frequency / total
                    if total else np.nan
                ),
            })

hmm21_patterns = pd.DataFrame(
    pattern_records21
)

_project_print("\n" + "=" * 78)
_project_print("21.8 — TRANSIÇÕES MAIS FREQUENTES")
_project_print("=" * 78)

for log_type in LOG_TYPES21:

    subset = hmm21_patterns[
        (hmm21_patterns["partition"] == "train")
        & (hmm21_patterns["event_log"] == log_type)
        & (hmm21_patterns["pattern_type"] == "pair")
    ].sort_values(
        "frequency",
        ascending=False,
    ).head(10)

    _project_print(f"\nLOG: {log_type.upper()}")

    _project_display(
        subset[
            [
                "pattern_json",
                "frequency",
                "relative_frequency",
            ]
        ].round(4)
    )


# ------------------------------------------------------------
# 21.9 — GRÁFICO: DIVERSIDADE DE VARIANTES
# ------------------------------------------------------------

fig, axes = plt.subplots(
    1, 3,
    figsize=(16, 4.5),
)

metrics_plot21 = [
    (
        "variant_case_ratio",
        "Variantes por caso",
    ),
    (
        "normalized_variant_entropy",
        "Entropia normalizada",
    ),
    (
        "median_events_per_case",
        "Mediana de eventos por caso",
    ),
]

for ax, (metric, title) in zip(
    axes, metrics_plot21
):

    pivot = (
        hmm21_summary
        .pivot(
            index="event_log",
            columns="partition",
            values=metric,
        )
        .reindex(LOG_TYPES21)
    )

    x = np.arange(len(pivot))
    width = 0.36

    ax.bar(
        x - width / 2,
        pivot["train"],
        width,
        label="Treinamento",
    )

    ax.bar(
        x + width / 2,
        pivot["validation"],
        width,
        label="Validação",
    )

    ax.set_xticks(x)
    ax.set_xticklabels(pivot.index)
    ax.set_title(title)
    ax.grid(
        axis="y",
        alpha=0.25,
    )

    ax.legend()

fig.suptitle(
    "Diversidade e complexidade dos Event Logs",
    fontsize=14,
)

plt.tight_layout()

fig.savefig(
    FIGURE_DIR21 / "cell21_variant_complexity.png",
    dpi=180,
    bbox_inches="tight",
)

plt.show()


# ------------------------------------------------------------
# 21.10 — GRÁFICO: COBERTURA DAS VARIANTES
# ------------------------------------------------------------

fig, axes = plt.subplots(
    1, 3,
    figsize=(17, 4.5),
)

for ax, log_type in zip(
    axes, LOG_TYPES21
):

    train = hmm21_variants[
        (hmm21_variants["partition"] == "train")
        & (hmm21_variants["event_log"] == log_type)
    ]

    validation = hmm21_variants[
        (hmm21_variants["partition"] == "validation")
        & (hmm21_variants["event_log"] == log_type)
    ]

    for label, subset in [
        ("Treinamento", train),
        ("Validação", validation),
    ]:

        frequencies = (
            subset["frequency_cases"]
            .sort_values(ascending=False)
            .to_numpy(dtype=float)
        )

        if len(frequencies) == 0:
            continue

        coverage = np.cumsum(
            frequencies
        ) / frequencies.sum()

        ax.step(
            np.arange(1, len(coverage) + 1),
            coverage,
            where="post",
            marker="o",
            markersize=3,
            label=label,
        )

    ax.set_title(log_type.upper())
    ax.set_xlabel("Número de variantes mais frequentes")
    ax.set_ylabel("Cobertura acumulada dos casos")
    ax.set_ylim(0, 1.05)
    ax.grid(alpha=0.25)
    ax.legend()

fig.suptitle(
    "Concentração de variantes por tipo de log",
    fontsize=14,
)

plt.tight_layout()

fig.savefig(
    FIGURE_DIR21 / "cell21_variant_coverage.png",
    dpi=180,
    bbox_inches="tight",
)

plt.show()


# ------------------------------------------------------------
# 21.11 — GRÁFICO: PADRÕES INÉDITOS
# ------------------------------------------------------------

plot21 = hmm21_generalization.set_index(
    "event_log"
).reindex(LOG_TYPES21)

fig, ax = plt.subplots(
    figsize=(10, 5),
)

x = np.arange(len(plot21))
width = 0.26

columns21 = [
    (
        "novel_validation_case_ratio",
        "Casos com variante exata inédita",
    ),
    (
        "novel_validation_pair_occurrence_ratio",
        "Ocorrências de pares inéditos",
    ),
    (
        "novel_validation_trigram_occurrence_ratio",
        "Ocorrências de trios inéditos",
    ),
]

for j, (column, label) in enumerate(columns21):

    ax.bar(
        x + (j - 1) * width,
        plot21[column],
        width,
        label=label,
    )

ax.set_xticks(x)
ax.set_xticklabels(plot21.index)
ax.set_ylabel("Proporção na validação")
ax.set_title(
    "Novidade estrutural em relação ao treinamento"
)

ax.set_ylim(
    0,
    max(
        0.1,
        float(
            np.nanmax(
                plot21[
                    [c for c, _ in columns21]
                ].to_numpy(dtype=float)
            )
        ) * 1.15,
    ),
)

ax.grid(
    axis="y",
    alpha=0.25,
)

ax.legend()

plt.tight_layout()

fig.savefig(
    FIGURE_DIR21 / "cell21_validation_novelty.png",
    dpi=180,
    bbox_inches="tight",
)

plt.show()


# ------------------------------------------------------------
# 21.12 — EXPORTAÇÃO DOS RESULTADOS
# ------------------------------------------------------------

_project_csv(hmm21_summary, 
    REPORT_DIR21 / "cell21_complexity_summary.csv",
    index=False,
)

_project_csv(hmm21_variants, 
    REPORT_DIR21 / "cell21_exact_variants.csv",
    index=False,
)

_project_csv(hmm21_generalization, 
    REPORT_DIR21 / "cell21_generalization.csv",
    index=False,
)

_project_csv(hmm21_patterns, 
    REPORT_DIR21 / "cell21_local_patterns.csv",
    index=False,
)

# Não exportar tuplas Python diretamente:
# serializar sequências para preservar reprodutibilidade.

cases_export21 = hmm21_cases.drop(
    columns=[
        "sequence",
        "transitions",
        "trigrams",
    ]
).copy()

_project_csv(cases_export21, 
    REPORT_DIR21 / "cell21_case_complexity.csv",
    index=False,
)

metadata21 = {
    "phase": 2,
    "cell": 21,
    "unit": "Iatan 6065 Unit 2",
    "year": 2025,
    "partitions": list(PARTITIONS21),
    "event_logs": list(LOG_TYPES21),
    "variant_definition": (
        "Exact ordered sequence of episode activities"
    ),
    "local_pattern_definition": (
        "Adjacent activity pairs and trigrams"
    ),
    "validation_policy": (
        "Training vocabulary frozen; "
        "validation used only for evaluation"
    ),
    "test_used": False,
    "co2_used": False,
    "hmm_retrained": False,
    "ramp_threshold_mw_per_hour": 20.0,
}

with open(
    REPORT_DIR21 / "cell21_metadata.json",
    "w",
    encoding="utf-8",
) as f:

    _project_dump(
        metadata21,
        f,
        indent=2,
        ensure_ascii=False,
    )


# ------------------------------------------------------------
# 21.13 — RESULTADO FINAL
# ------------------------------------------------------------

_project_print("\n" + "=" * 78)
_project_print("FASE 2 — CÉLULA 21/36 | RESULTADO FINAL")
_project_print("=" * 78)

_project_print("\nCOMPLEXIDADE")
_project_display(
    hmm21_summary[
        [
            "partition",
            "event_log",
            "n_cases",
            "n_variants",
            "variant_case_ratio",
            "normalized_variant_entropy",
            "median_events_per_case",
        ]
    ].round(4)
)

_project_print("\nGENERALIZAÇÃO")
_project_display(
    hmm21_generalization[
        [
            "event_log",
            "shared_exact_variants",
            "novel_validation_case_ratio",
            "novel_validation_distinct_pairs",
            "novel_validation_pair_occurrence_ratio",
            "novel_validation_distinct_trigrams",
            "novel_validation_trigram_occurrence_ratio",
        ]
    ].round(4)
)

_project_print("\nRelatórios:", REPORT_DIR21)
_project_print("Figuras:", FIGURE_DIR21)

_project_print("\nTeste utilizado: NÃO")
_project_print("HMM retreinado: NÃO")
_project_print("CO2 utilizado: NÃO")

_project_print("\nCÉLULA 21 CONCLUÍDA.")
_project_print(
    "Próxima etapa: Célula 22/36 — "
    "Descoberta e visualização dos DFGs."
)


## 22–25 — Descoberta de processos

O DFG resume sucessões diretamente observadas e suas frequências. O Inductive Miner produz árvores estruturadas e o PM4Py converte as árvores em redes de Petri para avaliação. O treino define o log de descoberta e os valores do limiar de ruído são opções locais da grade. [van der Aalst (2022), §§2–3](https://doi.org/10.1007/978-3-031-08848-3_1) · [Leemans, Fahland e van der Aalst (2014)](https://www.vdaalst.com/publications/p761.pdf) · [PM4Py](https://processintelligence.solutions/static/api/2.7.17/generated/pm4py.discovery.discover_process_tree_inductive.html)


In [ ]:

# ============================================================
# FASE 2 — CÉLULA 22/36
# DIRECTLY-FOLLOWS GRAPHS (DFGs)
# ============================================================
#
# OBJETIVOS
# 1. Construir DFGs a partir dos Event Logs de treinamento.
# 2. Visualizar atividades e transições com frequências.
# 3. Comparar frequências absolutas e condicionais.
# 4. Avaliar cobertura estrutural na validação.
# 5. Exportar tabelas, figuras e metadados.
#
# REGRAS METODOLÓGICAS
# - Descoberta: somente TRAIN.
# - Validação: somente avaliação, sem atualizar o DFG.
# - Teste: não utilizado.
# - CO2: não utilizado.
# - HMM: não retreinado.
# - As frequências são calculadas por episódio, não por hora.
# - As transições não atravessam fronteiras de ciclos ON.
# - DFG não é uma rede de Petri nem prova causalidade física.
# ============================================================

import os
import json
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx

from IPython.display import display

_project_print("=" * 76)
_project_print("FASE 2 — CÉLULA 22/36 | DIRECTLY-FOLLOWS GRAPHS")
_project_print("=" * 76)

# ------------------------------------------------------------
# 22.1 — CONFIGURAÇÃO
# ------------------------------------------------------------

LOG_TYPES22 = ("regime", "ramp", "combined")
PARTITIONS22 = ("train", "validation")

# Todos os arcos observados são preservados na descoberta.
# O filtro abaixo afeta apenas a visualização.
MIN_EDGE_DISPLAY22 = {
    "regime": 1,
    "ramp": 1,
    "combined": 5,
}

MAX_EDGES_DISPLAY22 = {
    "regime": 30,
    "ramp": 30,
    "combined": 40,
}

FIG_DPI22 = 170
SEED_LAYOUT22 = 42

# Não incluir início/fim artificiais nas matrizes de transição.
# As frequências de início/fim são armazenadas separadamente.

# ------------------------------------------------------------
# 22.2 — DIRETÓRIOS
# ------------------------------------------------------------

if "RUN_DIR" not in globals() or not Path(RUN_DIR).is_dir():
    raise RuntimeError("Execute a configuração da Fase 2 para recuperar RUN_DIR.")
RUN_DIR22 = Path(RUN_DIR)

REPORT_DIR22 = RUN_DIR22 / "reports"
FIG_DIR22 = RUN_DIR22 / "figures"

REPORT_DIR22.mkdir(parents=True, exist_ok=True)
FIG_DIR22.mkdir(parents=True, exist_ok=True)

_project_print("Execução:", RUN_DIR22)

# ------------------------------------------------------------
# 22.3 — RECUPERAÇÃO DOS EVENT LOGS
# ------------------------------------------------------------

if "hmm19_logs" not in globals():
    raise RuntimeError(
        "hmm19_logs não encontrado. "
        "Execute novamente a Célula 19."
    )

def get_log22(partition, log_type):
    """
    Suporta:
      hmm19_logs[partition][log_type]
      hmm19_logs[log_type][partition]
      hmm19_logs[(partition, log_type)]
    """

    source = hmm19_logs

    if (partition, log_type) in source:
        obj = source[(partition, log_type)]

    elif (
        partition in source
        and isinstance(source[partition], dict)
        and log_type in source[partition]
    ):
        obj = source[partition][log_type]

    elif (
        log_type in source
        and isinstance(source[log_type], dict)
        and partition in source[log_type]
    ):
        obj = source[log_type][partition]

    else:
        raise KeyError(
            f"Log não localizado: {partition}/{log_type}. "
            f"Chaves disponíveis: {list(source.keys())}"
        )

    if not isinstance(obj, pd.DataFrame):
        raise TypeError(
            f"{partition}/{log_type}: esperado DataFrame; "
            f"recebido {type(obj).__name__}."
        )

    return obj.copy()


def find_column22(df, candidates, purpose):
    for candidate in candidates:
        if candidate in df.columns:
            return candidate

    raise KeyError(
        f"Coluna de {purpose} não encontrada. "
        f"Disponíveis: {df.columns.tolist()}"
    )


def normalize_log22(partition, log_type):
    df = get_log22(partition, log_type)

    case_col = find_column22(
        df,
        ["case:concept:name", "case_id", "case", "cycle_id"],
        "identificação do caso",
    )

    activity_col = find_column22(
        df,
        ["concept:name", "activity", "activity_name"],
        "atividade",
    )

    timestamp_col = find_column22(
        df,
        ["time:timestamp", "timestamp", "start_timestamp",
         "event_start", "start_time"],
        "tempo",
    )

    df = df.rename(
        columns={
            case_col: "case_id_22",
            activity_col: "activity_22",
            timestamp_col: "timestamp_22",
        }
    )

    if df["case_id_22"].isna().any():
        raise ValueError("Existem eventos sem case_id.")

    if df["activity_22"].isna().any():
        raise ValueError("Existem eventos sem atividade.")

    df["case_id_22"] = df["case_id_22"].astype(str)
    df["activity_22"] = df["activity_22"].astype(str)

    df["timestamp_22"] = pd.to_datetime(
        df["timestamp_22"], errors="coerce"
    )

    if df["timestamp_22"].isna().any():
        raise ValueError(
            f"Timestamps inválidos em {partition}/{log_type}."
        )

    # Ordenação estável para preservar a ordem original
    # em eventuais empates de timestamp.
    df["_original_order22"] = np.arange(len(df))

    df = df.sort_values(
        ["case_id_22", "timestamp_22", "_original_order22"],
        kind="stable",
    ).reset_index(drop=True)

    # Uma atividade por episódio. A Célula 19 já realizou
    # a consolidação; aqui apenas preservamos seus eventos.
    if len(df) == 0:
        raise ValueError(
            f"Event Log vazio: {partition}/{log_type}"
        )

    return df


logs22 = {
    partition: {
        log_type: normalize_log22(partition, log_type)
        for log_type in LOG_TYPES22
    }
    for partition in PARTITIONS22
}

_project_print("\nEVENT LOGS RECUPERADOS")

for partition in PARTITIONS22:
    for log_type in LOG_TYPES22:
        df = logs22[partition][log_type]
        _project_print(
            f"{partition:10s} | {log_type:8s} | "
            f"{df['case_id_22'].nunique():3d} casos | "
            f"{len(df):5d} eventos"
        )

# ------------------------------------------------------------
# 22.4 — EXTRAÇÃO DAS SEQUÊNCIAS
# ------------------------------------------------------------

def extract_traces22(df):
    traces = {}

    for case_id, group in df.groupby(
        "case_id_22", sort=False
    ):
        sequence = tuple(group["activity_22"].tolist())

        if sequence:
            traces[case_id] = sequence

    return traces


traces22 = {
    partition: {
        log_type: extract_traces22(
            logs22[partition][log_type]
        )
        for log_type in LOG_TYPES22
    }
    for partition in PARTITIONS22
}

# ------------------------------------------------------------
# 22.5 — CONSTRUÇÃO DOS DFGs
# ------------------------------------------------------------

def discover_dfg22(traces):
    """
    Retorna frequências de atividades, arcos,
    início/fim e número de casos.

    Cada ocorrência consecutiva (a,b) contribui
    com uma unidade à frequência do arco a -> b.
    """

    activities = Counter()
    edges = Counter()
    starts = Counter()
    ends = Counter()

    for sequence in traces.values():
        if not sequence:
            continue

        activities.update(sequence)
        starts[sequence[0]] += 1
        ends[sequence[-1]] += 1

        edges.update(zip(sequence[:-1], sequence[1:]))

    return {
        "activities": activities,
        "edges": edges,
        "starts": starts,
        "ends": ends,
        "n_cases": len(traces),
        "n_events": sum(activities.values()),
        "n_transitions": sum(edges.values()),
    }


# IMPORTANTE:
# Os modelos descobertos são construídos apenas com TRAIN.

hmm22_dfg = {
    log_type: discover_dfg22(
        traces22["train"][log_type]
    )
    for log_type in LOG_TYPES22
}

# Dados de validação são mantidos em estrutura separada,
# exclusivamente para análise de cobertura.

hmm22_validation_dfg = {
    log_type: discover_dfg22(
        traces22["validation"][log_type]
    )
    for log_type in LOG_TYPES22
}

# ------------------------------------------------------------
# 22.6 — TABELAS DE ATIVIDADES E TRANSIÇÕES
# ------------------------------------------------------------

activity_rows22 = []
edge_rows22 = []
boundary_rows22 = []

for log_type, dfg in hmm22_dfg.items():

    total_events = dfg["n_events"]
    total_transitions = dfg["n_transitions"]

    outgoing = defaultdict(int)

    for (source, target), frequency in dfg["edges"].items():
        outgoing[source] += frequency

    for activity, frequency in dfg["activities"].items():
        activity_rows22.append({
            "event_log": log_type,
            "activity": activity,
            "frequency": frequency,
            "relative_frequency": (
                frequency / total_events
                if total_events else np.nan
            ),
        })

    for (source, target), frequency in dfg["edges"].items():
        edge_rows22.append({
            "event_log": log_type,
            "source": source,
            "target": target,
            "frequency": frequency,
            "global_relative_frequency": (
                frequency / total_transitions
                if total_transitions else np.nan
            ),
            "conditional_probability": (
                frequency / outgoing[source]
                if outgoing[source] else np.nan
            ),
        })

    for activity in sorted(dfg["activities"]):
        boundary_rows22.append({
            "event_log": log_type,
            "activity": activity,
            "start_count": dfg["starts"].get(activity, 0),
            "end_count": dfg["ends"].get(activity, 0),
            "start_case_fraction": (
                dfg["starts"].get(activity, 0) / dfg["n_cases"]
            ),
            "end_case_fraction": (
                dfg["ends"].get(activity, 0) / dfg["n_cases"]
            ),
        })


hmm22_activities = pd.DataFrame(activity_rows22)
hmm22_edges = pd.DataFrame(edge_rows22)
hmm22_boundaries = pd.DataFrame(boundary_rows22)

hmm22_edges = hmm22_edges.sort_values(
    ["event_log", "frequency"],
    ascending=[True, False],
).reset_index(drop=True)

_project_print("\n" + "=" * 76)
_project_print("22.6 — PRINCIPAIS TRANSIÇÕES DESCOBERTAS")
_project_print("=" * 76)

for log_type in LOG_TYPES22:
    _project_print(f"\nLOG: {log_type.upper()}")

    _project_display(
        hmm22_edges.loc[
            hmm22_edges["event_log"] == log_type
        ].head(12).round(4)
    )

# ------------------------------------------------------------
# 22.7 — MATRIZES DE FREQUÊNCIA E PROBABILIDADE
# ------------------------------------------------------------

hmm22_matrices = {}

for log_type, dfg in hmm22_dfg.items():

    activities = sorted(dfg["activities"])

    matrix = pd.DataFrame(
        0,
        index=activities,
        columns=activities,
        dtype=int,
    )

    for (source, target), frequency in dfg["edges"].items():
        matrix.loc[source, target] = frequency

    row_sums = matrix.sum(axis=1)

    probabilities = matrix.div(
        row_sums.replace(0, np.nan),
        axis=0,
    )

    hmm22_matrices[log_type] = {
        "frequency": matrix,
        "conditional_probability": probabilities,
    }

    _project_print("\n" + "-" * 76)
    _project_print(f"MATRIZ DE FREQUÊNCIAS — {log_type.upper()}")
    _project_display(matrix)

    _project_print("PROBABILIDADES CONDICIONAIS EMPÍRICAS")
    _project_display(probabilities.round(3))

    _project_csv(matrix, 
        REPORT_DIR22 / f"cell22_dfg_matrix_{log_type}.csv"
    )

    _project_csv(probabilities, 
        REPORT_DIR22 /
        f"cell22_dfg_conditional_{log_type}.csv"
    )

# ------------------------------------------------------------
# 22.8 — VISUALIZAÇÃO DOS DFGs
# ------------------------------------------------------------

def draw_dfg22(log_type):

    dfg = hmm22_dfg[log_type]

    edges_all = sorted(
        dfg["edges"].items(),
        key=lambda item: item[1],
        reverse=True,
    )

    min_frequency = MIN_EDGE_DISPLAY22[log_type]
    max_edges = MAX_EDGES_DISPLAY22[log_type]

    visible_edges = [
        (edge, freq)
        for edge, freq in edges_all
        if freq >= min_frequency
    ][:max_edges]

    G = nx.DiGraph()

    # Preservar todas as atividades, mesmo sem arco visível.
    for activity, frequency in dfg["activities"].items():
        G.add_node(activity, frequency=frequency)

    for (source, target), frequency in visible_edges:
        G.add_edge(source, target, weight=frequency)

    n_nodes = G.number_of_nodes()
    n_edges = G.number_of_edges()

    if n_nodes == 0:
        raise ValueError("DFG sem atividades.")

    fig_width = 13 if log_type != "combined" else 17
    fig_height = 8 if log_type != "combined" else 11

    fig, ax = plt.subplots(
        figsize=(fig_width, fig_height)
    )

    # Layout determinístico e reproduzível.
    pos = nx.spring_layout(
        G,
        seed=SEED_LAYOUT22,
        weight="weight",
        k=2.0 / np.sqrt(max(n_nodes, 1)),
        iterations=300,
    )

    node_freq = np.array([
        G.nodes[node]["frequency"]
        for node in G.nodes()
    ], dtype=float)

    max_node = max(node_freq.max(), 1)

    node_sizes = [
        1100 + 3300 * (frequency / max_node)
        for frequency in node_freq
    ]

    weights = np.array([
        G[u][v]["weight"]
        for u, v in G.edges()
    ], dtype=float)

    max_weight = max(weights.max(), 1) if len(weights) else 1

    edge_widths = [
        0.8 + 5.0 * (weight / max_weight)
        for weight in weights
    ]

    nx.draw_networkx_nodes(
        G,
        pos,
        node_size=node_sizes,
        node_color="#dbeafe",
        edgecolors="#1d4ed8",
        linewidths=1.6,
        ax=ax,
    )

    nx.draw_networkx_labels(
        G,
        pos,
        labels={
            node: (
                f"{node}\n"
                f"n={G.nodes[node]['frequency']}"
            )
            for node in G.nodes()
        },
        font_size=8 if log_type == "combined" else 10,
        font_weight="bold",
        ax=ax,
    )

    nx.draw_networkx_edges(
        G,
        pos,
        width=edge_widths,
        edge_color="#475569",
        arrows=True,
        arrowsize=19,
        arrowstyle="-|>",
        connectionstyle="arc3,rad=0.12",
        node_size=node_sizes,
        ax=ax,
    )

    edge_labels = {
        (u, v): G[u][v]["weight"]
        for u, v in G.edges()
    }

    nx.draw_networkx_edge_labels(
        G,
        pos,
        edge_labels=edge_labels,
        font_size=8,
        bbox={
            "facecolor": "white",
            "edgecolor": "none",
            "alpha": 0.85,
            "pad": 0.2,
        },
        rotate=False,
        ax=ax,
    )

    ax.set_title(
        f"DFG — {log_type.upper()} | TRAIN\n"
        f"{dfg['n_cases']} ciclos ON | "
        f"{dfg['n_events']} eventos | "
        f"{len(dfg['edges'])} transições distintas",
        fontsize=14,
        pad=20,
    )

    ax.axis("off")
    fig.tight_layout()

    stem = FIG_DIR22 / f"cell22_dfg_{log_type}"

    fig.savefig(
        str(stem) + ".png",
        dpi=FIG_DPI22,
        bbox_inches="tight",
    )

    fig.savefig(
        str(stem) + ".svg",
        bbox_inches="tight",
    )

    plt.show()
    plt.close(fig)

    return {
        "visible_nodes": n_nodes,
        "visible_edges": n_edges,
        "total_edges": len(dfg["edges"]),
        "display_filter": min_frequency,
    }


hmm22_graph_display = {}

for log_type in LOG_TYPES22:
    hmm22_graph_display[log_type] = draw_dfg22(
        log_type
    )

# ------------------------------------------------------------
# 22.9 — MATRIZES VISUAIS DE TRANSIÇÃO
# ------------------------------------------------------------

def plot_matrix22(log_type):

    matrix = hmm22_matrices[log_type]["frequency"]

    n = len(matrix)

    figsize = (
        (8, 7) if n <= 5
        else (12, 10)
    )

    fig, ax = plt.subplots(figsize=figsize)

    values = matrix.to_numpy()

    image = ax.imshow(
        values,
        cmap="Blues",
        aspect="auto",
    )

    ax.set_xticks(np.arange(n))
    ax.set_yticks(np.arange(n))

    ax.set_xticklabels(
        matrix.columns,
        rotation=65,
        ha="right",
        fontsize=8,
    )

    ax.set_yticklabels(
        matrix.index,
        fontsize=8,
    )

    ax.set_xlabel("Atividade destino")
    ax.set_ylabel("Atividade origem")

    ax.set_title(
        f"DFG — Matriz de transições | {log_type.upper()}"
    )

    threshold = values.max() / 2 if values.size else 0

    for i in range(n):
        for j in range(n):
            value = values[i, j]

            if value > 0:
                ax.text(
                    j, i, str(value),
                    ha="center",
                    va="center",
                    fontsize=7 if n > 5 else 10,
                    color=(
                        "white"
                        if value > threshold
                        else "black"
                    ),
                )

    fig.colorbar(
        image,
        ax=ax,
        label="Frequência de transições",
    )

    fig.tight_layout()

    fig.savefig(
        FIG_DIR22 /
        f"cell22_transition_heatmap_{log_type}.png",
        dpi=FIG_DPI22,
        bbox_inches="tight",
    )

    plt.show()
    plt.close(fig)


for log_type in LOG_TYPES22:
    plot_matrix22(log_type)

# ------------------------------------------------------------
# 22.10 — COBERTURA ESTRUTURAL NA VALIDAÇÃO
# ------------------------------------------------------------

coverage_rows22 = []
novel_edges_rows22 = []

for log_type in LOG_TYPES22:

    train_dfg = hmm22_dfg[log_type]
    val_dfg = hmm22_validation_dfg[log_type]

    train_activities = set(train_dfg["activities"])
    train_edges = set(train_dfg["edges"])

    val_activities = set(val_dfg["activities"])
    val_edges = set(val_dfg["edges"])

    novel_activities = val_activities - train_activities
    novel_edges = val_edges - train_edges

    total_val_occurrences = sum(
        val_dfg["edges"].values()
    )

    known_occurrences = sum(
        frequency
        for edge, frequency in val_dfg["edges"].items()
        if edge in train_edges
    )

    edge_coverage = (
        known_occurrences / total_val_occurrences
        if total_val_occurrences else np.nan
    )

    distinct_edge_coverage = (
        len(val_edges & train_edges) / len(val_edges)
        if val_edges else np.nan
    )

    # Cobertura de casos:
    # um caso é coberto quando todas as atividades
    # e todos os pares consecutivos pertencem ao DFG train.
    covered_cases = 0

    for sequence in traces22["validation"][log_type].values():

        activities_ok = all(
            activity in train_activities
            for activity in sequence
        )

        edges_ok = all(
            edge in train_edges
            for edge in zip(sequence[:-1], sequence[1:])
        )

        if activities_ok and edges_ok:
            covered_cases += 1

    n_val_cases = len(
        traces22["validation"][log_type]
    )

    case_coverage = (
        covered_cases / n_val_cases
        if n_val_cases else np.nan
    )

    coverage_rows22.append({
        "event_log": log_type,
        "train_cases": train_dfg["n_cases"],
        "validation_cases": val_dfg["n_cases"],
        "train_distinct_activities": len(train_activities),
        "validation_distinct_activities": len(val_activities),
        "novel_validation_activities": len(novel_activities),
        "train_distinct_edges": len(train_edges),
        "validation_distinct_edges": len(val_edges),
        "novel_validation_edges": len(novel_edges),
        "validation_edge_occurrence_coverage": edge_coverage,
        "validation_distinct_edge_coverage": distinct_edge_coverage,
        "validation_case_local_coverage": case_coverage,
    })

    for edge in sorted(novel_edges):
        novel_edges_rows22.append({
            "event_log": log_type,
            "source": edge[0],
            "target": edge[1],
            "validation_frequency": val_dfg["edges"][edge],
        })


hmm22_coverage = pd.DataFrame(coverage_rows22)

hmm22_novel_edges = pd.DataFrame(
    novel_edges_rows22,
    columns=[
        "event_log",
        "source",
        "target",
        "validation_frequency",
    ],
)

_project_print("\n" + "=" * 76)
_project_print("22.10 — COBERTURA ESTRUTURAL NA VALIDAÇÃO")
_project_print("=" * 76)

_project_display(hmm22_coverage.round(4))

_project_print(
    "\nATENÇÃO: cobertura local de DFG não equivale "
    "a fitness de uma rede de Petri."
)

# ------------------------------------------------------------
# 22.11 — VISUALIZAÇÃO DA COBERTURA
# ------------------------------------------------------------

fig, ax = plt.subplots(figsize=(10, 5))

labels = hmm22_coverage["event_log"].tolist()

metrics = [
    (
        "validation_edge_occurrence_coverage",
        "Cobertura das ocorrências",
    ),
    (
        "validation_distinct_edge_coverage",
        "Cobertura dos pares distintos",
    ),
    (
        "validation_case_local_coverage",
        "Cobertura local dos casos",
    ),
]

x = np.arange(len(labels))
width = 0.24

for i, (column, label) in enumerate(metrics):
    values = hmm22_coverage[column].to_numpy(dtype=float)

    ax.bar(
        x + (i - 1) * width,
        values,
        width,
        label=label,
    )

ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_ylim(0, 1.08)
ax.set_ylabel("Proporção")
ax.set_title(
    "Cobertura estrutural dos DFGs na validação"
)

ax.legend()
ax.grid(axis="y", alpha=0.2)
ax.set_axisbelow(True)

fig.tight_layout()

fig.savefig(
    FIG_DIR22 / "cell22_validation_coverage.png",
    dpi=FIG_DPI22,
    bbox_inches="tight",
)

plt.show()
plt.close(fig)

# ------------------------------------------------------------
# 22.12 — EXPORTAÇÃO DOS RESULTADOS
# ------------------------------------------------------------

_project_csv(hmm22_activities, 
    REPORT_DIR22 / "cell22_activities.csv",
    index=False,
)

_project_csv(hmm22_edges, 
    REPORT_DIR22 / "cell22_edges.csv",
    index=False,
)

_project_csv(hmm22_boundaries, 
    REPORT_DIR22 / "cell22_start_end_activities.csv",
    index=False,
)

_project_csv(hmm22_coverage, 
    REPORT_DIR22 / "cell22_validation_coverage.csv",
    index=False,
)

_project_csv(hmm22_novel_edges, 
    REPORT_DIR22 / "cell22_novel_validation_edges.csv",
    index=False,
)

# Serialização explícita dos DFGs para as próximas células.
# Não inclui observações individuais nem dados de teste.

hmm22_serializable = {}

for log_type, dfg in hmm22_dfg.items():

    hmm22_serializable[log_type] = {
        "n_cases": dfg["n_cases"],
        "n_events": dfg["n_events"],
        "n_transitions": dfg["n_transitions"],
        "activities": dict(dfg["activities"]),
        "starts": dict(dfg["starts"]),
        "ends": dict(dfg["ends"]),
        "edges": [
            {
                "source": source,
                "target": target,
                "frequency": int(frequency),
            }
            for (source, target), frequency
            in sorted(dfg["edges"].items())
        ],
    }

with open(
    REPORT_DIR22 / "cell22_dfg_models.json",
    "w",
    encoding="utf-8",
) as file:
    _project_dump(
        hmm22_serializable,
        file,
        ensure_ascii=False,
        indent=2,
    )

hmm22_metadata = {
    "phase": 2,
    "cell": 22,
    "method": "Directly-Follows Graph",
    "discovery_partition": "train",
    "evaluation_partition": "validation",
    "test_used": False,
    "co2_used": False,
    "hmm_retrained": False,
    "event_logs": list(LOG_TYPES22),
    "frequency_unit": "event_episode_transition",
    "case_boundary_policy": "no_cross_case_transitions",
    "visualization_thresholds": MIN_EDGE_DISPLAY22,
    "visualization_max_edges": MAX_EDGES_DISPLAY22,
    "random_seed_layout": SEED_LAYOUT22,
    "interpretation_warning": (
        "DFG local coverage is not process-model fitness "
        "and does not establish physical causality."
    ),
}

with open(
    REPORT_DIR22 / "cell22_metadata.json",
    "w",
    encoding="utf-8",
) as file:
    _project_dump(
        hmm22_metadata,
        file,
        ensure_ascii=False,
        indent=2,
    )

# ------------------------------------------------------------
# 22.13 — RESUMO FINAL
# ------------------------------------------------------------

summary_rows22 = []

for log_type in LOG_TYPES22:

    dfg = hmm22_dfg[log_type]

    summary_rows22.append({
        "event_log": log_type,
        "n_cases_train": dfg["n_cases"],
        "n_events_train": dfg["n_events"],
        "n_activities": len(dfg["activities"]),
        "n_distinct_edges": len(dfg["edges"]),
        "n_edge_occurrences": dfg["n_transitions"],
        "visible_edges": (
            hmm22_graph_display[log_type]["visible_edges"]
        ),
    })

hmm22_summary = pd.DataFrame(summary_rows22)

_project_print("\n" + "=" * 76)
_project_print("RESULTADO DA CÉLULA 22/36")
_project_print("=" * 76)

_project_display(hmm22_summary)

_project_print("\nMODELOS DISPONÍVEIS:")
_project_print("  hmm22_dfg")
_project_print("  hmm22_matrices")
_project_print("  hmm22_edges")
_project_print("  hmm22_activities")
_project_print("  hmm22_boundaries")
_project_print("  hmm22_coverage")
_project_print("  hmm22_summary")

_project_print("\nPOLÍTICA DE DADOS:")
_project_print("  Descoberta: TRAIN")
_project_print("  Avaliação: VALIDATION")
_project_print("  Teste utilizado: NÃO")
_project_print("  CO2 utilizado: NÃO")
_project_print("  HMM retreinado: NÃO")

_project_print("\nRELATÓRIOS:", REPORT_DIR22)
_project_print("FIGURAS:", FIG_DIR22)

_project_print("\nCÉLULA 22 CONCLUÍDA.")


In [ ]:

# ================================================================
# FASE 2 — CÉLULA 23/36
# DESCOBERTA DE MODELOS COM INDUCTIVE MINER
# ================================================================
#
# OBJETIVO:
#   Descobrir árvores de processo a partir dos Event Logs
#   produzidos na Célula 19, comparando níveis de ruído.
#
# ENTRADAS:
#   - Event Logs da Célula 19 (train)
#   - RUN_DIR / run_dir, quando disponíveis
#
# SAÍDAS:
#   - hmm23_models
#   - audit23_models
#   - audit23_summary
#   - CSV, JSON, PTML e figuras SVG/PNG
#
# RESTRIÇÕES METODOLÓGICAS:
#   - Descoberta exclusivamente no treinamento
#   - Sem seleção por desempenho no teste
#   - Sem CO2 na descoberta
#   - Sem retreinamento do HMM
#   - Sem seleção definitiva nesta célula
#
# ================================================================

import os
import sys
import json
import subprocess
import importlib.util
import warnings
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display

warnings.filterwarnings(
    "ignore",
    category=FutureWarning
)

_project_print("=" * 78)
_project_print("FASE 2 — CÉLULA 23/36 | INDUCTIVE MINER")
_project_print("=" * 78)


# ================================================================
# 23.1 — DEPENDÊNCIAS
# ================================================================

if importlib.util.find_spec("pm4py") is None:
    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "-q",
        "install",
        "pm4py"
    ])

import pm4py

from pm4py.objects.log.obj import EventLog, Trace, Event
from pm4py.objects.process_tree.obj import ProcessTree
from pm4py.objects.process_tree import semantics as tree_semantics
from pm4py.objects.process_tree.exporter import exporter as ptml_exporter

_project_print("PM4Py:", pm4py.__version__)


# ================================================================
# 23.2 — DIRETÓRIOS DA EXECUÇÃO
# ================================================================

if "RUN_DIR" not in globals() or not Path(RUN_DIR).is_dir():
    raise RuntimeError("Execute a configuração da Fase 2 para recuperar RUN_DIR.")
RUN_DIR23 = Path(RUN_DIR)

REPORT_DIR23 = RUN_DIR23 / "reports"
FIGURE_DIR23 = RUN_DIR23 / "figures"
MODEL_DIR23 = RUN_DIR23 / "models" / "cell23"

for directory in (
    REPORT_DIR23,
    FIGURE_DIR23,
    MODEL_DIR23
):
    directory.mkdir(parents=True, exist_ok=True)

_project_print("Execução:", RUN_DIR23)


# ================================================================
# 23.3 — CONFIGURAÇÃO EXPERIMENTAL
# ================================================================

LOG_TYPES23 = ("regime", "ramp", "combined")

# Limiar 0 corresponde à descoberta sem filtragem por ruído.
# Os demais valores são candidatos, não escolhas definitivas.

NOISE_THRESHOLDS23 = (
    0.00,
    0.05,
    0.10,
    0.20,
    0.30
)

DISCOVERY_PARTITION23 = "train"

_project_print("\nConfiguração:")
_project_print("Logs:", LOG_TYPES23)
_project_print("Limiar de ruído:", NOISE_THRESHOLDS23)
_project_print("Partição de descoberta:", DISCOVERY_PARTITION23)


# ================================================================
# 23.4 — RECUPERAÇÃO DOS EVENT LOGS
# ================================================================
#
# A célula procura primeiro os objetos já produzidos na
# Célula 19. Caso não estejam disponíveis, procura os CSVs
# exportados naquela etapa.
#
# Não utiliza validation nem test para descoberta.
# ================================================================

def normalize_event_dataframe23(df):
    df = df.copy()

    aliases = {
        "case:concept:name": "case_id",
        "concept:name": "activity",
        "time:timestamp": "timestamp",
        "case_id": "case_id",
        "activity": "activity",
        "timestamp": "timestamp"
    }

    for old, new in aliases.items():
        if old in df.columns and new not in df.columns:
            df = df.rename(columns={old: new})

    required = {"case_id", "activity"}

    missing = required - set(df.columns)

    if missing:
        raise ValueError(
            f"Event Log sem colunas obrigatórias: {missing}. "
            f"Colunas disponíveis: {list(df.columns)}"
        )

    df["case_id"] = df["case_id"].astype(str)
    df["activity"] = df["activity"].astype(str)

    if "timestamp" in df.columns:
        df["timestamp"] = pd.to_datetime(
            df["timestamp"],
            errors="coerce"
        )

    return df


def eventlog_to_dataframe23(log):
    records = []

    for trace in log:
        case_id = str(
            trace.attributes.get(
                "concept:name",
                len(records)
            )
        )

        for position, event in enumerate(trace):
            records.append({
                "case_id": case_id,
                "activity": event.get("concept:name"),
                "timestamp": event.get("time:timestamp"),
                "_event_position23": position
            })

    return normalize_event_dataframe23(
        pd.DataFrame(records)
    )


def candidate_objects23():
    names = [
        "hmm19_event_logs",
        "hmm19_logs",
        "event_logs19",
        "logs19",
        "hmm19_eventlog",
        "hmm19_events"
    ]

    return [
        (name, globals()[name])
        for name in names
        if name in globals()
    ]


def extract_log23(container, log_type):
    if not isinstance(container, dict):
        return None

    # Estrutura: container[partition][log_type]
    if DISCOVERY_PARTITION23 in container:
        section = container[DISCOVERY_PARTITION23]

        if isinstance(section, dict):
            if log_type in section:
                return section[log_type]

    # Estrutura: container[log_type][partition]
    if log_type in container:
        section = container[log_type]

        if isinstance(section, dict):
            if DISCOVERY_PARTITION23 in section:
                return section[DISCOVERY_PARTITION23]

    # Estrutura: container[(partition, log_type)]
    for key in (
        (DISCOVERY_PARTITION23, log_type),
        (log_type, DISCOVERY_PARTITION23)
    ):
        if key in container:
            return container[key]

    return None


def load_log23(log_type):
    # 1. Objetos em memória
    for object_name, container in candidate_objects23():
        result = extract_log23(container, log_type)

        if result is None:
            continue

        if isinstance(result, pd.DataFrame):
            return (
                normalize_event_dataframe23(result),
                object_name
            )

        if isinstance(result, EventLog):
            return (
                eventlog_to_dataframe23(result),
                object_name
            )

    # 2. CSVs exportados na Célula 19
    candidates = []

    for path in RUN_DIR23.rglob("*.csv"):
        name = path.name.lower()

        if (
            "cell19" in name
            and log_type in name
            and "train" in name
            and "validation" not in name
            and "test" not in name
        ):
            candidates.append(path)

    for path in sorted(candidates):
        try:
            df = pd.read_csv(path)

            return (
                normalize_event_dataframe23(df),
                str(path)
            )

        except (ValueError, KeyError, pd.errors.ParserError):
            continue

    raise RuntimeError(
        f"Event Log de treinamento '{log_type}' não encontrado. "
        "Execute a Célula 19 e verifique seus arquivos exportados."
    )


hmm23_input_logs = {}
hmm23_sources = {}

for log_type in LOG_TYPES23:
    df, source = load_log23(log_type)

    # Defesa contra uso acidental de outras partições.
    if "partition" in df.columns:
        df = df.loc[
            df["partition"].astype(str).str.lower()
            == DISCOVERY_PARTITION23
        ].copy()

    if df.empty:
        raise RuntimeError(
            f"Event Log '{log_type}' vazio após filtragem train."
        )

    hmm23_input_logs[log_type] = df
    hmm23_sources[log_type] = source

    _project_print(
        f"{log_type:8s} | "
        f"{df['case_id'].nunique():3d} casos | "
        f"{len(df):5d} eventos | "
        f"fonte: {source}"
    )


# ================================================================
# 23.5 — ORDENAÇÃO E CONSTRUÇÃO DOS EVENT LOGS PM4PY
# ================================================================
#
# Preserva a ordem dos eventos dentro de cada caso.
#
# Quando há índice de evento ou horário de início disponível,
# ele tem prioridade sobre ordenação exclusivamente textual.
# ================================================================

def prepare_pm4py_log23(df):
    df = df.copy()

    order_candidates = [
        "event_index",
        "event_order",
        "start_index",
        "start_index_global",
        "start_hour_index",
        "start_timestamp",
        "timestamp",
        "_event_position23"
    ]

    sort_cols = ["case_id"]

    for candidate in order_candidates:
        if candidate in df.columns:
            sort_cols.append(candidate)
            break

    df = df.sort_values(
        sort_cols,
        kind="stable"
    ).reset_index(drop=True)

    if "timestamp" not in df.columns:
        if "start_timestamp" in df.columns:
            df["timestamp"] = pd.to_datetime(
                df["start_timestamp"],
                errors="coerce"
            )
        else:
            # Timestamp sintético apenas para compatibilidade.
            # A ordem dos eventos já foi definida acima.
            df["timestamp"] = pd.Timestamp(
                "2025-01-01"
            ) + pd.to_timedelta(
                np.arange(len(df)),
                unit="h"
            )

    df["timestamp"] = pd.to_datetime(
        df["timestamp"],
        errors="coerce"
    )

    if df["timestamp"].isna().any():
        raise ValueError(
            "Há timestamps inválidos no Event Log."
        )

    log = EventLog()

    for case_id, group in df.groupby(
        "case_id",
        sort=False
    ):
        trace = Trace(
            attributes={"concept:name": str(case_id)}
        )

        for _, row in group.iterrows():
            trace.append(
                Event({
                    "concept:name": str(row["activity"]),
                    "time:timestamp": row["timestamp"].to_pydatetime()
                })
            )

        log.append(trace)

    return log, df


hmm23_pm4py_logs = {}

for log_type, df in hmm23_input_logs.items():
    log, ordered_df = prepare_pm4py_log23(df)

    hmm23_pm4py_logs[log_type] = log
    hmm23_input_logs[log_type] = ordered_df

    _project_print(
        f"PM4Py {log_type}: "
        f"{len(log)} traces"
    )


# ================================================================
# 23.6 — DESCOBERTA COM INDUCTIVE MINER
# ================================================================
#
# PM4Py aplica o Inductive Miner às sequências observadas.
#
# O limiar de ruído controla a tolerância a comportamentos
# menos frequentes, mas não é uma porcentagem garantida
# de eventos removidos.
#
# Não são utilizadas métricas de validation/test nesta etapa.
# ================================================================

hmm23_models = {}
discovery_errors23 = []

for log_type in LOG_TYPES23:

    log = hmm23_pm4py_logs[log_type]

    for noise in NOISE_THRESHOLDS23:

        key = (log_type, float(noise))

        _project_print(
            f"\nDescobrindo: {log_type} | "
            f"noise={noise:.2f}"
        )

        try:
            tree = pm4py.discover_process_tree_inductive(
                log,
                noise_threshold=float(noise)
            )

            hmm23_models[key] = tree

            _project_print("  OK:", str(tree)[:160])

        except Exception as exc:
            discovery_errors23.append({
                "event_log": log_type,
                "noise_threshold": noise,
                "error": repr(exc)
            })

            _project_print("  ERRO:", repr(exc))

if not hmm23_models:
    raise RuntimeError(
        "Nenhum modelo foi descoberto pelo Inductive Miner."
    )


# ================================================================
# 23.7 — MÉTRICAS ESTRUTURAIS DAS ÁRVORES
# ================================================================
#
# Métricas descritivas:
#   - nós totais
#   - atividades visíveis
#   - nós tau (silenciosos)
#   - operadores
#   - profundidade
#   - número de atividades distintas
#
# Essas métricas não substituem fitness ou precision.
# ================================================================

def inspect_tree23(tree):

    operators = Counter()
    activities = []
    depths = []
    n_nodes = 0
    n_tau = 0

    def visit(node, depth):
        nonlocal n_nodes, n_tau

        n_nodes += 1
        depths.append(depth)

        if node.operator is not None:
            operators[str(node.operator)] += 1

        elif node.label is None:
            n_tau += 1

        else:
            activities.append(str(node.label))

        for child in node.children:
            visit(child, depth + 1)

    visit(tree, 0)

    return {
        "n_nodes": n_nodes,
        "n_visible_leaves": len(activities),
        "n_tau_leaves": n_tau,
        "n_distinct_activities": len(set(activities)),
        "n_operator_nodes": sum(operators.values()),
        "tree_depth": max(depths) if depths else 0,
        "n_sequence": operators.get("->", 0),
        "n_xor": operators.get("X", 0),
        "n_parallel": operators.get("+", 0),
        "n_loop": operators.get("*", 0),
        "operator_signature": _project_dumps(
            dict(operators),
            ensure_ascii=False
        )
    }


rows23 = []

for (log_type, noise), tree in hmm23_models.items():

    metrics = inspect_tree23(tree)

    rows23.append({
        "event_log": log_type,
        "noise_threshold": noise,
        "n_cases_train": len(
            hmm23_pm4py_logs[log_type]
        ),
        "n_events_train": len(
            hmm23_input_logs[log_type]
        ),
        **metrics,
        "discovery_status": "success"
    })

audit23_models = pd.DataFrame(rows23).sort_values(
    ["event_log", "noise_threshold"]
).reset_index(drop=True)

_project_print("\n" + "=" * 78)
_project_print("23.7 — COMPLEXIDADE DOS MODELOS")
_project_print("=" * 78)

_project_display(
    audit23_models[
        [
            "event_log",
            "noise_threshold",
            "n_nodes",
            "n_visible_leaves",
            "n_tau_leaves",
            "n_operator_nodes",
            "tree_depth",
            "n_distinct_activities"
        ]
    ]
)


# ================================================================
# 23.8 — INTEGRIDADE DA DESCOBERTA
# ================================================================
#
# Verifica se as atividades dos modelos são conhecidas no log.
# A ausência de alguma atividade na árvore pode decorrer
# da filtragem de ruído e deve ser documentada.
# ================================================================

activity_rows23 = []

for (log_type, noise), tree in hmm23_models.items():

    observed = set(
        hmm23_input_logs[log_type]["activity"].astype(str)
    )

    def collect_labels(node):
        labels = []

        if node.operator is None and node.label is not None:
            labels.append(str(node.label))

        for child in node.children:
            labels.extend(collect_labels(child))

        return labels

    discovered = set(collect_labels(tree))

    activity_rows23.append({
        "event_log": log_type,
        "noise_threshold": noise,
        "n_observed_activities": len(observed),
        "n_discovered_activities": len(discovered),
        "activity_coverage": (
            len(observed & discovered) / len(observed)
            if observed else np.nan
        ),
        "missing_activities": _project_dumps(
            sorted(observed - discovered),
            ensure_ascii=False
        ),
        "unexpected_activities": _project_dumps(
            sorted(discovered - observed),
            ensure_ascii=False
        )
    })

audit23_activity_coverage = pd.DataFrame(
    activity_rows23
)

audit23_models = audit23_models.merge(
    audit23_activity_coverage,
    on=["event_log", "noise_threshold"],
    how="left"
)

if (
    audit23_models["unexpected_activities"]
    .ne("[]")
    .any()
):
    raise AssertionError(
        "O modelo contém atividades ausentes do treinamento."
    )


# ================================================================
# 23.9 — EXPORTAÇÃO DAS ÁRVORES
# ================================================================
#
# PTML é o formato principal para preservar a árvore.
# SVG/PNG são artefatos visuais.
#
# A exportação visual não altera o modelo descoberto.
# ================================================================

export_rows23 = []

for (log_type, noise), tree in hmm23_models.items():

    suffix = f"{log_type}_noise_{noise:.2f}".replace(
        ".", "p"
    )

    ptml_path = MODEL_DIR23 / f"cell23_{suffix}.ptml"
    svg_path = FIGURE_DIR23 / f"cell23_{suffix}.svg"
    png_path = FIGURE_DIR23 / f"cell23_{suffix}.png"

    ptml_exporter.apply(
        tree,
        str(ptml_path)
    )

    visual_status = "not_exported"
    visual_error = None

    try:
        from pm4py.visualization.process_tree import (
            visualizer as tree_visualizer
        )

        gviz = tree_visualizer.apply(tree)

        tree_visualizer.save(
            gviz,
            str(svg_path)
        )

        tree_visualizer.save(
            gviz,
            str(png_path)
        )

        visual_status = "success"

    except Exception as exc:
        visual_error = repr(exc)
        _project_print(
            f"Aviso: figura {suffix} não exportada: "
            f"{visual_error}"
        )

    export_rows23.append({
        "event_log": log_type,
        "noise_threshold": noise,
        "ptml_path": str(ptml_path),
        "svg_path": (
            str(svg_path)
            if svg_path.exists()
            else None
        ),
        "png_path": (
            str(png_path)
            if png_path.exists()
            else None
        ),
        "visual_status": visual_status,
        "visual_error": visual_error
    })

audit23_exports = pd.DataFrame(export_rows23)


# ================================================================
# 23.10 — COMPARAÇÃO GRÁFICA DA COMPLEXIDADE
# ================================================================

fig, axes = plt.subplots(
    1,
    3,
    figsize=(17, 5),
    constrained_layout=True
)

metrics_to_plot = [
    ("n_nodes", "Número de nós"),
    ("tree_depth", "Profundidade"),
    ("n_tau_leaves", "Atividades silenciosas (tau)")
]

for ax, (metric, title) in zip(
    axes,
    metrics_to_plot
):
    for log_type in LOG_TYPES23:

        subset = audit23_models.loc[
            audit23_models["event_log"] == log_type
        ].sort_values("noise_threshold")

        ax.plot(
            subset["noise_threshold"],
            subset[metric],
            marker="o",
            linewidth=2,
            label=log_type
        )

    ax.set_title(title)
    ax.set_xlabel("Limiar de ruído")
    ax.set_ylabel(metric)
    ax.grid(alpha=0.25)

axes[0].legend()

fig.suptitle(
    "Inductive Miner — complexidade estrutural por limiar",
    fontsize=14
)

fig_path = FIGURE_DIR23 / (
    "cell23_inductive_miner_complexity.png"
)

fig.savefig(
    fig_path,
    dpi=180,
    bbox_inches="tight"
)

plt.show()


# ================================================================
# 23.11 — VISUALIZAÇÃO RESUMIDA DAS ÁRVORES
# ================================================================
#
# Exibe uma árvore de referência por log para inspeção.
# Noise = 0.10 é apenas um ponto de visualização.
# Não representa seleção de modelo.
# ================================================================

REFERENCE_NOISE23 = 0.10

_project_print("\n" + "=" * 78)
_project_print("23.11 — ÁRVORES DE REFERÊNCIA")
_project_print("=" * 78)

for log_type in LOG_TYPES23:

    key = (log_type, REFERENCE_NOISE23)

    if key not in hmm23_models:
        continue

    tree = hmm23_models[key]

    _project_print(
        f"\n{log_type.upper()} | "
        f"noise={REFERENCE_NOISE23:.2f}"
    )

    try:
        from pm4py.visualization.process_tree import (
            visualizer as tree_visualizer
        )

        gviz = tree_visualizer.apply(tree)

        _project_display(gviz)

    except Exception as exc:
        _project_print(
            "Visualização inline indisponível:",
            repr(exc)
        )

        _project_print("Representação textual:")
        _project_print(str(tree))


# ================================================================
# 23.12 — RESUMO EXPERIMENTAL
# ================================================================

audit23_summary = (
    audit23_models
    .groupby("event_log", as_index=False)
    .agg(
        n_models=("noise_threshold", "count"),
        min_nodes=("n_nodes", "min"),
        max_nodes=("n_nodes", "max"),
        min_depth=("tree_depth", "min"),
        max_depth=("tree_depth", "max"),
        min_activity_coverage=("activity_coverage", "min"),
        max_activity_coverage=("activity_coverage", "max")
    )
)

_project_print("\n" + "=" * 78)
_project_print("23.12 — RESUMO")
_project_print("=" * 78)

_project_display(audit23_summary)


# ================================================================
# 23.13 — SALVAMENTO DOS RELATÓRIOS
# ================================================================

_project_csv(audit23_models, 
    REPORT_DIR23 / "cell23_inductive_miner_models.csv",
    index=False
)

_project_csv(audit23_activity_coverage, 
    REPORT_DIR23 / "cell23_activity_coverage.csv",
    index=False
)

_project_csv(audit23_exports, 
    REPORT_DIR23 / "cell23_model_exports.csv",
    index=False
)

_project_csv(audit23_summary, 
    REPORT_DIR23 / "cell23_summary.csv",
    index=False
)

_project_csv(pd.DataFrame(discovery_errors23), 
    REPORT_DIR23 / "cell23_discovery_errors.csv",
    index=False
)

metadata23 = {
    "phase": 2,
    "cell": 23,
    "method": "Inductive Miner",
    "pm4py_version": pm4py.__version__,
    "discovery_partition": "train",
    "logs": list(LOG_TYPES23),
    "noise_thresholds": list(NOISE_THRESHOLDS23),
    "n_models": len(hmm23_models),
    "n_errors": len(discovery_errors23),
    "input_sources": hmm23_sources,
    "uses_validation_for_discovery": False,
    "uses_test": False,
    "uses_co2": False,
    "retrains_hmm": False,
    "model_selection_performed": False
}

with open(
    REPORT_DIR23 / "cell23_metadata.json",
    "w",
    encoding="utf-8"
) as file:
    _project_dump(
        metadata23,
        file,
        indent=2,
        ensure_ascii=False
    )


# ================================================================
# 23.14 — VERIFICAÇÕES FINAIS
# ================================================================

expected_models23 = (
    len(LOG_TYPES23)
    * len(NOISE_THRESHOLDS23)
)

_project_print("\n" + "=" * 78)
_project_print("RESULTADO DA CÉLULA 23")
_project_print("=" * 78)

_project_print(
    "Modelos descobertos:",
    len(hmm23_models),
    "/",
    expected_models23
)

_project_print(
    "Falhas de descoberta:",
    len(discovery_errors23)
)

_project_print(
    "Treinamento utilizado: SIM"
)

_project_print(
    "Validação utilizada na descoberta: NÃO"
)

_project_print(
    "Teste utilizado: NÃO"
)

_project_print(
    "CO2 utilizado: NÃO"
)

_project_print(
    "HMM retreinado: NÃO"
)

_project_print(
    "Seleção definitiva realizada: NÃO"
)

_project_print(
    "Relatórios:",
    REPORT_DIR23
)

_project_print(
    "Modelos PTML:",
    MODEL_DIR23
)

_project_print(
    "Figuras:",
    FIGURE_DIR23
)

if len(hmm23_models) != expected_models23:
    _project_print(
        "\nATENÇÃO: alguns candidatos não foram descobertos. "
        "Consulte cell23_discovery_errors.csv."
    )
else:
    _project_print("\nCÉLULA 23 CONCLUÍDA.")


In [ ]:

# ============================================================
# FASE 2 — CÉLULA 24/36
# INTERPRETAÇÃO E VISUALIZAÇÃO DAS ÁRVORES DE PROCESSO
# ============================================================
#
# Objetivos:
# 1. Recuperar os modelos do Inductive Miner (Célula 23).
# 2. Caracterizar operadores e atividades.
# 3. Analisar relações estruturais entre atividades.
# 4. Visualizar árvores selecionadas para inspeção.
# 5. Comparar a complexidade por limiar de ruído.
# 6. Exportar resultados reproduzíveis.
#
# Restrições:
# - Não redescobrir modelos.
# - Não retreinar HMM.
# - Não utilizar CO2.
# - Não utilizar teste.
# - Não selecionar modelo definitivo.
# - Não confundir concorrência formal com simultaneidade física.
#
# ============================================================

import os
import json
import math
import re
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display, SVG, Image

import pm4py
from pm4py.objects.process_tree.obj import Operator, ProcessTree
from pm4py.visualization.process_tree import visualizer as pt_visualizer
from pm4py.objects.process_tree.exporter import exporter as pt_exporter

_project_print("=" * 78)
_project_print("FASE 2 — CÉLULA 24/36 | INTERPRETAÇÃO DAS ÁRVORES")
_project_print("=" * 78)

# ------------------------------------------------------------
# 24.1 — CONFIGURAÇÃO
# ------------------------------------------------------------

LOGS24 = ("regime", "ramp", "combined")
NOISE24 = (0.00, 0.05, 0.10, 0.20, 0.30)

REFERENCE_NOISE24 = 0.10

# Evitar diagramas excessivamente extensos no notebook.
DISPLAY_LOGS24 = ("regime", "ramp", "combined")

# A representação textual é truncada apenas na exibição.
# Os modelos e relatórios preservam a estrutura integral.
MAX_TEXT_DISPLAY24 = 450

# Identificar diretório de execução existente.
if "RUN_DIR" in globals():
    ROOT24 = Path(RUN_DIR)
elif "run_dir" in globals():
    ROOT24 = Path(run_dir)
else:
    raise RuntimeError(
        "RUN_DIR não encontrado. Execute as células anteriores "
        "para recuperar o diretório da execução."
    )

REPORTS24 = ROOT24 / "reports"
FIGURES24 = ROOT24 / "figures"
MODELS24 = ROOT24 / "models" / "cell24"

for folder in (REPORTS24, FIGURES24, MODELS24):
    folder.mkdir(parents=True, exist_ok=True)

_project_print("PM4Py:", pm4py.__version__)
_project_print("Execução:", ROOT24)
_project_print("Ruído de referência:", REFERENCE_NOISE24)

# ------------------------------------------------------------
# 24.2 — LOCALIZAR OS MODELOS DA CÉLULA 23
# ------------------------------------------------------------
#
# Primeiro utiliza os objetos em memória.
# Se necessário, procura arquivos PTML exportados.
#
# A célula não deve produzir modelos novos.
# ------------------------------------------------------------

def is_tree24(obj):
    return isinstance(obj, ProcessTree)


def normalize_log24(value):
    s = str(value).lower()
    for log in LOGS24:
        if log in s:
            return log
    return None


def normalize_noise24(value):
    try:
        v = float(value)
    except (ValueError, TypeError):
        s = str(value)
        matches = re.findall(r"(?<!\d)(?:0[.,]\d+|0)(?!\d)", s)
        if not matches:
            return None
        v = float(matches[-1].replace(",", "."))

    for target in NOISE24:
        if abs(v - target) < 1e-7:
            return target
    return None


def find_trees_recursive24(obj, prefix="", depth=0):
    """
    Descobre árvores em estruturas dict/list/tuple,
    sem percorrer internamente objetos ProcessTree.
    """
    found = []

    if depth > 5:
        return found

    if is_tree24(obj):
        found.append((prefix, obj))
        return found

    if isinstance(obj, dict):
        for key, value in obj.items():
            found.extend(
                find_trees_recursive24(
                    value,
                    prefix=f"{prefix}/{key}",
                    depth=depth + 1
                )
            )

    elif isinstance(obj, (list, tuple)):
        for i, value in enumerate(obj):
            found.extend(
                find_trees_recursive24(
                    value,
                    prefix=f"{prefix}/{i}",
                    depth=depth + 1
                )
            )

    return found


def identify_key24(path):
    """
    Extrai log e limiar de uma chave composta ou caminho.
    """
    log = normalize_log24(path)

    noise = None
    tokens = re.split(r"[/|;() \[\]]+", str(path))

    for token in reversed(tokens):
        if not token:
            continue
        candidate = normalize_noise24(token)
        if candidate is not None:
            noise = candidate
            break

    return log, noise


hmm24_trees = {}
sources24 = []

# Objetos conhecidos ou candidatos da Célula 23.
CANDIDATES24 = [
    "hmm23_trees",
    "hmm23_models",
    "hmm23_process_trees",
    "process_trees23",
    "trees23",
    "models23",
]

for varname in CANDIDATES24:
    if varname not in globals():
        continue

    obj = globals()[varname]

    for path, tree in find_trees_recursive24(obj, varname):
        log, noise = identify_key24(path)

        if log is not None and noise is not None:
            hmm24_trees.setdefault((log, noise), tree)
            sources24.append((log, noise, path))

# Recuperação adicional via PTML.
PTML_DIR24 = ROOT24 / "models" / "cell23"

if len(hmm24_trees) < len(LOGS24) * len(NOISE24):

    from pm4py.objects.process_tree.importer import (
        importer as pt_importer
    )

    if PTML_DIR24.exists():
        for file in sorted(PTML_DIR24.rglob("*.ptml")):

            log = normalize_log24(file.name)
            noise = None

            # Procura especificamente números no nome do arquivo.
            name = file.stem.lower()

            for value in NOISE24:
                candidates = [
                    f"{value:.2f}",
                    f"{value:.2f}".replace(".", "_"),
                    f"{value:.2f}".replace(".", "p"),
                ]

                if any(c in name for c in candidates):
                    noise = value
                    break

            if log is None or noise is None:
                continue

            key = (log, noise)

            if key not in hmm24_trees:
                hmm24_trees[key] = pt_importer.apply(str(file))
                sources24.append((log, noise, str(file)))

EXPECTED24 = {
    (log, noise)
    for log in LOGS24
    for noise in NOISE24
}

MISSING24 = sorted(EXPECTED24 - set(hmm24_trees))

_project_print("\nÁrvores recuperadas:", len(hmm24_trees), "/ 15")

if MISSING24:
    _project_print("Modelos ausentes:", MISSING24)
    _project_print("\nVariáveis relacionadas à Célula 23:")
    _project_print([
        name for name in globals()
        if "23" in name.lower()
        and not name.startswith("_")
    ])

    raise RuntimeError(
        "Não foi possível recuperar todas as árvores. "
        "Verifique os nomes dos objetos ou arquivos PTML "
        "produzidos na Célula 23. "
        "Nenhuma redescoberta será realizada."
    )

_project_print("Todos os 15 modelos recuperados.")

# ------------------------------------------------------------
# 24.3 — FUNÇÕES DE ANÁLISE ESTRUTURAL
# ------------------------------------------------------------

def operator_name24(node):
    if node.operator is None:
        return None

    mapping = {
        Operator.SEQUENCE: "sequence",
        Operator.XOR: "xor",
        Operator.PARALLEL: "parallel",
        Operator.LOOP: "loop",
    }

    return mapping.get(
        node.operator,
        str(node.operator)
    )


def traverse24(tree):
    """
    Percurso em profundidade.

    Retorna:
      node, depth, path
    """
    stack = [(tree, 0, "root")]

    while stack:
        node, depth, path = stack.pop()

        yield node, depth, path

        for i in reversed(range(len(node.children))):
            stack.append((
                node.children[i],
                depth + 1,
                f"{path}/{i}"
            ))


def visible_activities24(tree):
    return sorted({
        str(node.label)
        for node, _, _ in traverse24(tree)
        if node.operator is None
        and node.label is not None
    })


def tree_metrics24(tree):

    nodes = list(traverse24(tree))

    op_counts = Counter()
    activity_counts = Counter()

    tau = 0
    visible = 0
    depths = []
    branching = []

    for node, depth, _ in nodes:

        depths.append(depth)

        if node.operator is not None:
            op_counts[operator_name24(node)] += 1
            branching.append(len(node.children))

        elif node.label is None:
            tau += 1

        else:
            visible += 1
            activity_counts[str(node.label)] += 1

    result = {
        "n_nodes": len(nodes),
        "n_visible_leaves": visible,
        "n_tau_leaves": tau,
        "n_operator_nodes": sum(op_counts.values()),
        "tree_depth": max(depths) if depths else 0,
        "n_distinct_activities": len(activity_counts),
        "n_sequence": op_counts["sequence"],
        "n_xor": op_counts["xor"],
        "n_parallel": op_counts["parallel"],
        "n_loop": op_counts["loop"],
        "max_branching": max(branching) if branching else 0,
        "mean_branching": (
            float(np.mean(branching)) if branching else 0.0
        ),
        "tau_ratio": (
            tau / len(nodes) if nodes else np.nan
        ),
        "operator_ratio": (
            sum(op_counts.values()) / len(nodes)
            if nodes else np.nan
        ),
    }

    return result


def node_table24(tree, log, noise):

    rows = []

    for node, depth, path in traverse24(tree):

        if node.operator is not None:
            node_type = "operator"
            label = operator_name24(node)

        elif node.label is None:
            node_type = "tau"
            label = "tau"

        else:
            node_type = "activity"
            label = str(node.label)

        rows.append({
            "event_log": log,
            "noise_threshold": noise,
            "node_path": path,
            "parent_path": (
                path.rsplit("/", 1)[0]
                if path != "root"
                else None
            ),
            "depth": depth,
            "node_type": node_type,
            "label": label,
            "n_children": len(node.children),
        })

    return rows


def structural_relations24(tree, log, noise):
    """
    Relações locais pai-filho.

    Importante:
    - Não representam DFGs.
    - Não representam transições horárias observadas.
    - Não constituem caminhos de execução enumerados.
    """

    rows = []

    for node, depth, path in traverse24(tree):

        if node.operator is None:
            continue

        op = operator_name24(node)

        children = []

        for child in node.children:

            if child.operator is not None:
                children.append(operator_name24(child))

            elif child.label is None:
                children.append("tau")

            else:
                children.append(str(child.label))

        rows.append({
            "event_log": log,
            "noise_threshold": noise,
            "node_path": path,
            "depth": depth,
            "operator": op,
            "n_children": len(children),
            "children": " | ".join(children),
            "has_tau_child": any(
                child.operator is None
                and child.label is None
                for child in node.children
            ),
        })

    return rows


# ------------------------------------------------------------
# 24.4 — CARACTERIZAÇÃO DOS 15 MODELOS
# ------------------------------------------------------------

metrics_rows24 = []
node_rows24 = []
relation_rows24 = []

for log in LOGS24:
    for noise in NOISE24:

        tree = hmm24_trees[(log, noise)]

        metrics_rows24.append({
            "event_log": log,
            "noise_threshold": noise,
            **tree_metrics24(tree)
        })

        node_rows24.extend(
            node_table24(tree, log, noise)
        )

        relation_rows24.extend(
            structural_relations24(tree, log, noise)
        )

audit24_complexity = pd.DataFrame(metrics_rows24)
audit24_nodes = pd.DataFrame(node_rows24)
audit24_relations = pd.DataFrame(relation_rows24)

assert len(audit24_complexity) == 15

_project_print("\n" + "=" * 78)
_project_print("24.4 — COMPLEXIDADE E OPERADORES")
_project_print("=" * 78)

_project_display(
    audit24_complexity[
        [
            "event_log",
            "noise_threshold",
            "n_nodes",
            "tree_depth",
            "n_visible_leaves",
            "n_tau_leaves",
            "n_sequence",
            "n_xor",
            "n_parallel",
            "n_loop",
        ]
    ].round(3)
)

# ------------------------------------------------------------
# 24.5 — ANÁLISE DOS OPERADORES
# ------------------------------------------------------------

operator_cols24 = [
    "n_sequence",
    "n_xor",
    "n_parallel",
    "n_loop",
]

fig, axes = plt.subplots(
    1, 3,
    figsize=(17, 5),
    constrained_layout=True
)

for ax, log in zip(axes, LOGS24):

    subset = audit24_complexity[
        audit24_complexity["event_log"] == log
    ].sort_values("noise_threshold")

    bottom = np.zeros(len(subset))

    for col in operator_cols24:

        values = subset[col].to_numpy()

        ax.bar(
            subset["noise_threshold"].astype(str),
            values,
            bottom=bottom,
            label=col.replace("n_", "")
        )

        bottom += values

    ax.set_title(log.upper())
    ax.set_xlabel("Limiar de ruído")
    ax.set_ylabel("Quantidade de operadores")
    ax.grid(axis="y", alpha=0.2)
    ax.legend(fontsize=8)

fig.suptitle(
    "Composição estrutural das árvores de processo",
    fontsize=14
)

fig.savefig(
    FIGURES24 / "cell24_operator_composition.png",
    dpi=180,
    bbox_inches="tight"
)

fig.savefig(
    FIGURES24 / "cell24_operator_composition.svg",
    bbox_inches="tight"
)

plt.show()

# ------------------------------------------------------------
# 24.6 — ESTABILIDADE ESTRUTURAL
# ------------------------------------------------------------
#
# Compara métricas e representações textuais.
#
# Igualdade textual é um diagnóstico estrito:
# árvores semanticamente equivalentes podem ter
# representações textuais diferentes.
# ------------------------------------------------------------

stability_rows24 = []

for log in LOGS24:

    previous = None

    for noise in NOISE24:

        tree = hmm24_trees[(log, noise)]
        metrics = tree_metrics24(tree)

        current = str(tree)

        stability_rows24.append({
            "event_log": log,
            "noise_threshold": noise,
            "n_nodes": metrics["n_nodes"],
            "tree_depth": metrics["tree_depth"],
            "n_tau_leaves": metrics["n_tau_leaves"],
            "same_text_as_previous": (
                None if previous is None
                else current == previous
            ),
        })

        previous = current

audit24_stability = pd.DataFrame(stability_rows24)

_project_print("\n" + "=" * 78)
_project_print("24.6 — ESTABILIDADE ENTRE LIMIARES")
_project_print("=" * 78)

_project_display(audit24_stability)

# ------------------------------------------------------------
# 24.7 — INTERPRETAÇÃO DOS OPERADORES
# ------------------------------------------------------------

operator_dictionary24 = pd.DataFrame([
    {
        "operator": "sequence",
        "meaning": "Ordem estrutural entre subprocessos",
        "physical_caution": (
            "Não prova causalidade física entre etapas"
        )
    },
    {
        "operator": "xor",
        "meaning": "Escolha exclusiva entre caminhos",
        "physical_caution": (
            "Não identifica sozinho a causa da escolha"
        )
    },
    {
        "operator": "parallel",
        "meaning": "Intercalações permitidas pelo modelo",
        "physical_caution": (
            "Não implica execução física simultânea"
        )
    },
    {
        "operator": "loop",
        "meaning": "Repetição de atividades ou subprocessos",
        "physical_caution": (
            "Não determina duração nem frequência real"
        )
    },
    {
        "operator": "tau",
        "meaning": "Atividade silenciosa ou passagem interna",
        "physical_caution": (
            "Não corresponde necessariamente a um evento físico"
        )
    },
])

_project_print("\n" + "=" * 78)
_project_print("24.7 — DICIONÁRIO DE INTERPRETAÇÃO")
_project_print("=" * 78)

_project_display(operator_dictionary24)

# ------------------------------------------------------------
# 24.8 — ESTRUTURA DAS ÁRVORES DE REFERÊNCIA
# ------------------------------------------------------------

_project_print("\n" + "=" * 78)
_project_print("24.8 — ÁRVORES DE REFERÊNCIA")
_project_print("=" * 78)

for log in LOGS24:

    tree = hmm24_trees[
        (log, REFERENCE_NOISE24)
    ]

    _project_print(
        f"\n{log.upper()} | "
        f"noise={REFERENCE_NOISE24:.2f}"
    )

    _project_print(
        str(tree)[:MAX_TEXT_DISPLAY24]
        + (
            " ..."
            if len(str(tree)) > MAX_TEXT_DISPLAY24
            else ""
        )
    )

    _project_print("Atividades:")
    _project_print(visible_activities24(tree))

# ------------------------------------------------------------
# 24.9 — VISUALIZAÇÃO INTEGRAL DAS ÁRVORES
# ------------------------------------------------------------
#
# Salva SVG e PNG.
# O SVG preserva melhor a legibilidade de árvores grandes.
# ------------------------------------------------------------

_project_print("\n" + "=" * 78)
_project_print("24.9 — DIAGRAMAS DAS ÁRVORES")
_project_print("=" * 78)

for log in DISPLAY_LOGS24:

    tree = hmm24_trees[
        (log, REFERENCE_NOISE24)
    ]

    _project_print(
        f"\nÁrvore: {log.upper()} | "
        f"noise={REFERENCE_NOISE24:.2f}"
    )

    try:
        gviz = pt_visualizer.apply(
            tree,
            parameters={
                "format": "svg"
            }
        )

        svg_path = (
            FIGURES24
            / f"cell24_tree_{log}_noise010.svg"
        )

        pt_visualizer.save(
            gviz,
            str(svg_path)
        )

        # Exibição inline do SVG.
        _project_display(SVG(filename=str(svg_path)))

    except Exception as exc:
        _project_print(
            "Falha na visualização SVG:",
            type(exc).__name__,
            str(exc)
        )

    try:
        gviz_png = pt_visualizer.apply(
            tree,
            parameters={
                "format": "png"
            }
        )

        png_path = (
            FIGURES24
            / f"cell24_tree_{log}_noise010.png"
        )

        pt_visualizer.save(
            gviz_png,
            str(png_path)
        )

    except Exception as exc:
        _project_print(
            "Falha na exportação PNG:",
            type(exc).__name__,
            str(exc)
        )

# ------------------------------------------------------------
# 24.10 — MAPA DE OPERADORES POR PROFUNDIDADE
# ------------------------------------------------------------

depth_rows24 = []

for log in LOGS24:

    tree = hmm24_trees[
        (log, REFERENCE_NOISE24)
    ]

    for node, depth, path in traverse24(tree):

        if node.operator is None:
            continue

        depth_rows24.append({
            "event_log": log,
            "depth": depth,
            "operator": operator_name24(node),
        })

audit24_operator_depth = pd.DataFrame(depth_rows24)

fig, axes = plt.subplots(
    1, 3,
    figsize=(17, 5),
    constrained_layout=True
)

for ax, log in zip(axes, LOGS24):

    subset = audit24_operator_depth[
        audit24_operator_depth["event_log"] == log
    ]

    table = pd.crosstab(
        subset["depth"],
        subset["operator"]
    )

    table.plot(
        kind="bar",
        stacked=True,
        ax=ax,
        legend=True
    )

    ax.set_title(log.upper())
    ax.set_xlabel("Profundidade")
    ax.set_ylabel("Operadores")
    ax.grid(axis="y", alpha=0.2)

fig.suptitle(
    "Distribuição dos operadores por profundidade",
    fontsize=14
)

fig.savefig(
    FIGURES24 / "cell24_operator_depth.png",
    dpi=180,
    bbox_inches="tight"
)

fig.savefig(
    FIGURES24 / "cell24_operator_depth.svg",
    bbox_inches="tight"
)

plt.show()

# ------------------------------------------------------------
# 24.11 — COMPARAÇÃO DE COMPLEXIDADE
# ------------------------------------------------------------

fig, axes = plt.subplots(
    1, 3,
    figsize=(17, 5),
    constrained_layout=True
)

METRICS24 = [
    ("n_nodes", "Número de nós"),
    ("tree_depth", "Profundidade"),
    ("n_tau_leaves", "Atividades silenciosas"),
]

for ax, (metric, title) in zip(axes, METRICS24):

    for log in LOGS24:

        subset = audit24_complexity[
            audit24_complexity["event_log"] == log
        ].sort_values("noise_threshold")

        ax.plot(
            subset["noise_threshold"],
            subset[metric],
            marker="o",
            label=log
        )

    ax.set_title(title)
    ax.set_xlabel("Limiar de ruído")
    ax.grid(alpha=0.25)
    ax.legend()

fig.suptitle(
    "Sensibilidade estrutural ao limiar de ruído",
    fontsize=14
)

fig.savefig(
    FIGURES24 / "cell24_complexity_comparison.png",
    dpi=180,
    bbox_inches="tight"
)

fig.savefig(
    FIGURES24 / "cell24_complexity_comparison.svg",
    bbox_inches="tight"
)

plt.show()

# ------------------------------------------------------------
# 24.12 — DIAGNÓSTICOS DE INTERPRETABILIDADE
# ------------------------------------------------------------
#
# Indicadores descritivos, não escores de qualidade:
#
# - tau_ratio: fração de nós silenciosos;
# - operator_ratio: fração de operadores;
# - branching: ramificação média e máxima.
#
# Não utilizar esses indicadores isoladamente
# para selecionar o melhor modelo.
# ------------------------------------------------------------

audit24_interpretability = audit24_complexity[
    [
        "event_log",
        "noise_threshold",
        "n_nodes",
        "tree_depth",
        "tau_ratio",
        "operator_ratio",
        "mean_branching",
        "max_branching",
        "n_distinct_activities",
    ]
].copy()

_project_print("\n" + "=" * 78)
_project_print("24.12 — INDICADORES DE INTERPRETABILIDADE")
_project_print("=" * 78)

_project_display(
    audit24_interpretability.round(4)
)

# ------------------------------------------------------------
# 24.13 — VERIFICAÇÕES METODOLÓGICAS
# ------------------------------------------------------------

assert set(
    audit24_complexity["event_log"]
) == set(LOGS24)

assert set(
    audit24_complexity["noise_threshold"]
) == set(NOISE24)

assert (
    audit24_complexity["n_nodes"]
    ==
    audit24_complexity[
        [
            "n_visible_leaves",
            "n_tau_leaves",
            "n_operator_nodes",
        ]
    ].sum(axis=1)
).all()

assert (
    audit24_complexity["n_distinct_activities"]
    ==
    audit24_complexity["event_log"].map({
        "regime": 5,
        "ramp": 4,
        "combined": 10,
    })
).all()

# A representação de árvore deve ser acíclica.
# A repetição é expressa pelo operador LOOP.
for tree in hmm24_trees.values():
    paths = [
        path for _, _, path in traverse24(tree)
    ]
    assert len(paths) == len(set(paths))

_project_print("\nVerificações estruturais: OK")

# ------------------------------------------------------------
# 24.14 — EXPORTAÇÃO
# ------------------------------------------------------------

_project_csv(audit24_complexity, 
    REPORTS24 / "cell24_tree_complexity.csv",
    index=False
)

_project_csv(audit24_nodes, 
    REPORTS24 / "cell24_tree_nodes.csv",
    index=False
)

_project_csv(audit24_relations, 
    REPORTS24 / "cell24_tree_relations.csv",
    index=False
)

_project_csv(audit24_stability, 
    REPORTS24 / "cell24_tree_stability.csv",
    index=False
)

_project_csv(audit24_operator_depth, 
    REPORTS24 / "cell24_operator_depth.csv",
    index=False
)

_project_csv(audit24_interpretability, 
    REPORTS24 / "cell24_interpretability.csv",
    index=False
)

_project_csv(operator_dictionary24, 
    REPORTS24 / "cell24_operator_dictionary.csv",
    index=False
)

# Exporta os modelos em formato PTML.
for log in LOGS24:
    for noise in NOISE24:

        tree = hmm24_trees[(log, noise)]

        filename = (
            f"cell24_{log}_noise_"
            f"{noise:.2f}.ptml"
        )

        pt_exporter.apply(
            tree,
            str(MODELS24 / filename)
        )

metadata24 = {
    "phase": 2,
    "cell": 24,
    "method": "process_tree_structural_interpretation",
    "logs": list(LOGS24),
    "noise_thresholds": list(NOISE24),
    "reference_noise": REFERENCE_NOISE24,
    "models_recovered": len(hmm24_trees),
    "models_discovered_in_cell24": 0,
    "uses_training": True,
    "uses_validation_for_discovery": False,
    "uses_test": False,
    "uses_co2": False,
    "hmm_retrained": False,
    "definitive_selection": False,
    "interpretation_policy": (
        "Formal process-tree operators are not direct "
        "evidence of physical causality or simultaneity."
    ),
    "structural_relation_policy": (
        "Parent-child relations are not observed "
        "directly-follows relations."
    ),
}

with open(
    REPORTS24 / "cell24_metadata.json",
    "w",
    encoding="utf-8"
) as f:
    _project_dump(
        metadata24,
        f,
        ensure_ascii=False,
        indent=2
    )

# ------------------------------------------------------------
# 24.15 — RESUMO FINAL
# ------------------------------------------------------------

audit24_summary = (
    audit24_complexity
    .groupby("event_log", as_index=False)
    .agg(
        n_models=("noise_threshold", "count"),
        min_nodes=("n_nodes", "min"),
        max_nodes=("n_nodes", "max"),
        min_depth=("tree_depth", "min"),
        max_depth=("tree_depth", "max"),
        min_tau=("n_tau_leaves", "min"),
        max_tau=("n_tau_leaves", "max"),
    )
)

_project_print("\n" + "=" * 78)
_project_print("24.15 — RESUMO FINAL")
_project_print("=" * 78)

_project_display(audit24_summary)

_project_print("\n" + "=" * 78)
_project_print("RESULTADO DA CÉLULA 24")
_project_print("=" * 78)

_project_print("Árvores analisadas:", len(hmm24_trees))
_project_print("Modelos redescobertos: NÃO")
_project_print("HMM retreinado: NÃO")
_project_print("Teste utilizado: NÃO")
_project_print("CO2 utilizado: NÃO")
_project_print("Seleção definitiva: NÃO")
_project_print("Relatórios:", REPORTS24)
_project_print("Figuras:", FIGURES24)
_project_print("Modelos PTML:", MODELS24)

_project_print("\nCÉLULA 24 CONCLUÍDA.")


In [ ]:

# ============================================================
# FASE 2 — CÉLULA 25/36
# CONVERSÃO DAS ÁRVORES DE PROCESSO EM REDES DE PETRI
# ============================================================
#
# OBJETIVOS:
#   1. Recuperar as árvores PTML da Célula 23.
#   2. Interpretar corretamente os limiares de ruído.
#   3. Validar a integridade dos arquivos recuperados.
#   4. Converter árvores de processo em redes de Petri.
#   5. Preservar marcações inicial e final.
#   6. Exportar os modelos em PNML.
#   7. Produzir diagramas e métricas estruturais.
#   8. Preparar os modelos para a avaliação da Célula 26.
#
# RESTRIÇÕES METODOLÓGICAS:
#   - Não retreinar o HMM.
#   - Não executar novamente o Inductive Miner.
#   - Não utilizar o conjunto de teste.
#   - Não utilizar CO2 para seleção dos modelos.
#   - Não alterar os modelos descobertos.
#
# ============================================================

import re
import json
import warnings
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display, SVG, Image

import pm4py

from pm4py.objects.process_tree.importer import importer as ptml_importer
from pm4py.objects.conversion.process_tree import converter as pt_converter
from pm4py.objects.petri_net.exporter import exporter as pnml_exporter
from pm4py.visualization.petri_net import visualizer as pn_visualizer

_project_print("=" * 78)
_project_print("FASE 2 — CÉLULA 25/36 | REDES DE PETRI")
_project_print("=" * 78)
_project_print("PM4Py:", pm4py.__version__)


# ============================================================
# 25.1 — CONFIGURAÇÃO E DIRETÓRIOS
# ============================================================

LOG_TYPES25 = ("regime", "ramp", "combined")
NOISE_LEVELS25 = (0.00, 0.05, 0.10, 0.20, 0.30)

# Recuperar a pasta da execução corrente.
if "RUN_DIR" in globals():
    RUN_DIR25 = Path(RUN_DIR)
elif "run_dir" in globals():
    RUN_DIR25 = Path(run_dir)
else:
    raise RuntimeError(
        "RUN_DIR não encontrado. Execute a célula de "
        "configuração da Fase 2 antes da Célula 25."
    )

SOURCE_DIR25 = RUN_DIR25 / "models" / "cell23"
MODEL_DIR25 = RUN_DIR25 / "models" / "cell25"
FIG_DIR25 = RUN_DIR25 / "figures"
REPORT_DIR25 = RUN_DIR25 / "reports"

MODEL_DIR25.mkdir(parents=True, exist_ok=True)
FIG_DIR25.mkdir(parents=True, exist_ok=True)
REPORT_DIR25.mkdir(parents=True, exist_ok=True)

if not SOURCE_DIR25.is_dir():
    raise FileNotFoundError(
        f"Diretório PTML da Célula 23 não encontrado:\n"
        f"{SOURCE_DIR25}"
    )

_project_print("\nDiretório de origem:", SOURCE_DIR25)
_project_print("Diretório dos modelos:", MODEL_DIR25)


# ============================================================
# 25.2 — INTERPRETAÇÃO ROBUSTA DOS NOMES PTML
# ============================================================

def parse_key25(key):
    """
    Extrai (tipo_log, noise) de nomes como:

      cell23_combined_noise_0p00.ptml
      cell23_combined_noise_0p05.ptml
      cell23_regime_noise_0p10.ptml
      cell23_ramp_noise_0p20.ptml
      cell23_combined_noise_0p30.ptml

    Também aceita valores com ponto decimal.
    """

    if isinstance(key, tuple):
        parts = key
    else:
        parts = (key,)

    log_type = None
    noise = None

    for part in parts:

        if isinstance(
            part, (int, float, np.integer, np.floating)
        ):
            value = float(part)

            if any(
                np.isclose(value, n, atol=1e-9)
                for n in NOISE_LEVELS25
            ):
                noise = value

            continue

        text = str(part).lower()

        for kind in LOG_TYPES25:
            if re.search(
                rf"(?<![a-z]){kind}(?![a-z])",
                text
            ):
                log_type = kind
                break

        match = re.search(
            r"(?:noise|threshold)[_=\- ]*"
            r"(0(?:[p\.]\d+)?)"
            r"(?![0-9])",
            text
        )

        if match:
            value = float(
                match.group(1).replace("p", ".")
            )

            if any(
                np.isclose(value, n, atol=1e-9)
                for n in NOISE_LEVELS25
            ):
                noise = value

        elif re.fullmatch(r"0(?:[p\.]\d+)?", text):
            value = float(text.replace("p", "."))

            if any(
                np.isclose(value, n, atol=1e-9)
                for n in NOISE_LEVELS25
            ):
                noise = value

    if log_type is None or noise is None:
        return None

    return log_type, round(noise, 2)


# Teste unitário obrigatório da interpretação.
for expected in NOISE_LEVELS25:

    encoded = f"{expected:.2f}".replace(".", "p")
    filename = f"cell23_combined_noise_{encoded}.ptml"

    result = parse_key25(filename)

    assert result == ("combined", expected), (
        f"Falha na interpretação de {filename}: {result}"
    )

_project_print("\n25.2 — Parser PTML validado.")


# ============================================================
# 25.3 — INVENTÁRIO DOS ARQUIVOS PTML
# ============================================================

ptml_files25 = sorted(SOURCE_DIR25.glob("*.ptml"))

if not ptml_files25:
    raise FileNotFoundError(
        "Nenhum arquivo PTML encontrado em "
        f"{SOURCE_DIR25}"
    )

ptml_map25 = {}
inventory_rows25 = []

for filepath in ptml_files25:

    parsed = parse_key25(filepath.name)

    if parsed is None:
        inventory_rows25.append({
            "filename": filepath.name,
            "log_type": None,
            "noise": np.nan,
            "status": "ignored_unrecognized"
        })
        continue

    if parsed in ptml_map25:
        raise RuntimeError(
            "Arquivos PTML duplicados para a mesma chave:\n"
            f"Chave: {parsed}\n"
            f"Arquivo 1: {ptml_map25[parsed]}\n"
            f"Arquivo 2: {filepath}"
        )

    ptml_map25[parsed] = filepath

    inventory_rows25.append({
        "filename": filepath.name,
        "log_type": parsed[0],
        "noise": parsed[1],
        "status": "recognized"
    })

inventory25 = pd.DataFrame(inventory_rows25)

expected_keys25 = {
    (kind, round(float(noise), 2))
    for kind in LOG_TYPES25
    for noise in NOISE_LEVELS25
}

missing_keys25 = sorted(
    expected_keys25 - set(ptml_map25)
)

unexpected_keys25 = sorted(
    set(ptml_map25) - expected_keys25
)

_project_print("\n25.3 — INVENTÁRIO PTML")
_project_print("Arquivos encontrados:", len(ptml_files25))
_project_print("Modelos reconhecidos:", len(ptml_map25))
_project_print("Modelos esperados:", len(expected_keys25))

_project_display(inventory25)

if missing_keys25:
    raise RuntimeError(
        "Modelos PTML esperados não encontrados:\n"
        + "\n".join(map(str, missing_keys25))
    )

if unexpected_keys25:
    raise RuntimeError(
        "Modelos PTML inesperados:\n"
        + "\n".join(map(str, unexpected_keys25))
    )

_project_csv(inventory25, 
    REPORT_DIR25 / "cell25_ptml_inventory.csv",
    index=False
)

_project_print("Integridade do inventário: OK.")


# ============================================================
# 25.4 — IMPORTAÇÃO DAS ÁRVORES DE PROCESSO
# ============================================================

hmm25_trees = {}

for key in sorted(ptml_map25):

    filepath = ptml_map25[key]

    try:
        tree = ptml_importer.apply(str(filepath))

    except Exception as exc:
        raise RuntimeError(
            f"Falha ao importar árvore {key}:\n"
            f"{filepath}\n{exc}"
        ) from exc

    if tree is None:
        raise RuntimeError(
            f"Árvore vazia para {key}"
        )

    hmm25_trees[key] = tree

assert len(hmm25_trees) == 15

_project_print("\n25.4 — Árvores importadas:", len(hmm25_trees))


# ============================================================
# 25.5 — CONVERSÃO PARA REDES DE PETRI
# ============================================================

hmm25_petri_models = {}
conversion_rows25 = []

for key, tree in sorted(hmm25_trees.items()):

    kind, noise = key

    try:
        net, initial_marking, final_marking = (
            pt_converter.apply(tree)
        )

    except Exception as exc:
        raise RuntimeError(
            f"Erro na conversão da árvore {key}:\n{exc}"
        ) from exc

    if net is None:
        raise RuntimeError(
            f"Rede de Petri nula: {key}"
        )

    if initial_marking is None or final_marking is None:
        raise RuntimeError(
            f"Marcações ausentes: {key}"
        )

    if len(initial_marking) == 0:
        raise RuntimeError(
            f"Marcação inicial vazia: {key}"
        )

    if len(final_marking) == 0:
        raise RuntimeError(
            f"Marcação final vazia: {key}"
        )

    hmm25_petri_models[key] = {
        "net": net,
        "initial_marking": initial_marking,
        "final_marking": final_marking,
        "tree": tree,
        "source_ptml": str(ptml_map25[key])
    }

    transitions = list(net.transitions)

    visible = [
        t for t in transitions
        if t.label is not None
    ]

    silent = [
        t for t in transitions
        if t.label is None
    ]

    labels = sorted({
        str(t.label)
        for t in visible
    })

    conversion_rows25.append({
        "log_type": kind,
        "noise": noise,
        "n_places": len(net.places),
        "n_transitions": len(transitions),
        "n_visible_transitions": len(visible),
        "n_silent_transitions": len(silent),
        "n_arcs": len(net.arcs),
        "n_distinct_activities": len(labels),
        "n_initial_marked_places": len(initial_marking),
        "n_final_marked_places": len(final_marking),
        "conversion_status": "success"
    })

audit25_conversion = pd.DataFrame(
    conversion_rows25
).sort_values(
    ["log_type", "noise"]
).reset_index(drop=True)

_project_print("\n25.5 — CONVERSÃO CONCLUÍDA")
_project_display(audit25_conversion)


# ============================================================
# 25.6 — VERIFICAÇÃO DE INTEGRIDADE ESTRUTURAL
# ============================================================

def validate_petri25(net, im, fm):
    """
    Verifica integridade estrutural básica.

    Importante:
    esta função NÃO substitui verificação formal de
    soundness, boundedness ou conformance.
    """

    places = set(net.places)
    transitions = set(net.transitions)
    nodes = places | transitions

    invalid_arcs = []

    for arc in net.arcs:

        if arc.source not in nodes:
            invalid_arcs.append("invalid_source")

        if arc.target not in nodes:
            invalid_arcs.append("invalid_target")

        # Redes place/transition devem ser bipartidas.
        valid_bipartite = (
            (
                arc.source in places
                and arc.target in transitions
            )
            or
            (
                arc.source in transitions
                and arc.target in places
            )
        )

        if not valid_bipartite:
            invalid_arcs.append("non_bipartite_arc")

    initial_places_valid = all(
        p in places for p in im
    )

    final_places_valid = all(
        p in places for p in fm
    )

    initial_tokens_positive = all(
        int(v) > 0 for v in im.values()
    )

    final_tokens_positive = all(
        int(v) > 0 for v in fm.values()
    )

    return {
        "arcs_valid": len(invalid_arcs) == 0,
        "initial_places_valid": initial_places_valid,
        "final_places_valid": final_places_valid,
        "initial_tokens_positive": initial_tokens_positive,
        "final_tokens_positive": final_tokens_positive,
        "n_invalid_arcs": len(invalid_arcs)
    }


integrity_rows25 = []

for key, item in sorted(hmm25_petri_models.items()):

    result = validate_petri25(
        item["net"],
        item["initial_marking"],
        item["final_marking"]
    )

    integrity_rows25.append({
        "log_type": key[0],
        "noise": key[1],
        **result
    })

audit25_integrity = pd.DataFrame(integrity_rows25)

check_cols25 = [
    "arcs_valid",
    "initial_places_valid",
    "final_places_valid",
    "initial_tokens_positive",
    "final_tokens_positive"
]

if not audit25_integrity[check_cols25].all().all():
    _project_display(audit25_integrity)
    raise RuntimeError(
        "Falha na integridade estrutural de uma "
        "ou mais redes de Petri."
    )

_project_print("\n25.6 — Integridade estrutural básica: OK.")


# ============================================================
# 25.7 — EXPORTAÇÃO PNML
# ============================================================

export_rows25 = []

for key, item in sorted(hmm25_petri_models.items()):

    kind, noise = key
    encoded = f"{noise:.2f}".replace(".", "p")

    filename = (
        f"cell25_{kind}_noise_{encoded}.pnml"
    )

    destination = MODEL_DIR25 / filename

    try:
        pnml_exporter.apply(
            item["net"],
            item["initial_marking"],
            str(destination),
            final_marking=item["final_marking"]
        )

    except Exception as exc:
        raise RuntimeError(
            f"Falha na exportação PNML de {key}:\n"
            f"{exc}"
        ) from exc

    if not destination.is_file():
        raise RuntimeError(
            f"Arquivo PNML não criado: {destination}"
        )

    if destination.stat().st_size == 0:
        raise RuntimeError(
            f"Arquivo PNML vazio: {destination}"
        )

    item["pnml_path"] = str(destination)

    export_rows25.append({
        "log_type": kind,
        "noise": noise,
        "pnml_file": filename,
        "size_bytes": destination.stat().st_size,
        "export_status": "success"
    })

audit25_exports = pd.DataFrame(export_rows25)

_project_print("\n25.7 — Arquivos PNML exportados:",
      len(audit25_exports))

_project_display(audit25_exports)


# ============================================================
# 25.8 — VISUALIZAÇÃO DAS REDES DE PETRI
# ============================================================

# Diagramas principais: limiar de ruído 0.10.
SELECTED_NOISE25 = 0.10

hmm25_visualizations = {}

for kind in LOG_TYPES25:

    key = (kind, SELECTED_NOISE25)
    item = hmm25_petri_models[key]

    net = item["net"]
    im = item["initial_marking"]
    fm = item["final_marking"]

    encoded = (
        f"{SELECTED_NOISE25:.2f}"
        .replace(".", "p")
    )

    svg_path = (
        FIG_DIR25
        / f"cell25_petri_{kind}_noise_{encoded}.svg"
    )

    try:
        gviz = pn_visualizer.apply(
            net,
            im,
            fm,
            parameters={
                "format": "svg",
                "rankdir": "LR"
            }
        )

        pn_visualizer.save(
            gviz,
            str(svg_path)
        )

        if not svg_path.is_file():
            raise RuntimeError(
                "O arquivo SVG não foi criado."
            )

        hmm25_visualizations[key] = str(svg_path)

    except Exception as exc:
        warnings.warn(
            f"Não foi possível gerar o diagrama {key}: "
            f"{exc}"
        )

_project_print("\n25.8 — DIAGRAMAS DAS REDES DE PETRI")

for kind in LOG_TYPES25:

    key = (kind, SELECTED_NOISE25)

    if key not in hmm25_visualizations:
        continue

    _project_print(
        "\n",
        "=" * 65,
        f"\nREDE DE PETRI — {kind.upper()}",
        f"| noise={SELECTED_NOISE25:.2f}",
        "\n",
        "=" * 65
    )

    _project_display(
        SVG(filename=hmm25_visualizations[key])
    )


# ============================================================
# 25.9 — COMPLEXIDADE ESTRUTURAL
# ============================================================

audit25_complexity = audit25_conversion.copy()

audit25_complexity["transition_place_ratio"] = (
    audit25_complexity["n_transitions"]
    /
    audit25_complexity["n_places"].replace(0, np.nan)
)

audit25_complexity["silent_transition_ratio"] = (
    audit25_complexity["n_silent_transitions"]
    /
    audit25_complexity["n_transitions"].replace(0, np.nan)
)

audit25_complexity["arcs_per_node"] = (
    audit25_complexity["n_arcs"]
    /
    (
        audit25_complexity["n_places"]
        +
        audit25_complexity["n_transitions"]
    ).replace(0, np.nan)
)

fig, axes = plt.subplots(
    1, 3,
    figsize=(17, 4.8)
)

metrics25 = [
    ("n_places", "Lugares"),
    ("n_transitions", "Transições"),
    ("n_arcs", "Arcos")
]

for ax, (metric, title) in zip(axes, metrics25):

    for kind in LOG_TYPES25:

        subset = audit25_complexity[
            audit25_complexity["log_type"] == kind
        ].sort_values("noise")

        ax.plot(
            subset["noise"],
            subset[metric],
            marker="o",
            label=kind
        )

    ax.set_title(title)
    ax.set_xlabel("Limiar de ruído")
    ax.set_ylabel("Quantidade")
    ax.grid(alpha=0.25)
    ax.legend()

fig.suptitle(
    "Redes de Petri — complexidade estrutural por limiar",
    fontsize=14
)

plt.tight_layout()

complexity_figure25 = (
    FIG_DIR25
    / "cell25_petri_complexity.png"
)

plt.savefig(
    complexity_figure25,
    dpi=180,
    bbox_inches="tight"
)

plt.show()


# ============================================================
# 25.10 — TRANSIÇÕES VISÍVEIS E SILENCIOSAS
# ============================================================

fig, axes = plt.subplots(
    1, 3,
    figsize=(17, 4.8),
    sharey=False
)

for ax, kind in zip(axes, LOG_TYPES25):

    subset = audit25_complexity[
        audit25_complexity["log_type"] == kind
    ].sort_values("noise")

    x = np.arange(len(subset))

    ax.bar(
        x,
        subset["n_visible_transitions"],
        label="Visíveis"
    )

    ax.bar(
        x,
        subset["n_silent_transitions"],
        bottom=subset["n_visible_transitions"],
        label="Silenciosas (tau)"
    )

    ax.set_xticks(x)
    ax.set_xticklabels([
        f"{n:.2f}" for n in subset["noise"]
    ])

    ax.set_title(kind.upper())
    ax.set_xlabel("Limiar de ruído")
    ax.set_ylabel("Número de transições")
    ax.grid(axis="y", alpha=0.25)
    ax.legend()

fig.suptitle(
    "Composição das transições nas redes de Petri",
    fontsize=14
)

plt.tight_layout()

transition_figure25 = (
    FIG_DIR25
    / "cell25_petri_transition_composition.png"
)

plt.savefig(
    transition_figure25,
    dpi=180,
    bbox_inches="tight"
)

plt.show()


# ============================================================
# 25.11 — RELATÓRIOS E METADADOS
# ============================================================

_project_csv(audit25_conversion, 
    REPORT_DIR25 / "cell25_petri_conversion.csv",
    index=False
)

_project_csv(audit25_integrity, 
    REPORT_DIR25 / "cell25_petri_integrity.csv",
    index=False
)

_project_csv(audit25_exports, 
    REPORT_DIR25 / "cell25_petri_exports.csv",
    index=False
)

_project_csv(audit25_complexity, 
    REPORT_DIR25 / "cell25_petri_complexity.csv",
    index=False
)

hmm25_metadata = {
    "phase": 2,
    "cell": 25,
    "title": "Process trees to Petri nets",
    "pm4py_version": str(pm4py.__version__),
    "source_cell": 23,
    "source_directory": str(SOURCE_DIR25),
    "model_directory": str(MODEL_DIR25),
    "log_types": list(LOG_TYPES25),
    "noise_levels": list(NOISE_LEVELS25),
    "n_expected_models": len(expected_keys25),
    "n_imported_trees": len(hmm25_trees),
    "n_petri_nets": len(hmm25_petri_models),
    "n_exported_pnml": len(audit25_exports),
    "selected_visualization_noise": SELECTED_NOISE25,
    "hmm_retrained": False,
    "inductive_miner_rerun": False,
    "test_used": False,
    "co2_used": False,
    "formal_soundness_tested": False,
    "conformance_tested": False,
    "selection_performed": False,
    "next_cell": 26
}

metadata_path25 = (
    REPORT_DIR25
    / "cell25_petri_metadata.json"
)

with open(
    metadata_path25,
    "w",
    encoding="utf-8"
) as f:
    _project_dump(
        hmm25_metadata,
        f,
        ensure_ascii=False,
        indent=2
    )


# ============================================================
# 25.12 — RESUMO FINAL
# ============================================================

_project_print("\n" + "=" * 78)
_project_print("RESUMO — CÉLULA 25/36")
_project_print("=" * 78)

summary25 = audit25_complexity[
    audit25_complexity["noise"] == SELECTED_NOISE25
][[
    "log_type",
    "noise",
    "n_places",
    "n_transitions",
    "n_visible_transitions",
    "n_silent_transitions",
    "n_arcs",
    "n_distinct_activities"
]]

_project_display(summary25.reset_index(drop=True))

assert len(hmm25_trees) == 15
assert len(hmm25_petri_models) == 15
assert len(audit25_exports) == 15

assert audit25_exports[
    "export_status"
].eq("success").all()

_project_print("\nÁrvores PTML recuperadas:", len(hmm25_trees))
_project_print("Redes de Petri construídas:", len(hmm25_petri_models))
_project_print("Arquivos PNML exportados:", len(audit25_exports))
_project_print("Integridade estrutural básica: OK")

_project_print("\nHMM retreinado: NÃO")
_project_print("Inductive Miner reexecutado: NÃO")
_project_print("Conjunto de teste utilizado: NÃO")
_project_print("CO2 utilizado: NÃO")
_project_print("Seleção final realizada: NÃO")

_project_print("\nModelos:", MODEL_DIR25)
_project_print("Relatórios:", REPORT_DIR25)
_project_print("Figuras:", FIG_DIR25)

_project_print("\n" + "=" * 78)
_project_print("CÉLULA 25 CONCLUÍDA.")
_project_print("PRÓXIMA: CÉLULA 26/36 — FITNESS, PRECISION E COMPLEXIDADE")
_project_print("=" * 78)


## 26–29 — Conformidade, seleção e desvios

Avaliam-se fitness por alinhamento, precisão ETConformance por token, estrutura e complexidade. A Aula 07, pp. 107–115, distingue as dimensões de qualidade. Tax et al. explicam por que resultados de precisão dependem da medida. O F1, os limites de seleção e o compromisso entre medidas são escolhas diagnósticas do projeto. A avaliação na última partição é retrospectiva, feita com os modelos já escolhidos. [Tax et al. (2018)](https://doi.org/10.1016/j.ipl.2018.01.013) · [Aula 07, material da disciplina](#fontes-do-material-local)


In [ ]:

# ============================================================
# FASE 2 — CÉLULA 26/36
# FITNESS, PRECISION E COMPLEXIDADE
# VERSÃO REPRODUZÍVEL E OTIMIZADA — EXECUÇÃO LIMPA
# ============================================================
#
# PROJETO:
# Modelagem de emissões horárias de CO2
# Iatan — EPA ORISPL 6065 — Unit 2 — 2025
#
# OBJETIVOS:
# 1. Recuperar os seis Event Logs validados.
# 2. Importar as 15 redes PNML da Célula 25.
# 3. Identificar redes estruturalmente isomorfas.
# 4. Avaliar fitness por alinhamento.
# 5. Avaliar precision por ETConformance/token replay.
# 6. Comparar qualidade e complexidade.
# 7. Exportar resultados, gráficos e manifesto.
#
# REPRODUTIBILIDADE:
# - Não utiliza checkpoints anteriores.
# - Não depende de resultados da antiga Célula 26.
# - Não utiliza o conjunto de teste.
# - Não utiliza CO2 nas métricas.
# - Não retreina HMM.
# - Não redescobre modelos.
# - Usa apenas artefatos das células anteriores.
#
# ============================================================

import json
import time
import hashlib
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx
import pm4py

from pm4py.objects.log.obj import EventLog
from pm4py.objects.conversion.log import converter as log_converter
from pm4py.objects.petri_net.importer import importer as pnml_importer

from pm4py.algo.evaluation.replay_fitness import (
    algorithm as fitness_evaluator
)
from pm4py.algo.evaluation.precision import (
    algorithm as precision_evaluator
)

warnings.filterwarnings(
    "ignore",
    category=DeprecationWarning
)

_project_print("=" * 78)
_project_print("FASE 2 — CÉLULA 26/36")
_project_print("FITNESS, PRECISION E COMPLEXIDADE")
_project_print("EXECUÇÃO LIMPA E OTIMIZADA")
_project_print("=" * 78)

_project_print("PM4Py:", pm4py.__version__)


# ============================================================
# 26.1 — CONFIGURAÇÃO
# ============================================================

if "RUN_DIR" in globals():
    RUN_DIR26 = Path(RUN_DIR)

elif "RUN_DIR25" in globals():
    RUN_DIR26 = Path(RUN_DIR25)

else:
    raise RuntimeError(
        "RUN_DIR não encontrado. "
        "Execute as células anteriores."
    )

MODEL_DIR26 = RUN_DIR26 / "models" / "cell25"
REPORT_DIR26 = RUN_DIR26 / "reports"
FIG_DIR26 = RUN_DIR26 / "figures"

REPORT_DIR26.mkdir(
    parents=True,
    exist_ok=True
)

FIG_DIR26.mkdir(
    parents=True,
    exist_ok=True
)

LOG_TYPES26 = (
    "regime",
    "ramp",
    "combined"
)

PARTITIONS26 = (
    "train",
    "validation"
)

NOISE_LEVELS26 = (
    0.00,
    0.05,
    0.10,
    0.20,
    0.30
)

_project_print("\n26.1 — Diretórios")
_project_print("Execução:", RUN_DIR26)
_project_print("PNML:", MODEL_DIR26)


# ============================================================
# 26.2 — IMPORTAÇÃO DAS REDES PNML
# ============================================================

petri_models26 = {}
structure_rows26 = []

for log_type in LOG_TYPES26:

    for noise in NOISE_LEVELS26:

        tag = f"{noise:.2f}".replace(".", "p")

        filename = (
            f"cell25_{log_type}_noise_{tag}.pnml"
        )

        path = MODEL_DIR26 / filename

        if not path.exists():
            raise FileNotFoundError(
                f"PNML não encontrado: {path}"
            )

        net, im, fm = pnml_importer.apply(
            str(path)
        )

        key = (
            log_type,
            round(noise, 2)
        )

        petri_models26[key] = (
            net,
            im,
            fm
        )

        visible = [
            t for t in net.transitions
            if t.label is not None
        ]

        silent = [
            t for t in net.transitions
            if t.label is None
        ]

        structure_rows26.append({
            "log_type": log_type,
            "noise": noise,
            "n_places": len(net.places),
            "n_transitions": len(net.transitions),
            "n_visible_transitions": len(visible),
            "n_silent_transitions": len(silent),
            "n_arcs": len(net.arcs),
            "n_distinct_activities": len({
                t.label for t in visible
            }),
            "silent_fraction": (
                len(silent) / len(net.transitions)
            ),
            "pnml_file": filename,
            "pnml_sha256": hashlib.sha256(
                path.read_bytes()
            ).hexdigest()
        })

audit26_structure = pd.DataFrame(
    structure_rows26
)

_project_print(
    "\n26.2 — Redes importadas:",
    len(petri_models26)
)

_project_display(
    audit26_structure[
        [
            "log_type",
            "noise",
            "n_places",
            "n_transitions",
            "n_arcs"
        ]
    ]
)


# ============================================================
# 26.3 — RECUPERAÇÃO DOS EVENT LOGS
# ============================================================

def normalize_partition26(value):

    aliases = {
        "train": "train",
        "training": "train",
        "treinamento": "train",
        "validation": "validation",
        "val": "validation",
        "validacao": "validation",
        "validação": "validation"
    }

    return aliases.get(
        str(value).strip().lower()
    )


def normalize_type26(value):

    value = str(value).strip().lower()

    return (
        value
        if value in LOG_TYPES26
        else None
    )


def to_event_log26(obj):

    if isinstance(obj, EventLog):
        return obj

    if not isinstance(obj, pd.DataFrame):
        return None

    df = obj.copy()

    if "case:concept:name" not in df:

        if "case_id" not in df:
            return None

        df["case:concept:name"] = (
            df["case_id"].astype(str)
        )

    if "concept:name" not in df:

        if "activity" not in df:
            return None

        df["concept:name"] = (
            df["activity"].astype(str)
        )

    if "time:timestamp" not in df:

        source = next(
            (
                c for c in (
                    "timestamp",
                    "start_timestamp"
                )
                if c in df
            ),
            None
        )

        if source is None:
            return None

        df["time:timestamp"] = df[source]

    df["time:timestamp"] = pd.to_datetime(
        df["time:timestamp"],
        utc=True,
        errors="raise"
    )

    df = df.sort_values(
        ["case:concept:name", "time:timestamp"],
        kind="stable"
    )

    return log_converter.apply(
        df,
        variant=log_converter.Variants.TO_EVENT_LOG
    )


def extract_logs26(obj):

    found = {}

    if not isinstance(obj, dict):
        return found

    for outer_key, outer_value in obj.items():

        if (
            isinstance(outer_key, tuple)
            and len(outer_key) == 2
        ):

            a, b = outer_key

            lt = normalize_type26(a)
            pt = normalize_partition26(b)

            if lt is None or pt is None:

                lt = normalize_type26(b)
                pt = normalize_partition26(a)

            if lt is not None and pt is not None:

                log = to_event_log26(
                    outer_value
                )

                if log is not None:
                    found[(lt, pt)] = log

        elif isinstance(outer_value, dict):

            outer_lt = normalize_type26(
                outer_key
            )

            outer_pt = normalize_partition26(
                outer_key
            )

            for inner_key, value in (
                outer_value.items()
            ):

                lt = (
                    outer_lt
                    or normalize_type26(inner_key)
                )

                pt = (
                    outer_pt
                    or normalize_partition26(inner_key)
                )

                if lt is None or pt is None:
                    continue

                log = to_event_log26(value)

                if log is not None:
                    found[(lt, pt)] = log

    return found


LOG_SOURCES26 = [
    "hmm20_validated_logs",
    "hmm19_event_logs",
    "hmm19_logs",
    "hmm21_logs",
    "hmm23_pm4py_logs",
    "LOGS20",
    "logs22",
    "LOGS24",
    "EVENT_LOGS"
]

event_logs26 = {}
log_sources26 = {}

for source_name in LOG_SOURCES26:

    if source_name not in globals():
        continue

    candidates = extract_logs26(
        globals()[source_name]
    )

    for key, log in candidates.items():

        if key not in event_logs26:

            event_logs26[key] = log
            log_sources26[key] = source_name


EXPECTED26 = {
    ("regime", "train"): (27, 843),
    ("ramp", "train"): (27, 1111),
    ("combined", "train"): (27, 1611),
    ("regime", "validation"): (6, 147),
    ("ramp", "validation"): (6, 162),
    ("combined", "validation"): (6, 259)
}

log_rows26 = []

for key, expected in EXPECTED26.items():

    if key not in event_logs26:

        raise RuntimeError(
            f"Event Log ausente: {key}"
        )

    log = event_logs26[key]

    observed = (
        len(log),
        sum(len(trace) for trace in log)
    )

    if observed != expected:

        raise RuntimeError(
            f"Integridade divergente: {key} "
            f"{observed} != {expected}"
        )

    log_rows26.append({
        "log_type": key[0],
        "partition": key[1],
        "n_cases": observed[0],
        "n_events": observed[1],
        "source": log_sources26[key]
    })

audit26_logs = pd.DataFrame(
    log_rows26
)

_project_print("\n26.3 — Event Logs")
_project_display(audit26_logs)


# ============================================================
# 26.4 — ISOMORFISMO E REUTILIZAÇÃO SEGURA
# ============================================================

def petri_to_graph26(net, im, fm):

    G = nx.DiGraph()

    for place in net.places:

        G.add_node(
            place,
            kind="place",
            label="",
            initial_tokens=int(
                im.get(place, 0)
            ),
            final_tokens=int(
                fm.get(place, 0)
            )
        )

    for transition in net.transitions:

        G.add_node(
            transition,
            kind="transition",
            label=(
                transition.label
                if transition.label is not None
                else "__TAU__"
            ),
            initial_tokens=0,
            final_tokens=0
        )

    for arc in net.arcs:

        G.add_edge(
            arc.source,
            arc.target,
            weight=int(arc.weight)
        )

    return G


node_match26 = (
    nx.algorithms.isomorphism
    .categorical_node_match(
        [
            "kind",
            "label",
            "initial_tokens",
            "final_tokens"
        ],
        [
            None,
            None,
            0,
            0
        ]
    )
)

edge_match26 = (
    nx.algorithms.isomorphism
    .categorical_edge_match(
        "weight",
        1
    )
)

graphs26 = {
    key: petri_to_graph26(*model)
    for key, model in petri_models26.items()
}

representatives26 = {}
isomorphism_rows26 = []

for log_type in LOG_TYPES26:

    groups = []

    for noise in NOISE_LEVELS26:

        key = (
            log_type,
            round(noise, 2)
        )

        graph = graphs26[key]

        matched = None

        for group in groups:

            representative = group[0]

            if nx.is_isomorphic(
                graph,
                graphs26[representative],
                node_match=node_match26,
                edge_match=edge_match26
            ):

                matched = group
                break

        if matched is None:

            groups.append([key])
            representatives26[key] = key

        else:

            matched.append(key)
            representatives26[key] = matched[0]

    for group in groups:

        isomorphism_rows26.append({
            "log_type": log_type,
            "representative_noise": group[0][1],
            "member_noises": [
                key[1] for key in group
            ],
            "group_size": len(group)
        })

audit26_isomorphism = pd.DataFrame(
    isomorphism_rows26
)

_project_print("\n26.4 — Redes estruturalmente distintas")
_project_display(audit26_isomorphism)


# ============================================================
# 26.5 — MÉTODOS DE CONFORMIDADE
# ============================================================

# Fitness: alignment-based.
#
# Precision: ETConformance baseada em token replay.
#
# Essa variante evita o custo elevado de alinhamentos
# completos na avaliação da precisão.
#
# Os métodos são aplicados uniformemente a todos
# os tipos de log e limiares de ruído.

FITNESS_VARIANT26 = (
    fitness_evaluator.Variants.ALIGNMENT_BASED
)

PRECISION_VARIANT26 = (
    precision_evaluator.Variants.ETCONFORMANCE_TOKEN
)

_project_print("\n26.5 — Métodos")
_project_print("Fitness: alignment-based")
_project_print("Precision: ETConformance token-based")


# ============================================================
# 26.6 — FUNÇÃO DE AVALIAÇÃO
# ============================================================

def evaluate_model26(log, net, im, fm):

    start = time.perf_counter()

    fitness_result = fitness_evaluator.apply(
        log,
        net,
        im,
        fm,
        variant=FITNESS_VARIANT26
    )

    precision = precision_evaluator.apply(
        log,
        net,
        im,
        fm,
        variant=PRECISION_VARIANT26
    )

    elapsed = (
        time.perf_counter() - start
    )

    fitness = float(
        fitness_result["log_fitness"]
    )

    precision = float(precision)

    if not (
        np.isfinite(fitness)
        and np.isfinite(precision)
        and 0 <= fitness <= 1
        and 0 <= precision <= 1
    ):

        raise ValueError(
            "Métricas inválidas."
        )

    f1 = (
        2 * fitness * precision
        / (fitness + precision)
        if fitness + precision > 0
        else 0.0
    )

    return {
        "fitness_log": fitness,
        "fitness_avg_trace": (
            fitness_result.get(
                "average_trace_fitness",
                np.nan
            )
        ),
        "percentage_fitting_traces": (
            fitness_result.get(
                "percentage_of_fitting_traces",
                np.nan
            )
        ),
        "precision": precision,
        "f1_fitness_precision": f1,
        "elapsed_seconds": elapsed
    }


# ============================================================
# 26.7 — AVALIAÇÃO LIMPA
# ============================================================

# Nenhum checkpoint antigo é lido.
#
# Cada rede estruturalmente distinta é avaliada
# uma vez por partição.
#
# Redes isomorfas recebem os mesmos resultados,
# com registro explícito da origem.

results26 = []
computed26 = {}

for log_type in LOG_TYPES26:

    for noise in NOISE_LEVELS26:

        noise = round(noise, 2)

        representative = representatives26[
            (log_type, noise)
        ]

        for partition in PARTITIONS26:

            cache_key = (
                representative,
                partition
            )

            if cache_key not in computed26:

                net, im, fm = petri_models26[
                    representative
                ]

                log = event_logs26[
                    (log_type, partition)
                ]

                _project_print(
                    f"\nAvaliando {log_type.upper()} | "
                    f"noise={representative[1]:.2f} | "
                    f"{partition.upper()}",
                    flush=True
                )

                computed26[cache_key] = (
                    evaluate_model26(
                        log,
                        net,
                        im,
                        fm
                    )
                )

                source = "computed"

            else:

                source = "isomorphic_reuse"

            metrics = computed26[cache_key]

            structure = audit26_structure[
                (
                    audit26_structure["log_type"]
                    == log_type
                )
                &
                (
                    audit26_structure["noise"]
                    == noise
                )
            ].iloc[0]

            log = event_logs26[
                (log_type, partition)
            ]

            results26.append({
                "log_type": log_type,
                "noise": noise,
                "partition": partition,
                "n_cases": len(log),
                "n_events": sum(
                    len(trace)
                    for trace in log
                ),
                **metrics,
                "n_places": int(
                    structure["n_places"]
                ),
                "n_transitions": int(
                    structure["n_transitions"]
                ),
                "n_arcs": int(
                    structure["n_arcs"]
                ),
                "n_silent_transitions": int(
                    structure["n_silent_transitions"]
                ),
                "silent_fraction": float(
                    structure["silent_fraction"]
                ),
                "metric_source": source,
                "representative_noise": (
                    representative[1]
                ),
                "fitness_method": (
                    "alignment_based"
                ),
                "precision_method": (
                    "etconformance_token"
                )
            })

            _project_print(
                f"{log_type} | noise={noise:.2f} | "
                f"{partition} | "
                f"Fitness={metrics['fitness_log']:.4f} | "
                f"Precision={metrics['precision']:.4f} | "
                f"Fonte={source}",
                flush=True
            )


# ============================================================
# 26.8 — CONSOLIDAÇÃO
# ============================================================

audit26_metrics = pd.DataFrame(
    results26
)

audit26_metrics = (
    audit26_metrics
    .sort_values(
        [
            "log_type",
            "noise",
            "partition"
        ]
    )
    .reset_index(drop=True)
)

if len(audit26_metrics) != 30:

    raise RuntimeError(
        "Esperadas 30 avaliações."
    )

_project_print("\n26.8 — Métricas")
_project_display(
    audit26_metrics[
        [
            "log_type",
            "noise",
            "partition",
            "fitness_log",
            "precision",
            "f1_fitness_precision",
            "n_arcs",
            "metric_source"
        ]
    ].round(4)
)


# ============================================================
# 26.9 — TRAIN × VALIDATION
# ============================================================

comparison26 = audit26_metrics.pivot(
    index=[
        "log_type",
        "noise"
    ],
    columns="partition",
    values=[
        "fitness_log",
        "precision",
        "f1_fitness_precision"
    ]
)

comparison26.columns = [
    f"{metric}_{partition}"
    for metric, partition
    in comparison26.columns
]

comparison26 = (
    comparison26.reset_index()
)

for metric in [
    "fitness_log",
    "precision",
    "f1_fitness_precision"
]:

    comparison26[
        f"{metric}_gap_val_minus_train"
    ] = (
        comparison26[
            f"{metric}_validation"
        ]
        - comparison26[
            f"{metric}_train"
        ]
    )

comparison26 = comparison26.merge(
    audit26_structure.drop(
        columns=["pnml_sha256"]
    ),
    on=[
        "log_type",
        "noise"
    ],
    how="left"
)

_project_print("\n26.9 — Comparação")
_project_display(
    comparison26.round(4)
)


# ============================================================
# 26.10 — FRONTEIRA DE PARETO
# ============================================================

pareto_rows26 = []

for log_type in LOG_TYPES26:

    data = comparison26[
        comparison26["log_type"]
        == log_type
    ]

    for _, candidate in data.iterrows():

        dominated = False

        for _, other in data.iterrows():

            no_worse = (
                other[
                    "f1_fitness_precision_validation"
                ]
                >= candidate[
                    "f1_fitness_precision_validation"
                ]
                and
                other["n_arcs"]
                <= candidate["n_arcs"]
            )

            strictly_better = (
                other[
                    "f1_fitness_precision_validation"
                ]
                > candidate[
                    "f1_fitness_precision_validation"
                ]
                or
                other["n_arcs"]
                < candidate["n_arcs"]
            )

            if no_worse and strictly_better:
                dominated = True
                break

        row = candidate.to_dict()
        row["pareto_efficient"] = (
            not dominated
        )

        pareto_rows26.append(row)

audit26_pareto = pd.DataFrame(
    pareto_rows26
)

_project_print("\n26.10 — Pareto")
_project_display(
    audit26_pareto[
        [
            "log_type",
            "noise",
            "fitness_log_validation",
            "precision_validation",
            "f1_fitness_precision_validation",
            "n_arcs",
            "pareto_efficient"
        ]
    ].round(4)
)


# ============================================================
# 26.11 — GRÁFICOS
# ============================================================

fig, axes = plt.subplots(
    1,
    3,
    figsize=(18, 5),
    sharey=True
)

for ax, log_type in zip(
    axes,
    LOG_TYPES26
):

    data = audit26_metrics[
        (
            audit26_metrics["log_type"]
            == log_type
        )
        &
        (
            audit26_metrics["partition"]
            == "validation"
        )
    ].sort_values("noise")

    ax.plot(
        data["noise"],
        data["fitness_log"],
        marker="o",
        label="Fitness"
    )

    ax.plot(
        data["noise"],
        data["precision"],
        marker="s",
        label="Precision"
    )

    ax.plot(
        data["noise"],
        data["f1_fitness_precision"],
        marker="^",
        label="F1"
    )

    ax.set_title(
        log_type.upper()
    )

    ax.set_xlabel(
        "Limiar de ruído"
    )

    ax.set_ylim(
        0,
        1.05
    )

    ax.grid(
        alpha=0.25
    )

    ax.legend()

axes[0].set_ylabel(
    "Métricas de validação"
)

plt.tight_layout()

fig.savefig(
    FIG_DIR26 /
    "cell26_validation_metrics.png",
    dpi=180,
    bbox_inches="tight"
)

plt.show()


fig, ax = plt.subplots(
    figsize=(10, 6)
)

for log_type in LOG_TYPES26:

    data = audit26_pareto[
        audit26_pareto["log_type"]
        == log_type
    ]

    ax.scatter(
        data["n_arcs"],
        data[
            "f1_fitness_precision_validation"
        ],
        s=90,
        label=log_type
    )

    for _, row in data.iterrows():

        ax.annotate(
            f'{row["noise"]:.2f}',
            (
                row["n_arcs"],
                row[
                    "f1_fitness_precision_validation"
                ]
            ),
            xytext=(5, 5),
            textcoords="offset points",
            fontsize=8
        )

ax.set_xlabel(
    "Número de arcos"
)

ax.set_ylabel(
    "F1 na validação"
)

ax.set_title(
    "Qualidade × complexidade"
)

ax.grid(
    alpha=0.25
)

ax.legend()

plt.tight_layout()

fig.savefig(
    FIG_DIR26 /
    "cell26_quality_complexity.png",
    dpi=180,
    bbox_inches="tight"
)

plt.show()


# ============================================================
# 26.12 — EXPORTAÇÃO
# ============================================================

_project_csv(audit26_structure, 
    REPORT_DIR26 /
    "cell26_petri_structure.csv",
    index=False
)

_project_csv(audit26_logs, 
    REPORT_DIR26 /
    "cell26_event_log_sources.csv",
    index=False
)

_project_csv(audit26_isomorphism, 
    REPORT_DIR26 /
    "cell26_isomorphism_groups.csv",
    index=False
)

_project_csv(audit26_metrics, 
    REPORT_DIR26 /
    "cell26_fitness_precision_complexity.csv",
    index=False
)

_project_csv(comparison26, 
    REPORT_DIR26 /
    "cell26_train_validation_comparison.csv",
    index=False
)

_project_csv(audit26_pareto, 
    REPORT_DIR26 /
    "cell26_pareto.csv",
    index=False
)


# ============================================================
# 26.13 — MANIFESTO
# ============================================================

manifest26 = {
    "phase": 2,
    "cell": 26,
    "facility": "Iatan",
    "orispl": 6065,
    "unit": "2",
    "year": 2025,
    "pm4py_version": pm4py.__version__,
    "log_types": list(LOG_TYPES26),
    "partitions": list(PARTITIONS26),
    "noise_levels": list(NOISE_LEVELS26),
    "n_models": len(petri_models26),
    "n_isomorphism_groups": len(
        audit26_isomorphism
    ),
    "n_results": len(
        audit26_metrics
    ),
    "fitness_method": (
        "alignment_based"
    ),
    "precision_method": (
        "etconformance_token"
    ),
    "checkpoint_used": False,
    "test_used": False,
    "co2_used": False,
    "hmm_retrained": False,
    "process_rediscovered": False,
    "pnml_sha256": {
        f"{k[0]}_{k[1]:.2f}": hashlib.sha256(
            (
                MODEL_DIR26 /
                (
                    f"cell25_{k[0]}_noise_"
                    f"{k[1]:.2f}".replace(".", "p")
                    + ".pnml"
                )
            ).read_bytes()
        ).hexdigest()
        for k in petri_models26
    }
}

with open(
    REPORT_DIR26 /
    "cell26_reproducibility_manifest.json",
    "w",
    encoding="utf-8"
) as f:

    _project_dump(
        manifest26,
        f,
        indent=2,
        ensure_ascii=False
    )


# ============================================================
# 26.14 — RESUMO FINAL
# ============================================================

_project_print("\n" + "=" * 78)
_project_print("CÉLULA 26 CONCLUÍDA")
_project_print("=" * 78)

_project_print(
    "Redes PNML:",
    len(petri_models26)
)

_project_print(
    "Event Logs:",
    len(event_logs26)
)

_project_print(
    "Redes estruturalmente distintas:",
    len(audit26_isomorphism)
)

_project_print(
    "Avaliações consolidadas:",
    len(audit26_metrics)
)

_project_print("\nTeste utilizado: NÃO")
_project_print("CO2 utilizado: NÃO")
_project_print("Checkpoint utilizado: NÃO")

_project_print("\nRelatórios:", REPORT_DIR26)
_project_print("Figuras:", FIG_DIR26)

_project_print("\nPRÓXIMA ETAPA:")
_project_print(
    "CÉLULA 27/36 — SELEÇÃO E CONGELAMENTO "
    "DO MODELO DE PROCESSO"
)


In [ ]:

# ============================================================
# FASE 2 — CÉLULA 27/36
# SELEÇÃO, AUDITORIA E CONGELAMENTO DOS MODELOS DE PROCESSO
# ============================================================
#
# PROJETO:
# Modelagem das emissões horárias de CO2 de unidade
# termelétrica utilizando dados CEMS da EPA.
#
# UNIDADE:
# Iatan — ORISPL 6065 — Unit 2 — Ano 2025
#
# OBJETIVOS:
#
# 1. Carregar os resultados da Célula 26.
# 2. Validar a integridade dos arquivos e das métricas.
# 3. Selecionar modelos exclusivamente por validação.
# 4. Aplicar critérios explícitos e determinísticos.
# 5. Congelar os três modelos de processo.
# 6. Registrar hashes, critérios e justificativas.
# 7. Exportar relatórios e manifesto de reprodução.
#
# REGRAS METODOLÓGICAS:
#
# - Não utiliza o conjunto de teste.
# - Não utiliza emissões de CO2.
# - Não modifica o HMM.
# - Não redescobre redes de Petri.
# - Não recalcula fitness ou precision.
# - Não reutiliza checkpoints da Célula 26.
# - Não utiliza resultados de previsão.
# - Seleção separada por tipo de representação.
#
# CRITÉRIO DE SELEÇÃO:
#
# A. Fitness de validação >= 0.99.
# B. Maior F1 de validação entre elegíveis.
# C. Tolerância absoluta de F1 = 0.005.
# D. Entre modelos dentro da tolerância,
#    escolher o de menor número de arcos.
# E. Desempate: menor número de transições,
#    menor noise e nome do PNML.
#
# ATENÇÃO:
# F1 é a média harmônica entre fitness e precision,
# não a métrica F1 de classificação.
#
# ============================================================

import json
import hashlib
import shutil
import platform
import sys
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import pm4py

from pm4py.objects.petri_net.importer import (
    importer as pnml_importer
)

_project_print("=" * 78)
_project_print("FASE 2 — CÉLULA 27/36")
_project_print("SELEÇÃO E CONGELAMENTO DOS MODELOS DE PROCESSO")
_project_print("=" * 78)


# ============================================================
# 27.1 — CONFIGURAÇÃO E DIRETÓRIOS
# ============================================================

if "RUN_DIR" in globals():
    RUN_DIR27 = Path(RUN_DIR)

elif "RUN_DIR26" in globals():
    RUN_DIR27 = Path(RUN_DIR26)

else:
    raise RuntimeError(
        "RUN_DIR não encontrado. "
        "Execute as células anteriores."
    )

RUN_DIR27 = RUN_DIR27.expanduser().resolve()

MODEL_DIR27 = RUN_DIR27 / "models" / "cell25"
REPORT_DIR27 = RUN_DIR27 / "reports"
FIG_DIR27 = RUN_DIR27 / "figures"

FROZEN_DIR27 = (
    RUN_DIR27 / "models" / "cell27_frozen"
)

REPORT_DIR27.mkdir(
    parents=True,
    exist_ok=True
)

FROZEN_DIR27.mkdir(
    parents=True,
    exist_ok=True
)

LOG_TYPES27 = (
    "regime",
    "ramp",
    "combined"
)

NOISE_LEVELS27 = (
    0.00,
    0.05,
    0.10,
    0.20,
    0.30
)

MIN_VALIDATION_FITNESS27 = 0.99

F1_TOLERANCE27 = 0.005

SELECTION_POLICY27 = (
    "validation_fitness_ge_0.99__"
    "f1_within_0.005_of_best__"
    "min_arcs_then_transitions_then_noise"
)

_project_print("\n27.1 — Configuração")
_project_print("RUN_DIR:", RUN_DIR27)
_project_print("Modelos de origem:", MODEL_DIR27)
_project_print("Modelos congelados:", FROZEN_DIR27)
_project_print("Fitness mínimo:", MIN_VALIDATION_FITNESS27)
_project_print("Tolerância F1:", F1_TOLERANCE27)


# ============================================================
# 27.2 — FUNÇÕES DE INTEGRIDADE
# ============================================================

def sha256_file27(path, chunk_size=1024 * 1024):

    digest = hashlib.sha256()

    with open(path, "rb") as f:

        while True:

            chunk = f.read(chunk_size)

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


def ensure_finite27(df, columns):

    for col in columns:

        if col not in df.columns:

            raise RuntimeError(
                f"Coluna obrigatória ausente: {col}"
            )

        values = pd.to_numeric(
            df[col],
            errors="coerce"
        )

        if not np.isfinite(values.to_numpy()).all():

            raise RuntimeError(
                f"Valores inválidos na coluna: {col}"
            )

        df[col] = values

    return df


def json_ready27(value):

    if isinstance(value, Path):
        return str(value)

    if isinstance(value, (np.integer,)):
        return int(value)

    if isinstance(value, (np.floating,)):
        return float(value)

    if isinstance(value, (np.bool_,)):
        return bool(value)

    if isinstance(value, dict):
        return {
            str(k): json_ready27(v)
            for k, v in value.items()
        }

    if isinstance(value, (list, tuple)):
        return [
            json_ready27(v)
            for v in value
        ]

    return value


# ============================================================
# 27.3 — CARREGAMENTO DOS RESULTADOS DA CÉLULA 26
# ============================================================

METRICS_PATH27 = (
    REPORT_DIR27 /
    "cell26_fitness_precision_complexity.csv"
)

STRUCTURE_PATH27 = (
    REPORT_DIR27 /
    "cell26_petri_structure.csv"
)

MANIFEST26_PATH27 = (
    REPORT_DIR27 /
    "cell26_reproducibility_manifest.json"
)

for path in (
    METRICS_PATH27,
    STRUCTURE_PATH27,
    MANIFEST26_PATH27
):

    if not path.is_file():

        raise FileNotFoundError(
            f"Artefato obrigatório ausente: {path}"
        )

metrics27 = pd.read_csv(
    METRICS_PATH27
)

structure27 = pd.read_csv(
    STRUCTURE_PATH27
)

with open(
    MANIFEST26_PATH27,
    "r",
    encoding="utf-8"
) as f:

    manifest26_for27 = json.load(f)

_project_print("\n27.3 — Arquivos carregados")
_project_print("Avaliações:", len(metrics27))
_project_print("Estruturas:", len(structure27))


# ============================================================
# 27.4 — VALIDAÇÃO DO PROTOCOLO DA CÉLULA 26
# ============================================================

assert manifest26_for27.get(
    "test_used"
) is False, (
    "Manifesto indica utilização do teste."
)

assert manifest26_for27.get(
    "co2_used"
) is False, (
    "Manifesto indica utilização de CO2."
)

assert manifest26_for27.get(
    "checkpoint_used"
) is False, (
    "Manifesto indica uso de checkpoint."
)

if manifest26_for27.get(
    "fitness_method"
) != "alignment_based":

    raise RuntimeError(
        "Método de fitness inesperado."
    )

if manifest26_for27.get(
    "precision_method"
) != "etconformance_token":

    raise RuntimeError(
        "Método de precision inesperado."
    )

required_metrics27 = [
    "log_type",
    "noise",
    "partition",
    "fitness_log",
    "precision",
    "f1_fitness_precision",
    "n_places",
    "n_transitions",
    "n_arcs",
    "metric_source",
    "representative_noise",
    "fitness_method",
    "precision_method"
]

missing27 = [
    col
    for col in required_metrics27
    if col not in metrics27.columns
]

if missing27:

    raise RuntimeError(
        f"Colunas ausentes: {missing27}"
    )

metrics27 = ensure_finite27(
    metrics27,
    [
        "noise",
        "fitness_log",
        "precision",
        "f1_fitness_precision",
        "n_places",
        "n_transitions",
        "n_arcs",
        "representative_noise"
    ]
)

metrics27["log_type"] = (
    metrics27["log_type"]
    .astype(str)
    .str.strip()
    .str.lower()
)

metrics27["partition"] = (
    metrics27["partition"]
    .astype(str)
    .str.strip()
    .str.lower()
)

metrics27["noise"] = (
    metrics27["noise"].round(2)
)

metrics27["representative_noise"] = (
    metrics27["representative_noise"].round(2)
)

expected_keys27 = {
    (lt, noise, partition)
    for lt in LOG_TYPES27
    for noise in NOISE_LEVELS27
    for partition in ("train", "validation")
}

observed_keys27 = list(
    zip(
        metrics27["log_type"],
        metrics27["noise"],
        metrics27["partition"]
    )
)

if (
    len(metrics27) != 30
    or len(set(observed_keys27)) != 30
    or set(observed_keys27) != expected_keys27
):

    raise RuntimeError(
        "Grade de avaliações da Célula 26 "
        "incompleta, duplicada ou inconsistente."
    )

for col in (
    "fitness_log",
    "precision",
    "f1_fitness_precision"
):

    if not metrics27[col].between(
        0,
        1
    ).all():

        raise RuntimeError(
            f"Métrica fora do intervalo [0,1]: {col}"
        )

expected_f127 = np.where(
    (
        metrics27["fitness_log"]
        + metrics27["precision"]
    ) > 0,
    (
        2
        * metrics27["fitness_log"]
        * metrics27["precision"]
    )
    / (
        metrics27["fitness_log"]
        + metrics27["precision"]
    ).replace(0, np.nan),
    0.0
)

if not np.allclose(
    metrics27["f1_fitness_precision"],
    expected_f127,
    rtol=1e-6,
    atol=1e-6
):

    raise RuntimeError(
        "F1 inconsistente com fitness e precision."
    )

if not (
    metrics27["fitness_method"]
    == "alignment_based"
).all():

    raise RuntimeError(
        "Fitness calculado com métodos diferentes."
    )

if not (
    metrics27["precision_method"]
    == "etconformance_token"
).all():

    raise RuntimeError(
        "Precision calculada com métodos diferentes."
    )

_project_print("\n27.4 — Protocolo validado")
_project_print("Grade completa: SIM")
_project_print("Métodos consistentes: SIM")
_project_print("Teste utilizado: NÃO")
_project_print("CO2 utilizado: NÃO")


# ============================================================
# 27.5 — AUDITORIA DOS ARQUIVOS PNML
# ============================================================

required_structure27 = [
    "log_type",
    "noise",
    "n_places",
    "n_transitions",
    "n_arcs",
    "pnml_file",
    "pnml_sha256"
]

for col in required_structure27:

    if col not in structure27.columns:

        raise RuntimeError(
            f"Coluna estrutural ausente: {col}"
        )

structure27["noise"] = (
    pd.to_numeric(
        structure27["noise"],
        errors="raise"
    ).round(2)
)

if (
    len(structure27) != 15
    or structure27.duplicated(
        ["log_type", "noise"]
    ).any()
):

    raise RuntimeError(
        "Tabela estrutural inválida."
    )

pnml_audit_rows27 = []

for _, row in structure27.iterrows():

    log_type = str(
        row["log_type"]
    ).lower()

    noise = round(
        float(row["noise"]),
        2
    )

    filename = str(
        row["pnml_file"]
    )

    if Path(filename).name != filename:

        raise RuntimeError(
            "Nome PNML contém caminho inesperado."
        )

    source_path = (
        MODEL_DIR27 / filename
    )

    if not source_path.is_file():

        raise FileNotFoundError(
            f"PNML ausente: {source_path}"
        )

    actual_hash = sha256_file27(
        source_path
    )

    expected_hash = str(
        row["pnml_sha256"]
    ).lower()

    if actual_hash != expected_hash:

        raise RuntimeError(
            f"Hash divergente: {filename}"
        )

    manifest_key = (
        f"{log_type}_{noise:.2f}"
    )

    manifest_hash = (
        manifest26_for27
        .get("pnml_sha256", {})
        .get(manifest_key)
    )

    if manifest_hash != actual_hash:

        raise RuntimeError(
            "Hash divergente no manifesto "
            f"da Célula 26: {filename}"
        )

    net, im, fm = pnml_importer.apply(
        str(source_path)
    )

    observed_structure = {
        "n_places": len(net.places),
        "n_transitions": len(net.transitions),
        "n_arcs": len(net.arcs)
    }

    for field, observed in (
        observed_structure.items()
    ):

        if observed != int(row[field]):

            raise RuntimeError(
                f"Estrutura divergente em {filename}: "
                f"{field}"
            )

    pnml_audit_rows27.append({
        "log_type": log_type,
        "noise": noise,
        "pnml_file": filename,
        "sha256": actual_hash,
        **observed_structure,
        "integrity_ok": True
    })

audit27_pnml = pd.DataFrame(
    pnml_audit_rows27
)

_project_print("\n27.5 — Integridade PNML")
_project_print(
    "Arquivos auditados:",
    len(audit27_pnml)
)

_project_print(
    "Hashes válidos:",
    audit27_pnml["integrity_ok"].all()
)


# ============================================================
# 27.6 — PREPARAÇÃO DA TABELA DE SELEÇÃO
# ============================================================

validation27 = (
    metrics27[
        metrics27["partition"]
        == "validation"
    ]
    .copy()
    .reset_index(drop=True)
)

training27 = (
    metrics27[
        metrics27["partition"]
        == "train"
    ][
        [
            "log_type",
            "noise",
            "fitness_log",
            "precision",
            "f1_fitness_precision"
        ]
    ]
    .rename(
        columns={
            "fitness_log": "fitness_train",
            "precision": "precision_train",
            "f1_fitness_precision": "f1_train"
        }
    )
)

candidates27 = validation27.merge(
    training27,
    on=["log_type", "noise"],
    how="left",
    validate="one_to_one"
)

candidates27 = candidates27.rename(
    columns={
        "fitness_log": "fitness_validation",
        "precision": "precision_validation",
        "f1_fitness_precision": "f1_validation"
    }
)

candidates27["fitness_gap"] = (
    candidates27["fitness_validation"]
    - candidates27["fitness_train"]
)

candidates27["precision_gap"] = (
    candidates27["precision_validation"]
    - candidates27["precision_train"]
)

candidates27["f1_gap"] = (
    candidates27["f1_validation"]
    - candidates27["f1_train"]
)

candidates27["eligible_fitness"] = (
    candidates27["fitness_validation"]
    >= MIN_VALIDATION_FITNESS27
)

_project_print("\n27.6 — Candidatos")
_project_display(
    candidates27[
        [
            "log_type",
            "noise",
            "fitness_validation",
            "precision_validation",
            "f1_validation",
            "n_arcs",
            "eligible_fitness"
        ]
    ].sort_values(
        ["log_type", "noise"]
    ).round(5)
)


# ============================================================
# 27.7 — SELEÇÃO DETERMINÍSTICA
# ============================================================

selected_rows27 = []
decision_rows27 = []

for log_type in LOG_TYPES27:

    group = candidates27[
        candidates27["log_type"]
        == log_type
    ].copy()

    eligible = group[
        group["eligible_fitness"]
    ].copy()

    if eligible.empty:

        raise RuntimeError(
            f"Nenhum modelo {log_type} atende "
            "ao fitness mínimo. "
            "Não será aplicado fallback silencioso."
        )

    best_f1 = float(
        eligible["f1_validation"].max()
    )

    eligible["within_f1_tolerance"] = (
        eligible["f1_validation"]
        >= best_f1 - F1_TOLERANCE27 - 1e-12
    )

    shortlist = eligible[
        eligible["within_f1_tolerance"]
    ].copy()

    shortlist = shortlist.sort_values(
        [
            "n_arcs",
            "n_transitions",
            "noise"
        ],
        ascending=[
            True,
            True,
            True
        ],
        kind="stable"
    )

    winner = shortlist.iloc[0].copy()

    selected_rows27.append(
        winner.to_dict()
    )

    for _, row in group.iterrows():

        is_eligible = bool(
            row["eligible_fitness"]
        )

        within_tolerance = (
            is_eligible
            and
            float(row["f1_validation"])
            >= best_f1 - F1_TOLERANCE27 - 1e-12
        )

        is_selected = (
            abs(
                float(row["noise"])
                - float(winner["noise"])
            ) < 1e-12
        )

        if is_selected:

            reason = "selected"

        elif not is_eligible:

            reason = "fitness_below_threshold"

        elif not within_tolerance:

            reason = "f1_below_tolerance"

        else:

            reason = "higher_complexity_or_tiebreak"

        decision_rows27.append({
            "log_type": log_type,
            "noise": float(row["noise"]),
            "fitness_validation": float(
                row["fitness_validation"]
            ),
            "precision_validation": float(
                row["precision_validation"]
            ),
            "f1_validation": float(
                row["f1_validation"]
            ),
            "best_eligible_f1": best_f1,
            "f1_distance_to_best": (
                best_f1
                - float(row["f1_validation"])
            ),
            "n_arcs": int(row["n_arcs"]),
            "eligible_fitness": is_eligible,
            "within_f1_tolerance": (
                within_tolerance
            ),
            "selected": is_selected,
            "decision_reason": reason
        })

selected27 = pd.DataFrame(
    selected_rows27
)

decisions27 = pd.DataFrame(
    decision_rows27
)

if (
    len(selected27) != 3
    or selected27["log_type"].nunique() != 3
):

    raise RuntimeError(
        "A seleção deve produzir três modelos."
    )

_project_print("\n27.7 — MODELOS SELECIONADOS")

_project_display(
    selected27[
        [
            "log_type",
            "noise",
            "fitness_validation",
            "precision_validation",
            "f1_validation",
            "n_places",
            "n_transitions",
            "n_arcs"
        ]
    ].round(5)
)


# ============================================================
# 27.8 — JUSTIFICATIVAS DA SELEÇÃO
# ============================================================

justifications27 = {}

for _, row in selected27.iterrows():

    log_type = str(
        row["log_type"]
    )

    noise = float(
        row["noise"]
    )

    fitness = float(
        row["fitness_validation"]
    )

    precision = float(
        row["precision_validation"]
    )

    f1 = float(
        row["f1_validation"]
    )

    arcs = int(
        row["n_arcs"]
    )

    justification = (
        f"Modelo {log_type.upper()} selecionado "
        f"com noise={noise:.2f}. "
        f"Fitness de validação={fitness:.6f}, "
        f"precision={precision:.6f}, "
        f"F1={f1:.6f} e {arcs} arcos. "
        "Atende ao limite mínimo de fitness "
        "e à regra de proximidade ao maior F1 "
        "entre modelos elegíveis, priorizando "
        "menor complexidade estrutural."
    )

    justifications27[log_type] = (
        justification
    )

    _project_print("\n" + justification)


# ============================================================
# 27.9 — PLANEJAMENTO DO CONGELAMENTO
# ============================================================

# Primeiro validar todos os destinos.
# Somente depois realizar as cópias.
#
# Política:
# - Arquivo inexistente: criar.
# - Arquivo existente idêntico: reutilizar.
# - Arquivo existente diferente: interromper.
#
# Não sobrescrever silenciosamente.

freeze_plan27 = []

for _, row in selected27.iterrows():

    log_type = str(
        row["log_type"]
    )

    noise = round(
        float(row["noise"]),
        2
    )

    source_row = structure27[
        (
            structure27["log_type"]
            == log_type
        )
        &
        (
            structure27["noise"]
            == noise
        )
    ]

    if len(source_row) != 1:

        raise RuntimeError(
            "Origem PNML ambígua."
        )

    source_filename = str(
        source_row.iloc[0]["pnml_file"]
    )

    source_path = (
        MODEL_DIR27 / source_filename
    )

    frozen_filename = (
        f"cell27_frozen_{log_type}.pnml"
    )

    frozen_path = (
        FROZEN_DIR27 / frozen_filename
    )

    source_hash = sha256_file27(
        source_path
    )

    if frozen_path.exists():

        frozen_hash = sha256_file27(
            frozen_path
        )

        if frozen_hash != source_hash:

            raise RuntimeError(
                "Modelo congelado já existe "
                "com conteúdo diferente: "
                f"{frozen_path}. "
                "Crie uma nova execução para "
                "alterar o modelo."
            )

    freeze_plan27.append({
        "log_type": log_type,
        "noise": noise,
        "source_path": source_path,
        "frozen_path": frozen_path,
        "source_sha256": source_hash
    })


# ============================================================
# 27.10 — CONGELAMENTO DOS MODELOS
# ============================================================

frozen_models27 = {}
freeze_rows27 = []

for item in freeze_plan27:

    source_path = item[
        "source_path"
    ]

    frozen_path = item[
        "frozen_path"
    ]

    if not frozen_path.exists():

        shutil.copyfile(
            source_path,
            frozen_path
        )

        action = "created"

    else:

        action = "verified_existing"

    frozen_hash = sha256_file27(
        frozen_path
    )

    if frozen_hash != item[
        "source_sha256"
    ]:

        raise RuntimeError(
            "Falha na integridade após cópia."
        )

    net, im, fm = pnml_importer.apply(
        str(frozen_path)
    )

    log_type = item["log_type"]

    frozen_models27[log_type] = (
        net,
        im,
        fm
    )

    freeze_rows27.append({
        "log_type": log_type,
        "selected_noise": item["noise"],
        "source_file": source_path.name,
        "frozen_file": frozen_path.name,
        "sha256": frozen_hash,
        "action": action,
        "n_places": len(net.places),
        "n_transitions": len(net.transitions),
        "n_arcs": len(net.arcs)
    })

audit27_frozen = pd.DataFrame(
    freeze_rows27
)

_project_print("\n27.10 — Modelos congelados")

_project_display(
    audit27_frozen
)


# ============================================================
# 27.11 — MANIFESTO DE CONGELAMENTO
# ============================================================

input_files27 = {
    "cell26_metrics": {
        "filename": METRICS_PATH27.name,
        "sha256": sha256_file27(
            METRICS_PATH27
        )
    },
    "cell26_structure": {
        "filename": STRUCTURE_PATH27.name,
        "sha256": sha256_file27(
            STRUCTURE_PATH27
        )
    },
    "cell26_manifest": {
        "filename": MANIFEST26_PATH27.name,
        "sha256": sha256_file27(
            MANIFEST26_PATH27
        )
    }
}

selected_manifest27 = {}

for _, row in selected27.iterrows():

    log_type = str(
        row["log_type"]
    )

    frozen_row = audit27_frozen[
        audit27_frozen["log_type"]
        == log_type
    ].iloc[0]

    selected_manifest27[log_type] = {
        "noise": float(
            row["noise"]
        ),
        "fitness_train": float(
            row["fitness_train"]
        ),
        "fitness_validation": float(
            row["fitness_validation"]
        ),
        "precision_train": float(
            row["precision_train"]
        ),
        "precision_validation": float(
            row["precision_validation"]
        ),
        "f1_train": float(
            row["f1_train"]
        ),
        "f1_validation": float(
            row["f1_validation"]
        ),
        "n_places": int(
            row["n_places"]
        ),
        "n_transitions": int(
            row["n_transitions"]
        ),
        "n_arcs": int(
            row["n_arcs"]
        ),
        "frozen_file": str(
            frozen_row["frozen_file"]
        ),
        "sha256": str(
            frozen_row["sha256"]
        ),
        "justification": (
            justifications27[log_type]
        )
    }

manifest27 = {
    "project": "co2-cems-thermal-modeling",
    "phase": 2,
    "cell": 27,
    "facility": "Iatan",
    "orispl": 6065,
    "unit": "2",
    "year": 2025,
    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "selection_policy": (
        SELECTION_POLICY27
    ),
    "selection_parameters": {
        "minimum_validation_fitness": (
            MIN_VALIDATION_FITNESS27
        ),
        "f1_absolute_tolerance": (
            F1_TOLERANCE27
        ),
        "complexity_metric": "n_arcs",
        "secondary_complexity_metric": (
            "n_transitions"
        ),
        "final_tiebreak": "minimum_noise"
    },
    "metric_definitions": {
        "fitness": "alignment_based",
        "precision": "etconformance_token",
        "f1": (
            "harmonic_mean_fitness_precision"
        )
    },
    "selected_models": (
        selected_manifest27
    ),
    "input_files": input_files27,
    "environment": {
        "python": sys.version.split()[0],
        "pm4py": pm4py.__version__,
        "numpy": np.__version__,
        "pandas": pd.__version__
    },
    "test_used": False,
    "co2_used": False,
    "hmm_retrained": False,
    "process_rediscovered": False,
    "model_selection_completed": True,
    "models_frozen": True
}


# ============================================================
# 27.12 — EXPORTAÇÃO DOS RELATÓRIOS
# ============================================================

_project_csv(selected27, 
    REPORT_DIR27 /
    "cell27_selected_models.csv",
    index=False
)

_project_csv(decisions27, 
    REPORT_DIR27 /
    "cell27_selection_decisions.csv",
    index=False
)

_project_csv(audit27_pnml, 
    REPORT_DIR27 /
    "cell27_pnml_integrity_audit.csv",
    index=False
)

_project_csv(audit27_frozen, 
    REPORT_DIR27 /
    "cell27_frozen_models_audit.csv",
    index=False
)

MANIFEST27_PATH = (
    REPORT_DIR27 /
    "cell27_frozen_models_manifest.json"
)

# Evitar alterar silenciosamente o manifesto
# de uma execução anterior.
#
# A data de criação é excluída da comparação
# semântica para permitir reexecução idempotente.

if MANIFEST27_PATH.exists():

    with open(
        MANIFEST27_PATH,
        "r",
        encoding="utf-8"
    ) as f:

        previous_manifest27 = json.load(f)

    current_compare27 = dict(
        manifest27
    )

    previous_compare27 = dict(
        previous_manifest27
    )

    current_compare27.pop(
        "created_at_utc",
        None
    )

    previous_compare27.pop(
        "created_at_utc",
        None
    )

    if current_compare27 != previous_compare27:

        raise RuntimeError(
            "Manifesto de congelamento já "
            "existe com conteúdo diferente. "
            "Não será sobrescrito."
        )

    manifest27 = previous_manifest27

else:

    with open(
        MANIFEST27_PATH,
        "w",
        encoding="utf-8"
    ) as f:

        _project_dump(
            json_ready27(manifest27),
            f,
            indent=2,
            ensure_ascii=False
        )


# ============================================================
# 27.13 — VERIFICAÇÃO FINAL
# ============================================================

for log_type in LOG_TYPES27:

    selected_info = (
        manifest27["selected_models"][log_type]
    )

    frozen_path = (
        FROZEN_DIR27 /
        selected_info["frozen_file"]
    )

    if not frozen_path.is_file():

        raise RuntimeError(
            f"Modelo congelado ausente: {log_type}"
        )

    if sha256_file27(
        frozen_path
    ) != selected_info["sha256"]:

        raise RuntimeError(
            "Integridade final inválida: "
            f"{log_type}"
        )

_project_print("\n" + "=" * 78)
_project_print("CÉLULA 27 CONCLUÍDA")
_project_print("=" * 78)

_project_print("\nMODELOS SELECIONADOS:")

for log_type in LOG_TYPES27:

    info = (
        manifest27["selected_models"][log_type]
    )

    _project_print(
        f"{log_type.upper():10s} | "
        f"noise={info['noise']:.2f} | "
        f"fitness={info['fitness_validation']:.4f} | "
        f"precision={info['precision_validation']:.4f} | "
        f"F1={info['f1_validation']:.4f} | "
        f"arcos={info['n_arcs']}"
    )

_project_print("\nMODELOS CONGELADOS:")
_project_print(FROZEN_DIR27)

_project_print("\nMANIFESTO:")
_project_print(MANIFEST27_PATH)

_project_print("\nTeste utilizado: NÃO")
_project_print("CO2 utilizado: NÃO")
_project_print("Modelos redescobertos: NÃO")

_project_print("\nPRÓXIMA ETAPA:")
_project_print(
    "CÉLULA 28/36 — AVALIAÇÃO FINAL "
    "DOS MODELOS CONGELADOS NO CONJUNTO DE TESTE"
)


In [ ]:

# ============================================================
# FASE 2 — CÉLULA 28/36
# AVALIAÇÃO FINAL DOS MODELOS CONGELADOS NO TESTE
# ============================================================
#
# PROJETO:
# CO2-CEMS Thermal Modeling
#
# UNIDADE:
# Iatan | ORISPL 6065 | Unit 2 | 2025
#
# OBJETIVOS:
# 1. Recuperar a inferência HMM congelada no teste.
# 2. Construir os três Event Logs de teste.
# 3. Preservar a semântica das Células 18–20.
# 4. Auditar a integridade temporal e dos casos.
# 5. Importar os três modelos congelados.
# 6. Avaliar fitness, precision e F1 no teste.
# 7. Comparar train, validation e test.
# 8. Exportar resultados e manifesto reproduzível.
#
# RESTRIÇÕES:
# - Não retreinar o HMM.
# - Não reajustar scaler.
# - Não redescobrir processos.
# - Não modificar os PNML congelados.
# - Não utilizar CO2.
# - Não selecionar modelos usando o teste.
#
# ============================================================

import json
import hashlib
import time
import sys
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pm4py

from IPython.display import display

from pm4py.objects.conversion.log import (
    converter as log_converter
)

from pm4py.objects.petri_net.importer import (
    importer as pnml_importer
)

from pm4py.algo.evaluation.replay_fitness import (
    algorithm as fitness_evaluator
)

from pm4py.algo.evaluation.precision import (
    algorithm as precision_evaluator
)

_project_print("=" * 78)
_project_print("FASE 2 — CÉLULA 28/36")
_project_print("AVALIAÇÃO FINAL DOS MODELOS CONGELADOS NO TESTE")
_project_print("=" * 78)


# ============================================================
# 28.1 — CONFIGURAÇÃO E PRÉ-CONDIÇÕES
# ============================================================

required28 = [
    "RUN_DIR",
    "hmm13_inference",
    "hmm08_data",
    "hmm18_semantics",
    "hmm20_validated_logs",
    "locate_hourly19",
    "build_episodes19",
    "ramp_activity19",
    "RAMP_THRESHOLD_MW19",
    "oper19"
]

missing28 = [
    name for name in required28
    if name not in globals()
]

if missing28:
    raise RuntimeError(
        "Objetos necessários ausentes: "
        + ", ".join(missing28)
        + "\nExecute as Células 13–27 na sessão atual."
    )

RUN_DIR28 = Path(RUN_DIR).expanduser().resolve()

REPORT_DIR28 = RUN_DIR28 / "reports"
FIG_DIR28 = RUN_DIR28 / "figures"
EVENT_DIR28 = RUN_DIR28 / "event_logs"

FROZEN_DIR28 = (
    RUN_DIR28 / "models" / "cell27_frozen"
)

for folder in (
    REPORT_DIR28,
    FIG_DIR28,
    EVENT_DIR28
):
    folder.mkdir(parents=True, exist_ok=True)

LOG_TYPES28 = (
    "regime",
    "ramp",
    "combined"
)

MANIFEST27_PATH28 = (
    REPORT_DIR28 /
    "cell27_frozen_models_manifest.json"
)

METRICS26_PATH28 = (
    REPORT_DIR28 /
    "cell26_fitness_precision_complexity.csv"
)

if not MANIFEST27_PATH28.is_file():
    raise FileNotFoundError(
        MANIFEST27_PATH28
    )

if not METRICS26_PATH28.is_file():
    raise FileNotFoundError(
        METRICS26_PATH28
    )

with open(
    MANIFEST27_PATH28,
    encoding="utf-8"
) as f:
    manifest27_28 = json.load(f)

if manifest27_28.get("models_frozen") is not True:
    raise RuntimeError(
        "Modelos da Célula 27 não estão congelados."
    )

if manifest27_28.get("test_used") is not False:
    raise RuntimeError(
        "A seleção anterior utilizou o teste."
    )

if not np.isclose(
    float(RAMP_THRESHOLD_MW19),
    20.0
):
    raise RuntimeError(
        "Limiar de rampa diferente do definido "
        "na Célula 19."
    )

_project_print("\n28.1 — Pré-condições verificadas")
_project_print("Execução:", RUN_DIR28)
_project_print("PM4Py:", pm4py.__version__)


# ============================================================
# 28.2 — FUNÇÕES DE INTEGRIDADE
# ============================================================

def sha256_file28(path):

    h = hashlib.sha256()

    with open(path, "rb") as f:
        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            h.update(chunk)

    return h.hexdigest()


def fingerprint_log28(log):

    h = hashlib.sha256()

    for trace in log:

        case_id = str(
            trace.attributes.get(
                "concept:name",
                ""
            )
        )

        h.update(case_id.encode("utf-8"))
        h.update(b"\x00")

        for event in trace:

            activity = str(
                event.get("concept:name", "")
            )

            timestamp = pd.Timestamp(
                event["time:timestamp"]
            ).isoformat()

            h.update(activity.encode("utf-8"))
            h.update(b"\x1f")
            h.update(timestamp.encode("utf-8"))
            h.update(b"\x1e")

        h.update(b"\x0a")

    return h.hexdigest()


# ============================================================
# 28.3 — RECUPERAÇÃO DA INFERÊNCIA DE TESTE
# ============================================================

_project_print("\n28.3 — Recuperando inferência HMM")

inference28 = hmm13_inference.get("test")

if inference28 is None:
    raise RuntimeError(
        "hmm13_inference['test'] não encontrado."
    )

states28 = np.asarray(
    inference28["filtered_state"],
    dtype=int
)

confidence28 = np.asarray(
    inference28["filtered_confidence"],
    dtype=float
)

entropy28 = np.asarray(
    inference28["filtered_entropy"],
    dtype=float
)

lengths28 = np.asarray(
    inference28["lengths"],
    dtype=int
)

cycle_index28 = np.asarray(
    inference28["cycle_index"],
    dtype=int
)

n28 = len(states28)

if n28 == 0:
    raise RuntimeError("Teste HMM vazio.")

if (
    lengths28.sum() != n28
    or np.any(lengths28 <= 0)
):
    raise RuntimeError(
        "Comprimentos dos ciclos inconsistentes."
    )

if not (
    len(confidence28)
    == len(entropy28)
    == len(cycle_index28)
    == n28
):
    raise RuntimeError(
        "Vetores HMM com comprimentos diferentes."
    )

if not np.isfinite(confidence28).all():
    raise RuntimeError(
        "Confiança HMM inválida."
    )

if not np.all(
    (states28 >= 0) & (states28 < 5)
):
    raise RuntimeError(
        "Estados HMM fora do intervalo 0–4."
    )

expected_cycle_index28 = np.repeat(
    np.arange(1, len(lengths28) + 1),
    lengths28
)

if not np.array_equal(
    cycle_index28,
    expected_cycle_index28
):
    raise RuntimeError(
        "Índice de ciclos HMM inconsistente."
    )

_project_print("Horas no teste:", n28)
_project_print("Ciclos ON:", len(lengths28))
_project_print("Estados:", sorted(set(states28)))


# ============================================================
# 28.4 — ALINHAMENTO TEMPORAL ORIGINAL
# ============================================================
#
# Reutiliza a função da Célula 19.
#
# Não atribui timestamps pela posição.
# A função verifica a correspondência da matriz X
# com a inferência HMM e recupera o índice temporal.
#
# ============================================================

_project_print("\n28.4 — Alinhamento temporal")

frame28, alignment_source28 = (
    locate_hourly19("test")
)

frame28 = frame28.reset_index(drop=True).copy()

if len(frame28) != n28:
    raise RuntimeError(
        "Número de horas incompatível com HMM."
    )

if frame28["timestamp19"].isna().any():
    raise RuntimeError(
        "Timestamps ausentes no teste."
    )

if frame28["timestamp19"].duplicated().any():
    raise RuntimeError(
        "Timestamps duplicados no teste."
    )

if not frame28["timestamp19"].is_monotonic_increasing:
    raise RuntimeError(
        "Observações do teste fora de ordem temporal."
    )

if frame28["load_mw19"].isna().any():
    raise RuntimeError(
        "Carga ausente no conjunto de teste."
    )

_project_print("Fonte do alinhamento:", alignment_source28)
_project_print("Primeira hora:", frame28["timestamp19"].min())
_project_print("Última hora:", frame28["timestamp19"].max())


# ============================================================
# 28.5 — CONSTRUÇÃO DOS CASOS DO TESTE
# ============================================================
#
# Mesma definição da Célula 18:
#
# - Um caso por ciclo operacional ON.
# - Identificadores independentes por partição.
# - Nenhum evento atravessa fronteiras de ciclos.
#
# ============================================================

_project_print("\n28.5 — Construindo casos")

offsets28 = np.r_[
    0,
    np.cumsum(lengths28)
]

case_ids28 = np.empty(
    n28,
    dtype=object
)

case_rows28 = []

for cycle_number, (start, end) in enumerate(
    zip(offsets28[:-1], offsets28[1:]),
    start=1
):

    case_id = (
        f"test_ON_{cycle_number:04d}"
    )

    case_ids28[start:end] = case_id

    case_rows28.append({
        "case_id": case_id,
        "partition": "test",
        "cycle_id": cycle_number,
        "start_index": int(start),
        "end_index_exclusive": int(end),
        "duration_hours": int(end - start),
        "initial_state": int(states28[start]),
        "final_state": int(states28[end - 1])
    })

cases28 = pd.DataFrame(case_rows28)

frame28["partition"] = "test"
frame28["case_id"] = case_ids28
frame28["state"] = states28
frame28["confidence"] = confidence28
frame28["entropy"] = entropy28
frame28["cycle_index_hmm"] = cycle_index28

if frame28["case_id"].isna().any():
    raise RuntimeError(
        "Existem horas sem case_id."
    )

for _, case in cases28.iterrows():

    start = int(case["start_index"])
    end = int(case["end_index_exclusive"])

    segment = frame28.iloc[start:end]

    if len(segment) != int(case["duration_hours"]):
        raise RuntimeError(
            f"Duração inconsistente: {case['case_id']}"
        )

    if not (
        segment["case_id"] == case["case_id"]
    ).all():
        raise RuntimeError(
            f"Fronteira de caso inconsistente: "
            f"{case['case_id']}"
        )

    steps = (
        segment["timestamp19"]
        .diff()
        .dropna()
    )

    if not (
        steps == pd.Timedelta(hours=1)
    ).all():
        raise RuntimeError(
            f"Lacuna temporal no caso "
            f"{case['case_id']}"
        )

_project_print("Casos construídos:", len(cases28))


# ============================================================
# 28.6 — ATIVIDADES DE REGIME E RAMPA
# ============================================================
#
# A classificação de rampa é realizada somente
# dentro de cada caso.
#
# Primeira hora do ciclo:
# RAMP_UNDEFINED
#
# Limiar original:
# +20 MW/h e -20 MW/h
#
# ============================================================

_project_print("\n28.6 — Construindo atividades")

frame28["delta_load_mw"] = (
    frame28.groupby(
        "case_id",
        sort=False
    )["load_mw19"].diff()
)

frame28["regime_activity"] = (
    "REGIME_S"
    + frame28["state"].astype(str)
)

frame28["ramp_activity"] = (
    frame28["delta_load_mw"]
    .apply(ramp_activity19)
)

frame28["combined_activity"] = (
    frame28["regime_activity"]
    + "__"
    + frame28["ramp_activity"]
)

first_rows28 = (
    frame28.groupby(
        "case_id",
        sort=False
    ).head(1)
)

if not (
    first_rows28["ramp_activity"]
    == "RAMP_UNDEFINED"
).all():
    raise RuntimeError(
        "Primeira hora do ciclo com rampa definida."
    )

_project_print(
    "\nAtividades REGIME:",
    frame28["regime_activity"].nunique()
)

_project_print(
    "Atividades RAMP:",
    frame28["ramp_activity"].nunique()
)

_project_print(
    "Atividades COMBINED:",
    frame28["combined_activity"].nunique()
)


# ============================================================
# 28.7 — CONSTRUÇÃO DOS EVENT LOGS
# ============================================================
#
# Reutiliza a função build_episodes19.
#
# Não altera:
# - atividades;
# - limiares;
# - fronteiras dos casos;
# - regra de consolidação.
#
# ============================================================

_project_print("\n28.7 — Construindo Event Logs")

event_frames28 = {
    "regime": build_episodes19(
        frame28,
        "regime_activity"
    ),
    "ramp": build_episodes19(
        frame28,
        "ramp_activity"
    ),
    "combined": build_episodes19(
        frame28,
        "combined_activity"
    )
}

event_logs28 = {}

# Preserva os logs de desenvolvimento já validados.
if "event_logs26" in globals():
    event_logs28.update(event_logs26)

audit_rows28 = []

for log_type in LOG_TYPES28:

    events = event_frames28[log_type].copy()

    if events.empty:
        raise RuntimeError(
            f"Event Log vazio: {log_type}"
        )

    if int(events["duration_hours"].sum()) != n28:
        raise RuntimeError(
            f"Cobertura horária incorreta: {log_type}"
        )

    if (
        events["case:concept:name"].nunique()
        != len(cases28)
    ):
        raise RuntimeError(
            f"Número de casos incorreto: {log_type}"
        )

    events["time:timestamp"] = pd.to_datetime(
        events["time:timestamp"],
        utc=True,
        errors="raise"
    )

    events["end_timestamp"] = pd.to_datetime(
        events["end_timestamp"],
        utc=True,
        errors="raise"
    )

    if (
        events["time:timestamp"].isna().any()
        or events["end_timestamp"].isna().any()
    ):
        raise RuntimeError(
            f"Timestamps inválidos: {log_type}"
        )

    if (
        events["end_timestamp"]
        < events["time:timestamp"]
    ).any():
        raise RuntimeError(
            f"Intervalos inválidos: {log_type}"
        )

    events = events.sort_values(
        [
            "case:concept:name",
            "time:timestamp"
        ],
        kind="stable"
    ).reset_index(drop=True)

    # Confere reconstrução exata das atividades
    # horárias, respeitando a ordem dos casos.
    activity_column = {
        "regime": "regime_activity",
        "ramp": "ramp_activity",
        "combined": "combined_activity"
    }[log_type]

    for case_id, group in frame28.groupby(
        "case_id",
        sort=False
    ):

        expected = (
            group[activity_column]
            .astype(str)
            .tolist()
        )

        observed_events = events.loc[
            events["case:concept:name"]
            == case_id
        ]

        reconstructed = []

        for _, event in observed_events.iterrows():

            reconstructed.extend(
                [str(event["concept:name"])]
                * int(event["duration_hours"])
            )

        if reconstructed != expected:
            raise RuntimeError(
                f"Reconstrução divergente: "
                f"{log_type}/{case_id}"
            )

    log = log_converter.apply(
        events,
        variant=(
            log_converter.Variants.TO_EVENT_LOG
        )
    )

    event_logs28[
        (log_type, "test")
    ] = log

    audit_rows28.append({
        "log_type": log_type,
        "partition": "test",
        "n_cases": len(log),
        "n_events": len(events),
        "n_activities": (
            events["concept:name"].nunique()
        ),
        "covered_hours": int(
            events["duration_hours"].sum()
        ),
        "exact_reconstruction": True,
        "log_sha256": fingerprint_log28(log)
    })

audit28_logs = pd.DataFrame(audit_rows28)

_project_display(audit28_logs)


# ============================================================
# 28.8 — INTEGRIDADE DOS MODELOS CONGELADOS
# ============================================================

_project_print("\n28.8 — Verificando modelos congelados")

selected28 = manifest27_28[
    "selected_models"
]

frozen_models28 = {}

for log_type in LOG_TYPES28:

    info = selected28[log_type]

    filename = info["frozen_file"]

    if Path(filename).name != filename:
        raise RuntimeError(
            "Nome PNML inválido no manifesto."
        )

    path = FROZEN_DIR28 / filename

    if not path.is_file():
        raise FileNotFoundError(path)

    observed_hash = sha256_file28(path)

    if observed_hash != info["sha256"]:
        raise RuntimeError(
            f"SHA256 divergente: {log_type}"
        )

    net, im, fm = pnml_importer.apply(
        str(path)
    )

    structure = (
        len(net.places),
        len(net.transitions),
        len(net.arcs)
    )

    expected_structure = (
        int(info["n_places"]),
        int(info["n_transitions"]),
        int(info["n_arcs"])
    )

    if structure != expected_structure:
        raise RuntimeError(
            f"Estrutura PNML divergente: {log_type}"
        )

    frozen_models28[log_type] = (
        net,
        im,
        fm
    )

    _project_print(
        f"{log_type.upper():10s} | "
        f"noise={info['noise']:.2f} | "
        f"SHA256 OK"
    )


# ============================================================
# 28.9 — AVALIAÇÃO FINAL NO TESTE
# ============================================================
#
# Fitness: alignment-based
# Precision: ETConformance token-based
#
# Mesmas métricas da Célula 26.
#
# Nenhum resultado do teste altera os modelos.
#
# ============================================================

_project_print("\n28.9 — Avaliação final")

FITNESS_VARIANT28 = (
    fitness_evaluator.Variants.ALIGNMENT_BASED
)

PRECISION_VARIANT28 = (
    precision_evaluator.Variants.ETCONFORMANCE_TOKEN
)

test_rows28 = []

for log_type in LOG_TYPES28:

    _project_print(
        f"\nAvaliando {log_type.upper()}...",
        flush=True
    )

    log = event_logs28[
        (log_type, "test")
    ]

    net, im, fm = frozen_models28[
        log_type
    ]

    start_time = time.perf_counter()

    fitness_result = fitness_evaluator.apply(
        log,
        net,
        im,
        fm,
        variant=FITNESS_VARIANT28
    )

    precision = precision_evaluator.apply(
        log,
        net,
        im,
        fm,
        variant=PRECISION_VARIANT28
    )

    elapsed = (
        time.perf_counter()
        - start_time
    )

    fitness = float(
        fitness_result["log_fitness"]
    )

    precision = float(precision)

    if not (
        np.isfinite(fitness)
        and np.isfinite(precision)
        and 0 <= fitness <= 1
        and 0 <= precision <= 1
    ):
        raise RuntimeError(
            f"Métricas inválidas: {log_type}"
        )

    f1 = (
        2 * fitness * precision
        / (fitness + precision)
        if fitness + precision > 0
        else 0.0
    )

    test_rows28.append({
        "log_type": log_type,
        "partition": "test",
        "noise": float(
            selected28[log_type]["noise"]
        ),
        "fitness_log": fitness,
        "fitness_avg_trace": (
            fitness_result.get(
                "average_trace_fitness",
                np.nan
            )
        ),
        "percentage_fitting_traces": (
            fitness_result.get(
                "percentage_of_fitting_traces",
                np.nan
            )
        ),
        "precision": precision,
        "f1_fitness_precision": f1,
        "elapsed_seconds": elapsed,
        "n_cases": len(log),
        "n_events": sum(
            len(trace)
            for trace in log
        ),
        "pnml_sha256": (
            selected28[log_type]["sha256"]
        ),
        "log_sha256": (
            fingerprint_log28(log)
        )
    })

    _project_print(
        f"Fitness: {fitness:.6f} | "
        f"Precision: {precision:.6f} | "
        f"F1: {f1:.6f} | "
        f"Tempo: {elapsed:.1f}s",
        flush=True
    )

audit28_test = pd.DataFrame(
    test_rows28
)

if len(audit28_test) != 3:
    raise RuntimeError(
        "Avaliação incompleta: esperados três modelos."
    )


# ============================================================
# 28.10 — COMPARAÇÃO TRAIN / VALIDATION / TEST
# ============================================================

_project_print("\n28.10 — Comparação de generalização")

metrics26_28 = pd.read_csv(
    METRICS26_PATH28
)

history_rows28 = []

for log_type in LOG_TYPES28:

    noise = float(
        selected28[log_type]["noise"]
    )

    subset = metrics26_28.loc[
        (
            metrics26_28["log_type"]
            == log_type
        )
        &
        (
            np.isclose(
                pd.to_numeric(
                    metrics26_28["noise"],
                    errors="coerce"
                ),
                noise
            )
        )
        &
        (
            metrics26_28["partition"].isin(
                ["train", "validation"]
            )
        )
    ].copy()

    if (
        len(subset) != 2
        or set(subset["partition"])
        != {"train", "validation"}
    ):
        raise RuntimeError(
            f"Histórico incompleto: {log_type}"
        )

    history_rows28.extend(
        subset[
            [
                "log_type",
                "partition",
                "noise",
                "fitness_log",
                "precision",
                "f1_fitness_precision"
            ]
        ].to_dict("records")
    )

comparison_long28 = pd.concat(
    [
        pd.DataFrame(history_rows28),
        audit28_test[
            [
                "log_type",
                "partition",
                "noise",
                "fitness_log",
                "precision",
                "f1_fitness_precision"
            ]
        ]
    ],
    ignore_index=True
)

comparison_wide28 = comparison_long28.pivot(
    index="log_type",
    columns="partition",
    values=[
        "fitness_log",
        "precision",
        "f1_fitness_precision"
    ]
)

comparison_wide28.columns = [
    f"{metric}_{partition}"
    for metric, partition
    in comparison_wide28.columns
]

comparison_wide28 = (
    comparison_wide28.reset_index()
)

for metric in (
    "fitness_log",
    "precision",
    "f1_fitness_precision"
):
    comparison_wide28[
        f"{metric}_gap_test_minus_validation"
    ] = (
        comparison_wide28[
            f"{metric}_test"
        ]
        -
        comparison_wide28[
            f"{metric}_validation"
        ]
    )

_project_display(
    comparison_long28.sort_values(
        ["log_type", "partition"]
    ).round(6)
)

_project_display(
    comparison_wide28.round(6)
)


# ============================================================
# 28.11 — VISUALIZAÇÃO
# ============================================================

fig, axes = plt.subplots(
    1,
    3,
    figsize=(17, 5)
)

metric_specs28 = [
    ("fitness_log", "Fitness"),
    ("precision", "Precision"),
    (
        "f1_fitness_precision",
        "F1 fitness–precision"
    )
]

for ax, (metric, title) in zip(
    axes,
    metric_specs28
):

    pivot = comparison_long28.pivot(
        index="log_type",
        columns="partition",
        values=metric
    )

    pivot = pivot.reindex(
        index=LOG_TYPES28,
        columns=[
            "train",
            "validation",
            "test"
        ]
    )

    x = np.arange(
        len(LOG_TYPES28)
    )

    width = 0.25

    for i, partition in enumerate(
        ["train", "validation", "test"]
    ):

        ax.bar(
            x + (i - 1) * width,
            pivot[partition],
            width,
            label=partition
        )

    ax.set_xticks(x)

    ax.set_xticklabels(
        [lt.upper() for lt in LOG_TYPES28]
    )

    ax.set_ylim(0, 1.05)
    ax.set_title(title)
    ax.grid(axis="y", alpha=0.25)
    ax.legend()

plt.tight_layout()

FIG28_PATH = (
    FIG_DIR28 /
    "cell28_test_conformance_comparison.png"
)

fig.savefig(
    FIG28_PATH,
    dpi=180,
    bbox_inches="tight"
)

plt.show()


# ============================================================
# 28.12 — EXPORTAÇÃO DOS EVENT LOGS DE TESTE
# ============================================================

_project_print("\n28.12 — Exportando Event Logs")

event_paths28 = {}

for log_type in LOG_TYPES28:

    path = (
        EVENT_DIR28 /
        f"cell28_test_{log_type}.csv"
    )

    _project_csv(event_frames28[log_type], 
        path,
        index=False
    )

    event_paths28[log_type] = path

    _project_print(
        f"{log_type.upper()}: {path.name}"
    )


# ============================================================
# 28.13 — EXPORTAÇÃO DOS RELATÓRIOS
# ============================================================

OUTPUTS28 = {
    "cases": (
        REPORT_DIR28 /
        "cell28_test_cases.csv"
    ),
    "log_audit": (
        REPORT_DIR28 /
        "cell28_test_log_audit.csv"
    ),
    "test_metrics": (
        REPORT_DIR28 /
        "cell28_test_conformance.csv"
    ),
    "comparison_long": (
        REPORT_DIR28 /
        "cell28_train_validation_test.csv"
    ),
    "comparison_wide": (
        REPORT_DIR28 /
        "cell28_partition_comparison.csv"
    )
}

frames_to_save28 = {
    "cases": cases28,
    "log_audit": audit28_logs,
    "test_metrics": audit28_test,
    "comparison_long": comparison_long28,
    "comparison_wide": comparison_wide28
}

for key, path in OUTPUTS28.items():

    _project_csv(frames_to_save28[key], 
        path,
        index=False
    )


# ============================================================
# 28.14 — MANIFESTO DE REPRODUTIBILIDADE
# ============================================================

manifest28 = {
    "project": "co2-cems-thermal-modeling",
    "phase": 2,
    "cell": 28,
    "facility": "Iatan",
    "orispl": 6065,
    "unit": "2",
    "year": 2025,
    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "python_version": (
        sys.version.split()[0]
    ),
    "pm4py_version": (
        pm4py.__version__
    ),
    "selection_manifest_sha256": (
        sha256_file28(
            MANIFEST27_PATH28
        )
    ),
    "hmm_inference_source": (
        "hmm13_inference.test.filtered_state"
    ),
    "state_inference_method": (
        "forward_filtering"
    ),
    "temporal_alignment_source": (
        alignment_source28
    ),
    "case_definition": (
        "ON_operational_cycle"
    ),
    "ramp_threshold_mw": float(
        RAMP_THRESHOLD_MW19
    ),
    "event_construction": (
        "build_episodes19"
    ),
    "fitness_method": (
        "alignment_based"
    ),
    "precision_method": (
        "etconformance_token"
    ),
    "frozen_models": {
        lt: {
            "noise": float(
                selected28[lt]["noise"]
            ),
            "pnml_file": (
                selected28[lt]["frozen_file"]
            ),
            "pnml_sha256": (
                selected28[lt]["sha256"]
            )
        }
        for lt in LOG_TYPES28
    },
    "test_logs": {
        row["log_type"]: {
            "cases": int(
                row["n_cases"]
            ),
            "events": int(
                row["n_events"]
            ),
            "covered_hours": int(
                row["covered_hours"]
            ),
            "sha256": row["log_sha256"]
        }
        for _, row in audit28_logs.iterrows()
    },
    "n_test_models": int(
        len(audit28_test)
    ),
    "model_selection_modified": False,
    "test_used_for_selection": False,
    "co2_used": False,
    "hmm_retrained": False,
    "scaler_refitted": False,
    "process_rediscovered": False,
    "models_modified": False,
    "output_files_sha256": {
        key: sha256_file28(path)
        for key, path in OUTPUTS28.items()
    },
    "event_files_sha256": {
        key: sha256_file28(path)
        for key, path in event_paths28.items()
    },
    "figure_sha256": (
        sha256_file28(FIG28_PATH)
    )
}

MANIFEST28_PATH = (
    REPORT_DIR28 /
    "cell28_test_evaluation_manifest.json"
)

with open(
    MANIFEST28_PATH,
    "w",
    encoding="utf-8"
) as f:

    _project_dump(
        manifest28,
        f,
        indent=2,
        ensure_ascii=False
    )


# ============================================================
# 28.15 — RESUMO FINAL
# ============================================================

_project_print("\n" + "=" * 78)
_project_print("CÉLULA 28/36 CONCLUÍDA")
_project_print("=" * 78)

_project_print("\nAVALIAÇÃO FINAL NO TESTE")

_project_display(
    audit28_test[
        [
            "log_type",
            "noise",
            "n_cases",
            "n_events",
            "fitness_log",
            "precision",
            "f1_fitness_precision",
            "elapsed_seconds"
        ]
    ].round(6)
)

_project_print("\nAUDITORIA METODOLÓGICA")
_project_print("Estados: HMM forward filtering")
_project_print("Limiar de rampa: ±20 MW/h")
_project_print("Modelos congelados preservados: SIM")
_project_print("CO2 utilizado: NÃO")
_project_print("HMM retreinado: NÃO")
_project_print("Scaler reajustado: NÃO")
_project_print("Processos redescobertos: NÃO")
_project_print("Teste usado para seleção: NÃO")

_project_print("\nRELATÓRIOS")
_project_print(REPORT_DIR28)

_project_print("\nEVENT LOGS")
_project_print(EVENT_DIR28)

_project_print("\nMANIFESTO")
_project_print(MANIFEST28_PATH)

_project_print("\nPRÓXIMA ETAPA:")
_project_print(
    "CÉLULA 29/36 — ANÁLISE DE DESVIOS "
    "DOS MODELOS CONGELADOS"
)


In [ ]:

# ============================================================
# FASE 2 — CÉLULA 29/36
# ANÁLISE DE DESVIOS DOS MODELOS CONGELADOS
# ============================================================
#
# IATAN | ORISPL 6065 | UNIT 2 | 2025
#
# OBJETIVOS
# 1. Auditar os artefatos das Células 27 e 28.
# 2. Recuperar os três Event Logs independentes de teste.
# 3. Calcular alinhamentos por trace.
# 4. Identificar movimentos síncronos e desvios.
# 5. Separar transições silenciosas (tau).
# 6. Investigar casos e atividades desviantes.
# 7. Exportar relatórios e manifesto reproduzível.
#
# RESTRIÇÕES
# - Nenhum CO2 utilizado.
# - Nenhum HMM retreinado.
# - Nenhum processo redescoberto.
# - Nenhum PNML alterado.
# - Nenhuma seleção usando o teste.
# - Nenhum ajuste de parâmetros após observar o teste.
#
# ============================================================

import json
import hashlib
import time
import sys
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pm4py

from IPython.display import display

from pm4py.objects.log.obj import EventLog
from pm4py.objects.petri_net.importer import (
    importer as pnml_importer
)
from pm4py.objects.conversion.log import (
    converter as log_converter
)
from pm4py.algo.conformance.alignments.petri_net import (
    algorithm as alignments_algorithm
)

_project_print("=" * 78)
_project_print("FASE 2 — CÉLULA 29/36")
_project_print("ANÁLISE DE DESVIOS DOS MODELOS CONGELADOS")
_project_print("=" * 78)


# ============================================================
# 29.1 — CONFIGURAÇÃO
# ============================================================

if "RUN_DIR" not in globals():
    raise RuntimeError("RUN_DIR não encontrado.")

RUN_DIR29 = Path(RUN_DIR).expanduser().resolve()

REPORT_DIR29 = RUN_DIR29 / "reports"
EVENT_DIR29 = RUN_DIR29 / "event_logs"
FIG_DIR29 = RUN_DIR29 / "figures"
MODEL_DIR29 = RUN_DIR29 / "models" / "cell27_frozen"

REPORT_DIR29.mkdir(parents=True, exist_ok=True)
FIG_DIR29.mkdir(parents=True, exist_ok=True)

LOG_TYPES29 = ("regime", "ramp", "combined")

MANIFEST27_PATH29 = (
    REPORT_DIR29 /
    "cell27_frozen_models_manifest.json"
)

MANIFEST28_PATH29 = (
    REPORT_DIR29 /
    "cell28_test_evaluation_manifest.json"
)

METRICS28_PATH29 = (
    REPORT_DIR29 /
    "cell28_test_conformance.csv"
)

_project_print("\n29.1 — Diretório:", RUN_DIR29)
_project_print("PM4Py:", pm4py.__version__)


# ============================================================
# 29.2 — FUNÇÕES AUXILIARES
# ============================================================

def sha256_file29(path):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            h.update(chunk)

    return h.hexdigest()


def fingerprint_log29(log):
    # Mesmo algoritmo de fingerprint da Célula 28.
    h = hashlib.sha256()

    for trace in log:
        case_id = str(
            trace.attributes.get("concept:name", "")
        )
        h.update(case_id.encode("utf-8"))
        h.update(b"\x00")

        for event in trace:
            activity = str(
                event.get("concept:name", "")
            )
            timestamp = pd.Timestamp(
                event["time:timestamp"]
            ).isoformat()

            h.update(activity.encode("utf-8"))
            h.update(b"\x1f")
            h.update(timestamp.encode("utf-8"))
            h.update(b"\x1e")

        h.update(b"\x0a")

    return h.hexdigest()


def require29(path):
    if not path.is_file():
        raise FileNotFoundError(
            f"Artefato obrigatório ausente: {path}"
        )


def decode_move29(move):
    """
    Suporta formatos usuais do PM4Py:

    (log_label, model_label)

    ou

    ((log_id, model_id),
     (log_label, model_label))

    A classificação utiliza os rótulos,
    não os identificadores internos.
    """

    if not isinstance(move, (tuple, list)):
        raise ValueError(
            f"Movimento inesperado: {move!r}"
        )

    if len(move) != 2:
        raise ValueError(
            f"Movimento com formato inválido: {move!r}"
        )

    a, b = move

    if (
        isinstance(a, (tuple, list))
        and isinstance(b, (tuple, list))
        and len(a) == 2
        and len(b) == 2
    ):
        log_label, model_label = b
    else:
        log_label, model_label = a, b

    log_skip = log_label == ">>"
    model_skip = model_label == ">>"

    if log_skip and model_skip:
        raise ValueError(
            f"Movimento duplamente vazio: {move!r}"
        )

    if not log_skip and not model_skip:
        if log_label != model_label:
            raise ValueError(
                "Movimento síncrono com rótulos "
                f"diferentes: {move!r}"
            )
        move_type = "synchronous"

    elif not log_skip and model_skip:
        move_type = "log_only"

    elif log_skip and (
        model_label is None
        or model_label == "tau"
        or model_label == "τ"
    ):
        move_type = "silent_model"

    else:
        move_type = "model_only"

    return (
        move_type,
        log_label,
        model_label
    )


# ============================================================
# 29.3 — VALIDAÇÃO DOS MANIFESTOS
# ============================================================

for path in (
    MANIFEST27_PATH29,
    MANIFEST28_PATH29,
    METRICS28_PATH29
):
    require29(path)

with open(
    MANIFEST27_PATH29,
    encoding="utf-8"
) as f:
    manifest27_29 = json.load(f)

with open(
    MANIFEST28_PATH29,
    encoding="utf-8"
) as f:
    manifest28_29 = json.load(f)

assert manifest27_29["models_frozen"] is True

assert manifest28_29["n_test_models"] == 3

assert manifest28_29["co2_used"] is False

assert (
    manifest28_29["test_used_for_selection"]
    is False
)

if (
    sha256_file29(MANIFEST27_PATH29)
    != manifest28_29["selection_manifest_sha256"]
):
    raise RuntimeError(
        "Manifesto da Célula 27 foi alterado."
    )

metrics28_29 = pd.read_csv(
    METRICS28_PATH29
)

if set(metrics28_29["log_type"]) != set(LOG_TYPES29):
    raise RuntimeError(
        "Resultados da Célula 28 incompletos."
    )

_project_print("\n29.3 — Manifestos validados")


# ============================================================
# 29.4 — CARREGAMENTO DOS MODELOS CONGELADOS
# ============================================================

models29 = {}

for log_type in LOG_TYPES29:

    info = manifest27_29[
        "selected_models"
    ][log_type]

    filename = info["frozen_file"]

    if Path(filename).name != filename:
        raise RuntimeError("Nome PNML inválido.")

    path = MODEL_DIR29 / filename
    require29(path)

    if sha256_file29(path) != info["sha256"]:
        raise RuntimeError(
            f"Hash PNML divergente: {log_type}"
        )

    net, im, fm = pnml_importer.apply(str(path))

    structure = (
        len(net.places),
        len(net.transitions),
        len(net.arcs)
    )

    expected = (
        int(info["n_places"]),
        int(info["n_transitions"]),
        int(info["n_arcs"])
    )

    if structure != expected:
        raise RuntimeError(
            f"Estrutura divergente: {log_type}"
        )

    models29[log_type] = (net, im, fm)

_project_print("\n29.4 — Três PNML verificados")


# ============================================================
# 29.5 — RECUPERAÇÃO DOS LOGS DE TESTE
# ============================================================

logs29 = {}

for log_type in LOG_TYPES29:

    key = (log_type, "test")

    if (
        "event_logs28" in globals()
        and key in event_logs28
    ):
        log = event_logs28[key]

    else:
        # Recuperação dos CSVs exportados
        # pela Célula 28, sem reconstruir
        # atividades ou modificar os eventos.

        path = (
            EVENT_DIR29 /
            f"cell28_test_{log_type}.csv"
        )
        require29(path)

        df = pd.read_csv(path)

        required_cols = {
            "case:concept:name",
            "concept:name",
            "time:timestamp"
        }

        if not required_cols.issubset(df.columns):
            raise RuntimeError(
                f"Colunas ausentes no log: {log_type}"
            )

        df["time:timestamp"] = pd.to_datetime(
            df["time:timestamp"],
            utc=True,
            errors="raise"
        )

        df = df.sort_values(
            [
                "case:concept:name",
                "time:timestamp"
            ],
            kind="stable"
        )

        log = log_converter.apply(
            df,
            variant=(
                log_converter.Variants.TO_EVENT_LOG
            )
        )

    if not isinstance(log, EventLog):
        raise RuntimeError(
            f"Formato inválido: {log_type}"
        )

    expected_hash = manifest28_29[
        "test_logs"
    ][log_type]["sha256"]

    actual_hash = fingerprint_log29(log)

    if actual_hash != expected_hash:
        raise RuntimeError(
            f"Fingerprint divergente: {log_type}. "
            "Verifique a serialização dos timestamps."
        )

    if (
        len(log)
        != manifest28_29[
            "test_logs"
        ][log_type]["cases"]
    ):
        raise RuntimeError(
            f"Número de casos divergente: {log_type}"
        )

    logs29[log_type] = log

    _project_print(
        f"{log_type.upper():10s} | "
        f"{len(log)} casos | SHA256 OK"
    )


# ============================================================
# 29.6 — ALINHAMENTOS POR TRACE
# ============================================================

trace_rows29 = []
move_rows29 = []

for log_type in LOG_TYPES29:

    _project_print(
        f"\n29.6 — Alinhando {log_type.upper()}",
        flush=True
    )

    log = logs29[log_type]
    net, im, fm = models29[log_type]

    start_time = time.perf_counter()

    results = alignments_algorithm.apply_log(
        log,
        net,
        im,
        fm
    )

    if len(results) != len(log):
        raise RuntimeError(
            f"Alinhamentos incompletos: {log_type}"
        )

    for trace_index, (trace, result) in enumerate(
        zip(log, results)
    ):

        case_id = str(
            trace.attributes.get(
                "concept:name",
                f"trace_{trace_index}"
            )
        )

        fitness = result.get(
            "fitness",
            np.nan
        )

        cost = result.get(
            "cost",
            np.nan
        )

        counters = {
            "synchronous": 0,
            "log_only": 0,
            "model_only": 0,
            "silent_model": 0
        }

        alignment = result.get(
            "alignment",
            []
        )

        for move_index, move in enumerate(alignment):

            (
                move_type,
                log_label,
                model_label
            ) = decode_move29(move)

            counters[move_type] += 1

            move_rows29.append({
                "log_type": log_type,
                "case_id": case_id,
                "trace_index": trace_index,
                "move_index": move_index,
                "move_type": move_type,
                "log_activity": (
                    None
                    if log_label == ">>"
                    else log_label
                ),
                "model_activity": (
                    None
                    if model_label in (">>", None)
                    else model_label
                ),
                "is_deviation": (
                    move_type in (
                        "log_only",
                        "model_only"
                    )
                )
            })

        consumed_log_events = (
            counters["synchronous"]
            + counters["log_only"]
        )

        if consumed_log_events != len(trace):
            raise RuntimeError(
                f"Alinhamento não consumiu todos "
                f"os eventos: {log_type}/{case_id}"
            )

        n_deviations = (
            counters["log_only"]
            + counters["model_only"]
        )

        trace_rows29.append({
            "log_type": log_type,
            "case_id": case_id,
            "trace_index": trace_index,
            "n_events": len(trace),
            "fitness": fitness,
            "alignment_cost": cost,
            "synchronous_moves": (
                counters["synchronous"]
            ),
            "log_only_moves": (
                counters["log_only"]
            ),
            "model_only_moves": (
                counters["model_only"]
            ),
            "silent_model_moves": (
                counters["silent_model"]
            ),
            "n_deviations": n_deviations,
            "has_deviation": n_deviations > 0
        })

    _project_print(
        f"Concluído em "
        f"{time.perf_counter() - start_time:.2f}s",
        flush=True
    )

audit29_traces = pd.DataFrame(trace_rows29)
audit29_moves = pd.DataFrame(move_rows29)

audit29_deviations = audit29_moves.loc[
    audit29_moves["is_deviation"]
].copy()

_project_print("\nAlinhamentos concluídos")
_project_print("Traces:", len(audit29_traces))
_project_print("Movimentos:", len(audit29_moves))
_project_print("Desvios visíveis:", len(audit29_deviations))


# ============================================================
# 29.7 — RESUMO POR MODELO
# ============================================================

summary29 = (
    audit29_traces.groupby(
        "log_type",
        as_index=False
    )
    .agg(
        n_cases=("case_id", "count"),
        n_events=("n_events", "sum"),
        mean_trace_fitness=("fitness", "mean"),
        median_trace_fitness=("fitness", "median"),
        min_trace_fitness=("fitness", "min"),
        mean_alignment_cost=(
            "alignment_cost",
            "mean"
        ),
        total_synchronous=(
            "synchronous_moves",
            "sum"
        ),
        total_log_only=(
            "log_only_moves",
            "sum"
        ),
        total_model_only=(
            "model_only_moves",
            "sum"
        ),
        total_silent_model=(
            "silent_model_moves",
            "sum"
        ),
        traces_with_deviation=(
            "has_deviation",
            "sum"
        )
    )
)

summary29["share_traces_with_deviation"] = (
    summary29["traces_with_deviation"]
    / summary29["n_cases"]
)

# Nota: mean_trace_fitness não é necessariamente
# igual ao log_fitness da Célula 28.
# As agregações têm definições diferentes.

_project_display(summary29.round(6))


# ============================================================
# 29.8 — FREQUÊNCIA DOS DESVIOS
# ============================================================

if not audit29_deviations.empty:

    deviation_frequency29 = (
        audit29_deviations.groupby(
            [
                "log_type",
                "move_type",
                "log_activity",
                "model_activity"
            ],
            dropna=False
        )
        .size()
        .reset_index(name="frequency")
        .sort_values(
            ["log_type", "frequency"],
            ascending=[True, False]
        )
    )

else:
    deviation_frequency29 = pd.DataFrame(
        columns=[
            "log_type",
            "move_type",
            "log_activity",
            "model_activity",
            "frequency"
        ]
    )

_project_print("\n29.8 — Frequência dos desvios")
_project_display(deviation_frequency29.head(30))


# ============================================================
# 29.9 — CASOS COM MAIORES DESVIOS
# ============================================================

cases_ranked29 = audit29_traces.sort_values(
    [
        "log_type",
        "n_deviations",
        "alignment_cost"
    ],
    ascending=[
        True,
        False,
        False
    ]
).copy()

_project_print("\n29.9 — Casos com maiores desvios")

_project_display(
    cases_ranked29[
        [
            "log_type",
            "case_id",
            "n_events",
            "fitness",
            "alignment_cost",
            "log_only_moves",
            "model_only_moves",
            "n_deviations"
        ]
    ].round(6)
)


# ============================================================
# 29.10 — VISUALIZAÇÃO
# ============================================================

fig, axes = plt.subplots(
    1,
    2,
    figsize=(13, 5)
)

fitness_data = [
    audit29_traces.loc[
        audit29_traces["log_type"] == lt,
        "fitness"
    ].dropna().to_numpy()
    for lt in LOG_TYPES29
]

axes[0].boxplot(
    fitness_data,
    tick_labels=[
        lt.upper()
        for lt in LOG_TYPES29
    ]
)

axes[0].set_title(
    "Fitness por ciclo operacional"
)
axes[0].set_ylabel("Fitness por trace")
axes[0].set_ylim(0, 1.05)
axes[0].grid(axis="y", alpha=0.25)

plot_data = summary29.set_index(
    "log_type"
).reindex(LOG_TYPES29)

x = np.arange(len(LOG_TYPES29))
width = 0.35

axes[1].bar(
    x - width / 2,
    plot_data["total_log_only"],
    width,
    label="Log only"
)

axes[1].bar(
    x + width / 2,
    plot_data["total_model_only"],
    width,
    label="Model only"
)

axes[1].set_xticks(x)
axes[1].set_xticklabels(
    [lt.upper() for lt in LOG_TYPES29]
)
axes[1].set_ylabel(
    "Quantidade de movimentos desviantes"
)
axes[1].set_title(
    "Desvios visíveis por representação"
)
axes[1].legend()
axes[1].grid(axis="y", alpha=0.25)

plt.tight_layout()

FIG29_PATH = (
    FIG_DIR29 /
    "cell29_deviation_diagnostics.png"
)

fig.savefig(
    FIG29_PATH,
    dpi=180,
    bbox_inches="tight"
)

plt.show()


# ============================================================
# 29.11 — EXPORTAÇÃO DOS RELATÓRIOS
# ============================================================

OUTPUTS29 = {
    "trace_alignments": (
        REPORT_DIR29 /
        "cell29_trace_alignments.csv"
    ),
    "all_moves": (
        REPORT_DIR29 /
        "cell29_all_alignment_moves.csv"
    ),
    "deviations": (
        REPORT_DIR29 /
        "cell29_deviation_moves.csv"
    ),
    "deviation_frequency": (
        REPORT_DIR29 /
        "cell29_deviation_frequency.csv"
    ),
    "cases_ranked": (
        REPORT_DIR29 /
        "cell29_cases_ranked.csv"
    ),
    "summary": (
        REPORT_DIR29 /
        "cell29_deviation_summary.csv"
    )
}

FRAMES29 = {
    "trace_alignments": audit29_traces,
    "all_moves": audit29_moves,
    "deviations": audit29_deviations,
    "deviation_frequency": deviation_frequency29,
    "cases_ranked": cases_ranked29,
    "summary": summary29
}

for key, path in OUTPUTS29.items():
    _project_csv(FRAMES29[key], 
        path,
        index=False
    )


# ============================================================
# 29.12 — MANIFESTO
# ============================================================

manifest29 = {
    "project": "co2-cems-thermal-modeling",
    "phase": 2,
    "cell": 29,
    "facility": "Iatan",
    "orispl": 6065,
    "unit": "2",
    "year": 2025,
    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "python_version": sys.version.split()[0],
    "pm4py_version": pm4py.__version__,
    "cell27_manifest_sha256": (
        sha256_file29(MANIFEST27_PATH29)
    ),
    "cell28_manifest_sha256": (
        sha256_file29(MANIFEST28_PATH29)
    ),
    "alignment_method": (
        "pm4py_petri_net_alignment"
    ),
    "move_classification": {
        "synchronous": (
            "observed activity matched by visible "
            "model transition"
        ),
        "log_only": (
            "observed activity not matched by "
            "model transition"
        ),
        "model_only": (
            "visible model transition without "
            "observed activity"
        ),
        "silent_model": (
            "invisible model transition, "
            "not counted as visible deviation"
        )
    },
    "test_logs": {
        lt: {
            "n_cases": len(logs29[lt]),
            "sha256": fingerprint_log29(
                logs29[lt]
            )
        }
        for lt in LOG_TYPES29
    },
    "co2_used": False,
    "hmm_retrained": False,
    "process_rediscovered": False,
    "models_modified": False,
    "test_used_for_selection": False,
    "n_traces": int(len(audit29_traces)),
    "n_visible_deviations": int(
        len(audit29_deviations)
    ),
    "output_files_sha256": {
        key: sha256_file29(path)
        for key, path in OUTPUTS29.items()
    },
    "figure_sha256": (
        sha256_file29(FIG29_PATH)
    )
}

MANIFEST29_PATH = (
    REPORT_DIR29 /
    "cell29_deviation_analysis_manifest.json"
)

with open(
    MANIFEST29_PATH,
    "w",
    encoding="utf-8"
) as f:
    _project_dump(
        manifest29,
        f,
        indent=2,
        ensure_ascii=False
    )


# ============================================================
# 29.13 — RESUMO FINAL
# ============================================================

_project_print("\n" + "=" * 78)
_project_print("CÉLULA 29/36 CONCLUÍDA")
_project_print("=" * 78)

_project_display(summary29.round(6))

_project_print("\nAUDITORIA")
_project_print("Modelos congelados preservados: SIM")
_project_print("CO2 utilizado: NÃO")
_project_print("HMM retreinado: NÃO")
_project_print("Processos redescobertos: NÃO")
_project_print("Teste utilizado para seleção: NÃO")

_project_print("\nRelatórios:", REPORT_DIR29)
_project_print("Figura:", FIG29_PATH)
_project_print("Manifesto:", MANIFEST29_PATH)

_project_print("\nPRÓXIMA ETAPA:")
_project_print(
    "CÉLULA 30/36 — CONSOLIDAÇÃO "
    "DOS ATRIBUTOS DE PROCESSO"
)


## 30–31 — Atributos disponíveis no instante t

Os atributos usam estado filtrado, carga e histórico dentro do ciclo. As durações usadas são decorridas até t. A duração total e o alinhamento da trajetória completa são reservados ao diagnóstico retrospectivo. Os lags exportados são candidatos. A validação do prefixo cobre a engenharia de atributos com entradas congeladas. O manifesto guarda os limites de treino e seleção da Fase 2 para comparação explícita com os cortes da modelagem. [Rabiner (1989)](https://doi.org/10.1109/5.18626) · [Aula 08, pp. 53–56, proveniência](#fontes-do-material-local)


In [ ]:

# ============================================================
# FASE 2 — CÉLULA 30/31
# CONSOLIDAÇÃO DOS ATRIBUTOS HORÁRIOS DE PROCESSO
# ============================================================
#
# IATAN | ORISPL 6065 | UNIT 2 | 2025
#
# OBJETIVO
# Consolidar atributos horários de operação, regimes HMM e atividades
# de Process Mining para a análise de dependências e lags na Fase 3.
# Utilizar somente informações disponíveis até a hora t.
#
# FONTES
# - HMM filtrado da Célula 13
# - Semântica operacional das Células 18–20
# - Dados horários de processo das Células 19 e 28
# - Manifestos dos modelos congelados (auditoria)
#
# NÃO INCORPORAR ÀS FEATURES DE PROCESSOS:
# - CO2 observado (será associado na Fase 3, sem alterar os processos)
# - estados Viterbi suavizados com dados futuros
# - duração final de episódios
# - informações futuras do ciclo operacional
# - fitness/alinhamento do ciclo completo
# - estatísticas globais aprendidas com validação ou teste
#
# Os lags pré-computados são históricos candidatos, NÃO lags selecionados.
#
# ============================================================

import json
import hashlib
import sys
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import pm4py

from IPython.display import display

_project_print("=" * 78)
_project_print("FASE 2 — CÉLULA 30/31")
_project_print("CONSOLIDAÇÃO DOS ATRIBUTOS HORÁRIOS DE PROCESSO")
_project_print("=" * 78)


# ============================================================
# 30.1 — CONFIGURAÇÃO
# ============================================================

required30 = [
    "RUN_DIR",
    "hmm13_inference",
    "hmm19_hourly",
    "frame28",
    "RAMP_THRESHOLD_MW19",
    "ramp_activity19"
]

missing30 = [
    name for name in required30
    if name not in globals()
]

if missing30:
    raise RuntimeError(
        "Objetos ausentes: "
        + ", ".join(missing30)
        + "\nExecute as células anteriores na sessão atual."
    )

RUN_DIR30 = Path(RUN_DIR).expanduser().resolve()

REPORT_DIR30 = RUN_DIR30 / "reports"
PROCESSED_DIR30 = RUN_DIR30 / "data" / "processed"
MODEL_DIR30 = RUN_DIR30 / "models" / "cell27_frozen"

REPORT_DIR30.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR30.mkdir(parents=True, exist_ok=True)

PARTITIONS30 = ("train", "validation", "test")
LOG_TYPES30 = ("regime", "ramp", "combined")

MANIFEST27_PATH30 = (
    REPORT_DIR30 / "cell27_frozen_models_manifest.json"
)
MANIFEST28_PATH30 = (
    REPORT_DIR30 / "cell28_test_evaluation_manifest.json"
)
MANIFEST29_PATH30 = (
    REPORT_DIR30 / "cell29_deviation_analysis_manifest.json"
)

for path in (
    MANIFEST27_PATH30,
    MANIFEST28_PATH30,
    MANIFEST29_PATH30
):
    if not path.is_file():
        raise FileNotFoundError(
            f"Manifesto obrigatório ausente: {path}"
        )

if not np.isclose(
    float(RAMP_THRESHOLD_MW19), 20.0
):
    raise RuntimeError(
        "Limiar de rampa diferente de 20 MW/h."
    )


# ============================================================
# 30.2 — FUNÇÕES AUXILIARES
# ============================================================

def sha256_file30(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            h.update(block)
    return h.hexdigest()


def get_partition_frame30(source, partition):
    """
    Recupera uma partição sem reconstruir os dados.
    Aceita dicionário indexado por partição ou DataFrame
    com coluna explícita de partição.
    """
    if isinstance(source, dict):
        if partition not in source:
            raise RuntimeError(
                f"Partição {partition} ausente em hmm19_hourly. "
                f"Chaves disponíveis: {list(source.keys())}"
            )
        obj = source[partition]

    elif isinstance(source, pd.DataFrame):
        partition_col = next(
            (
                c for c in ("partition", "split")
                if c in source.columns
            ),
            None
        )
        if partition_col is None:
            raise RuntimeError(
                "hmm19_hourly não identifica as partições."
            )
        obj = source.loc[
            source[partition_col].astype(str) == partition
        ]
    else:
        raise RuntimeError(
            "Estrutura não reconhecida para hmm19_hourly."
        )

    if not isinstance(obj, pd.DataFrame):
        raise RuntimeError(
            f"Partição {partition} não é DataFrame: "
            f"{type(obj).__name__}"
        )

    return obj.copy().reset_index(drop=True)


def resolve_column30(df, candidates, description):
    found = [c for c in candidates if c in df.columns]

    if not found:
        raise RuntimeError(
            f"Coluna de {description} não encontrada.\n"
            f"Candidatas: {candidates}\n"
            f"Disponíveis: {list(df.columns)}"
        )

    return found[0]


def get_inference_array30(inference, key, n):
    if key not in inference:
        raise RuntimeError(
            f"Campo HMM obrigatório ausente: {key}"
        )

    arr = np.asarray(inference[key])

    if len(arr) != n:
        raise RuntimeError(
            f"Comprimento incompatível em {key}: "
            f"{len(arr)} != {n}"
        )

    return arr


def safe_number30(values, description):
    result = pd.to_numeric(
        values, errors="coerce"
    )

    if result.isna().any():
        raise RuntimeError(
            f"Valores ausentes/não numéricos: {description}"
        )

    if not np.isfinite(result.to_numpy(dtype=float)).all():
        raise RuntimeError(
            f"Valores não finitos: {description}"
        )

    return result


# ============================================================
# 30.3 — AUDITORIA DOS MANIFESTOS
# ============================================================

with open(
    MANIFEST27_PATH30, encoding="utf-8"
) as f:
    manifest27_30 = json.load(f)

with open(
    MANIFEST28_PATH30, encoding="utf-8"
) as f:
    manifest28_30 = json.load(f)

with open(
    MANIFEST29_PATH30, encoding="utf-8"
) as f:
    manifest29_30 = json.load(f)

if manifest27_30.get("models_frozen") is not True:
    raise RuntimeError("Modelos não congelados.")

if manifest28_30.get("n_test_models") != 3:
    raise RuntimeError("Avaliação de teste incompleta.")

if (
    manifest28_30.get("selection_manifest_sha256")
    != sha256_file30(MANIFEST27_PATH30)
):
    raise RuntimeError(
        "Manifesto 27 diverge do registrado na Célula 28."
    )

if (
    manifest29_30.get("cell28_manifest_sha256")
    != sha256_file30(MANIFEST28_PATH30)
):
    raise RuntimeError(
        "Manifesto 28 diverge do registrado na Célula 29."
    )

for partition_manifest in (
    manifest28_30,
    manifest29_30
):
    if partition_manifest.get("co2_used") is not False:
        raise RuntimeError(
            "Auditoria de exclusão de CO2 não confirmada."
        )

_project_print("\n30.3 — Manifestos 27–29 verificados")


# ============================================================
# 30.4 — RECUPERAÇÃO DOS DADOS HORÁRIOS
# ============================================================

hourly_sources30 = {}

for partition in PARTITIONS30:

    if partition == "test":
        raw = frame28.copy().reset_index(drop=True)
    else:
        raw = get_partition_frame30(
            hmm19_hourly, partition
        )

    if raw.empty:
        raise RuntimeError(
            f"Dados horários vazios: {partition}"
        )

    hourly_sources30[partition] = raw

_project_print("\n30.4 — Dados horários recuperados")

for partition, raw in hourly_sources30.items():
    _project_print(
        f"{partition:10s}: {len(raw):6d} linhas"
    )


# ============================================================
# 30.5 — NORMALIZAÇÃO E ALINHAMENTO HMM
# ============================================================

hourly_normalized30 = {}
audit_sources30 = []

for partition in PARTITIONS30:

    raw = hourly_sources30[partition]
    inference = hmm13_inference.get(partition)

    if not isinstance(inference, dict):
        raise RuntimeError(
            f"Inferência HMM ausente: {partition}"
        )

    n = len(raw)

    timestamp_col = resolve_column30(
        raw,
        ("timestamp19", "timestamp", "datetime"),
        "timestamp"
    )

    load_col = resolve_column30(
        raw,
        ("load_mw19", "load_mw", "gross_load_mw"),
        "carga MW"
    )

    # Estados causais da Célula 13.
    states = get_inference_array30(
        inference, "filtered_state", n
    ).astype(int)

    confidence = get_inference_array30(
        inference, "filtered_confidence", n
    ).astype(float)

    entropy = get_inference_array30(
        inference, "filtered_entropy", n
    ).astype(float)

    lengths = np.asarray(
        inference["lengths"], dtype=int
    )

    cycle_index = get_inference_array30(
        inference, "cycle_index", n
    ).astype(int)

    if (
        len(lengths) == 0
        or np.any(lengths <= 0)
        or int(lengths.sum()) != n
    ):
        raise RuntimeError(
            f"Ciclos HMM inconsistentes: {partition}"
        )

    expected_cycle_index = np.repeat(
        np.arange(1, len(lengths) + 1),
        lengths
    )

    if not np.array_equal(
        cycle_index, expected_cycle_index
    ):
        raise RuntimeError(
            f"Índice de ciclos divergente: {partition}"
        )

    if (
        np.any(states < 0)
        or np.any(states >= 5)
        or not np.isfinite(confidence).all()
        or not np.isfinite(entropy).all()
    ):
        raise RuntimeError(
            f"Estados ou diagnósticos HMM inválidos: {partition}"
        )

    if np.any(
        (confidence < 0) | (confidence > 1)
    ):
        raise RuntimeError(
            f"Confiança fora de [0,1]: {partition}"
        )

    timestamps = pd.to_datetime(
        raw[timestamp_col],
        utc=True,
        errors="raise"
    )

    if (
        timestamps.isna().any()
        or timestamps.duplicated().any()
        or not timestamps.is_monotonic_increasing
    ):
        raise RuntimeError(
            f"Timestamps inválidos: {partition}"
        )

    load = safe_number30(
        raw[load_col],
        f"carga {partition}"
    ).to_numpy(dtype=float)

    # Conferência contra as colunas existentes,
    # quando a etapa anterior já as registrou.
    if "state" in raw.columns:
        old = pd.to_numeric(
            raw["state"], errors="raise"
        ).to_numpy(dtype=int)

        if not np.array_equal(old, states):
            raise RuntimeError(
                f"Estado HMM divergente: {partition}"
            )

    if "cycle_index_hmm" in raw.columns:
        old_cycles = pd.to_numeric(
            raw["cycle_index_hmm"],
            errors="raise"
        ).to_numpy(dtype=int)

        if not np.array_equal(
            old_cycles, cycle_index
        ):
            raise RuntimeError(
                f"Ciclos divergentes: {partition}"
            )

    case_ids = np.array(
        [
            f"{partition}_ON_{idx:04d}"
            for idx in cycle_index
        ],
        dtype=object
    )

    if "case_id" in raw.columns:
        if not np.array_equal(
            raw["case_id"].astype(str).to_numpy(),
            case_ids.astype(str)
        ):
            raise RuntimeError(
                f"case_id divergente: {partition}"
            )

    df = pd.DataFrame({
        "partition": partition,
        "timestamp": timestamps,
        "case_id": case_ids,
        "cycle_index": cycle_index,
        "load_mw": load,
        "hmm_state": states,
        "hmm_confidence": confidence,
        "hmm_entropy": entropy
    })

    # O vínculo posicional é aceito apenas quando
    # os dados horários foram preservados na mesma
    # ordem da inferência HMM original.
    #
    # A Célula 19 já realizou o alinhamento original;
    # este bloco não reconstrói timestamps do zero.

    for case_id, group in df.groupby(
        "case_id", sort=False
    ):
        diffs = group["timestamp"].diff().dropna()

        if not (
            diffs == pd.Timedelta(hours=1)
        ).all():
            raise RuntimeError(
                f"Lacuna temporal: {case_id}"
            )

    hourly_normalized30[partition] = df

    audit_sources30.append({
        "partition": partition,
        "n_hours": n,
        "n_cycles": len(lengths),
        "first_timestamp": str(timestamps.iloc[0]),
        "last_timestamp": str(timestamps.iloc[-1]),
        "source_timestamp_column": timestamp_col,
        "source_load_column": load_col,
        "hmm_state_source": "filtered_state",
        "temporal_continuity_ok": True
    })

audit_sources30 = pd.DataFrame(
    audit_sources30
)

_project_print("\n30.5 — Alinhamento horário")
_project_display(audit_sources30)


# ============================================================
# 30.6 — CONSTRUÇÃO DOS ATRIBUTOS CAUSAIS
# ============================================================

def build_causal_features30(df):

    df = df.copy()

    df = df.sort_values(
        ["cycle_index", "timestamp"],
        kind="stable"
    ).reset_index(drop=True)

    group = df.groupby(
        "case_id", sort=False
    )

    # --------------------------------------------------------
    # Tempo decorrido desde o início do ciclo
    # --------------------------------------------------------

    df["hours_since_cycle_start"] = (
        group.cumcount().astype(int)
    )

    # --------------------------------------------------------
    # Variação de carga: somente t e t-1
    # --------------------------------------------------------

    df["delta_load_mw"] = (
        group["load_mw"].diff()
    )

    df["ramp_defined"] = (
        df["delta_load_mw"].notna()
    ).astype(int)

    df["ramp_activity"] = (
        df["delta_load_mw"]
        .apply(ramp_activity19)
        .astype(str)
    )

    # --------------------------------------------------------
    # Estado e atividades atuais
    # --------------------------------------------------------

    df["regime_activity"] = (
        "REGIME_S"
        + df["hmm_state"].astype(str)
    )

    df["combined_activity"] = (
        df["regime_activity"]
        + "__"
        + df["ramp_activity"]
    )

    # --------------------------------------------------------
    # Histórico de estados e transições
    # --------------------------------------------------------

    df["previous_hmm_state"] = (
        group["hmm_state"].shift(1)
    )

    df["has_previous_state"] = (
        df["previous_hmm_state"].notna()
    ).astype(int)

    df["state_changed"] = (
        (
            df["hmm_state"]
            != df["previous_hmm_state"]
        )
        &
        df["previous_hmm_state"].notna()
    ).astype(int)

    df["cumulative_state_changes"] = (
        df.groupby(
            "case_id", sort=False
        )["state_changed"].cumsum()
    )

    # --------------------------------------------------------
    # Duração transcorrida no estado atual
    # Não utiliza a duração final do episódio.
    # --------------------------------------------------------

    change_block = (
        df.groupby(
            "case_id", sort=False
        )["state_changed"].cumsum()
    )

    df["hours_in_current_state"] = (
        df.groupby(
            ["case_id", change_block],
            sort=False
        ).cumcount() + 1
    )

    # --------------------------------------------------------
    # Duração transcorrida na atividade de rampa
    # --------------------------------------------------------

    previous_ramp = (
        df.groupby(
            "case_id", sort=False
        )["ramp_activity"].shift(1)
    )

    ramp_changed = (
        df["ramp_activity"] != previous_ramp
    ).astype(int)

    ramp_block = (
        ramp_changed.groupby(
            df["case_id"], sort=False
        ).cumsum()
    )

    df["hours_in_current_ramp"] = (
        df.groupby(
            ["case_id", ramp_block],
            sort=False
        ).cumcount() + 1
    )

    # --------------------------------------------------------
    # Estatísticas retrospectivas
    # Incluem a hora t, nunca t+1.
    # --------------------------------------------------------

    for window in (3, 6, 12, 24):

        df[f"load_mean_last_{window}h"] = (
            df.groupby(
                "case_id", sort=False
            )["load_mw"]
            .transform(
                lambda s: s.rolling(
                    window=window,
                    min_periods=1
                ).mean()
            )
        )

        df[f"load_std_last_{window}h"] = (
            df.groupby(
                "case_id", sort=False
            )["load_mw"]
            .transform(
                lambda s: s.rolling(
                    window=window,
                    min_periods=2
                ).std(ddof=0)
            )
            .fillna(0.0)
        )

        df[f"state_changes_last_{window}h"] = (
            df.groupby(
                "case_id", sort=False
            )["state_changed"]
            .transform(
                lambda s: s.rolling(
                    window=window,
                    min_periods=1
                ).sum()
            )
        )

    # --------------------------------------------------------
    # Lags históricos candidatos: nunca cruzam ciclos
    # Não representam seleção estatística de defasagens.
    # --------------------------------------------------------

    for lag in (1, 2, 3, 6, 12, 24):

        df[f"load_lag_{lag}h"] = (
            df.groupby(
                "case_id", sort=False
            )["load_mw"].shift(lag)
        )

        df[f"hmm_state_lag_{lag}h"] = (
            df.groupby(
                "case_id", sort=False
            )["hmm_state"].shift(lag)
        )

    # --------------------------------------------------------
    # Indicadores temporais conhecidos em t
    # --------------------------------------------------------

    df["hour_of_day"] = (
        df["timestamp"].dt.hour
    )

    df["day_of_week"] = (
        df["timestamp"].dt.dayofweek
    )

    df["month"] = (
        df["timestamp"].dt.month
    )

    df["hour_sin"] = np.sin(
        2 * np.pi * df["hour_of_day"] / 24
    )

    df["hour_cos"] = np.cos(
        2 * np.pi * df["hour_of_day"] / 24
    )

    # --------------------------------------------------------
    # Indicadores das atividades atuais
    # Categorias definidas sem observar distribuições
    # de validação ou teste.
    # --------------------------------------------------------

    for state in range(5):
        df[f"is_regime_s{state}"] = (
            df["hmm_state"] == state
        ).astype(int)

    ramp_categories = (
        "RAMP_UP",
        "RAMP_DOWN",
        "RAMP_STABLE",
        "RAMP_UNDEFINED"
    )

    for activity in ramp_categories:
        df[
            f"is_{activity.lower()}"
        ] = (
            df["ramp_activity"] == activity
        ).astype(int)

    return df


features_by_partition30 = {}

for partition in PARTITIONS30:

    features_by_partition30[partition] = (
        build_causal_features30(
            hourly_normalized30[partition]
        )
    )

_project_print("\n30.6 — Atributos causais construídos")


# ============================================================
# 30.7 — AUDITORIA CONTRA OS LOGS DE TESTE
# ============================================================
#
# A atividade horária consolidada deve ser idêntica
# à utilizada na Célula 28.
#
# Os resultados de alinhamento da Célula 29 NÃO
# são incorporados às variáveis explicativas.
#
# ============================================================

test30 = features_by_partition30["test"]

for col in (
    "regime_activity",
    "ramp_activity",
    "combined_activity"
):
    if col not in frame28.columns:
        raise RuntimeError(
            f"Coluna ausente em frame28: {col}"
        )

    expected = (
        frame28[col]
        .astype(str)
        .reset_index(drop=True)
    )

    observed = (
        test30[col]
        .astype(str)
        .reset_index(drop=True)
    )

    if not observed.equals(expected):
        raise RuntimeError(
            f"Atividades divergentes da Célula 28: {col}"
        )

expected_test_hours30 = {
    int(v["covered_hours"])
    for v in manifest28_30["test_logs"].values()
}

if expected_test_hours30 != {len(test30)}:
    raise RuntimeError(
        "Cobertura horária do teste diverge "
        "do manifesto da Célula 28."
    )

_project_print("\n30.7 — Atividades de teste verificadas")


# ============================================================
# 30.8 — VALIDAÇÃO DAS FRONTEIRAS DOS CICLOS
# ============================================================

for partition, df in features_by_partition30.items():

    first = (
        df.groupby(
            "case_id", sort=False
        ).head(1)
    )

    if not (
        first["hours_since_cycle_start"] == 0
    ).all():
        raise RuntimeError(
            f"Contador inicial inválido: {partition}"
        )

    if not (
        first["hours_in_current_state"] == 1
    ).all():
        raise RuntimeError(
            f"Duração inicial de estado inválida: {partition}"
        )

    if not (
        first["ramp_activity"]
        == "RAMP_UNDEFINED"
    ).all():
        raise RuntimeError(
            f"Rampa inicial inválida: {partition}"
        )

    if not (
        first["cumulative_state_changes"] == 0
    ).all():
        raise RuntimeError(
            f"Transições iniciais inválidas: {partition}"
        )

    for lag in (1, 2, 3, 6, 12, 24):
        if not first[
            f"load_lag_{lag}h"
        ].isna().all():
            raise RuntimeError(
                f"Vazamento entre ciclos no lag {lag}: "
                f"{partition}"
            )

    if (
        df["hours_in_current_state"] < 1
    ).any():
        raise RuntimeError(
            f"Duração de estado inválida: {partition}"
        )

    if (
        df["hours_in_current_ramp"] < 1
    ).any():
        raise RuntimeError(
            f"Duração de rampa inválida: {partition}"
        )

_project_print("\n30.8 — Fronteiras dos ciclos verificadas")


# ============================================================
# 30.9 — DEFINIÇÃO EXPLÍCITA DAS FEATURES
# ============================================================

metadata_columns30 = [
    "partition",
    "timestamp",
    "case_id",
    "cycle_index"
]

categorical_columns30 = [
    "regime_activity",
    "ramp_activity",
    "combined_activity"
]

numeric_feature_columns30 = [
    "load_mw",
    "hmm_state",
    "hmm_confidence",
    "hmm_entropy",
    "hours_since_cycle_start",
    "ramp_defined",
    "delta_load_mw",
    "has_previous_state",
    "previous_hmm_state",
    "state_changed",
    "cumulative_state_changes",
    "hours_in_current_state",
    "hours_in_current_ramp",
    "hour_of_day",
    "day_of_week",
    "month",
    "hour_sin",
    "hour_cos"
]

for window in (3, 6, 12, 24):
    numeric_feature_columns30.extend([
        f"load_mean_last_{window}h",
        f"load_std_last_{window}h",
        f"state_changes_last_{window}h"
    ])

for lag in (1, 2, 3, 6, 12, 24):
    numeric_feature_columns30.extend([
        f"load_lag_{lag}h",
        f"hmm_state_lag_{lag}h"
    ])

numeric_feature_columns30.extend(
    f"is_regime_s{s}" for s in range(5)
)

numeric_feature_columns30.extend([
    "is_ramp_up",
    "is_ramp_down",
    "is_ramp_stable",
    "is_ramp_undefined"
])

# Identificadores permanecem como metadados.
# CO2 observado será associado posteriormente, na Fase 3.
# Não incluir fitness, precision, F1, alinhamentos ou duração final.

feature_columns30 = (
    numeric_feature_columns30
    + categorical_columns30
)

for partition, df in features_by_partition30.items():

    missing = (
        set(metadata_columns30 + feature_columns30)
        - set(df.columns)
    )

    if missing:
        raise RuntimeError(
            f"Features ausentes em {partition}: "
            f"{sorted(missing)}"
        )

    prohibited = (
        "co2",
        "emission",
        "fitness",
        "precision",
        "alignment",
        "final_duration",
        "future",
        "viterbi"
    )

    for col in feature_columns30:
        if any(
            term in col.lower()
            for term in prohibited
        ):
            raise RuntimeError(
                f"Feature proibida: {col}"
            )

_project_print(
    "\n30.9 — Total de features candidatas:",
    len(feature_columns30)
)


# ============================================================
# 30.10 — MATRIZ HORÁRIA CONSOLIDADA
# ============================================================

ordered_columns30 = (
    metadata_columns30
    + feature_columns30
)

frames30 = []

for partition in PARTITIONS30:

    df = features_by_partition30[partition]

    frames30.append(
        df[ordered_columns30].copy()
    )

consolidated30 = pd.concat(
    frames30,
    ignore_index=True
)

if consolidated30.duplicated(
    subset=["partition", "timestamp"]
).any():
    raise RuntimeError(
        "Duplicação de timestamp por partição."
    )

if len(consolidated30) != sum(
    len(df) for df in frames30
):
    raise RuntimeError(
        "Perda de linhas na consolidação."
    )

_project_print("\n30.10 — Matriz consolidada")
_project_print("Dimensões:", consolidated30.shape)

_project_display(
    consolidated30.groupby(
        "partition",
        as_index=False
    ).agg(
        n_hours=("timestamp", "count"),
        n_cycles=("case_id", "nunique"),
        n_regimes=("regime_activity", "nunique"),
        n_ramp_activities=(
            "ramp_activity", "nunique"
        ),
        n_combined_activities=(
            "combined_activity", "nunique"
        )
    )
)


# ============================================================
# 30.11 — AUDITORIA DE VALORES AUSENTES
# ============================================================

missing_rows30 = []

for partition in PARTITIONS30:

    df = features_by_partition30[partition]

    for col in numeric_feature_columns30:

        missing_rows30.append({
            "partition": partition,
            "feature": col,
            "n_missing": int(
                df[col].isna().sum()
            ),
            "missing_fraction": float(
                df[col].isna().mean()
            )
        })

missing_audit30 = pd.DataFrame(
    missing_rows30
)

# Valores ausentes esperados:
# - delta_load_mw na primeira hora do ciclo
# - previous_hmm_state na primeira hora
# - lags antes de existir histórico suficiente
#
# Nenhuma imputação baseada em teste é feita aqui.

expected_missing30 = {
    "delta_load_mw",
    "previous_hmm_state"
}

expected_missing30.update(
    f"load_lag_{lag}h"
    for lag in (1, 2, 3, 6, 12, 24)
)

expected_missing30.update(
    f"hmm_state_lag_{lag}h"
    for lag in (1, 2, 3, 6, 12, 24)
)

unexpected_missing30 = missing_audit30.loc[
    (missing_audit30["n_missing"] > 0)
    &
    (~missing_audit30["feature"].isin(
        expected_missing30
    ))
]

if not unexpected_missing30.empty:
    _project_display(unexpected_missing30)
    raise RuntimeError(
        "Valores ausentes inesperados encontrados."
    )

_project_print("\n30.11 — Auditoria de valores ausentes")
_project_display(
    missing_audit30.loc[
        missing_audit30["n_missing"] > 0
    ]
)


# ============================================================
# 30.12 — EXPORTAÇÃO DAS PARTIÇÕES
# ============================================================

output_paths30 = {}

for partition in PARTITIONS30:

    path = (
        PROCESSED_DIR30 /
        f"cell30_process_features_{partition}.parquet"
    )

    features_by_partition30[partition][
        ordered_columns30
    ].to_parquet(
        path,
        index=False
    )

    output_paths30[partition] = path

    _project_print(
        f"{partition.upper():10s}: {path.name}"
    )

consolidated_path30 = (
    PROCESSED_DIR30 /
    "cell30_process_features_all.parquet"
)

consolidated30.to_parquet(
    consolidated_path30,
    index=False
)

output_paths30["all"] = consolidated_path30


# ============================================================
# 30.13 — DICIONÁRIO DE ATRIBUTOS
# ============================================================

dictionary_rows30 = []

descriptions30 = {
    "load_mw": "Carga operacional observada na hora t",
    "hmm_state": "Estado HMM filtrado na hora t",
    "hmm_confidence": "Confiança do estado filtrado",
    "hmm_entropy": "Entropia da distribuição filtrada",
    "hours_since_cycle_start": (
        "Horas decorridas desde o início do ciclo ON"
    ),
    "delta_load_mw": (
        "Variação de carga entre t e t-1"
    ),
    "state_changed": (
        "Indicador de mudança de estado em t"
    ),
    "cumulative_state_changes": (
        "Número acumulado de mudanças até t"
    ),
    "hours_in_current_state": (
        "Duração transcorrida do estado atual até t"
    ),
    "hours_in_current_ramp": (
        "Duração transcorrida da atividade de rampa até t"
    ),
    "regime_activity": (
        "Atividade de regime conhecida em t"
    ),
    "ramp_activity": (
        "Atividade de rampa conhecida em t"
    ),
    "combined_activity": (
        "Combinação de regime e rampa em t"
    )
}

# Taxonomia documental, sem modificar os valores das 54 features.
# Categorias de origem e derivação não são necessariamente disjuntas
# do ponto de vista físico; a classificação aqui é operacional.
def feature_group30(col):
    if (col == "load_mw" or col == "delta_load_mw"
        or col.startswith("load_mean_last_")
        or col.startswith("load_std_last_")
        or col.startswith("load_lag_")
        or col in ("ramp_defined", "hour_of_day", "day_of_week",
                   "month", "hour_sin", "hour_cos")):
        return "operational"
    if (col in ("hmm_state", "hmm_confidence", "hmm_entropy",
                "previous_hmm_state", "has_previous_state",
                "state_changed", "cumulative_state_changes",
                "hours_in_current_state")
        or col.startswith("hmm_state_lag_")
        or col.startswith("state_changes_last_")
        or col.startswith("is_regime_s")):
        return "hmm_regimes"
    if (col in ("regime_activity", "ramp_activity",
                "combined_activity", "hours_in_current_ramp")
        or col.startswith("is_ramp_")):
        return "process_activities"
    if col == "hours_since_cycle_start":
        return "cycle_context"
    raise RuntimeError(f"Feature sem classificação: {col}")

def lag_status30(col):
    if col.startswith(("load_lag_", "hmm_state_lag_")):
        return "candidate_not_selected"
    if col.startswith(("load_mean_last_", "load_std_last_",
                       "state_changes_last_")):
        return "historical_window_not_selected"
    return "not_applicable"

for col in feature_columns30:

    if col in categorical_columns30:
        kind = "categorical"
    else:
        kind = "numeric"

    dictionary_rows30.append({
        "feature": col,
        "type": kind,
        "feature_group": feature_group30(col),
        "lag_status": lag_status30(col),
        "description": descriptions30.get(
            col,
            "Atributo histórico causal calculado até t"
        ),
        "available_at": "t",
        "target_used": False,
        "uses_future_information": False
    })

feature_dictionary30 = pd.DataFrame(
    dictionary_rows30
)

DICTIONARY_PATH30 = (
    REPORT_DIR30 /
    "cell30_feature_dictionary.csv"
)

_project_csv(feature_dictionary30, 
    DICTIONARY_PATH30,
    index=False
)

MISSING_PATH30 = (
    REPORT_DIR30 /
    "cell30_missingness_audit.csv"
)

_project_csv(missing_audit30, 
    MISSING_PATH30,
    index=False
)

SOURCES_PATH30 = (
    REPORT_DIR30 /
    "cell30_source_audit.csv"
)

_project_csv(audit_sources30, 
    SOURCES_PATH30,
    index=False
)


# ============================================================
# 30.14 — TESTE DE CAUSALIDADE COMPUTACIONAL
# ============================================================
#
# Recalcula as features utilizando somente um prefixo
# temporal de cada partição.
#
# As features das horas anteriores devem permanecer
# idênticas após a inclusão das horas posteriores.
#
# Esse teste detecta dependência futura na engenharia
# de atributos desta célula.
#
# Não substitui a auditoria da inferência HMM original.
#
# ============================================================

causality_rows30 = []

for partition in PARTITIONS30:

    full = features_by_partition30[partition]

    for case_id, case_df in (
        hourly_normalized30[partition]
        .groupby("case_id", sort=False)
    ):

        if len(case_df) < 4:
            continue

        prefix_length = max(
            2,
            len(case_df) // 2
        )

        prefix_input = (
            case_df.iloc[:prefix_length].copy()
        )

        prefix_features = build_causal_features30(
            prefix_input
        )

        full_prefix = (
            full.loc[
                full["case_id"] == case_id
            ]
            .iloc[:prefix_length]
            .reset_index(drop=True)
        )

        cols = numeric_feature_columns30 + (
            categorical_columns30
        )

        pd.testing.assert_frame_equal(
            prefix_features[cols].reset_index(
                drop=True
            ),
            full_prefix[cols],
            check_dtype=False,
            check_exact=False,
            rtol=1e-12,
            atol=1e-12
        )

        causality_rows30.append({
            "partition": partition,
            "case_id": case_id,
            "prefix_hours": prefix_length,
            "prefix_invariance": True
        })

causality_audit30 = pd.DataFrame(
    causality_rows30
)

CAUSALITY_PATH30 = (
    REPORT_DIR30 /
    "cell30_prefix_causality_audit.csv"
)

_project_csv(causality_audit30, 
    CAUSALITY_PATH30,
    index=False
)

_project_print("\n30.14 — Teste de invariância de prefixo")
_project_print(
    "Ciclos auditados:",
    len(causality_audit30)
)
_project_print("Divergências encontradas: 0")


# ============================================================
# 30.15 — MANIFESTO DE REPRODUTIBILIDADE
# ============================================================

manifest30 = {
    "project": "co2-cems-thermal-modeling",
    "phase": 2,
    "cell": 30,
    "facility": "Iatan",
    "orispl": 6065,
    "unit": "2",
    "year": 2025,
    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "python_version": sys.version.split()[0],
    "pm4py_version": pm4py.__version__,
    "source_manifest_sha256": {
        "cell27": sha256_file30(
            MANIFEST27_PATH30
        ),
        "cell28": sha256_file30(
            MANIFEST28_PATH30
        ),
        "cell29": sha256_file30(
            MANIFEST29_PATH30
        )
    },
    "partitions": {
        partition: {
            "n_hours": int(
                len(features_by_partition30[partition])
            ),
            "n_cycles": int(
                features_by_partition30[
                    partition
                ]["case_id"].nunique()
            )
        }
        for partition in PARTITIONS30
    },
    "n_features": len(feature_columns30),
    "feature_columns": feature_columns30,
    "metadata_columns": metadata_columns30,
    "hmm_inference": "filtered_state",
    "run_id": RUN_DIR.name,
    "hmm_temporal_provenance": {
        "scaler_and_hmm_fit_partition": "train",
        "model_selection_partition": "validation",
        "fit_end": pd.Timestamp(train_end).isoformat(),
        "selection_end": pd.Timestamp(validation_end).isoformat(),
        "source_timestamp_convention": (
            "Mesma convenção horária da Célula 06. O tratamento de "
            "timestamps sem fuso segue o contrato de exportação original."
        ),
        "hmm_model_sha256": HMM_SELECTION_12["artifacts"]["hmm_sha256"],
        "scaler_sha256": HMM_SELECTION_12["artifacts"]["scaler_sha256"],
        "note": (
            "Limites da execução Fase 2. Comparar com cada corte externo "
            "antes de interpretar desempenho fora da amostra."
        ),
    },
    "ramp_threshold_mw": float(
        RAMP_THRESHOLD_MW19
    ),
    "causal_feature_policy": (
        "information_available_up_to_t_only"
    ),
    "scientific_scope": "process_mining_features_for_temporal_lag_analysis",
    "next_phase": "phase_3_temporal_lag_analysis",
    "lag_selection_performed": False,
    "precomputed_lags_are_candidates": True,
    "co2_join_policy": "join_observed_co2_in_phase_3_by_timestamp_and_unit",
    "feature_groups": {
        group: [c for c in feature_columns30 if feature_group30(c) == group]
        for group in ("operational", "hmm_regimes",
                      "process_activities", "cycle_context")
    },
    "target_created": False,
    "predictive_model_trained": False,
    "co2_used": False,
    "hmm_retrained": False,
    "scaler_refitted": False,
    "process_rediscovered": False,
    "petri_models_modified": False,
    "test_used_for_selection": False,
    "full_trace_alignment_as_feature": False,
    "final_episode_duration_as_feature": False,
    "prefix_invariance_passed": True,
    "output_sha256": {
        name: sha256_file30(path)
        for name, path in output_paths30.items()
    },
    "audit_sha256": {
        "dictionary": sha256_file30(
            DICTIONARY_PATH30
        ),
        "missingness": sha256_file30(
            MISSING_PATH30
        ),
        "sources": sha256_file30(
            SOURCES_PATH30
        ),
        "causality": sha256_file30(
            CAUSALITY_PATH30
        )
    }
}

MANIFEST30_PATH = (
    REPORT_DIR30 /
    "cell30_process_features_manifest.json"
)

with open(
    MANIFEST30_PATH,
    "w",
    encoding="utf-8"
) as f:
    _project_dump(
        manifest30,
        f,
        indent=2,
        ensure_ascii=False
    )


# ============================================================
# 30.16 — RESUMO FINAL
# ============================================================

_project_print("\n" + "=" * 78)
_project_print("CÉLULA 30/31 CONCLUÍDA")
_project_print("=" * 78)

_project_print("\nDIMENSÕES POR PARTIÇÃO")

for partition in PARTITIONS30:

    df = features_by_partition30[partition]

    _project_print(
        f"{partition.upper():10s} | "
        f"{len(df):6d} horas | "
        f"{df['case_id'].nunique():3d} ciclos"
    )

_project_print(
    "\nTotal de features candidatas:",
    len(feature_columns30)
)

_project_print("\nAUDITORIA METODOLÓGICA")
_project_print("Estados HMM filtrados: SIM")
_project_print("Features retrospectivas causais: SIM")
_project_print("Duração final dos episódios: NÃO")
_project_print("Alinhamentos completos como features: NÃO")
_project_print("CO2 utilizado: NÃO")
_project_print("HMM retreinado: NÃO")
_project_print("Modelos de Petri alterados: NÃO")
_project_print("Teste usado para seleção: NÃO")
_project_print("Invariância de prefixo: APROVADA")
_project_print("Seleção de lags: NÃO — apenas candidatos históricos")
_project_print("Modelagem preditiva de CO2: NÃO")
_project_print("Próxima fase: análise de dependências e lags temporais")

_project_print("\nARQUIVOS")
_project_print("Dados:", PROCESSED_DIR30)
_project_print("Relatórios:", REPORT_DIR30)
_project_print("Manifesto:", MANIFEST30_PATH)

_project_print("\nPRÓXIMA ETAPA:")
_project_print(
    "CÉLULA 31/31 — ENCERRAMENTO METODOLÓGICO "
    "E CONTRATO DE DADOS PARA A FASE 3"
)


In [ ]:

# ============================================================
# FASE 2 — CÉLULA 31/31
# ENCERRAMENTO METODOLÓGICO E CONTRATO PARA A FASE 3
# IATAN | ORISPL 6065 | UNIT 2 | 2025
# ============================================================
# Objetivo: verificar e documentar artefatos da Célula 30.
# Não calcula lags novos, não seleciona defasagens,
# não usa CO2 e não treina nem ajusta modelos.
# Executar após a Célula 30 revisada, no mesmo Colab.

import json
import hashlib
from pathlib import Path
from datetime import datetime, timezone
import pandas as pd
import numpy as np
from IPython.display import display

_project_print('=' * 78)
_project_print('FASE 2 — CÉLULA 31/31 | ENCERRAMENTO E CONTRATO PARA A FASE 3')
_project_print('=' * 78)

# 31.1 — Localização dos artefatos (sem reconstruir resultados)
if 'RUN_DIR' not in globals():
    raise RuntimeError('RUN_DIR ausente. Execute as células anteriores.')
RUN_DIR31 = Path(RUN_DIR).expanduser().resolve()
PROCESSED31 = RUN_DIR31 / 'data' / 'processed'
REPORTS31 = RUN_DIR31 / 'reports'
MANIFEST30_31 = REPORTS31 / 'cell30_process_features_manifest.json'
if not MANIFEST30_31.is_file():
    raise FileNotFoundError(f'Manifesto da Célula 30 ausente: {MANIFEST30_31}')


def sha256_31(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()


def assert_31(condition, message):
    if not condition:
        raise RuntimeError('AUDITORIA 31: ' + message)

with open(MANIFEST30_31, encoding='utf-8') as f:
    m30 = json.load(f)

assert_31(m30.get('phase') == 2 and m30.get('cell') == 30,
          'Manifesto não corresponde à Fase 2 / Célula 30.')
assert_31(m30.get('orispl') == 6065 and str(m30.get('unit')) == '2'
          and m30.get('year') == 2025, 'Identificação da unidade divergente.')
for field in ('co2_used', 'target_created', 'predictive_model_trained',
              'lag_selection_performed', 'test_used_for_selection'):
    assert_31(m30.get(field) is False, f'Política metodológica inválida: {field}')
assert_31(m30.get('prefix_invariance_passed') is True,
          'Invariância de prefixo não confirmada.')
assert_31(m30.get('hmm_inference') == 'filtered_state',
          'Origem causal dos estados HMM não confirmada.')

# 31.2 — Integridade criptográfica dos arquivos de entrada
paths31 = {
    p: PROCESSED31 / f'cell30_process_features_{p}.parquet'
    for p in ('train', 'validation', 'test')
}
paths31['all'] = PROCESSED31 / 'cell30_process_features_all.parquet'
checks31 = {
    'dictionary': REPORTS31 / 'cell30_feature_dictionary.csv',
    'missingness': REPORTS31 / 'cell30_missingness_audit.csv',
    'sources': REPORTS31 / 'cell30_source_audit.csv',
    'causality': REPORTS31 / 'cell30_prefix_causality_audit.csv',
}
for label, path in {**paths31, **checks31}.items():
    assert_31(path.is_file(), f'Arquivo ausente: {label} — {path}')
for label, path in paths31.items():
    assert_31(sha256_31(path) == m30['output_sha256'].get(label),
              f'SHA-256 divergente: {path.name}')
for label, path in checks31.items():
    assert_31(sha256_31(path) == m30['audit_sha256'].get(label),
              f'SHA-256 divergente: {path.name}')
for cell, digest in m30.get('source_manifest_sha256', {}).items():
    names = {
        'cell27': 'cell27_frozen_models_manifest.json',
        'cell28': 'cell28_test_evaluation_manifest.json',
        'cell29': 'cell29_deviation_analysis_manifest.json',
    }
    if cell in names:
        p = REPORTS31 / names[cell]
        assert_31(p.is_file() and sha256_31(p) == digest,
                  f'Cadeia de proveniência divergente: {cell}')
_project_print('\n31.2 — Integridade SHA-256 e proveniência: APROVADAS')

# 31.3 — Esquema e consistência das séries horárias
meta31 = list(m30['metadata_columns'])
features31 = list(m30['feature_columns'])
expected31 = meta31 + features31
assert_31(len(features31) == len(set(features31)) == m30['n_features'],
          'Contagem ou unicidade das features divergente.')
assert_31(all(c in meta31 for c in ('partition', 'timestamp', 'case_id', 'cycle_index')),
          'Chaves de identificação incompletas.')
assert_31(not any('co2' in c.lower() or 'emission' in c.lower()
                  for c in features31), 'CO2/emissões presentes nas features.')

frames31 = {}
summary31 = []
for part in ('train', 'validation', 'test'):
    df = pd.read_parquet(paths31[part])
    assert_31(list(df.columns) == expected31,
              f'Esquema/ordem de colunas divergente: {part}')
    assert_31(not df.empty and df['partition'].eq(part).all(),
              f'Partição inválida: {part}')
    df['timestamp'] = pd.to_datetime(df['timestamp'], utc=True, errors='raise')
    assert_31(df['timestamp'].notna().all(), f'Timestamp nulo: {part}')
    assert_31(not df.duplicated(['timestamp']).any(),
              f'Timestamps duplicados: {part}')
    assert_31(df['timestamp'].is_monotonic_increasing,
              f'Ordem cronológica inválida: {part}')
    assert_31(not df.duplicated(['case_id', 'timestamp']).any(),
              f'Chave de ciclo duplicada: {part}')
    for case_id, cycle in df.groupby('case_id', sort=False):
        delta = cycle['timestamp'].diff().dropna()
        assert_31(delta.eq(pd.Timedelta(hours=1)).all(),
                  f'Intervalo não horário dentro do ciclo {case_id}')
    assert_31(len(df) == m30['partitions'][part]['n_hours'],
              f'Quantidade de horas divergente: {part}')
    assert_31(df['case_id'].nunique() == m30['partitions'][part]['n_cycles'],
              f'Quantidade de ciclos divergente: {part}')
    frames31[part] = df
    summary31.append({
        'partition': part, 'hours': len(df),
        'cycles': df['case_id'].nunique(),
        'first_utc': df['timestamp'].min().isoformat(),
        'last_utc': df['timestamp'].max().isoformat(),
        'features': len(features31)
    })

all31 = pd.read_parquet(paths31['all'])
concat31 = pd.concat([frames31[p] for p in ('train', 'validation', 'test')],
                     ignore_index=True)
pd.testing.assert_frame_equal(all31, concat31,
                              check_dtype=True, check_exact=True)
assert_31(all31['case_id'].nunique() == sum(x['cycles'] for x in summary31),
          'Identificadores de ciclos repetidos entre partições.')
_project_print('\n31.3 — Dados e chaves horárias: APROVADOS')
_project_display(pd.DataFrame(summary31))

# 31.4 — Dicionário e agrupamento metodológico
dictionary31 = pd.read_csv(checks31['dictionary'])
assert_31('feature' in dictionary31.columns and
          dictionary31['feature'].tolist() == features31,
          'Dicionário de features não coincide com o manifesto.')
assert_31('feature_group' in dictionary31.columns and
          'lag_status' in dictionary31.columns,
          'Dicionário revisado sem grupos/status dos lags.')
groups31 = m30.get('feature_groups', {})
assert_31(set(groups31) == {'operational', 'hmm_regimes',
                             'process_activities', 'cycle_context'},
          'Grupos de features incompletos.')
assert_31(sorted(c for cols in groups31.values() for c in cols)
          == sorted(features31), 'Grupos não particionam as 54 features.')
assert_31(dictionary31['feature_group'].notna().all(),
          'Há features sem classificação metodológica.')
assert_31(all(dictionary31.set_index('feature').loc[cols, 'feature_group']
              .eq(group).all() for group, cols in groups31.items()),
          'Dicionário e grupos do manifesto divergentes.')
causal31 = pd.read_csv(checks31['causality'])
assert_31(len(causal31) > 0 and causal31['prefix_invariance'].eq(True).all(),
          'Auditoria de invariância de prefixo inválida.')
_project_print('\n31.4 — Dicionário, grupos e invariância: APROVADOS')
_project_display(dictionary31.groupby('feature_group').size()
        .rename('n_features').reset_index())

# 31.5 — Contrato de entrada da Fase 3
# Não copiar, modificar ou imputar dados nesta etapa.
contract31 = {
    'contract_version': '1.0',
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'project': m30['project'],
    'source_phase': 2,
    'next_phase': 3,
    'scientific_objective': 'temporal_dependency_and_lag_analysis',
    'facility': m30['facility'],
    'orispl': m30['orispl'],
    'unit': str(m30['unit']),
    'year': m30['year'],
    'source_manifest': str(MANIFEST30_31),
    'source_manifest_sha256': sha256_31(MANIFEST30_31),
    'input_files': {name: {'path': str(path), 'sha256': sha256_31(path)}
                    for name, path in paths31.items()},
    'feature_dictionary': {'path': str(checks31['dictionary']),
                           'sha256': sha256_31(checks31['dictionary'])},
    'row_counts': {x['partition']: x['hours'] for x in summary31},
    'cycle_counts': {x['partition']: x['cycles'] for x in summary31},
    'n_features': len(features31),
    'feature_columns': features31,
    'metadata_columns': meta31,
    'feature_groups': groups31,
    'timestamp_timezone': 'UTC',
    'timestamp_source_interpretation': (
        'Timestamps sem fuso são rotulados como UTC no formato de exportação '
        'original. A convenção horária da fonte EPA precisa ser confirmada '
        'antes de combinar fontes com fusos diferentes.'
    ),
    'temporal_granularity': '1 hour within each ON cycle',
    'case_id_policy': 'do_not_cross_ON_cycle_boundaries',
    'hmm_states': 'causal_filtered_states',
    'precomputed_lags': 'candidates_only_not_selected',
    'lag_selection_performed': False,
    'co2_present_in_feature_matrix': False,
    'co2_join_in_phase3': {
        'status': 'pending',
        'source': 'original_verified_hourly_CEMS_data',
        'join_keys': ['timestamp_utc', 'orispl', 'unit'],
        'requirements': [
            'verify_CEMS_CO2_measurement_column_and_units',
            'convert_short_tons_to_metric_tons_if_needed',
            'ensure_unique_timestamp_per_unit',
            'audit_missing_and_unmatched_hours',
            'preserve_partition_and_case_id',
            'never_shift_across_ON_cycle_boundaries'
        ]
    },
    'phase3_restrictions': [
        'lag_models_are_development_diagnostics_and_selection_tools',
        'keep_the_declared_test_out_of_feature_and_model_selection',
        'do_not_use_future_information_for_causal_features',
        'do_not_retrain_HMM_or_rediscover_processes',
        'do_not_treat_candidate_lags_as_selected',
        'record_previous_exploratory_test_exposure_before_future_evaluation'
    ],
    'phase2_status': 'closed_after_integrity_checks',
}

contract_path31 = REPORTS31 / 'cell31_phase3_data_contract.json'
report_path31 = REPORTS31 / 'cell31_phase2_closure_report.md'
manifest_path31 = REPORTS31 / 'cell31_phase2_closure_manifest.json'

_project_write_text(contract_path31, _project_dumps(contract31, indent=2,
                                     ensure_ascii=False), encoding='utf-8')

report31 = f'''# Fase 2 — Encerramento metodológico

**Unidade:** Iatan, ORISPL 6065, Unit 2 (2025).
**Objetivo cumprido:** descoberta/caracterização dos processos e construção
reprodutível de atributos horários para análise temporal.

## Entregas verificadas

- {len(concat31):,} horas e {sum(x['cycles'] for x in summary31)} ciclos ON.
- {len(features31)} features classificadas em quatro grupos.
- Estados HMM filtrados e atributos retrospectivos.
- SHA-256 de todos os artefatos conferido com o manifesto da Célula 30.
- Invariância de prefixo confirmada em {len(causal31)} ciclos auditados.
- Sem treino preditivo, sem seleção de lags e sem uso de CO2 nesta célula.

## Interface para a Fase 3

- Ler `cell30_process_features_all.parquet` ou as três partições individuais.
- Preservar `partition`, `timestamp` (UTC), `case_id` e `cycle_index`.
- Associar CO2 observado somente após verificar a fonte CEMS, unidade de medida,
  identidade da unidade e correspondência horária.
- Não calcular defasagens cruzando fronteiras de ciclos ON.
- Lags preexistentes são candidatos, não resultados de seleção estatística.
- Documentar que o teste já foi exposto em experimentos preditivos exploratórios
  anteriores, caso venha a ser reutilizado em uma futura avaliação preditiva.

**Contrato de dados:** `{contract_path31.name}`.
'''
_project_write_text(report_path31, report31, encoding='utf-8')

manifest31 = {
    'phase': 2, 'cell': 31, 'status': 'completed',
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'cell30_manifest_sha256': sha256_31(MANIFEST30_31),
    'contract_sha256': sha256_31(contract_path31),
    'report_sha256': sha256_31(report_path31),
    'n_hours': len(concat31),
    'n_cycles': sum(x['cycles'] for x in summary31),
    'n_features': len(features31),
    'n_prefix_cycles_checked': len(causal31),
    'predictive_model_trained': False,
    'lag_selection_performed': False,
    'co2_used': False,
    'phase3_ready_for_source_alignment': True,
}
_project_write_text(manifest_path31, _project_dumps(manifest31, indent=2,
                                     ensure_ascii=False), encoding='utf-8')

_project_print('\n' + '=' * 78)
_project_print('CÉLULA 31/31 CONCLUÍDA — FASE 2 ENCERRADA')
_project_print('=' * 78)
_project_print(f'Horas: {len(concat31)} | Ciclos: {manifest31["n_cycles"]} '
      f'| Features: {len(features31)}')
_project_print(f'Invariância: {len(causal31)} ciclos previamente auditados')
_project_print('Treinamento preditivo: NÃO | Seleção de lags: NÃO | CO2 utilizado: NÃO')
_project_print('\nARQUIVOS GERADOS')
_project_print('Contrato:', contract_path31)
_project_print('Relatório:', report_path31)
_project_print('Manifesto:', manifest_path31)
_project_print('\nPRÓXIMA ETAPA: FASE 3 — ALINHAMENTO DO CO2 OBSERVADO E ANÁLISE DOS LAGS')


## 32 — Estados e emissões na amostra

O ajuste quadrático com e sem estados é uma análise associativa no treino e na validação da Fase 2. O R² descreve ajuste na amostra. Os erros-padrão agrupados por ciclo tratam dependência dentro do grupo sob os pressupostos da estimativa. O número de ciclos limita a leitura do teste de Wald. A associação e os rótulos dos estados são hipóteses para avaliação preditiva posterior. [Gu et al. (2023), emissões com dados públicos](https://doi.org/10.1039/D3VA00191A) · [statsmodels, covariância por grupo](https://www.statsmodels.org/stable/generated/statsmodels.stats.sandwich_covariance.cov_cluster.html)


In [ ]:

# ============================================================
# FASE 2 — CÉLULA 32 (ANÁLISE COMPLEMENTAR)
# PERFIS DE EMISSÃO DE CO2 POR ESTADO HMM
# IATAN — ORISPL 6065 — UNIT 2 — 2025
# ============================================================
#
# QUESTÕES CIENTÍFICAS
#
# Q1: Os cinco estados apresentam emissões diferentes?
#
# Q2: As diferenças permanecem após controlar a carga?
#
# Q3: Existem estados com emissões positivas e geração zero?
#
# RESTRIÇÕES
# - Não retreinar o HMM
# - Não modificar estados
# - Não redescobrir processos
# - Não selecionar modelos pelo conjunto de teste
# - Usar apenas TRAIN e VALIDATION
# - Não realizar modelagem preditiva
#
# ============================================================

import json
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.formula.api as smf

from IPython.display import display

# ============================================================
# 32.1 — CONFIGURAÇÃO
# ============================================================

RUN32 = Path(RUN_DIR)
REPORT32 = RUN32 / "reports" / "cell32_emissions"
FIG32 = RUN32 / "figures" / "cell32_emissions"

REPORT32.mkdir(parents=True, exist_ok=True)
FIG32.mkdir(parents=True, exist_ok=True)

PARTITIONS32 = ("train", "validation")
SHORT_TON_TO_METRIC = 0.90718474

_project_print("=" * 76)
_project_print("CÉLULA 32 — PERFIS DE CO2 POR ESTADO HMM")
_project_print("=" * 76)

assert "features_by_partition30" in globals()
assert "df_fase2" in globals()

# ============================================================
# 32.2 — RECUPERAR ESTADOS CONGELADOS
# ============================================================

frames32 = []

for partition in PARTITIONS32:

    df = features_by_partition30[partition].copy()

    required = [
        "timestamp",
        "case_id",
        "load_mw",
        "hmm_state"
    ]

    missing = [
        c for c in required
        if c not in df.columns
    ]

    if missing:
        raise ValueError(
            f"Colunas ausentes em {partition}: {missing}"
        )

    df["partition"] = partition

    frames32.append(
        df[
            [
                "partition",
                "timestamp",
                "case_id",
                "load_mw",
                "hmm_state"
            ]
        ]
    )

states32 = pd.concat(
    frames32,
    ignore_index=True
)

states32["timestamp"] = pd.to_datetime(
    states32["timestamp"],
    utc=True
)

states32["hmm_state"] = (
    pd.to_numeric(
        states32["hmm_state"],
        errors="raise"
    ).astype(int)
)

states32["case_id"] = (
    states32["case_id"].astype(str)
)

if states32.duplicated("timestamp").any():
    raise RuntimeError(
        "Timestamps duplicados nos estados HMM."
    )

K32 = int(HMM_SELECTION_12["selected_k"])

EXPECTED_STATES32 = set(range(K32))

if not set(states32["hmm_state"]).issubset(
    EXPECTED_STATES32
):
    raise RuntimeError(
        "Estados incompatíveis com o HMM congelado."
    )

_project_print("\nEstados selecionados:", K32)
_project_print("Horas para análise:", len(states32))
_project_print(
    "Ciclos:",
    states32[
        ["partition", "case_id"]
    ].drop_duplicates().shape[0]
)

# ============================================================
# 32.3 — RECUPERAR EMISSÕES OBSERVADAS
# ============================================================

required_cems32 = [
    "timestamp",
    "CO2 Mass (short tons)",
    "Gross Load (MW)"
]

missing = [
    c for c in required_cems32
    if c not in df_fase2.columns
]

if missing:
    raise ValueError(
        f"Colunas CEMS ausentes: {missing}"
    )

cems32 = df_fase2[
    required_cems32
].copy()

cems32["timestamp"] = pd.to_datetime(
    cems32["timestamp"],
    utc=True
)

if cems32["timestamp"].duplicated().any():
    raise RuntimeError(
        "CEMS contém timestamps duplicados."
    )

cems32["co2_short_tons"] = pd.to_numeric(
    cems32["CO2 Mass (short tons)"],
    errors="coerce"
)

cems32["gross_load_mw"] = pd.to_numeric(
    cems32["Gross Load (MW)"],
    errors="coerce"
)

cems32["co2_tonnes"] = (
    cems32["co2_short_tons"]
    * SHORT_TON_TO_METRIC
)

# CEMS é horário: massa por registro de uma hora.
cems32["co2_tph"] = cems32["co2_tonnes"]

# ============================================================
# 32.4 — ASSOCIAÇÃO TEMPORAL E AUDITORIA
# ============================================================

data32 = states32.merge(
    cems32[
        [
            "timestamp",
            "co2_short_tons",
            "co2_tonnes",
            "co2_tph",
            "gross_load_mw"
        ]
    ],
    on="timestamp",
    how="left",
    validate="one_to_one",
    indicator=True
)

if not data32["_merge"].eq("both").all():
    raise RuntimeError(
        "Há timestamps HMM sem correspondência CEMS."
    )

data32 = data32.drop(columns="_merge")

if data32["co2_tph"].isna().any():
    raise RuntimeError(
        "Existem emissões ausentes. "
        "Não serão imputadas automaticamente."
    )

if (data32["co2_tph"] < 0).any():
    raise RuntimeError(
        "Foram encontradas emissões negativas."
    )

if data32["gross_load_mw"].isna().any():
    raise RuntimeError(
        "Há valores ausentes de carga CEMS."
    )

if (data32["gross_load_mw"] < 0).any():
    raise RuntimeError(
        "Carga bruta negativa encontrada."
    )

# Verificar consistência entre carga da Fase 2 e CEMS.
load_diff32 = (
    data32["load_mw"]
    - data32["gross_load_mw"]
).abs()

if load_diff32.max() > 1e-5:
    raise RuntimeError(
        "Carga das features HMM diverge da carga CEMS. "
        f"Máxima diferença: {load_diff32.max():.6f} MW."
    )

# Uma hora de registro = uma hora de energia.
# Intensidade: toneladas métricas por MWh.
data32["co2_t_per_mwh"] = np.where(
    data32["gross_load_mw"] > 0,
    data32["co2_tonnes"]
    / data32["gross_load_mw"],
    np.nan
)

data32["zero_generation"] = (
    data32["gross_load_mw"] == 0
)

data32["zero_generation_positive_co2"] = (
    data32["zero_generation"]
    & (data32["co2_tph"] > 0)
)

# Identificador único de ciclo, incluindo a partição.
data32["cycle_group"] = (
    data32["partition"].astype(str)
    + "::"
    + data32["case_id"].astype(str)
)

_project_print("\nASSOCIAÇÃO CEMS × HMM")
_project_print("Correspondências:", len(data32))
_project_print(
    "Emissões positivas com geração zero:",
    int(data32["zero_generation_positive_co2"].sum())
)
_project_print(
    "Maior divergência de carga:",
    float(load_diff32.max())
)

# ============================================================
# 32.5 — PERFIL DESCRITIVO DOS ESTADOS
# ============================================================

profile32 = (
    data32
    .groupby(
        ["partition", "hmm_state"],
        observed=True
    )
    .agg(
        n_hours=("co2_tph", "size"),
        n_cycles=("cycle_group", "nunique"),
        load_mean_mw=("gross_load_mw", "mean"),
        load_median_mw=("gross_load_mw", "median"),
        co2_mean_tph=("co2_tph", "mean"),
        co2_median_tph=("co2_tph", "median"),
        co2_std_tph=("co2_tph", "std"),
        co2_p10_tph=(
            "co2_tph",
            lambda x: x.quantile(0.10)
        ),
        co2_p90_tph=(
            "co2_tph",
            lambda x: x.quantile(0.90)
        ),
        intensity_mean=(
            "co2_t_per_mwh",
            "mean"
        ),
        intensity_median=(
            "co2_t_per_mwh",
            "median"
        ),
        zero_load_hours=(
            "zero_generation",
            "sum"
        ),
        zero_load_co2_positive=(
            "zero_generation_positive_co2",
            "sum"
        )
    )
    .reset_index()
)

_project_print("\nPERFIL DE EMISSÕES POR ESTADO")
_project_display(profile32.round(4))

# ============================================================
# 32.6 — VISUALIZAÇÕES DESCRITIVAS
# ============================================================

sns.set_theme(style="whitegrid")

fig, axes = plt.subplots(
    1, 2,
    figsize=(15, 5)
)

sns.boxplot(
    data=data32,
    x="hmm_state",
    y="co2_tph",
    hue="partition",
    showfliers=False,
    ax=axes[0]
)

axes[0].set_title(
    "Emissões horárias por estado HMM"
)
axes[0].set_xlabel("Estado")
axes[0].set_ylabel("CO2 (t métricas/h)")

positive32 = data32.loc[
    data32["gross_load_mw"] > 0
].copy()

sns.boxplot(
    data=positive32,
    x="hmm_state",
    y="co2_t_per_mwh",
    hue="partition",
    showfliers=False,
    ax=axes[1]
)

axes[1].set_title(
    "Intensidade de CO2 por estado"
)
axes[1].set_xlabel("Estado")
axes[1].set_ylabel("CO2 (t métricas/MWh)")

plt.tight_layout()

plt.savefig(
    FIG32 / "cell32_emission_profiles.png",
    dpi=180,
    bbox_inches="tight"
)

plt.show()

# ============================================================
# 32.7 — ANÁLISE INFERENCIAL
# ============================================================
#
# Modelo 0:
#   CO2 ~ carga + carga²
#
# Modelo 1:
#   CO2 ~ carga + carga² + estado HMM
#
# Erros-padrão robustos agrupados por ciclo.
#
# Esta é uma análise associativa, não causal.
# O R² é descritivo dentro da amostra.
# ============================================================

infer32 = data32.copy()

infer32["load_sq"] = (
    infer32["gross_load_mw"] ** 2
)

n_clusters32 = (
    infer32["cycle_group"].nunique()
)

if n_clusters32 < 10:
    raise RuntimeError(
        "Número insuficiente de ciclos "
        "para esta análise inferencial."
    )

observed_states32 = set(
    infer32["hmm_state"].unique()
)

if observed_states32 != EXPECTED_STATES32:
    raise RuntimeError(
        "Nem todos os estados estão representados "
        "na amostra inferencial."
    )

model_base32 = smf.ols(
    "co2_tph ~ gross_load_mw + load_sq",
    data=infer32
).fit(
    cov_type="cluster",
    cov_kwds={
        "groups": infer32["cycle_group"]
    }
)

model_state32 = smf.ols(
    "co2_tph ~ gross_load_mw + load_sq "
    "+ C(hmm_state)",
    data=infer32
).fit(
    cov_type="cluster",
    cov_kwds={
        "groups": infer32["cycle_group"]
    }
)

# Teste conjunto dos coeficientes dos estados.
state_terms32 = [
    name
    for name in model_state32.params.index
    if name.startswith("C(hmm_state)")
]

if len(state_terms32) != K32 - 1:
    raise RuntimeError(
        "Quantidade inesperada de coeficientes HMM."
    )

R32 = np.zeros(
    (
        len(state_terms32),
        len(model_state32.params)
    )
)

for i, term in enumerate(state_terms32):
    j = list(
        model_state32.params.index
    ).index(term)

    R32[i, j] = 1

wald32 = model_state32.wald_test(
    R32,
    scalar=True
)

p_value32 = float(wald32.pvalue)

comparison32 = pd.DataFrame([
    {
        "model": "Carga + carga²",
        "r2_in_sample": model_base32.rsquared,
        "adjusted_r2": model_base32.rsquared_adj,
        "n_observations": int(model_base32.nobs),
        "n_cycles": n_clusters32
    },
    {
        "model": "Carga + carga² + estados HMM",
        "r2_in_sample": model_state32.rsquared,
        "adjusted_r2": model_state32.rsquared_adj,
        "n_observations": int(model_state32.nobs),
        "n_cycles": n_clusters32
    }
])

delta_r2_32 = (
    model_state32.rsquared
    - model_base32.rsquared
)

_project_print("\nCOMPARAÇÃO DOS MODELOS")
_project_display(comparison32.round(5))

_project_print(
    "\nIncremento descritivo de R²:",
    round(delta_r2_32, 6)
)

_project_print(
    "Teste conjunto dos estados (Wald):",
    round(float(wald32.statistic), 4)
)

_project_print(
    "p-valor (aproximação assintótica):",
    p_value32
)

_project_print(
    "\nATENÇÃO: os erros-padrão são agrupados "
    "por ciclo. O p-valor não elimina possíveis "
    "problemas de confundimento, extrapolação "
    "ou pequeno número de ciclos."
)

# ============================================================
# 32.8 — ANÁLISE ESPECÍFICA DE GERAÇÃO ZERO
# ============================================================

zero32 = data32.loc[
    data32["zero_generation"]
].copy()

zero_profile32 = (
    data32
    .groupby("hmm_state")
    .agg(
        total_hours=("co2_tph", "size"),
        zero_load_hours=("zero_generation", "sum"),
        zero_load_positive_co2=(
            "zero_generation_positive_co2",
            "sum"
        )
    )
    .reset_index()
)

zero_profile32["zero_load_share"] = (
    zero_profile32["zero_load_hours"]
    / zero_profile32["total_hours"]
)

_project_print("\nEMISSÕES COM GERAÇÃO ZERO")
_project_display(zero_profile32.round(4))

if len(zero32):
    _project_display(
        zero32.groupby("hmm_state")[
            "co2_tph"
        ].agg(
            ["count", "mean", "median", "max"]
        ).round(4)
    )

# ============================================================
# 32.9 — EXPORTAÇÃO
# ============================================================

data32.to_parquet(
    REPORT32 / "cell32_hourly_states_co2.parquet",
    index=False
)

_project_csv(profile32, 
    REPORT32 / "cell32_state_profiles.csv",
    index=False
)

_project_csv(comparison32, 
    REPORT32 / "cell32_model_comparison.csv",
    index=False
)

_project_csv(zero_profile32, 
    REPORT32 / "cell32_zero_generation.csv",
    index=False
)

with open(
    REPORT32 / "cell32_regression_results.txt",
    "w",
    encoding="utf-8"
) as f:
    f.write(
        model_base32.summary().as_text()
    )
    f.write("\n\n")
    f.write(
        model_state32.summary().as_text()
    )

manifest32 = {
    "cell": 32,
    "analysis": "CO2_emission_profiles_by_HMM_state",
    "facility_id": 6065,
    "unit_id": "2",
    "year": 2025,
    "selected_k": K32,
    "partitions": list(PARTITIONS32),
    "n_hours": int(len(data32)),
    "n_cycles": int(n_clusters32),
    "co2_original_unit": "short_tons_per_hour",
    "co2_analysis_unit": "metric_tonnes_per_hour",
    "conversion_factor": SHORT_TON_TO_METRIC,
    "delta_r2_in_sample": float(delta_r2_32),
    "wald_pvalue": p_value32,
    "inference_covariance": "cluster_by_operating_cycle",
    "hmm_retrained": False,
    "states_modified": False,
    "process_rediscovered": False,
    "test_used": False,
    "predictive_model_trained": False,
    "interpretation": "associative_not_causal",
    "created_utc": datetime.now(
        timezone.utc
    ).isoformat()
}

with open(
    REPORT32 / "cell32_manifest.json",
    "w",
    encoding="utf-8"
) as f:
    _project_dump(
        manifest32,
        f,
        indent=2,
        ensure_ascii=False
    )

# ============================================================
# 32.10 — CONCLUSÃO AUTOMÁTICA
# ============================================================

_project_print("\n" + "=" * 76)
_project_print("RESULTADO FINAL — CÉLULA 32")
_project_print("=" * 76)

_project_print("Estados avaliados:", K32)
_project_print("Horas avaliadas:", len(data32))
_project_print("Ciclos avaliados:", n_clusters32)

_project_print("\nHIPÓTESE TESTADA:")
_project_print(
    "Os estados HMM acrescentam associação "
    "com CO2 após controlar carga e carga²."
)

_project_print("\nResultado do teste conjunto:")

if p_value32 < 0.05:
    _project_print(
        "Diferença estatisticamente detectável "
        "entre os estados, condicionada à carga."
    )
else:
    _project_print(
        "Não foi detectada diferença estatística "
        "conjunta ao nível de 5%."
    )

_project_print(
    "\nIncremento de R² na amostra:",
    round(delta_r2_32, 6)
)

_project_print("\nLIMITAÇÕES:")
_project_print("- Associação não implica causalidade.")
_project_print("- O teste não demonstra ganho preditivo.")
_project_print("- A dependência entre ciclos exige cautela.")
_project_print("- O conjunto de teste permaneceu isolado.")
_project_print("- A sobreposição de carga entre estados deve ser auditada.")

_project_print("\nRelatórios:", REPORT32)
_project_print("Figuras:", FIG32)

_project_print("\nCÉLULA 32 CONCLUÍDA")


In [ ]:
# V2-C — Auditoria científica final de modelos de processo e rastreabilidade.
# Executar após todas as células originais (inclusive 32).
from pathlib import Path
import json, hashlib
import pandas as pd
from IPython.display import display

if "RUN_DIR" not in globals():
    raise RuntimeError("RUN_DIR ausente: execute o notebook completo.")
v2_dir = Path(RUN_DIR) / "reports" / "v2_scientific_audit"
v2_dir.mkdir(parents=True, exist_ok=True)
base26 = Path(RUN_DIR) / "reports" / "cell26_fitness_precision_complexity.csv"
if not base26.exists():
    raise FileNotFoundError(f"Dados de conformidade não encontrados: {base26}")
metrics_v2 = pd.read_csv(base26)
required_v2 = {"log_type", "partition", "fitness_log", "precision", "f1_fitness_precision"}
missing_v2 = required_v2 - set(metrics_v2.columns)
if missing_v2:
    raise ValueError(f"Colunas de conformidade ausentes: {sorted(missing_v2)}")
metrics_v2["fitness_minus_precision"] = (
    metrics_v2["fitness_log"] - metrics_v2["precision"]
)
metrics_v2["interpretation_v2"] = metrics_v2.apply(
    lambda r: ("Modelo permissivo: verificar atividades e variantes"
               if r["precision"] < 0.5 and r["fitness_log"] >= 0.99
               else "Inspecionar compromisso fitness/precision"), axis=1
)
_project_csv(metrics_v2, v2_dir / "process_conformance_review.csv", index=False)
_project_print("Conformidade por partição e log:")
_project_display(metrics_v2[["log_type", "partition", "fitness_log",
                    "precision", "f1_fitness_precision",
                    "fitness_minus_precision", "interpretation_v2"]])
manifest_v2 = {
    "project": "Iatan / ORISPL 6065 / unit 2",
    "year": 2025,
    "notebook_version": "2.0_scientific_audit",
    "original_selection_preserved": True,
    "hmm_family_comparison_caveat": "LL entre dimensões não comparável diretamente",
    "test_status": "Já consultado anteriormente; não é teste final virgem",
    "outputs": ["hmm_105_fits_diagnostics.csv", "hmm_family_sensitivity.csv",
                "selected_hmm_diagnostic.json", "process_conformance_review.csv"],
    "open_questions": [
        "Justificar M2 frente a M1 com evidência operacional",
        "Examinar convergência do HMM selecionado",
        "Explicar baixa precision do log combined",
        "Validar interpretação física de S4 sem alegação causal",
        "Distinguir significância estatística de magnitude do efeito nas emissões"
    ],
}
with open(v2_dir / "scientific_review_manifest.json", "w", encoding="utf-8") as f:
    _project_dump(manifest_v2, f, ensure_ascii=False, indent=2)
_project_print("Auditoria científica v2 salva em:", v2_dir)


## Auditoria adicional de encerramento — HMM e causalidade
Esta auditoria **não altera** o HMM selecionado nem os 54 atributos. Verifica a convergência com critério conservador e a consistência das features exportadas. Uma advertência de convergência exige interpretação, não retreinamento automático.

In [ ]:
# AUDITORIA ADICIONAL — HMM selecionado e atributos causais (sem retreinamento)
# Esta célula não modifica modelos, partições ou arquivos exportados.
import numpy as np
import pandas as pd

required = ["HMM_SELECTED", "hmm13_inference", "features_by_partition30", "feature_columns30"]
missing = [name for name in required if name not in globals()]
if missing:
    raise RuntimeError(f"Execute as células anteriores antes desta auditoria: {missing}")

monitor = HMM_SELECTED.monitor_
history = np.asarray(list(monitor.history), dtype=float)
max_iter = int(monitor.n_iter)
tolerance = float(monitor.tol)
iterations = int(getattr(monitor, "iter", len(history)))
finite_history = bool(len(history) >= 2 and np.isfinite(history).all())
last_delta = float(history[-1] - history[-2]) if finite_history else np.nan
# O hmmlearn pode reportar converged=True ao alcançar o limite de iterações.
reached_limit = bool(max_iter > 0 and iterations >= max_iter)
strict_convergence = bool(
    finite_history
    and max_iter > 0
    and not reached_limit
    and np.isfinite(tolerance)
    and tolerance > 0
    and monitor.converged
    and -1e-8 <= last_delta < tolerance
)

_project_print("=== AUDITORIA DO HMM SELECIONADO ===")
_project_print(f"Estados: {HMM_SELECTED.n_components}; iterações: {iterations}/{max_iter}")
_project_print(f"monitor_.converged: {bool(monitor.converged)}")
_project_print(f"Variação final da log-verossimilhança: {last_delta:.8g}")
_project_print(f"Tolerância de parada: {tolerance:.8g}")
_project_print(f"Convergência estrita (critério conservador): {strict_convergence}")
if not strict_convergence:
    _project_print("ATENÇÃO: convergência estrita não demonstrada; investigar histórico antes de afirmar convergência.")

_project_print("\n=== AUDITORIA DOS ATRIBUTOS EXPORTADOS ===")
assert len(feature_columns30) == len(set(feature_columns30)), "Nomes de atributos duplicados"
assert not any("viterbi" in c.lower() for c in feature_columns30), "Atributo Viterbi exportado"
assert not any("co2" in c.lower() or "emission" in c.lower() for c in feature_columns30), "Alvo nas features"
assert set(features_by_partition30) >= {"train", "validation", "test"}, "Partições ausentes"
for partition in ("train", "validation", "test"):
    df = features_by_partition30[partition]
    assert set(feature_columns30).issubset(df.columns), f"Colunas ausentes: {partition}"
    if "hmm_state_source" in df.columns:
        assert df["hmm_state_source"].dropna().eq("filtered_state").all(), f"Fonte não causal: {partition}"
    _project_print(f"{partition}: {len(df)} linhas; {len(feature_columns30)} atributos; esquema OK")

# Teste de invariância do prefixo: usa a auditoria já calculada no notebook.
if "causality_audit30" not in globals():
    raise RuntimeError("Execute a Célula 30 e recupere a auditoria de prefixo.")
audit = causality_audit30
assert not audit.empty, "Nenhum ciclo com prefixo auditado"
assert audit["prefix_invariance"].eq(True).all(), "Falha de invariância de prefixo"
_project_print("Auditoria de prefixo aprovada:", len(audit), "registros")
_project_print("Auditoria concluída. Nenhum dado ou modelo foi alterado.")


## Fontes do material local

- **Aulas 04, 07 e 08**, IND2622. Preparação de logs, qualidade de modelos de processo e proveniência, respectivamente.
- **Diba, Batoulis, Weidlich e Weske (2020)**. Extraction, correlation, and abstraction of event data for process mining. DOI 10.1002/widm.1346.
- **van der Aalst (2022)**. Process Mining: A 360 Degree Overview. DOI 10.1007/978-3-031-08848-3_1.
- **Tax et al. (2018)**. The Imprecisions of Precision Measures in Process Mining. DOI 10.1016/j.ipl.2018.01.013.
- **Moradbeikie et al. (2025)**. Process Mining of Sensor Data for Predictive Process Monitoring. DOI 10.3390/systems13110935.
- **Gu, Sward e Zhang (2023)**. Predicting power plant emissions using public data and machine learning. DOI 10.1039/D3VA00191A.
- **U.S. EPA**. CAMD’s Power Sector Emissions Data Guide, p. 1.

A pasta Referências da entrega reúne os PDFs utilizados, incluindo Rabiner e Leemans et al. A documentação técnica é vinculada junto às respectivas chamadas. O registro de revisão identifica as células modificadas e os limites de validação.
